# 零售場景的生成式 AI：從 Prompt 到可上線的評測流程

**Google Cloud & Generative AI Applications**



---

這份 notebook 要證明一件事：

> **生成式 AI 的難處不在叫模型，而在你怎麼知道它有沒有變好。**

我們會用一個真實的零售任務（商品文案生成）走完整條路：

| 節 | 內容 | 重點 |
|---|---|---|
| §1 | 商品資料與法規禁詞 | 規則要跟著商品屬性走 |
| §2 | Prompt v0 → v3，每版只加一件事 | Prompt 是規格書，不是咒語 |
| §3 | 三層評測，產出版本 × 指標對比表 | 怎麼證明它變好了 |
| §4 | 場景 B：評論洞察 | 同一套流程，反方向用 |
| §5 | 成本：實際花了多少，外推到 10 萬 SKU | 教公式，不背數字 |

**§3 的對比表是整份 notebook 的重點。** 其他都是為了讓那張表可信。

---

### 執行前須知

- 需要一個啟用了 Vertex AI API 與計費的 GCP 專案。
- 全部跑完的成本很低（見 §5 實際印出的數字），但**不是零**。
- 若沒有 GCP 專案，可在 §0 的 CONFIG 把 `USE_VERTEX` 改成 `False`
  並填入 AI Studio API key。

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **Demo 全長 12 分鐘（30:00–42:00）。開場前先 Run all 一次，讓輸出都在畫面上。**
>
> 先確認 CONFIG 的 `OFFLINE_MODE = True` —— 這是最容易忘的一項。
>
> 開場第一句（不要道歉，一句帶過）：
> 「我先說明，這些輸出是我昨天跑好存下來的，現在是重播 —— 因為會場網路我不敢賭。
> 程式碼跟評測都是當場執行的，只有呼叫模型那一步是查表。」

</div>

## §0 環境設定

### 相依套件

這一格**只在缺少套件時才安裝**，不是無條件 `pip install`。

| 環境 | 行為 |
|---|---|
| Colab | 預裝了 pandas / jinja2 / matplotlib，會補裝 `google-genai` |
| 本機 uv 環境 | 全部已在 `.venv`，直接跳過 |

會做這個判斷是因為：在 uv 管理的環境裡無條件 `pip install`，套件會裝進
`.venv` 卻不在 `uv.lock` 裡，環境就跟鎖定檔對不上了。與其寫註解叫人自己
判斷該不該跑，不如讓它自己偵測。

In [ ]:
import sys

# 渲染用，任何模式都需要。§3 的對比表用 df.style.background_gradient() 上色：
# .style 需要 jinja2、background_gradient 需要 matplotlib，缺一就會 AttributeError。
CORE = {"pandas": "pandas", "jinja2": "jinja2", "matplotlib": "matplotlib"}

# 只有要「真的呼叫 API」才需要。離線重播完全用不到它 ——
# 這是刻意設計的：要裝它就得有網路，一旦需要網路，零網路重播的前提就破功了。
LIVE = {"google.genai": "google-genai"}


def _importable(mod: str) -> bool:
    try:
        __import__(mod)
        return True
    except ImportError:
        return False


_missing = [pkg for mod, pkg in {**CORE, **LIVE}.items() if not _importable(mod)]

if _missing:
    print("缺少套件，嘗試安裝：", " ".join(_missing))
    import subprocess
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *_missing], check=True
        )
        print("✓ 安裝完成（若 import 仍失敗，重啟 runtime 再跑一次）")
    except Exception as e:
        # 安裝失敗不一定是問題：離線重播根本不需要 google-genai。
        print(f"⚠ 安裝失敗：{e}")
        if not _importable("google.genai"):
            print("  google-genai 未安裝 —— 離線重播（OFFLINE_MODE=True）不需要它，可以繼續。")
            print("  但若要真的呼叫 API，就必須在有網路的環境先裝好。")
        _still = [p for m, p in CORE.items() if not _importable(m)]
        if _still:
            print(f"  ✗ 仍缺渲染必要套件 {_still} —— §3 的對比表會無法上色。")
else:
    print("✓ 相依套件齊全，跳過安裝")

# BigQuery 只有 §4 要用；[pandas] extra 才會帶入 to_dataframe() 需要的 pyarrow
if not _importable("google.cloud.bigquery"):
    print('  （§4 若要實際寫入 BigQuery：pip install "google-cloud-bigquery[pandas]"）')

### CONFIG — 只改這一格

價格常數**使用前必須到官方頁面現查更新**：
<https://cloud.google.com/vertex-ai/generative-ai/pricing>

模型與價格變動頻繁，任何寫死的數字都會很快過期。
要看實際花費，請以 §5 執行後印出來的金額為準。

> `PROJECT_ID`、`LOCATION` 與 BigQuery 那幾個值是**建構這份 notebook 時注入的**。
> 要換成你自己的環境，改這一格就好；若你是從原始 repo 重新產生，
> 用 `python3 poc/build_notebook.py --project-id 你的專案` 一次帶進來。

In [ ]:
"""集中設定：模型、價格、通路限制。

所有環境相關設定集中在這裡。換專案、換模型、換價格都只改這個檔案。
價格必須到官方頁面現查後更新，不要沿用寫死的數字。

**專案識別資訊不寫死在 repo 裡。** GCP 專案、BigQuery 位置這幾個值
「換一個人用就要換」，所以走環境變數（見下方各項），notebook 則在
`build_notebook.py` 建構時以參數注入成字面值。
這樣別人 clone 下來只要設好自己的環境變數就能跑，不必改這個檔案。
"""

# --- GCP ---
# 依序取：環境變數 GCP_PROJECT_ID → 佔位字串。
# 佔位字串留著沒關係，check_env.py 會擋下來；離線重播用不到專案 ID。
PROJECT_ID = "your-gcp-project-id"

# ⚠️ 必須是 "global"。實測 asia-east1 上沒有任何 Gemini publisher model，
#    所有呼叫都會 404。這是很容易踩的坑：compute 資源設在 asia-east1，
#    不代表 Gemini 模型在那裡可用。
#    開放成環境變數是為了讓別人能換，不是鼓勵你換 —— 換完務必跑 check_env.py。
LOCATION = "global"

# 使用 Vertex AI（走 GCP 專案計費、資料落地可控）
# 若現場 GCP 權限失效，把 USE_VERTEX 改成 False 並填入 AI Studio key 作為備援
USE_VERTEX = True
FALLBACK_API_KEY = ""  # 僅備援用，勿 commit 真實金鑰

# --- 離線重播（會場網路不穩時的主要策略）---
# 在網路不穩的場合（會議室、教室、展場）即時打 API 是不能接受的風險。
#
#   RECORD_FIXTURES = True   有網路時先跑一次，把真實輸出錄下來
#   OFFLINE_MODE    = True   之後零網路重播錄好的輸出
#
# 重播時所有 cell 一樣會執行、表格一樣是當場算出來的，只有 API 呼叫被換成
# 查表。畫面上看不出差別，但完全不依賴網路。
#
# ⚠ 兩個不能同時為 True。
OFFLINE_MODE = False
RECORD_FIXTURES = False
FIXTURES_FILE = "demo_outputs.json"

# --- 模型 ---
# 以下型號皆已在 location="global" 實測可用（2026-08-13）。
# 換專案或換 region 前請重跑 poc/check_env.py 確認。
GEN_MODEL = "gemini-flash-latest"   # 生成用：吞吐量大、要便宜
JUDGE_MODEL = "gemini-2.5-pro"      # 評審用：刻意與生成模型不同，降低 self-preference bias
CHEAP_MODEL = "gemini-2.5-flash-lite"  # 成本段示範「降級」用

GEN_TEMPERATURE = 0.4  # 用低溫，降低失敗機率
JUDGE_TEMPERATURE = 0.0

# 並行度。評審模型單次要十幾到二十幾秒，瓶頸完全在等待回應，不在 CPU，
# 所以拉高執行緒數幾乎是線性加速。
#
# 實測吞吐（gemini-2.5-pro，單一專案預設配額，2026-08-13，零失敗）：
#     8 → 10.3 次/分     16 → 26.1 次/分
#    32 → 46.5 次/分     64 → 80.7 次/分
#
# 到 64 都還沒撞到配額。設 32 是留一點餘裕的預設值；
# 要更快就調到 64（評測腳本可用 --workers 覆寫，不必改這裡）。
#
# 遇到 429 時不用緊張：生成與評審兩條路徑都有指數退避重試，
# 會自動降速而不是整批失敗。真的一直撞才把這個數字調小。
MAX_WORKERS = 32

# --- 價格（USD / 每百萬 token）---
# ⚠️ 這些是佔位值。使用前請到官方頁面現查更新：
#    https://cloud.google.com/vertex-ai/generative-ai/pricing
# 不要引用這裡的數字，要看 §5 實際跑出來的花費。
PRICING = {
    "gemini-flash-latest": {"input": 0.30, "output": 2.50},
    "gemini-2.5-pro": {"input": 1.25, "output": 10.00},
    "gemini-2.5-flash-lite": {"input": 0.10, "output": 0.40},
}
PRICE_LAST_CHECKED = "尚未查證 — 上場前必須更新"

# --- BigQuery（場景 B）---
# 預設關閉：重播模式不涵蓋 BigQuery 呼叫。
# 錄製時可開啟一次，把「AI 當 ETL」這件事真的做完。
USE_BIGQUERY = False
BQ_DATASET = "retail_genai_demo"
BQ_TABLE = "review_insights"
# BigQuery 的 region 與 Gemini 無關，這裡可以用亞洲
BQ_LOCATION = "asia-east1"

USD_TO_TWD = 32.0

# --- 通路限制（來自 products.json 的 _meta.channel_limits）---
SHOPEE_TITLE_MAX = 60
SEO_DESC_MAX = 120
BULLET_MAX = 30
BULLET_COUNT = 4

### 呼叫層與離線重播

底下這段程式碼做兩件事。

**第一，切換 Vertex AI 與 AI Studio 只差 Client 的建構參數**，
其他程式碼一行都不用改。選哪一個是**部署決策**
（資料落地、計費歸屬、VPC-SC、合規），不是**程式碼決策**。
先用 AI Studio 做原型，要簽 DPA 時再切到 Vertex。

**第二，離線重播。** 在網路不穩的場合即時打 API 是不能接受的風險，
所以有兩個模式：

| 設定 | 何時用 | 行為 |
|---|---|---|
| `RECORD_FIXTURES=True` | 有網路時，先跑一次 | 正常呼叫，同時把輸出錄下來 |
| `OFFLINE_MODE=True` | 之後任何時候 | 零網路，重播錄好的輸出 |

重播時**所有 cell 一樣會執行、表格一樣是當場算出來的**，
只有 API 呼叫被換成查表。`ReplayClient` 找不到對應輸出時會明確報錯，
不會靜默拿到過期資料。

In [ ]:
"""呼叫 Gemini 產生文案，並記錄 token 用量。

用 google-genai 統一 SDK。同一份程式碼可切換 Vertex AI 與 Gemini API ——
只差 Client 的建構參數。重點是：
選 Vertex 還是 AI Studio 是**部署決策**，不是**程式碼決策**。
"""

from __future__ import annotations

import time
from dataclasses import dataclass, field
from types import SimpleNamespace



def gen_config(**kwargs):
    """建立 generate_content 的 config 物件。

    為什麼不直接用 `types.GenerateContentConfig`：

    離線重播的使用情境是「把單一個 .ipynb 丟到一台沒有網路的機器」——
    這種時候 google-genai 很可能根本沒安裝，而安裝它就需要網路，
    整個「零網路」的前提就破功了。

    但重播其實不需要真的 SDK 型別：`ReplayClient` 只會讀 `response_schema`
    這一個屬性來判斷是不是結構化輸出。所以 SDK 不在時退回一個等效的
    簡單物件即可。

    只有 OFFLINE_MODE 才容許退回。連線模式下缺 SDK 是真的有問題，
    要讓它直接炸出來，不能靜默降級。
    """
    try:
        from google.genai import types

        return types.GenerateContentConfig(**kwargs)
    except ImportError:
        if not OFFLINE_MODE:
            raise
        return SimpleNamespace(**kwargs)


@dataclass
class Usage:
    """單次呼叫的 token 用量。所有成本計算都從這裡出發。"""

    model: str
    input_tokens: int
    output_tokens: int
    latency_s: float

    @property
    def cost_usd(self) -> float:
        price = PRICING.get(self.model)
        if price is None:
            return 0.0
        return (
            self.input_tokens * price["input"] + self.output_tokens * price["output"]
        ) / 1_000_000


@dataclass
class GenResult:
    text: str
    usage: Usage
    error: str | None = None


@dataclass
class UsageLedger:
    """累計所有呼叫的用量 —— demo §5 成本計算的資料來源。

    刻意把 judge 的呼叫也記進來。評測本身要花錢，這是最常被漏算的一筆。
    """

    calls: list[tuple[str, Usage]] = field(default_factory=list)

    def record(self, tag: str, usage: Usage) -> None:
        self.calls.append((tag, usage))

    def total_cost_usd(self, tag_prefix: str | None = None) -> float:
        return sum(
            u.cost_usd
            for t, u in self.calls
            if tag_prefix is None or t.startswith(tag_prefix)
        )

    def total_tokens(self, tag_prefix: str | None = None) -> tuple[int, int]:
        rows = [u for t, u in self.calls if tag_prefix is None or t.startswith(tag_prefix)]
        return sum(u.input_tokens for u in rows), sum(u.output_tokens for u in rows)

    def summary(self) -> str:
        gen_in, gen_out = self.total_tokens("gen")
        jdg_in, jdg_out = self.total_tokens("judge")
        gen_cost = self.total_cost_usd("gen")
        jdg_cost = self.total_cost_usd("judge")
        total = gen_cost + jdg_cost
        judge_share = (jdg_cost / total * 100) if total else 0.0
        return (
            f"呼叫次數：{len(self.calls)}\n"
            f"  生成  input {gen_in:>7,} / output {gen_out:>7,} tokens → US${gen_cost:.4f}\n"
            f"  評審  input {jdg_in:>7,} / output {jdg_out:>7,} tokens → US${jdg_cost:.4f}\n"
            f"  合計                                        → US${total:.4f}"
            f"（約 NT${total * USD_TO_TWD:.2f}）\n"
            f"\n  ⚠ 評審佔總成本 {judge_share:.0f}% —— 這筆最常被漏算。"
        )


# --------------------------------------------------------------------------
# 離線重播 —— 會場網路不穩時的主要策略
# --------------------------------------------------------------------------
def fixture_key(model: str, contents: str, structured: bool) -> str:
    """以 (模型, prompt, 是否結構化) 當索引鍵。

    prompt 完全相同才會命中，所以只要 prompt 有任何改動，重播就會失敗並
    明確報錯 —— 不會靜默拿到舊資料。這是刻意的：寧可在測試時炸掉，
    也不要產出跟程式碼對不上的結果。
    """
    import hashlib

    raw = f"{model}\x00{int(structured)}\x00{contents}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]


class _ReplayModels:
    def __init__(self, fixtures: dict):
        self._f = fixtures
        self.hits = 0

    def generate_content(self, *, model, contents, config=None):
        structured = bool(config is not None and getattr(config, "response_schema", None))
        key = fixture_key(model, contents if isinstance(contents, str) else str(contents), structured)
        row = self._f.get("calls", {}).get(key)
        if row is None:
            raise KeyError(
                f"離線重播找不到對應輸出（key={key}, model={model}）。\n"
                f"代表 prompt 或模型在錄製之後被改過。\n"
                f"請重新錄製：把 RECORD_FIXTURES 設為 True、在有網路的環境跑一次。"
            )
        self.hits += 1
        return _Recorded(row["text"], row["input_tokens"], row["output_tokens"])

    def count_tokens(self, *, model, contents):
        key = "tok:" + fixture_key(model, contents, False)
        row = self._f.get("token_counts", {}).get(key)
        if row is None:
            raise KeyError(f"離線重播找不到 count_tokens 結果（key={key}）。請重新錄製。")
        self.hits += 1
        return _RecordedTokens(row)


class _Recorded:
    def __init__(self, text: str, pin: int, pout: int):
        self.text = text
        self.usage_metadata = _RecordedUsage(pin, pout)


class _RecordedUsage:
    def __init__(self, pin: int, pout: int):
        self.prompt_token_count = pin
        self.candidates_token_count = pout


class _RecordedTokens:
    def __init__(self, n: int):
        self.total_tokens = n


class ReplayClient:
    """從錄好的 fixtures 回放，完全不碰網路。"""

    def __init__(self, fixtures: dict):
        self.models = _ReplayModels(fixtures)


class _RecordingModels:
    def __init__(self, real):
        self._real = real
        self.fixtures = {"calls": {}, "token_counts": {}}

    def generate_content(self, *, model, contents, config=None):
        resp = self._real.generate_content(model=model, contents=contents, config=config)
        structured = bool(config is not None and getattr(config, "response_schema", None))
        key = fixture_key(model, contents if isinstance(contents, str) else str(contents), structured)
        meta = resp.usage_metadata
        self.fixtures["calls"][key] = {
            "text": resp.text or "",
            "input_tokens": getattr(meta, "prompt_token_count", 0) or 0,
            "output_tokens": getattr(meta, "candidates_token_count", 0) or 0,
        }
        return resp

    def count_tokens(self, *, model, contents):
        resp = self._real.count_tokens(model=model, contents=contents)
        self.fixtures["token_counts"]["tok:" + fixture_key(model, contents, False)] = (
            resp.total_tokens
        )
        return resp


class RecordingClient:
    """包住真實 client，一邊正常呼叫一邊把結果錄下來。"""

    def __init__(self, real_client):
        # ⚠️ 一定要保留 real_client 本身的參照，不能只留 .models。
        #
        # 只留 .models 的話，genai.Client 物件在 make_client() 回傳後就沒有人
        # 參照，會被 GC 回收並關閉底層連線；之後每一次呼叫都會拿到
        # 「Cannot send a request, as the client has been closed.」，
        # 而且是**每一次都失敗、token 全為 0**，看起來像 API 壞掉。
        #
        # 這個 bug 只在真實 client 上會發生（測試用的假 client 沒有連線可關），
        # 所以單元測試抓不到 —— 實際錄製 50 筆時整批炸掉才發現。
        self._real_client = real_client
        self.models = _RecordingModels(real_client.models)

    def dump(self) -> dict:
        return self.models.fixtures


def make_client(fixtures: dict | None = None):
    """建立 client。

    三種模式：
      OFFLINE_MODE    → ReplayClient，零網路，重播錄好的輸出（無網路時）
      RECORD_FIXTURES → RecordingClient，正常呼叫並錄下來（有網路時）
      預設             → 真實 client
    """
    if OFFLINE_MODE and RECORD_FIXTURES:
        raise RuntimeError("OFFLINE_MODE 與 RECORD_FIXTURES 不能同時為 True。")

    if OFFLINE_MODE:
        if not fixtures or not fixtures.get("calls"):
            raise RuntimeError(
                "OFFLINE_MODE=True 但沒有可用的 fixtures。\n"
                "請先在有網路的環境設定 RECORD_FIXTURES=True 跑一次，產生 "
                f"{FIXTURES_FILE}，再重新產生 notebook。"
            )
        return ReplayClient(fixtures)

    from google import genai

    if USE_VERTEX:
        real = genai.Client(
            vertexai=True,
            project=PROJECT_ID,
            location=LOCATION,
        )
    else:
        if not FALLBACK_API_KEY:
            raise RuntimeError(
                "USE_VERTEX=False 但 FALLBACK_API_KEY 是空的。"
                "請在 CONFIG cell 填入 AI Studio key，或改回 USE_VERTEX=True。"
            )
        real = genai.Client(api_key=FALLBACK_API_KEY)

    return RecordingClient(real) if RECORD_FIXTURES else real


def generate(
    client,
    prompt: str,
    *,
    structured: bool = False,
    model: str | None = None,
    temperature: float | None = None,
    max_retries: int = 3,
) -> GenResult:
    """單次生成。

    structured=True 時啟用 API 原生的 responseSchema —— 這是 v3 與 v2 的唯一差別。
    重點：不要用 prompt 硬凹 JSON 格式，要用 API 參數約束，模型才真的被限制在 schema 內。
    """

    model = model or GEN_MODEL
    temperature = GEN_TEMPERATURE if temperature is None else temperature

    cfg_kwargs = {"temperature": temperature}
    if structured:
        cfg_kwargs["response_mime_type"] = "application/json"
        cfg_kwargs["response_schema"] = COPY_SCHEMA

    last_err = None
    for attempt in range(max_retries):
        started = time.time()
        try:
            resp = client.models.generate_content(
                model=model,
                contents=prompt,
                config=gen_config(**cfg_kwargs),
            )
            meta = resp.usage_metadata
            return GenResult(
                text=resp.text or "",
                usage=Usage(
                    model=model,
                    input_tokens=getattr(meta, "prompt_token_count", 0) or 0,
                    output_tokens=getattr(meta, "candidates_token_count", 0) or 0,
                    latency_s=round(time.time() - started, 2),
                ),
            )
        except Exception as e:  # noqa: BLE001 — demo 要能撐過暫時性錯誤
            last_err = e
            if attempt < max_retries - 1:
                time.sleep(2**attempt)  # 指數退避

    return GenResult(
        text="",
        usage=Usage(model=model, input_tokens=0, output_tokens=0, latency_s=0.0),
        error=str(last_err),
    )


def run_parallel(fn, items, *, workers: int | None = None, label: str = "處理中"):
    """對每個項目呼叫 fn，並行執行，**保持原本的順序**。

    為什麼需要這個：評審模型單次要十幾秒，整份 demo 序列跑完要十幾分鐘
    （光是 rubric 逐條檢查那一段就約 9 分鐘）。這些呼叫彼此獨立，
    並行之後整份只剩兩三分鐘。錄製 fixtures 時差別很有感。

    刻意保持順序（用 index 回填而非 as_completed 的完成順序），
    這樣結果與序列版本完全一致，重播時的 key 也不會受影響。

    離線重播時每次呼叫都是查表、瞬間完成，並行與否沒差；但仍然走同一條
    程式路徑，避免「錄製走 A、重播走 B」這種不一致。
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    items = list(items)
    if not items:
        return []

    workers = workers or MAX_WORKERS
    results: list = [None] * len(items)
    step = max(1, len(items) // 5)
    done = 0

    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(fn, item): i for i, item in enumerate(items)}
        for fut in as_completed(futures):
            idx = futures[fut]
            results[idx] = fut.result()
            done += 1
            if done % step == 0 or done == len(items):
                print(f"  {label} {done}/{len(items)}")
    return results


def count_tokens(client, text: str, model: str | None = None) -> int:
    """實測中文的 token 數。

    不要背「中文一個字約幾個 token」這種經驗法則 —— 用這個函式直接量。
    量出來的數字比任何通用經驗法則可靠。
    """
    model = model or GEN_MODEL
    resp = client.models.count_tokens(model=model, contents=text)
    return resp.total_tokens

In [ ]:
# 已錄製 465 筆真實輸出，直接內嵌在這一格。
# 只要有這份資料，這個 notebook 單獨一個檔案就能零網路跑完。
OFFLINE_MODE = True   # ← 有 fixtures 就預設離線；要真的打 API 改成 False
FIXTURES = {"calls": {"a8e1f2a048f293548695": {"text": "好的，這是一份為「青研 CHINGYEN 多重玻尿酸保濕面膜」設計的電商文案，涵蓋了標題、痛點、產品特色、成分解析、規格資訊和行動呼籲。\n\n---\n\n## 💧 青研 CHINGYEN 多重玻尿酸保濕面膜 💧\n\n**✨ 售價：NT$350 (5片入) ✨**\n\n### 💎 文案主標題：\n\n**【青研 CHINGYEN】告別乾荒！五重玻尿酸深度補水，敷出水光肌的秘密武器 — 多重玻尿酸保濕面膜 (5片入)**\n\n---\n\n### 💡 第一部分：抓住痛點，引發共鳴\n\n**你是否也有這些肌膚困擾？**\n\n* ❌ 上妝容易卡粉、浮粉，底妝不服貼？\n* ❌ 肌膚經常感到緊繃、乾癢，甚至出現細紋？\n* ❌ 換季時肌膚特別敏感，保濕產品怎麼擦都覺得不夠力？\n\n**👉 缺水，是所有肌膚問題的根源！** 青研 CHINGYEN 專為乾燥、缺水肌設計，讓你的肌膚從此告別「渴」望！\n\n---\n\n### 🔬 第二部分：產品核心特色與優勢\n\n#### 🌊 核心科技：五重玻尿酸，層層滲透鎖水\n\n我們不只給你單一保濕，而是提供全方位的「立體補水網」！\n\n1. **表層補水：** 迅速舒緩乾燥，提升肌膚柔軟度。\n2. **中層滲透：** 將水分帶入肌底，長效儲存。\n3. **深層鎖水：** 形成水潤保護膜，防止水分流失。\n\n#### 🌿 奢華成分加乘：海藻醣 — 肌膚的天然防護罩\n\n添加被譽為「生命之糖」的**海藻醣**，有效保護細胞結構，抵抗環境壓力，讓肌膚在補水同時，強化自身保濕屏障。\n\n#### ☁️ 頂級膜布體驗：天絲蠶絲膜布\n\n* **輕薄服貼：** 彷彿第二層肌膚，完美貼合臉部輪廓，不浪費一滴精華。\n* **高效導入：** 極佳的吸附與釋放能力，確保 25ml 的精華液能完整導入肌膚。\n* **親膚透氣：** 溫和不刺激，讓敷臉成為一種享受。\n\n---\n\n### 🎯 第三部分：產品規格與承諾\n\n| 項目 | 規格說明 |\n| :--- | :--- |\n| **商品名稱** | 多重玻尿酸保濕面膜 |\n| **品牌** | 青研 CHINGYEN |\n| **售價** | NT$350 |\n| **容量** | 5片 / 盒 (每片含 25ml 飽滿精華) |\n| **主成分** | 五重玻尿酸、海藻醣 |\n| **膜布材質** | 天絲蠶絲膜布 |\n| **適用膚質** | 特別推薦乾燥肌、缺水肌、換季不適肌 |\n| **產地** | 台灣製造，品質安心保證 |\n\n---\n\n### 🛒 第四部分：立即行動呼籲\n\n**給肌膚最頂級的「水」療 SPA！**\n\n每週 2-3 次，讓青研多重玻尿酸保濕面膜，為你的肌膚注入源源不絕的生命力。\n\n**現在就加入購物車，體驗從肌底透出的水潤光澤！**\n\n**👉 點擊購買，開啟你的水光肌之旅！**", "input_tokens": 125, "output_tokens": 848}, "be8954da37ac5107d674": {"text": "好的，這是一份為您的「溯源焙所 魚膠原蛋白粉」量身打造的電商文案。\n\n---\n\n## 🐟 溯源焙所 | 深海魚膠原蛋白粉 (3000mg)\n\n**✨ 韓國原裝進口 | 睡前黃金修護 | 啟動你的美麗時光機 ✨**\n\n---\n\n### 💎 【產品速覽與價格】\n\n| 產品名稱 | 溯源焙所 魚膠原蛋白粉 |\n| :--- | :--- |\n| **主要成分** | 深海魚膠原蛋白胜肽 3000mg |\n| **規格** | 90粒 / 盒 (獨立包裝) |\n| **建議售價** | **NT$490** (超值體驗價！) |\n| **產地** | 韓國 (原裝進口) |\n| **分類** | 保健食品 / 美容保養 |\n\n---\n\n### 🌟 【文案主標題】\n\n**睡前一包，喚醒澎潤光采！韓國熱銷 3000mg 高濃度魚膠原，讓美麗從「深」層綻放！**\n\n---\n\n### 💖 【核心賣點與痛點解決】\n\n您是否覺得：上妝開始浮粉？素顏時少了光澤？面對歲月痕跡感到無力？\n\n**別擔心！溯源焙所為您帶來最高效的「睡美人」保養法！**\n\n#### 💡 **為什麼選擇溯源焙所魚膠原？**\n\n1. **高濃度、高效率吸收：**\n   * **單包足量 3000mg！** 採用深海魚膠原蛋白胜肽，分子極小，無需等待，身體能快速吸收利用，直達美麗核心。\n2. **韓國原裝進口 品質保證：**\n   * 嚴選韓國優質原料與製程，確保每一包膠原蛋白都達到最高標準。\n3. **黃金修護時間，效果加乘：**\n   * 建議「**睡前食用**」！把握夜間身體修復的黃金時段，讓膠原蛋白在您熟睡時，默默為您的美麗打底。\n4. **獨立包裝設計，方便衛生：**\n   * 90粒獨立分裝，方便攜帶，無論是辦公室、旅行或外出，隨時隨地都能輕鬆補充，確保每次食用都新鮮有效。\n5. **超值體驗價：**\n   * 僅需 **NT$490**，就能享受來自韓國的高品質美容保養，輕鬆入門無負擔！\n\n---\n\n### 📋 【產品細節與食用建議】\n\n| 特色 | 說明 |\n| :--- | :--- |\n| **主要成分** | 深海魚膠原蛋白胜肽 (3000mg) |\n| **型態** | 獨立小包裝粉末 (方便直接食用或加入飲品) |\n| **包裝容量** | 90粒 / 盒 (約3個月份量) |\n| **建議用量** | 每日 1 份 (1包/粒) |\n| **最佳食用時間** | **睡前食用** (搭配溫水或直接食用) |\n| **適用對象** | 追求澎潤光澤、注重日常保養、長時間待在冷氣房、希望提升素顏狀態者。 |\n\n---\n\n### 🛒 【立即行動呼籲】\n\n**別讓時間偷走你的光采！**\n\n現在就將【溯源焙所 魚膠原蛋白粉】加入購物車，每天睡前花一分鐘，投資未來的自己。\n\n**NT$490 限量體驗價，美麗無需等待！**\n\n**👉 點擊購買，啟動你的逆齡保養計畫！**\n\n---\n\n### 🔒 【安心保證】\n\n*   韓國原裝進口，品質嚴格把關。\n*   本產品已投保產品責任險。\n*   如有任何疑問，歡迎隨時聯繫客服。", "input_tokens": 126, "output_tokens": 916}, "13914b0f573fd57c034c": {"text": "## 青研 CHINGYEN 胺基酸溫和洗面乳\n\n### 商品標題\n**青研 CHINGYEN 胺基酸溫和洗面乳 120ml 弱酸性 pH5.5 敏弱肌友善**\n\n### 賣點\n1. 高濃度胺基酸，溫和潔淨不刺激。\n2. 貼近肌膚弱酸性 pH5.5，維持健康屏障。\n3. 綿密細緻泡沫，洗後清爽不緊繃。\n4. 添加洋甘菊萃取，洗臉同時舒緩肌膚。\n\n### SEO 描述\n專為敏弱肌設計的青研胺基酸溫和洗面乳。採用高純度胺基酸界面活性劑，搭配弱酸性pH5.5配方，溫和洗淨不乾澀。洗後肌膚柔軟舒適，成分黨安心選擇。規格120ml，台灣製造。\n\n### 標籤\n胺基酸洗面乳, 敏弱肌, pH5.5, 弱酸性, 青研", "input_tokens": 815, "output_tokens": 240}, "49a530dc757b4cc93e87": {"text": "{\n  \"title\": \"晨光研選 小分子魚膠原蛋白粉 蔓越莓風味 30入 每日補充 2500mg\",\n  \"bullets\": [\n    \"深海魚膠原蛋白胜肽，足量 2500mg\",\n    \"清新蔓越莓風味，順口無腥味\",\n    \"小分子胜肽形式，好吸收利用\",\n    \"睡前一包，輕鬆養成美麗習慣\"\n  ],\n  \"seo_description\": \"晨光研選專為注重日常保養的女性設計。每包含足量 2500mg 小分子魚膠原蛋白胜肽，幫助您維持青春光采。搭配酸甜蔓越莓風味，口感佳，粉末隨身包方便每日持續補充。\",\n  \"hashtags\": [\n    \"膠原蛋白粉\",\n    \"魚膠原胜肽\",\n    \"蔓越莓風味\",\n    \"日常保養\",\n    \"美容保健\"\n  ]\n}", "input_tokens": 929, "output_tokens": 227}, "68c443f602248c9a2d1c": {"text": "## 【青研 CHINGYEN】五重玻尿酸超導水潤面膜 5片入｜敏弱肌專屬、天絲服貼\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1.  五重玻尿酸精華，層層注水直達肌底。\n2.  無添加酒精香精，成分黨安心首選。\n3.  輕薄天絲蠶絲膜布，極致服貼親膚透氣。\n4.  密集修護乾燥缺水，一盒 5片超值組。\n\n---\n\n### SEO 描述 (不超過 120 字)\n\n專為敏弱肌設計的青研 CHINGYEN 多重玻尿酸保濕面膜。嚴選五重玻尿酸與海藻醣，深層補水鎖水。採用輕薄天絲蠶絲膜布，溫和不刺激。無添加爭議成分，成分黨必備。超值組合 5片入，讓肌膚重現水潤光澤。\n\n---\n\n### 標籤 (3-5 個)\n\n#玻尿酸面膜\n#敏弱肌保養\n#天絲面膜\n#成分黨推薦\n#CHINGYEN", "input_tokens": 262, "output_tokens": 278}, "b7da0279ca8b6b47b426": {"text": "{\n  \"title\": \"溯源焙所 職人選豆 低溫慢焙 綜合堅果 350g 無調味 原型食物健康零食\",\n  \"bullets\": [\n    \"低溫烘焙製程，非油炸，保留堅果完整營養\",\n    \"嚴選四種頂級堅果，完全無調味、無添加糖鹽\",\n    \"內含腰果、核桃、杏仁、珍貴夏威夷豆\",\n    \"健身族群與家庭常備，補充優質油脂與膳食纖維\"\n  ],\n  \"seo_description\": \"溯源焙所嚴選頂級綜合堅果，採用低溫慢焙工法，完整保留堅果原始風味與營養。本產品為無調味 350g 大容量包裝，無添加糖鹽、無油炸，是注重原型食物的家庭與健身族群最佳選擇。每日適量攝取，享受天然堅果的健康美味。\",\n  \"hashtags\": [\n    \"綜合堅果\",\n    \"無調味堅果\",\n    \"原型食物\",\n    \"低溫烘焙\",\n    \"健身飲食\"\n  ]\n}", "input_tokens": 920, "output_tokens": 269}, "02324d580ce63f368202": {"text": "{\n  \"title\": \"Volta 手持無線吸塵器 Lite 18000Pa強勁吸力 輕量1.4kg 小坪數/寵物家庭適用\",\n  \"bullets\": [\n    \"18000Pa颶風吸力，深入清潔毛髮與微塵。\",\n    \"輕量化機身僅1.4kg，單手操作不費力，高處清潔無負擔。\",\n    \"標準模式續航35分鐘，滿足小坪數空間快速清掃需求。\",\n    \"配備床墊吸頭與軟絨滾筒，有效處理寵物毛髮與皮屑。\"\n  ],\n  \"seo_description\": \"Volta Lite 無線吸塵器，專為小坪數家庭與寵物族群打造。具備18000Pa強勁吸力，機身僅1.4kg，輕鬆應對高處與縫隙清潔。標準模式續航達35分鐘，集塵盒可水洗，維持居家潔淨。\",\n  \"hashtags\": [\n    \"無線吸塵器\",\n    \"18000Pa\",\n    \"輕量吸塵器\",\n    \"寵物家庭\",\n    \"小坪數家電\"\n  ]\n}", "input_tokens": 783, "output_tokens": 284}, "b130b88268a1a340f6e0": {"text": "## Volta 無線降噪藍牙耳機 Pro 2\n\n---\n\n### 商品標題 (38 字)\n\n**Volta Pro 2 旗艦無線降噪耳機｜42dB深度靜音艙 藍牙5.4 續航32小時**\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1.  **42dB 深度降噪**，隔絕捷運喧囂，為通勤族打造專屬靜音艙。\n2.  超長續航力，總計 **32小時** 電力，滿足一週高效工作不間斷。\n3.  採用 **藍牙5.4** 技術，連線極速穩定，支援雙裝置無縫切換。\n4.  單耳僅 4.8g 輕量設計，IPX4 防水，長時間配戴依然舒適。\n\n---\n\n### SEO 描述 (109 字)\n\nVolta Pro 2 是通勤族與遠距工作者的最佳選擇。搭載 42dB 深度**降噪**技術，有效消除環境雜音，提升專注力。**藍牙5.4** 提供極速穩定連線，並支援多裝置切換。總續航力長達 **32小時**，讓您從早到晚保持高效。輕巧舒適，隨時隨地打造您的個人靜音工作空間。\n\n---\n\n### 標籤 (5 個)\n\n#降噪耳機\n#藍牙5.4\n#通勤必備\n#遠距工作\n#Volta\n---", "input_tokens": 260, "output_tokens": 349}, "bd4b6acdad4b34fd8900": {"text": "{\n  \"title\": \"晨光研選 緩釋B群錠 維生素B群複方 100mg 20粒 睡前補充\",\n  \"bullets\": [\n    \"採用緩釋技術，持續釋放營養，幫助維持整日精神。\",\n    \"足量維生素B群複方，支持身體正常能量代謝。\",\n    \"每粒含足量 100mg B群，每日睡前一錠，輕鬆納入日常。\",\n    \"適合長時間用腦、經常感到疲勞的上班族群持續補充。\"\n  ],\n  \"seo_description\": \"晨光研選緩釋B群錠，專為忙碌的上班族設計。採用緩釋技術，穩定提供維生素B群複方，幫助維持一整天的精神旺盛。每盒 20粒，每粒含 100mg B群。睡前輕鬆補充，為您的日常活力打底。\",\n  \"hashtags\": [\n    \"B群\",\n    \"緩釋B群\",\n    \"上班族必備\",\n    \"精神體力\",\n    \"維生素B群複方\"\n  ]\n}", "input_tokens": 908, "output_tokens": 260}, "149ef919c2da7c353c52": {"text": "{\n  \"title\": \"白日光 SUNDAY 純淨卸妝油 100ml 1入 | 韓國植萃植物油脂，溫和卸妝，乾燥肌適用。\",\n  \"bullets\": [\n    \"嚴選天然植物油脂，親膚溫和不刺激。\",\n    \"輕盈油狀質地，快速乳化，清爽無負擔。\",\n    \"專為乾燥缺水肌設計，卸後水潤不緊繃。\",\n    \"韓國原裝進口，享受純淨卸妝儀式感。\"\n  ],\n  \"seo_description\": \"白日光 SUNDAY 純淨卸妝油 100ml 1入，專為30-45歲重視保濕的族群打造。採用天然植物油脂配方，溫和溶解彩妝與髒污。油狀質地輕盈好推，快速乳化，卸後肌膚維持水潤感，告別乾澀緊繃。每日純淨卸妝，讓肌膚深呼吸。\",\n  \"hashtags\": [\n    \"卸妝油\",\n    \"植物油脂\",\n    \"溫和卸妝\",\n    \"韓國美妝\",\n    \"乾燥肌保濕\"\n  ]\n}", "input_tokens": 779, "output_tokens": 270}, "99a6cae296211492f3d6": {"text": "{\n  \"title\": \"Volta 65W 氮化鎵三孔快充頭 雙USB-C+USB-A 輕巧高效差旅首選\",\n  \"bullets\": [\n    \"65W大功率輸出，滿足筆電、平板等高耗能裝置快充需求。\",\n    \"採用第三代氮化鎵技術，實現輕量化設計，尺寸僅5.2 × 5.2 × 2.9 cm。\",\n    \"配置三孔輸出 (雙C一A)，可同時為三台裝置供電，提升充電效率。\",\n    \"重量僅112g，內建過壓、過流、過熱保護機制，確保充電安全。\"\n  ],\n  \"seo_description\": \"Volta 65W 氮化鎵三孔快充充電器，專為多裝置使用者與出差族設計。提供65W總輸出，配置USB-C×2與USB-A×1，可同時為筆電、手機、平板充電。採用GaN技術，體積小巧，高效穩定，一機滿足所有快充需求。\",\n  \"hashtags\": [\n    \"65W充電器\",\n    \"氮化鎵GaN\",\n    \"三孔快充\",\n    \"差旅必備\",\n    \"多裝置充電\"\n  ]\n}", "input_tokens": 790, "output_tokens": 301}, "b9add45dae454183c5e1": {"text": "## 溯源焙所 冷萃掛耳咖啡\n\n### 商品標題\n\n**溯源焙所 耶加雪菲 淺焙 水果調 冷萃掛耳咖啡 10包入**\n\n---\n\n### 賣點\n\n1. 經典耶加雪菲，柑橘莓果風味。\n2. 精選淺焙，保留細緻花香。\n3. 輕巧掛耳設計，一盒內含 10包。\n4. 浸泡 8 小時，輕鬆享受冷萃醇厚。\n\n---\n\n### SEO 描述\n\n溯源焙所嚴選衣索比亞耶加雪菲，以專業淺焙技術，展現乾淨明亮的柑橘與莓果調性。專為冷萃設計的掛耳包，只需常溫浸泡，即可享受精品咖啡的甘甜醇厚。每盒 10包，讓您隨時隨地享受高品質冷萃咖啡。\n\n---\n\n### 標籤\n\n#冷萃咖啡 #掛耳包 #耶加雪菲 #淺焙 #精品咖啡", "input_tokens": 812, "output_tokens": 233}, "201a78f781c90a372273": {"text": "## 台灣紅烏龍茶包 禮盒組\n\n---\n\n### 商品標題 (不超過 60 個字)\n\n**溯源焙所｜台東鹿野 紅烏龍三角茶包 蜜香熟果重發酵 20包 禮盒組**\n\n---\n\n### 賣點 (4 條，每條不超過 30 個字)\n\n1. **台東鹿野紅烏龍，重發酵蜜香熟果。**\n2. **三角立體茶包，90秒快速回甘。**\n3. **醇厚蜜香尾韻，溫潤回甘不澀口。**\n4. **內含足量20包，送禮體面大方。**\n\n---\n\n### SEO 描述 (不超過 120 個字)\n\n來自台東鹿野的溯源焙所紅烏龍茶包，採用重發酵工藝，散發天然蜜香與熟果風味。三角立體茶包設計，讓您隨時享受醇厚回甘的頂級茶湯。每盒內含20包，精緻包裝，是送禮或日常品茗的最佳選擇。\n\n---\n\n### 標籤 (3-5 個)\n\n#紅烏龍 #鹿野茶 #蜜香茶 #送禮禮盒 #三角茶包", "input_tokens": 271, "output_tokens": 294}, "98c756e43637c8ede624": {"text": "## 青研 CHINGYEN 積雪草舒緩精華液 30ml 文案\n\n---\n\n### 商品標題\n\n**青研 CHINGYEN 5%高濃度積雪草舒緩精華 30ml | 敏弱泛紅急救，成分黨必收！**\n\n---\n\n### 賣點\n\n1. 5%高濃度積雪草，快速穩定泛紅膚況。\n2. 水感輕盈精華，秒速吸收不黏膩。\n3. 強化屏障，添加泛醇與神經醯胺。\n4. 韓國原裝進口，敏弱肌安心適用。\n\n---\n\n### SEO 描述\n\n**青研 CHINGYEN 積雪草舒緩精華液 30ml，成分黨必備！內含 5% 高濃度積雪草萃取，有效急救泛紅、修護受損。搭配泛醇與神經醯胺，深層補水並強化肌膚屏障。韓國製造，敏弱肌安心適用。**\n\n---\n\n### 標籤\n\n積雪草精華、敏弱肌、泛紅、修護、韓國保養", "input_tokens": 262, "output_tokens": 257}, "51b1980034fa13f8420a": {"text": "{\n  \"title\": \"晨光研選 游離型葉黃素膠囊 30mg高濃度配方 60粒\",\n  \"bullets\": [\n    \"採用游離型葉黃素，分子小好吸收，適合每日持續補充。\",\n    \"高濃度配方：每粒含足量葉黃素 30mg 及玉米黃素 6mg。\",\n    \"適合長時間使用電腦、手機的上班族，維持舒適晶亮。\",\n    \"每日一粒，一盒 60粒裝，輕鬆持續補充兩個月。\"\n  ],\n  \"seo_description\": \"晨光研選葉黃素膠囊，專為長時間使用3C產品的上班族設計。每粒提供高劑量游離型葉黃素30mg，搭配玉米黃素。每日只需一粒，一盒60粒，輕鬆維持晶亮舒適。溫和配方，為您的日常保養打下基礎。\",\n  \"hashtags\": [\n    \"葉黃素\",\n    \"游離型\",\n    \"30mg\",\n    \"晶亮舒適\",\n    \"3C族必備\"\n  ]\n}", "input_tokens": 908, "output_tokens": 261}, "87a8f4cd2581dd2d95e3": {"text": "## 白日光 SUNDAY 純淨卸妝油 100ml | 溫和植物油脂配方 卸妝同時滋潤乾燥肌\n\n---\n\n### 賣點\n\n1.  嚴選親膚**植物油脂**，溫和卸妝無負擔。\n2.  專為乾燥缺水肌設計，洗後水潤不緊繃。\n3.  油狀質地輕盈好乳化，快速帶走彩妝髒污。\n4.  韓國原裝進口，**100ml** 小巧容量，旅行必備**1入**。\n\n---\n\n### SEO 描述\n\n30-45歲重視保濕的您，一定要試試白日光 SUNDAY 純淨卸妝油！採用親膚**植物油脂**基底，溫和卸除彩妝，同時維持肌膚水感。**100ml** 輕巧瓶身，韓國原裝進口，每盒**1入**。給予乾燥缺水肌最溫柔的清潔體驗，卸妝不再是肌膚的負擔。\n\n---\n\n### 標籤\n\n#卸妝油\n#乾燥肌保濕\n#植物油脂\n#溫和卸妝\n#韓國美妝", "input_tokens": 672, "output_tokens": 267}, "4315e70a09479382b85b": {"text": "## 溯源焙所 魚膠原蛋白粉\n\n### 商品標題\n**【溯源焙所】韓國專利 深海魚膠原蛋白胜肽 3000mg｜90粒裝｜睡前黃金美容配方，重現Q彈緊緻**\n\n---\n\n### 賣點\n1. 嚴選深海魚膠原蛋白胜肽，分子小好吸收。\n2. 足量添加 3000mg，每日補足青春能量。\n3. 韓國原裝進口，獨立包裝，隨身攜帶超方便。\n4. 建議睡前食用，把握夜間黃金修護期。\n\n---\n\n### SEO 描述\n專為30-50歲熟齡女性設計。溯源焙所嚴選韓國專利**深海魚膠原蛋白胜肽**，每日足量**3000mg**，分子小吸收快。一盒**90粒**獨立包裝，方便攜帶。睡前一包，輕鬆補充流失的膠原，維持肌膚彈潤與光采，由內而外散發青春美麗。\n\n---\n\n### 標籤\n膠原蛋白 / 膠原胜肽 / 美容保養 / 熟齡保養 / 魚膠原", "input_tokens": 272, "output_tokens": 284}, "0187998e7c9d08bc107a": {"text": "{\n  \"title\": \"溯源焙所 深海魚膠原蛋白胜肽 3000mg 90粒 每日青春補給\",\n  \"bullets\": [\n    \"深海魚膠原蛋白胜肽，純粹來源。\",\n    \"足量 3000mg，每日定額補充。\",\n    \"韓國進口原料，獨立膠囊包裝。\",\n    \"90粒裝，建議睡前輕鬆食用。\"\n  ],\n  \"seo_description\": \"專為注重日常保養的女性設計。溯源焙所嚴選韓國進口深海魚膠原蛋白胜肽，每份足量3000mg。90粒膠囊包裝，方便每日睡前輕鬆補充，由內而外維持光采。\",\n  \"hashtags\": [\n    \"膠原蛋白\",\n    \"魚膠原胜肽\",\n    \"每日保養\",\n    \"3000mg\",\n    \"溯源焙所\"\n  ]\n}", "input_tokens": 923, "output_tokens": 226}, "2f7050ae6cae4f2cb5b5": {"text": "## Volta 65W 氮化鎵三孔快充充電器 文案\n\n---\n\n### 商品標題\n**【Volta】迷你口袋型 65W 氮化鎵三孔快充頭｜PD/QC 極速充電，筆電手機一次搞定 (48字)**\n\n---\n\n### 賣點\n1. **65W PD極速快充，筆電電力瞬間回血。**\n2. **獨家三孔設計，手機、平板、筆電同步滿電。**\n3. **採用氮化鎵技術，體積縮小50%，輕巧好攜帶。**\n4. **僅112g超輕量，出差旅行一顆抵三顆。**\n\n---\n\n### SEO 描述\nVolta 65W 氮化鎵快充頭，專為多裝置使用者設計。具備USB-C×2+USB-A三孔輸出，最高65W PD快充，輕鬆應付筆電、手機、平板。體積輕巧，出差旅行必備，支援多重安全保護，讓您告別笨重變壓器。\n\n---\n\n### 標籤\n#65W快充\n#氮化鎵GaN\n#出差旅行\n#多孔充電器\n#PD充電", "input_tokens": 261, "output_tokens": 286}, "8cc2ed63ae4a1e974e27": {"text": "## 晨光研選 小分子魚膠原蛋白粉 2500mg 蔓越莓風味 30入\n\n---\n\n### 賣點\n1. 每日足量補充 2500mg 膠原蛋白\n2. 嚴選小分子魚膠原蛋白胜肽，好吸收\n3. 清新蔓越莓風味，粉末細緻好入口\n4. 獨立隨身包裝，輕鬆融入日常保養\n\n---\n\n### SEO 描述\n專為30-50歲女性設計的日常保養。晨光研選小分子魚膠原蛋白粉，每包提供足量2500mg膠原蛋白胜肽，搭配清新蔓越莓風味，酸甜順口。粉末隨身包，睡前一包，持續為美麗打底。\n\n---\n\n### 標籤\n膠原蛋白、魚膠原、蔓越莓、美容保養、日常補充", "input_tokens": 822, "output_tokens": 207}, "25b652acacbe93e21b9f": {"text": "{\n  \"title\": \"青研 CHINGYEN 積雪草舒緩精華液 30ml 溫和配方 敏弱肌適用\",\n  \"bullets\": [\n    \"高濃度 5% 積雪草萃取，溫和舒緩肌膚。\",\n    \"搭配泛醇與神經醯胺，維持肌膚健康屏障。\",\n    \"水感輕盈精華質地，快速吸收不黏膩。\",\n    \"無添加酒精香料，敏弱肌友善配方。\"\n  ],\n  \"seo_description\": \"青研積雪草舒緩精華液 30ml，專為敏弱肌設計。蘊含 5% 高濃度積雪草萃取，搭配泛醇與神經醯胺，幫助肌膚穩定。水感質地輕盈好吸收，不造成肌膚負擔。成分透明，給追求安心保養的你。\",\n  \"hashtags\": [\n    \"青研CHINGYEN\",\n    \"積雪草精華\",\n    \"敏弱肌保養\",\n    \"成分黨\",\n    \"舒緩\"\n  ]\n}", "input_tokens": 924, "output_tokens": 256}, "c02b06a830e22129c4e0": {"text": "## 青研 CHINGYEN 胺基酸溫和潔顏乳\n\n---\n\n### 商品標題\n\n**青研 CHINGYEN 胺基酸溫和潔顏乳 120ml | pH5.5弱酸配方 敏弱肌專屬的修護級洗顏** (55字)\n\n---\n\n### 賣點\n\n1.  高濃度胺基酸界面活性劑，溫和淨膚。\n2.  弱酸性 pH5.5，維持肌膚健康屏障。\n3.  添加洋甘菊萃取，舒緩敏弱不適。\n4.  豐盈綿密泡沫，洗後水潤不緊繃。\n\n---\n\n### SEO 描述\n\n**青研 CHINGYEN 專為敏弱肌設計。使用高純度胺基酸配方，溫和清潔不刺激。pH5.5 弱酸性，洗淨同時修護肌膚屏障。120ml 大容量，成分黨安心首選。** (99字)\n\n---\n\n### 標籤\n\n#胺基酸洗面乳 #敏弱肌 #pH5.5 #弱酸性 #成分黨", "input_tokens": 260, "output_tokens": 260}, "af7563768f3867b2e5a6": {"text": "{\n  \"title\": \"青研 CHINGYEN 多重玻尿酸保濕面膜 5片入｜五重玻尿酸精華 敏弱肌適用\",\n  \"bullets\": [\n    \"五重玻尿酸配方，層層補水，維持肌膚水嫩。\",\n    \"採用輕薄服貼的天絲蠶絲膜布，親膚透氣。\",\n    \"一盒內含5片，每片25ml精華，高效保濕。\",\n    \"溫和成分，質地清爽，乾燥敏弱肌適用。\"\n  ],\n  \"seo_description\": \"青研多重玻尿酸保濕面膜，專為敏弱肌設計。蘊含五重玻尿酸精華，提供深層且持久的保濕效果。輕盈的天絲蠶絲膜布，服貼臉部輪廓。一盒5片裝，幫助肌膚維持健康水循環。\",\n  \"hashtags\": [\n    \"青研CHINGYEN\",\n    \"保濕面膜\",\n    \"玻尿酸\",\n    \"敏弱肌保養\",\n    \"成分黨\"\n  ]\n}", "input_tokens": 924, "output_tokens": 257}, "07ddd306c0c58d7cfd30": {"text": "## 溯源焙所 低溫烘焙綜合堅果 無調味 350g\n\n---\n\n### 商品標題\n**【溯源焙所】原型食物首選！低溫烘焙綜合堅果 350g 無調味，純淨營養補給。** (54 字)\n\n---\n\n### 賣點\n1. 堅持純粹無添加，完全無調味、無糖鹽。 (20 字)\n2. 低溫慢焙鎖住營養，非油炸更健康。 (15 字)\n3. 健身族群、家庭常備的優質原型食物。 (18 字)\n4. 嚴選四種頂級綜合堅果，包含珍貴夏威夷豆。 (20 字)\n\n---\n\n### SEO 描述\n尋找最純淨的健康零食？【溯源焙所】低溫烘焙綜合堅果 350g，堅持無調味、無糖鹽，完整保留堅果的天然風味與營養。是健身族群的能量補充、更是家庭常備的原型食物首選。每日一把，輕鬆攝取優質脂肪與膳食纖維。 (98 字)\n\n---\n\n### 標籤\n#綜合堅果\n#原型食物\n#無調味\n#低溫烘焙\n#健身零食", "input_tokens": 269, "output_tokens": 297}, "1c41a7afbd847ca288a5": {"text": "{\n  \"title\": \"晨光研選 五益菌複方益生菌粉 30包入 | 每包100億CFU 益生菌 | 調整體質\",\n  \"bullets\": [\n    \"五株專利益生菌配方，幫助維持消化道機能。\",\n    \"每包足量 100億 CFU 益生菌，穩定性高。\",\n    \"每日一包，共 30包 入，方便持續補充。\",\n    \"粉末隨身包設計，適合忙碌外食族群。\"\n  ],\n  \"seo_description\": \"晨光研選五益菌複方益生菌粉，專為外食族及作息不規律者設計。每包含有 100億 CFU 活性益生菌，五株專利菌種，幫助維持消化道機能。一盒 30包 份量，輕巧好攜帶，每日持續補充，輕鬆做好日常保健。\",\n  \"hashtags\": [\n    \"益生菌\",\n    \"消化道保健\",\n    \"晨光研選\",\n    \"外食族必備\",\n    \"100億\"\n  ]\n}", "input_tokens": 915, "output_tokens": 269}, "7fd2bee00c7da7327416": {"text": "## 晨光研選 小分子魚膠原蛋白粉 | 2500mg高濃度添加 | 蔓越莓風味 | 睡前美妍配方\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1.  胜肽級小分子，吸收率極佳，美麗直達。\n2.  足量添加 2500mg 膠原蛋白，高效有感。\n3.  嚴選天然蔓越莓風味，酸甜順口無腥味。\n4.  獨立隨身包裝，每日一包，輕鬆養成美妍習慣。\n\n---\n\n### SEO 描述 (不超過 120 字)\n\n30-50歲熟齡女性的青春秘密！晨光研選小分子魚**膠原蛋白**粉，採用高吸收率的胜肽級膠原蛋白，每包足量添加 **2500mg**，搭配維生素C協同作用。酸甜**蔓越莓**風味，睡前飲用，輕鬆為美麗打底。告別歲月痕跡，由內而外散發光采。\n\n---\n\n### 標籤 (3-5 個)\n\n#膠原蛋白\n#小分子胜肽\n#熟齡保養\n#蔓越莓\n#美容飲品", "input_tokens": 285, "output_tokens": 292}, "e5a84726738a1a5e3f3c": {"text": "**商品標題**\n本木 BENMU 16L 高效除濕機 | 美國陽極處理機身 | 樸實耐用 首購族CP值首選\n\n**賣點**\n1. 每日除濕量達 16L，輕鬆應對台灣潮濕季節。\n2. 採用美國陽極處理工法，機身結構紮實耐用。\n3. 終身技術支援服務，讓您買得安心，用得長久。\n4. 高CP值入門款，功能到位，為居家環境帶來乾爽。\n\n**SEO 描述**\n正在找耐用又划算的除濕機嗎？本木 BENMU 16L 除濕機，專為台灣潮濕氣候設計。採用紮實的陽極處理工法，提供穩定高效的除濕能力。高CP值首選，終身技術支援，讓您的居家環境保持乾爽舒適。\n\n**標籤**\n除濕機、16L、高CP值、首購族、BENMU", "input_tokens": 501, "output_tokens": 229}, "71063809d5d7a348272b": {"text": "{\n  \"title\": \"本木 BENMU 16L 高效能除濕機，陽極耐用機身，首購族高CP值入門款\",\n  \"bullets\": [\n    \"每日除濕量達 16L，快速改善室內濕度，告別潮濕感。\",\n    \"採用陽極處理機身，結構紮實，符合樸實耐用的品牌調性。\",\n    \"終身技術支援服務，提供長久後勤保障，使用更安心無虞。\",\n    \"美國製造工藝標準，品質穩定可靠，是小資族群的實惠選擇。\"\n  ],\n  \"seo_description\": \"本木 BENMU 16L 除濕機專為重視 CP 值的首購族設計。提供每日 16L 強勁除濕力，有效調節家中濕氣。機身採陽極處理，結構堅固耐用。搭配終身技術支援，讓您以實惠價格享受穩定可靠的除濕體驗。\",\n  \"hashtags\": [\n    \"除濕機\",\n    \"16L\",\n    \"本木BENMU\",\n    \"CP值首選\",\n    \"耐用家電\"\n  ]\n}", "input_tokens": 608, "output_tokens": 271}, "37a8b4a2a4e20e4b61c2": {"text": "{\n  \"title\": \"Volta 無線降噪藍牙耳機 Pro 2。42dB ANC，藍牙5.4，總續航32小時。\",\n  \"bullets\": [\n    \"42dB 主動降噪，有效隔絕環境噪音，保持專注。\",\n    \"總續航達 32小時，單次使用可達 8 小時。\",\n    \"藍牙5.4 穩定連線，支援雙裝置快速切換。\",\n    \"單耳 4.8g 輕量設計，具備 IPX4 防水等級。\"\n  ],\n  \"seo_description\": \"Volta Pro 2 專為通勤與遠距工作設計。搭載 42dB 強效降噪技術，在吵雜環境中也能保持專注。採用最新的藍牙5.4 晶片，連線穩定。搭配充電盒提供總續航 32小時，滿足全日使用需求。\",\n  \"hashtags\": [\n    \"降噪耳機\",\n    \"藍牙5.4\",\n    \"32小時續航\",\n    \"通勤必備\",\n    \"遠距工作\"\n  ]\n}", "input_tokens": 789, "output_tokens": 263}, "9a570092b5786fc66265": {"text": "**商品標題**\n溯源焙所 職人嚴選 深海魚膠原蛋白胜肽 3000mg 90粒 韓國原裝\n\n**賣點**\n1. 每日足量 3000mg 深海魚膠原蛋白胜肽\n2. 胜肽小分子結構，吸收利用率佳\n3. 韓國原裝進口，獨立包裝好攜帶\n4. 睡前補充，輕鬆納入日常保養程序\n\n**SEO 描述**\n溯源焙所嚴選高品質深海魚膠原蛋白胜肽，專為30-50歲注重保養的女性設計。每日份量足量3000mg，小分子胜肽結構，好吸收。一盒90粒，建議睡前食用，輕鬆維持日常光采。職人精神，為您的美麗打底。\n\n**標籤**\n膠原蛋白、胜肽、熟齡保養、韓國進口", "input_tokens": 816, "output_tokens": 214}, "fe26e02333506ff90761": {"text": "身為台灣電商平台的資深商品文案編輯，我為您撰寫以下文案，確保符合所有法規與格式要求。\n\n---\n\n## 晨光研選 金盞花萃取葉黃素膠囊\n\n### 商品標題 (29 字)\n**晨光研選 金盞花游離型葉黃素膠囊 30mg 60粒**\n\n### 賣點 (4 條，每條不超過 30 字)\n1. 足量游離型葉黃素 30mg，吸收率佳。\n2. 採用葉黃素：玉米黃素 5:1 黃金比例。\n3. 專為長時間使用螢幕的上班族設計。\n4. 每盒 60粒，每日一粒，持續補充兩個月。\n\n### SEO 描述 (91 字，不超過 120 字)\n晨光研選金盞花萃取葉黃素，提供足量 30mg 游離型葉黃素與玉米黃素，採用黃金比例配方。適合長時間面對電腦、手機的上班族，幫助維持晶亮舒適。植物膠囊，素食可食。每盒 60粒，輕鬆維持日常保養。\n\n### 標籤 (3-5 個)\n葉黃素、游離型葉黃素、眼部保養、30mg、上班族", "input_tokens": 801, "output_tokens": 309}, "8a26786006c4e77006fa": {"text": "好的，這是一份針對「溯源焙所 冷萃掛耳咖啡 淺焙水果調 10入」所撰寫的電商文案，包含了標題、痛點切入、產品介紹、風味描述、沖泡指引和行動呼籲。\n\n---\n\n## 溯源焙所 冷萃掛耳咖啡 淺焙水果調 10入 電商文案\n\n### 標題選項 (擇一或組合使用)\n\n**主標題：**\n【溯源焙所】夏日限定！冷萃掛耳咖啡 淺焙水果調 10入 — 冰鎮耶加雪菲，在家享受精品級果香！\n\n**副標題 (強調便利與風味)：**\n免磨豆、免濾紙！5分鐘完成冰咖啡，柑橘莓果花香一應俱全。\n\n---\n\n### 內文：告別苦澀，迎接清爽！\n\n#### 💡 痛點切入：你還在喝苦澀的即溶咖啡嗎？\n\n炎炎夏日，想來一杯清爽解渴的冰咖啡，卻又擔心：\n* **❌ 傳統熱沖咖啡** 降溫慢，風味容易跑掉？\n* **❌ 傳統冰咖啡** 設備複雜，手沖技巧門檻高？\n* **❌ 即溶咖啡** 只有焦苦味，缺乏層次感？\n\n**【溯源焙所】聽到你的心聲了！** 我們將精品咖啡的極致風味，濃縮進這款專為「冷萃」設計的掛耳包中，讓你在家也能輕鬆享受清爽、無負擔的夏日咖啡盛宴。\n\n---\n\n#### 🌟 產品亮點：精品級冷萃，隨時隨地享受\n\n**商品名稱：** 溯源焙所 冷萃掛耳咖啡 淺焙水果調 10入\n**售價：** NT$460 (平均每杯不到 $50，享受精品級體驗)\n\n| 特色 | 描述 |\n| :--- | :--- |\n| **頂級產區** | 嚴選咖啡故鄉 **衣索比亞 耶加雪菲**，得天獨厚的風土孕育出迷人的花果香氣。 |\n| **淺焙工藝** | 專業淺焙，完整保留咖啡豆中最珍貴的 **柑橘、莓果、花香**，告別焦苦，只留清甜。 |\n| **冷萃專用** | 獨家研磨度與烘焙曲線，專為冷水浸泡設計，**5分鐘快速萃取**，風味乾淨、酸甜平衡。 |\n| **極致便利** | 獨立掛耳包裝（12g/包），免磨豆、免濾紙，隨時隨地都能享受，辦公室或戶外露營的最佳選擇。 |\n\n---\n\n#### 🍊 風味解鎖：一口入魂的夏日協奏曲\n\n這款淺焙水果調冷萃咖啡，是為喜愛明亮酸質與豐富層次的你量身打造：\n\n* **前調：** 清新的**柑橘**香氣率先引爆味蕾，帶來夏日般的活力。\n* **中調：** 接著是飽滿而甜美的**莓果**氣息，如同一顆顆紅寶石在口中綻放。\n* **尾韻：** 乾淨優雅的**花香**緩緩收尾，留下綿長、令人愉悅的回甘。\n\n**口感特色：** 乾淨、輕盈、酸甜平衡。冷萃後，咖啡的苦澀感降到最低，酸質轉化為清爽的果酸，即使不加糖奶也風味十足。\n\n---\n\n#### 🧊 輕鬆上手：5分鐘完成一杯冰咖啡！\n\n**【冷萃/冰沖 建議步驟】**\n\n1. **撕開包裝：** 取出掛耳包，沿線撕開。\n2. **注水：** 將掛耳包掛在杯緣，緩慢注入 **150-180ml 的冰水或常溫水**。\n3. **浸泡：** 輕輕搖晃杯子，確保咖啡粉完全浸濕。\n4. **冰鎮/享用：**\n    * **快速冰沖 (5分鐘)：** 浸泡 5-8 分鐘後取出掛耳包，加入冰塊即可享用。\n    * **經典冷萃 (隔夜)：** 將掛耳包留在水中，放入冰箱冷藏 **8-12 小時**，風味更濃郁醇厚。\n\n**小提醒：** 建議使用純淨水，風味更佳。開封後請盡快飲用，享受最佳賞味期。\n\n---\n\n#### 🛒 立即行動：讓你的夏天充滿果香！\n\n**NT$460 / 10入**\n\n別再讓苦澀的咖啡佔據你的夏天！\n\n**溯源焙所 冷萃掛耳咖啡**，不僅是咖啡，更是你對生活品質的堅持。\n\n👉 **[點擊此處] 立即加入購物車**，將衣索比亞的陽光與果香帶回家！\n\n**【貼心提醒】**\n* 淨含量：10包 / 盒，每包 12g\n* 保存方式：請存放於室溫陰涼處，避免陽光直射。", "input_tokens": 131, "output_tokens": 1197}, "412e7c4c200f9ae62e28": {"text": "## 白日光 SUNDAY 純淨植萃卸妝油 100ml 1入｜溫和不刺激，乾燥缺水肌專用，深層淨化同時鎖水保濕。\n\n---\n\n### 賣點\n\n1.  嚴選**植物油脂**基底，溫和卸除彩妝，不熏眼，敏感肌適用。\n2.  專為乾燥缺水肌設計，卸妝後肌膚依然水潤，告別緊繃乾澀感。\n3.  韓國原裝進口，油狀質地，乳化快速，清爽無殘留負擔。\n4.  輕巧**100ml**瓶裝，**1入**販售，高CP值入門體驗首選。\n\n---\n\n### SEO 描述\n\n30-45歲重視保濕與溫和配方的您，絕不能錯過白日光 SUNDAY 純淨植萃卸妝油。採用高純度**植物油脂**，能徹底溶解防水彩妝與毛孔髒污，同時呵護乾燥缺水肌。韓國製造，**100ml**輕巧容量，**1入**體驗價NT$390。溫和不刺激，讓卸妝成為每日最療癒的保養步驟。\n\n---\n\n### 標籤\n\n#卸妝油\n#保濕卸妝\n#溫和配方\n#乾燥肌\n#植萃保養", "input_tokens": 256, "output_tokens": 310}, "57197bf029d65c67a868": {"text": "{\n  \"title\": \"青研 CHINGYEN 胺基酸溫和洗面乳 120ml 弱酸性 pH5.5 敏弱肌適用\",\n  \"bullets\": [\n    \"採用溫和胺基酸界面活性劑，親膚潔淨。\",\n    \"弱酸性 pH5.5，維持肌膚健康平衡。\",\n    \"輕盈綿密泡沫，減少摩擦感，好沖洗。\",\n    \"添加洋甘菊萃取，洗後膚感溫和不緊繃。\"\n  ],\n  \"seo_description\": \"專為敏弱肌設計的青研胺基酸溫和洗面乳，容量 120ml。採用胺基酸界面活性劑，溫和清潔不刺激。弱酸性 pH5.5 配方，貼近肌膚酸鹼值，洗後不乾澀。成分透明，給予肌膚最純粹的溫柔呵護。\",\n  \"hashtags\": [\n    \"敏弱肌\",\n    \"胺基酸洗面乳\",\n    \"弱酸性\",\n    \"成分黨\",\n    \"CHINGYEN\"\n  ]\n}", "input_tokens": 922, "output_tokens": 259}, "898774be01fd7077c0d9": {"text": "## Volta 65W 氮化鎵 GaN 三孔快充頭 雙C單A輸出 輕巧差旅必備\n\n---\n\n### 賣點\n1. **65W高功率輸出 筆電也能快速回血** (18 字)\n2. **採用第三代氮化鎵 體積縮小50%** (17 字)\n3. **三孔配置 C2A 同時為三裝置供電** (18 字)\n4. **僅112g輕量化設計 差旅無負擔** (16 字)\n\n---\n\n### SEO 描述\nVolta 65W 氮化鎵快充頭，專為多裝置使用者與出差族設計。提供三孔（雙C一A）輸出，可同時為筆電、平板、手機進行PD/QC快充。採用GaN技術，實現65W高功率與極致輕巧體積，是您簡化行李、高效充電的首選配件。\n\n---\n\n### 標籤\n#65W快充\n#氮化鎵\n#差旅必備\n#PD快充\n#三孔充電", "input_tokens": 683, "output_tokens": 258}, "cbc0fd23ac246a656f6a": {"text": "**【晨光研選】五益菌複方益生菌粉 30入**\n\n---\n\n### 商品標題\n\n外食族救星！晨光研選 五益菌複方**益生菌**粉，每包**100億**順暢有感，一盒**30包**，告別卡關人生。\n\n---\n\n### 賣點\n\n1.  專利五益菌配方，每包足量**100億**CFU，高效有感。\n2.  專為外食族設計，調整消化道機能，維持健康。\n3.  獨立隨身包，方便攜帶，一盒**30包**足月補給。\n4.  台灣製造，粉末劑型，空腹或睡前食用效果更佳。\n\n---\n\n### SEO 描述\n\n忙碌的上班族與外食族必備！晨光研選五益菌複方**益生菌**粉，嚴選五株專利益生菌，每包提供高達**100億**CFU，有效調整體質。獨立包裝設計，一盒**30包**，方便您隨時補充好菌。幫助消化道機能，告別作息不規律帶來的脹氣困擾，輕鬆維持健康。\n\n---\n\n### 標籤\n\n#益生菌 #外食族必備 #消化保健 #順暢有感 #晨光研選", "input_tokens": 271, "output_tokens": 319}, "8641c4eb712aa95909af": {"text": "## 青研 CHINGYEN 多重玻尿酸保濕面膜 5片入\n\n---\n\n### 商品標題 (45 字)\n\n**青研 CHINGYEN 五重玻尿酸保濕面膜 5片入｜無添加、高親膚天絲膜布 敏弱肌適用**\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1.  五重**玻尿酸**配方，分層滲透，深度補水。\n2.  高親膚**天絲**膜布，輕薄服貼，精華導入更高效。\n3.  溫和純淨配方，無多餘添加，敏弱肌安心使用。\n4.  添加海藻醣精華，幫助肌膚維持長效水潤感。\n\n---\n\n### SEO 描述 (106 字)\n\n敏弱肌專屬保濕對策。青研 CHINGYEN 多重**玻尿酸**保濕面膜，專為乾燥缺水肌設計。採用五重玻尿酸與海藻醣精華，提供肌膚長效水潤。搭配輕盈**天絲**膜布，服貼且不刺激。一盒**5片**入，讓您輕鬆享受高規格保養，成分黨必備。\n\n---\n\n### 標籤 (5 個)\n\n#玻尿酸面膜\n#敏弱肌保養\n#保濕面膜\n#成分黨\n#天絲面膜", "input_tokens": 817, "output_tokens": 326}, "00e813accc61a17d1b6d": {"text": "## 溯源焙所 低溫烘焙綜合堅果 無調味 350g\n\n---\n\n### 商品標題\n**溯源焙所 職人嚴選 低溫烘焙綜合堅果 350g 無調味**\n\n### 賣點\n1.  低溫慢焙，保留堅果原始風味。\n2.  嚴選四種頂級堅果，完全無調味。\n3.  健身族群與家庭常備的優質點心。\n4.  充氮保鮮，鎖住堅果酥脆口感。\n\n### SEO 描述\n溯源焙所綜合堅果350g，低溫慢焙，完全無調味、無添加糖鹽。嚴選腰果、核桃、杏仁、夏威夷豆，是健身與家庭常備的純淨原型食物。\n\n### 標籤\n原型食物、綜合堅果、低溫烘焙、健身飲食、無調味", "input_tokens": 813, "output_tokens": 212}, "00aea5e3c4732a06a5fa": {"text": "## 青研 CHINGYEN 5%積雪草舒緩精華 30ml | 溫和水感精華 敏弱肌安心首選\n\n---\n\n### 賣點\n\n1. **5%高純度積雪草萃取，溫和穩定膚況。**\n2. **水感清爽質地，快速吸收不黏膩。**\n3. **泛醇與神經醯胺，提升肌膚舒適度。**\n4. **容量 30ml，成分透明，韓國原裝。**\n\n---\n\n### SEO 描述\n\n青研 CHINGYEN 積雪草舒緩精華液 **30ml** 專為敏弱肌設計。含 **5%** 高濃度**積雪草**萃取，搭配泛醇與神經醯胺，提供肌膚溫和的穩定支持。水感精華質地，清爽好吸收，成分黨必備。\n\n---\n\n### 標籤\n\n#積雪草 #敏弱肌 #精華液 #成分黨 #青研CHINGYEN", "input_tokens": 817, "output_tokens": 235}, "424e9a38698359516bb7": {"text": "## 【本木 BENMU】超值首選！高效 16L 除濕機，美國原廠終身保固，告別潮濕霉味\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1. 每日強效**除濕 16L**，小資價享受大容量。\n2. **首購族必搶**！NT$490 入門價，輕鬆告別濕氣。\n3. 美國製造，陽極處理機身，耐用度大升級。\n4. 業界唯一！享終身技術支援，買得最安心。\n\n---\n\n### SEO 描述 (不超過 120 字)\n\n正在找高CP值的**除濕**機嗎？【本木 BENMU】專為**首購族**設計！擁有**16L**日除濕量，超強效能卻只要NT$490。美國原廠品質保證，提供終身技術支援，讓您輕鬆對抗台灣潮濕氣候，不再擔心霉味與衣物曬不乾！\n\n---\n\n### 標籤 (3-5 個)\n\n#除濕機 #16L #CP值 #首購 #本木BENMU", "input_tokens": 219, "output_tokens": 278}, "570f6430aeee0634abd7": {"text": "## 晨光研選 緩釋B群錠 銷售文案\n\n---\n\n### 商品標題\n\n**【晨光研選】長效緩釋B群錠｜夜間修護，日間續航，告別螢幕疲勞**\n\n---\n\n### 賣點\n\n1. 高單位**維生素B群複方**，完整補給所需能量。\n2. 獨特緩釋技術，長效釋放不浪費，持續有感。\n3. 專為長時間螢幕族設計，幫助舒緩緊繃壓力。\n4. 睡前一顆，啟動夜間修復，醒來精神更飽滿。\n\n---\n\n### SEO 描述\n\n上班族必備！晨光研選緩釋B群錠，採用高單位**維生素B群複方**，每錠高達**100mg**。獨家緩釋技術，持續釋放能量，告別慢性疲勞。一盒**20粒**，每日睡前食用，啟動夜間修護機制，讓您白天精神飽滿，效率加倍，輕鬆應對工作挑戰。\n\n---\n\n### 標籤\n\n\\#緩釋B群\n\\#上班族必備\n\\#螢幕疲勞\n\\#精神體力\n\\#長效有感", "input_tokens": 264, "output_tokens": 282}, "ca6bd3edb03197a8cb99": {"text": "{\n  \"title\": \"溯源焙所 台東鹿野 紅烏龍茶包 蜜香熟果風味 三角立體茶包 20包入\",\n  \"bullets\": [\n    \"重發酵工藝，呈現獨特蜜香與熟果韻味。\",\n    \"嚴選台東鹿野產區茶葉，職人級焙製。\",\n    \"三角立體茶包設計，完整釋放茶葉精華。\",\n    \"精緻盒裝，內含 20包，送禮自用兩相宜。\"\n  ],\n  \"seo_description\": \"溯源焙所嚴選台東鹿野紅烏龍，重發酵工藝打造出濃郁蜜香與熟果風味。三角立體茶包方便沖泡，95°C熱水浸泡90秒即可享受回甘滋味。每盒 20包，是茶葉愛好者或商務送禮的優雅選擇。\",\n  \"hashtags\": [\n    \"紅烏龍茶\",\n    \"鹿野茶\",\n    \"台灣茶葉\",\n    \"送禮推薦\",\n    \"蜜香紅茶\"\n  ]\n}", "input_tokens": 922, "output_tokens": 263}, "a48895f6e83460497cf9": {"text": "## 台灣紅烏龍茶包 (台東鹿野產)\n\n---\n\n### 商品標題\n\n**溯源焙所 台東鹿野 紅烏龍 三角立體茶包 蜜香熟果風味 20包入**\n\n---\n\n### 賣點\n\n1.  台東鹿野產地，重發酵工藝醇厚\n2.  獨特蜜香與熟果韻，尾韻回甘\n3.  三角立體茶包，完整釋放茶葉原味\n4.  輕巧盒裝，隨時享用或送禮皆宜\n\n---\n\n### SEO 描述\n\n溯源焙所嚴選台東鹿野紅烏龍，以重發酵工藝成就獨特的蜜香熟果風味。三角立體茶包設計，讓茶葉完整舒展。每盒內含20包，方便快速沖泡，是茶葉愛好者日常品飲或送禮的最佳選擇。\n\n---\n\n### 標籤\n\n紅烏龍、鹿野茶、三角茶包、送禮、台灣好茶", "input_tokens": 815, "output_tokens": 234}, "f9813e982fb6e4bf07df": {"text": "## 晨光研選 緩釋B群錠\n\n### 商品標題\n晨光研選 緩釋B群錠 100mg 維生素B群複方 螢幕族群的活力準備 (49字)\n\n### 賣點\n1. 緩釋劑型設計，穩定釋放營養。\n2. 足量維生素B群複方，每份含100mg。\n3. 支援長時間工作與用眼，維持精神。\n4. 輕巧20粒盒裝，方便攜帶與嘗試。\n\n### SEO 描述\n專為25-45歲上班族設計的晨光研選緩釋B群錠。長時間使用螢幕，需要穩定補充能量。本產品採用緩釋技術，提供足量**維生素B群複方**，每份**100mg**。輕巧**20粒**包裝，適合睡前食用，為隔日工作做好準備。維持代謝，是您日常的專業選擇。(118字)\n\n### 標籤\nB群, 緩釋, 上班族, 螢幕族, 維生素B", "input_tokens": 801, "output_tokens": 252}, "11920f8198e28572b8aa": {"text": "## 晨光研選 金盞花萃取葉黃素膠囊 (60粒)\n\n---\n\n### 商品標題 (不超過 60 字)\n\n**晨光研選｜高濃度游離型葉黃素 30mg｜晶亮守護 60粒｜長時間螢幕族必備**\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1.  **足量高濃度：** 每粒含游離型葉黃素 30mg，高效晶亮有感。\n2.  **黃金比例：** 國際認證 5:1 比例，搭配玉米黃素，吸收更完整。\n3.  **輕鬆補給：** 每日只需一粒，一盒 60粒 滿足雙月保養需求。\n4.  **安心素食：** 台灣製造，採用植物膠囊，素食者也能安心食用。\n\n---\n\n### SEO 描述 (不超過 120 字)\n\n專為長時間使用螢幕的 25-45 歲上班族設計。晨光研選高濃度游離型**葉黃素**，每粒足量 **30mg**，提供清晰視野的關鍵守護。採用黃金比例配方，有效舒緩數位疲勞。一盒 **60粒**，每日一顆，輕鬆維持晶亮健康。台灣研製，品質安心，是您數位生活必備的眼部保養品。\n\n---\n\n### 標籤 (3-5 個)\n\n葉黃素、游離型、眼部保養、30mg、螢幕族", "input_tokens": 264, "output_tokens": 364}, "b9f0fa13db69aa5e2721": {"text": "## Volta 手持無線吸塵器 Lite\n\n### 商品標題\n**Volta 手持無線吸塵器 Lite | 18000Pa強吸力 1.4kg輕量化 小空間寵物家庭專用**\n\n### 賣點\n1.  **18000Pa強勁吸力，高效捕捉寵物毛髮。**\n2.  **1.4kg輕量化設計，收納不佔空間。**\n3.  **標準模式續航35分鐘，清掃不中斷。**\n4.  **軟絨滾筒與床墊吸頭，專攻地板與織物。**\n\n### SEO 描述\nVolta Lite 專為小坪數及寵物家庭設計。擁有 18000Pa 強大吸力，輕鬆吸除毛髮與灰塵。機身僅 1.4kg，輕巧好操作。標準模式提供 35分鐘 續航力，快速完成日常清潔。\n\n### 標籤\n\\#無線吸塵器\n\\#寵物清潔\n\\#小家庭\n\\#Volta\n\\#輕量化", "input_tokens": 676, "output_tokens": 248}, "d268b76ec3424ee95aaa": {"text": "## Volta 無線降噪藍牙耳機 Pro 2\n\n---\n\n### 商品標題 (不超過 60 個字)\n\n**Volta Pro 2 旗艦降噪耳機 | 42dB深度ANC | 藍牙5.4 | 32小時長效續航 | 通勤遠距工作必備** (49 字)\n\n---\n\n### 賣點 (4 條，每條不超過 30 個字)\n\n1. **42dB深度降噪，高效隔絕環境雜音**\n2. **總續航 32小時，滿足全日工作與通勤**\n3. **藍牙5.4穩定連線，支援雙裝置切換**\n4. **單耳4.8g輕量設計，IPX4防水防汗**\n\n---\n\n### SEO 描述 (不超過 120 個字)\n\nVolta Pro 2 專為高效率人士設計。搭載 42dB 主動降噪 技術，讓您在通勤或遠距辦公中保持專注。採用最新 藍牙5.4 晶片，連線穩定低延遲。搭配充電盒，總續航力長達 32小時，電力持久。支援雙裝置切換，工作娛樂無縫銜接。\n\n---\n\n### 標籤 (3-5 個)\n\n#降噪耳機 #藍牙5.4 #32小時續航 #遠距工作 #通勤必備", "input_tokens": 682, "output_tokens": 323}, "aed7e2d027dd8b64fb00": {"text": "身為資深商品文案編輯，我為此款精品冷萃掛耳咖啡設計以下文案，強調其高品質與便利性，以吸引追求生活品味的精品咖啡愛好者。\n\n---\n\n## 溯源焙所 冷萃掛耳咖啡 淺焙水果調 10入\n\n### 商品標題\n**【溯源焙所】耶加雪菲 淺焙水果調 冷萃掛耳咖啡 10包入 | 隨時享受精品冰咖啡** (49 字)\n\n### 賣點\n1. 嚴選衣索比亞耶加雪菲，淺焙保留花果香。\n2. 柑橘莓果調性，口感清爽無負擔。\n3. 一盒 10包 獨立包裝，冷熱沖泡皆宜。\n4. 精品級冷萃，輕鬆製作冰咖啡儀式感。\n\n### SEO 描述\n溯源焙所推出精品級冷萃掛耳咖啡，嚴選衣索比亞**耶加雪菲**，採用**淺焙**工藝，完整保留柑橘與莓果的細緻花香。每盒 **10包** 獨立包裝，讓您在辦公室或家中都能輕鬆製作冰咖啡。享受高品質水果調性，告別苦澀，隨時隨地享受清爽精品咖啡。 (99 字)\n\n### 標籤\n#冷萃咖啡 #耶加雪菲 #精品咖啡 #淺焙 #掛耳包", "input_tokens": 268, "output_tokens": 325}, "61b7f48be0071d568721": {"text": "**【晨光研選】五益菌複方益生菌粉 30入**\n\n---\n\n### 商品標題\n\n**晨光研選 五益菌複方益生菌粉 | 30包/盒 | 每包100億CFU | 幫助維持消化道機能**\n\n---\n\n### 賣點\n\n1.  足量 100億 活性益生菌，幫助維持戰力。\n2.  嚴選五株專利菌株，打造優勢菌叢生態。\n3.  粉末隨身包設計，空腹或睡前皆可食。\n4.  每日一包，輕鬆補給，調整體質好幫手。\n\n---\n\n### SEO 描述\n\n專為外食族與作息不規律的上班族設計。晨光研選五益菌複方**益生菌**粉，每包含有高達**100億**CFU活性菌，嚴選專利菌株，幫助維持消化道機能。一盒**30包**，輕巧隨身包，持續補充，輕鬆做好體內環保。\n\n---\n\n### 標籤\n\n益生菌, 消化道保健, 調整體質, 晨光研選, 100億", "input_tokens": 808, "output_tokens": 281}, "de7bba1e22f9615d2006": {"text": "## Volta 手持無線吸塵器 Lite\n\n---\n\n### 商品標題\n\n**Volta Lite 1.4kg 超輕量無線吸塵器 | 18000Pa 強勁吸力，小家庭、寵物毛髮救星！**\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1. **1.4kg 極輕量設計，小坪數家庭輕鬆移動無負擔。**\n2. **18000Pa 瞬效吸力，徹底清除寵物毛髮與塵蟎。**\n3. **標準模式續航 35 分鐘，滿足日常快速清潔需求。**\n4. **附贈床墊吸頭，有效對抗沙發與寢具過敏源。**\n\n---\n\n### SEO 描述 (不超過 120 字)\n\n**Volta Lite 專為小坪數與寵物家庭設計！機身僅 1.4kg，收納輕巧。搭載 18000Pa 瞬效吸力，輕鬆吸除毛髮與灰塵。標準模式續航 35 分鐘，清潔不中斷。高 CP 值入門款，讓您輕鬆享受無塵生活。**\n\n---\n\n### 標籤 (3-5 個)\n\n#無線吸塵器\n#寵物家庭必備\n#小坪數\n#輕量吸塵器\n#Volta", "input_tokens": 254, "output_tokens": 316}, "7bedbd96d70fe3e72980": {"text": "{\n  \"title\": \"溯源焙所 耶加雪菲 淺焙 冷萃掛耳咖啡 10包入 水果調性\",\n  \"bullets\": [\n    \"衣索比亞 耶加雪菲產區，風味純淨明亮。\",\n    \"精品淺焙工藝，完整保留柑橘與莓果調性。\",\n    \"冷萃專用掛耳包，輕鬆製作冰鎮清爽咖啡。\",\n    \"每盒內含 10包，隨時隨地享受職人級風味。\"\n  ],\n  \"seo_description\": \"享受頂級精品咖啡的便利選擇。溯源焙所嚴選衣索比亞 耶加雪菲 淺焙 咖啡豆，以精準烘焙帶出柑橘、莓果與花香的明亮調性。專為冷萃設計的掛耳包，每盒 10包，只需浸泡即可輕鬆製作冰咖啡，隨時品味清爽果香。\",\n  \"hashtags\": [\n    \"冷萃咖啡\",\n    \"耶加雪菲\",\n    \"淺焙咖啡\",\n    \"精品咖啡\",\n    \"掛耳包\"\n  ]\n}", "input_tokens": 919, "output_tokens": 266}, "c6e51f3ca000450eb13f": {"text": "好的，這是一個針對「緩釋型B群」的電商文案，強調其獨特的吸收機制和食用時間（睡前）。\n\n以下提供三種不同風格的文案，您可以根據您的目標客群和平台特性選擇最適合的：\n\n---\n\n## 晨光研選 緩釋B群錠 (100mg, 20粒) 電商文案\n\n### 方案一：【強調功效與緩釋技術】— 適合追求效率和科學依據的客群\n\n**主標題：** 告別無效補充！「晨光研選」緩釋B群：睡前吃，醒來持續有精神！\n\n**副標題：** 台灣製造🇹🇼 獨家100mg緩釋配方，讓活力不再曇花一現！\n\n---\n\n#### 💡 文案內容：\n\n**【開頭痛點切入】**\n您是否覺得B群「吃了沒感覺」？或是早上吃，下午就電力耗盡？\n傳統B群容易快速吸收、快速代謝，導致效果短暫。\n\n**【核心賣點：緩釋技術】**\n**「晨光研選 緩釋B群錠」** 專為現代人設計！我們採用**獨家緩釋技術**，讓維生素B群不再一次性釋放，而是像「定時鬧鐘」一樣，在您體內緩慢、穩定地釋放長達數小時。\n\n**【獨特食用時間：睡前】**\n誰說B群只能早上吃？\n我們的緩釋配方，**建議「睡前」食用**。當您進入睡眠時，B群開始穩定作用，幫助身體在夜間進行新陳代謝與修復。隔天醒來，您會感受到前所未有的**「持續有感」**，精神飽滿，迎接挑戰！\n\n**【產品規格與保證】**\n*   **規格：** 100mg 高效複方 B群\n*   **型態：** 易吞食錠劑\n*   **產地：** 台灣嚴選製造\n*   **售價：** NT$290 (20粒/盒)\n\n**【行動呼籲】**\n別再讓短暫的提神飲料綁架您的精神！選擇「晨光研選」，從根本改善您的精神體力。\n👉 **立即體驗，讓您的活力從早晨持續到夜晚！**\n\n---\n\n### 方案二：【情境式生活提案】— 適合忙碌、追求生活品質的上班族\n\n**主標題：** 終結「下午三點疲勞魔咒」！您的活力續航專家：晨光研選 緩釋B群\n\n**副標題：** 每天睡前一錠，為隔日電量預先充飽！\n\n---\n\n#### 💡 文案內容：\n\n**【情境描寫】**\n早上咖啡一杯，精神勉強撐到中午。但一到下午，眼皮開始沉重，工作效率直線下降... 這是許多現代人的日常寫照。\n\n**【產品定位與承諾】**\n**「晨光研選 緩釋B群錠」** 不是讓您亢奮，而是幫助您**「穩定輸出」**。\n\n我們知道您需要的是持久的專注力與穩定的好精神。這款緩釋配方，讓B群在體內緩慢釋放，就像為您的身體安裝了**「隱形電池」**：\n\n1.  **夜間修復：** 睡前服用，協助身體在睡眠中有效代謝，為隔日儲備能量。\n2.  **持續有感：** 告別B群「衝高點、秒降落」的窘境，精神體力穩定維持一整天。\n3.  **高CP值選擇：** 台灣製造，NT$290 輕鬆入手，為您的健康投資。\n\n**【適用對象】**\n*   經常加班、壓力大的上班族\n*   需要長時間專注的學生或考生\n*   想改善精神不濟、但又怕B群影響睡眠者 (本品建議睡前服用)\n\n**【規格速覽】**\n*   **分類：** 保健食品 / 精神體力\n*   **建議：** 每日睡前 1 錠\n*   **售價：** NT$290 / 20粒\n\n**【行動呼籲】**\n讓疲勞感成為過去式！現在就加入購物車，開啟您的「持續有感」新生活！\n\n---\n\n### 方案三：【簡潔有力，強調差異化】— 適合社群媒體或廣告投放\n\n**主標題：** 顛覆傳統B群吃法！【晨光研選】緩釋B群：唯一推薦「睡前吃」！\n\n**副標題：** 100mg高效複方 | 台灣製造 | 20粒 NT$290\n\n---\n\n#### 💡 文案內容：\n\n**【核心差異化】**\n**為什麼要睡前吃？** 因為我們是**「緩釋型」**！\n\n傳統B群早上吃，中午就代謝光了。\n「晨光研選 緩釋B群」採用特殊技術，讓營養素在體內緩慢釋放：\n\n✅ **夜間代謝不間斷：** 睡前補充，幫助身體在休息時持續進行能量轉換。\n✅ **隔日精神不斷電：** 醒來後，B群仍穩定作用，維持一整天的好狀態。\n✅ **告別短暫興奮感：** 溫和持續，不造成身體負擔。\n\n**【產品資訊】**\n*   **商品：** 晨光研選 緩釋B群錠\n*   **規格：** 100mg 複方 B群\n*   **數量：** 20粒 / 盒\n*   **價格：** NT$290\n*   **建議吃法：** 每日 1 錠，**睡前食用**。\n\n**【立即搶購】**\n用最聰明的方式補充B群！點擊下方連結，讓您每天都充滿「晨光」！", "input_tokens": 121, "output_tokens": 1377}, "95c025fe11f1a8b2bf63": {"text": "這是一份為「本木 BENMU 除濕機 16L」設計的電商文案，強調其高效能、美式工藝與終身保固的價值。\n\n---\n\n# 【本木 BENMU 16L 強效除濕機】\n\n## 標題：告別黏膩潮濕！美國血統 BENMU 16L 強效除濕機，陽極工藝，終身技術支援，給您頂級乾爽體驗。\n\n---\n\n## 💡 痛點：您受夠了台灣的濕氣了嗎？\n\n梅雨季衣物曬不乾、牆角開始發霉、空氣中總帶著一股悶悶的霉味？潮濕不僅影響心情，更是過敏原和塵蟎滋生的溫床。\n\n是時候讓您的居家環境升級了！**本木 BENMU 16L 除濕機**，專為應對極端濕度而生，讓您輕鬆享受美式乾爽生活。\n\n---\n\n## 🔥 核心賣點：強悍性能與頂級工藝\n\n### 1. 每日 16L 超強除濕力 (Power Performance)\n\n無論是客廳、臥室還是儲藏室，BENMU 都能迅速降低濕度。每日高達 **16 公升** 的除濕量，效率遠超小型除濕盒，快速達到人體最舒適的濕度範圍（約 50%-60%），有效保護您的傢俱和電子產品。\n\n### 2. 頂級陽極處理機身 (Premium Craftsmanship)\n\nBENMU 不只是一台家電，更是家中的質感擺設。機身採用高成本的**陽極處理 (Anodized Finish)**，這不僅賦予機身極佳的耐腐蝕性和耐磨性，更帶來低調奢華的霧面質感，完美融入任何現代裝潢風格。\n\n### 3. 美國血統，品質保證 (USA Origin)\n\n源自美國的設計與技術標準，確保了 BENMU 除濕機的穩定性和耐用性。我們堅持提供最高標準的產品，讓您買得安心，用得放心。\n\n---\n\n## 🛡️ 業界承諾：終身技術支援\n\n家電最怕成為孤兒機！本木 BENMU 承諾提供業界罕見的 **「終身技術支援」**。無論您使用多久，只要遇到任何技術問題，我們專業的團隊隨時為您服務，確保您的除濕機始終保持最佳狀態。\n\n---\n\n## ✨ 選擇 BENMU 的三大理由\n\n| 特色 | 帶給您的好處 |\n| :--- | :--- |\n| **16L 強效除濕** | 快速乾燥衣物，抑制塵蟎與霉菌生長。 |\n| **陽極處理外觀** | 質感升級，耐用抗腐蝕，清潔保養更容易。 |\n| **終身技術支援** | 零後顧之憂，享受長久、穩定的乾爽體驗。 |\n\n---\n\n## 🛒 限時體驗價：NT$490\n\n**【本木 BENMU 16L 除濕機】**\n*   **分類：** 家電 / 空氣\n*   **主要規格：** 每日除濕量 16L\n*   **型態：** 陽極處理機身\n*   **產地：** 美國\n*   **保固：** 終身技術支援\n\n**現在就投資您的健康與舒適！** 立即下單，以超值價格 NT$490，將這台高效能、高質感的 BENMU 除濕機帶回家，徹底終結家中的潮濕困擾！\n\n**👉 點擊購買，開啟您的乾爽新生活！**", "input_tokens": 92, "output_tokens": 833}, "798d6cac5bb47d306dea": {"text": "這是一份針對「低溫烘焙綜合堅果」設計的電商文案，強調其健康、純淨與頂級製程。\n\n---\n\n## 【溯源焙所】頂級低溫烘焙綜合堅果 | 零添加・純淨原味 | 350g 健康能量罐\n\n**💰 售價：NT$590**\n**📦 規格：350g / 罐**\n\n---\n\n### 🌟 主標題：找回堅果最純粹的美味！告別油炸與重鹹，享受低溫烘焙的頂級健康能量。\n\n### 📌 抓住痛點：您還在吃高溫油炸、重油重鹹的堅果嗎？\n\n市售堅果為了追求口感和保存，往往經過高溫油炸或大量調味，讓您在補充營養的同時，也攝取了多餘的負擔。\n\n**「溯源焙所」堅持從源頭把關，只為追求健康與美味的完美平衡。**\n\n---\n\n### ✨ 核心賣點：四大堅持，成就頂級純淨堅果\n\n#### 1. 【低溫慢焙，鎖住營養】\n我們採用耗時費工的**「低溫烘焙」**製程，而非高溫油炸。\n*   **優勢：** 溫和的熱度能最大限度地保留堅果中珍貴的不飽和脂肪酸、維生素E及礦物質，避免高溫破壞營養結構。\n*   **口感：** 雖然無油，卻能達到清脆、不燥熱的完美口感。\n\n#### 2. 【頂級配方，黃金比例】\n嚴選四種高品質堅果，提供全面均衡的營養：\n*   **👑 夏威夷豆：** 堅果中的奶油，口感滑順，價值最高。\n*   **🧠 核桃：** 富含Omega-3，是優質的健腦食物。\n*   **💪 腰果：** 豐富的蛋白質和礦物質，提供持久能量。\n*   **🌿 杏仁：** 高纖維，幫助維持消化道機能。\n\n#### 3. 【純淨無調味，零添加負擔】\n我們承諾：**完全無調味、無添加糖、無添加鹽、無防腐劑。**\n讓您品嚐到的每一顆堅果，都是最原始、最自然的香氣與甜度。適合生酮飲食、低碳飲食、健身族群及注重健康的全家大小。\n\n#### 4. 【溯源把關，品質保證】\n「溯源焙所」堅持追溯原料來源，確保每一批堅果都新鮮、飽滿、無劣質品混雜。\n\n---\n\n### 💡 多元享用方式：隨時隨地，補充健康能量\n\n*   **🏃 辦公室零食：** 下午茶時間，取代高熱量甜點的健康選擇。\n*   **🥣 早餐良伴：** 加入優格、牛奶或燕麥片中，增加飽足感與風味。\n*   **🥗 沙拉點綴：** 撒在輕食沙拉上，增添酥脆口感與營養。\n*   **👶 兒童點心：** 給予孩子最純淨、無負擔的天然零食。\n\n---\n\n### 📋 商品資訊與保存建議\n\n| 項目 | 內容說明 |\n| :--- | :--- |\n| **商品名稱** | 低溫烘焙綜合堅果 (無調味) |\n| **品牌** | 溯源焙所 |\n| **淨含量** | 350g / 罐 |\n| **成分** | 腰果、核桃、杏仁、夏威夷豆 |\n| **製程** | 低溫烘焙，無油炸 |\n| **調味** | 完全無調味、無添加糖鹽、無防腐劑 |\n| **保存方式** | 請存放於陰涼乾燥處，避免陽光直射。 |\n| **開封後建議** | 為保持最佳風味與脆度，**請務必冷藏**，並於兩週內食用完畢。 |\n| **產地** | 台灣加工烘焙 |\n\n---\n\n### 🛒 立即下單：給自己和家人一個最純淨、最健康的選擇！\n\n**【溯源焙所】低溫烘焙綜合堅果，讓您每一口都吃得安心、吃得美味。**\n\n**👉 點擊加入購物車，今天就開始您的純淨堅果之旅！**", "input_tokens": 134, "output_tokens": 1015}, "2ca1cc70742ccdf6d586": {"text": "這是一份為您的「白日光 SUNDAY 純淨卸妝油」設計的電商文案，適用於商品頁面、社群媒體或廣告投放。\n\n---\n\n## 💡 白日光 SUNDAY 純淨卸妝油 (100ml)\n\n**【乾肌救星！卸妝不再是拉扯與乾燥的開始】**\n\n💰 **限時優惠價：NT$390**\n\n---\n\n### 🌟 主標題：告別乾澀緊繃！韓國白日光SUNDAY 純淨卸妝油：乾肌的療癒級卸妝體驗\n\n### 💧 產品痛點與承諾 (Introduction)\n\n您是否受夠了卸妝後肌膚的緊繃感？對於乾燥缺水肌而言，傳統卸妝產品往往會帶走肌膚必需的水分與油脂，導致乾燥、泛紅甚至細紋。\n\n**白日光 SUNDAY 純淨卸妝油**，專為渴望溫和卻高效卸妝的乾性肌膚設計。我們相信，卸妝應該是療癒的開始，而不是水分流失的終點。讓您的肌膚享受如同週日早晨般純淨、放鬆的呵護。\n\n### ✨ 三大核心賣點 (Why You Need This)\n\n#### 1. 【100% 植物油脂基底】溫和溶解，不傷屏障\n採用高純度植物油脂配方，能迅速滲透毛孔，溫和溶解頑固彩妝、防曬與空氣髒污。不含礦物油，不堵塞毛孔，在徹底清潔的同時，為乾燥肌膚提供必需的滋養。\n\n#### 2. 【專為乾燥缺水肌設計】洗後水潤不緊繃\n不同於傳統卸妝油的乾澀感，SUNDAY 卸妝油在乳化後能形成水潤保護膜。即使是極度乾燥的肌膚，使用後也能感受到肌膚的柔軟與彈性，徹底告別卸妝後的「沙漠肌」狀態。\n\n#### 3. 【韓國製造，純淨安心】\n堅持韓國原裝進口，嚴選純淨成分。質地輕盈，易於乳化，不需過度搓揉拉扯，保護您脆弱的肌膚屏障。\n\n### 🌿 產品特色與使用感受\n\n*   **型態：** 輕盈油狀質地，延展性極佳。\n*   **關鍵機制：** 遇水瞬間乳化成牛奶狀，快速帶走臉部髒污，沖洗後無油膩殘留感。\n*   **適用對象：** 特別推薦給乾燥肌、缺水肌，以及追求溫和卸妝體驗的敏感肌膚。\n\n### 📝 使用步驟建議\n\n1.  **乾手乾臉：** 取適量卸妝油（約按壓2-3下），均勻塗抹於全臉。\n2.  **按摩溶解：** 輕柔畫圓按摩約30秒，使彩妝完全溶解。\n3.  **關鍵乳化：** 沾取少量清水，在臉上輕柔按摩，使卸妝油完全乳化成白色乳液狀。\n4.  **沖洗：** 最後以大量清水徹底洗淨。\n\n---\n\n### 🛒 商品規格一覽\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **商品名稱** | 白日光 SUNDAY 純淨卸妝油 |\n| **品牌** | 白日光 SUNDAY |\n| **分類** | 美妝保養 / 卸妝 |\n| **主要規格** | 植物油脂 100ml |\n| **型態** | 油狀質地 |\n| **適用膚質** | 乾燥缺水肌 (乾性肌、極乾肌) |\n| **產地** | 韓國 (原裝進口) |\n| **包裝** | 1入 / 盒 |\n| **售價** | **NT$390** |\n\n**現在就將這份來自韓國的純淨呵護帶回家，讓卸妝成為一天中最放鬆的時刻！**\n**👉 立即加入購物車！**", "input_tokens": 115, "output_tokens": 915}, "a81e8f5e35c97068099c": {"text": "好的，這是一個非常優質且規格明確的保健食品。\n\n我將為您撰寫一系列電商文案，涵蓋**標題/短文案、長文案（商品詳情頁）、社群文案**，以應對不同平台和行銷需求。\n\n---\n\n## 晨光研選｜金盞花萃取葉黃素膠囊 (60粒) 電商文案\n\n### 一、 標題與短文案 (適用於廣告、Banner、商品主標)\n\n| 類型 | 文案內容 | 訴求重點 |\n| :--- | :--- | :--- |\n| **主標題** | 【晨光研選】高濃度游離型葉黃素膠囊 (30mg)｜晶亮守護神 60粒 | 強調品牌、高劑量、游離型、功能性。 |\n| **促銷標** | 30mg高劑量！金盞花萃取葉黃素｜買二送一限時優惠 NT$890 | 突出價格與優惠，吸引點擊。 |\n| **功能標** | 數位族必備！游離型葉黃素 30mg：6mg 黃金比例，清晰有感 | 鎖定目標客群（數位族），強調黃金比例。 |\n| **短文案A** | 每日一粒，告別模糊！高濃度游離型葉黃素，給你清晰新視界。 | 簡潔有力，強調效果。 |\n| **短文案B** | 晨光研選｜專利金盞花萃取，足量 30mg 游離型葉黃素，台灣製造。 | 突出成分來源與產地，建立信任。 |\n\n---\n\n### 二、 長文案 (商品詳情頁 / 產品介紹)\n\n#### 💎 標題：【晨光研選】金盞花萃取葉黃素膠囊｜30mg 高濃度游離型，數位生活晶亮守護神\n\n---\n\n#### 🌟 前言：您是否面臨這些困擾？\n\n長時間面對手機、電腦，感到疲勞乾澀？\n閱讀時常需要瞇眼，清晰度大不如前？\n家中的長輩，需要更強力的晶亮保養？\n\n**別擔心！晨光研選為您帶來最高規格的晶亮守護方案！**\n\n---\n\n#### ✨ 產品核心優勢 (賣點拆解)\n\n**【高濃度足量】30mg 游離型葉黃素，一次到位**\n\n*   **國際認可黃金劑量：** 每日一粒，足量提供 30mg 游離型葉黃素，達到國際公認的有效保養劑量。\n*   **吸收率極佳：** 採用分子量小、無需酵素分解的「游離型」葉黃素，吸收利用率遠高於酯化型。\n\n**【完美黃金比例】葉黃素：玉米黃素 = 5：1**\n\n*   嚴選專利金盞花萃取物，精準調配 **30mg 葉黃素：6mg 玉米黃素** 的黃金比例。\n*   雙重防護，協同作用，提供最完整的晶亮保護網。\n\n**【純淨安心】植物膠囊，素食者可食**\n\n*   採用天然植物膠囊，不含動物性成分，素食者也能安心食用。\n*   台灣在地製造，嚴格品管，品質安心有保障。\n\n**【食用便利】每日一粒，輕鬆維持晶亮狀態**\n\n*   一盒 60 粒裝，可食用兩個月，高 CP 值。\n*   建議隨餐食用，搭配油脂能有效提升吸收效果。\n\n---\n\n#### 🔬 產品規格與成分\n\n| 項目 | 內容 |\n| :--- | :--- |\n| **商品名稱** | 晨光研選 金盞花萃取葉黃素膠囊 |\n| **淨含量** | 60 粒 / 盒 (素食膠囊) |\n| **主成分** | 游離型葉黃素 30mg、玉米黃素 6mg (來自金盞花萃取) |\n| **建議用量** | 每日 1 粒，隨餐食用 |\n| **適用對象** | 3C 族、學生族、銀髮族、長時間閱讀者 |\n| **產地** | 台灣 (Made in Taiwan) |\n| **售價** | NT$890 |\n\n---\n\n#### 💡 誰最需要晨光研選葉黃素？\n\n1.  **💻 數位工作者：** 每天超過 8 小時面對電腦的上班族。\n2.  **📚 學生族群：** 課業壓力大，長時間閱讀或使用平板學習。\n3.  **👵 銀髮族群：** 想要維持長久清晰，做好基礎保養。\n4.  **📱 手機重度使用者：** 睡前、通勤都離不開手機的您。\n\n---\n\n#### 🛒 立即下單，給自己和家人最清晰的守護！\n\n**NT$890 / 60粒 (兩個月份)**\n\n---\n\n### 三、 社群媒體文案 (Facebook / Instagram)\n\n#### 📱 文案 A：針對 3C 族群 (強調痛點與解決方案)\n\n**【滑手機滑到眼花？你需要高濃度葉黃素救援！】**\n\n你是不是常常覺得：螢幕亮度開到最大，還是看不清楚？\n\n別再讓疲勞拖垮你的工作效率和生活品質！\n\n**✨ 晨光研選 金盞花萃取葉黃素膠囊，給你最有感的晶亮升級！**\n\n✅ **足量 30mg 游離型：** 吸收快、效率高，每天一粒就達標。\n✅ **5:1 黃金比例：** 葉黃素 + 玉米黃素，雙重防護更完整。\n✅ **植物膠囊：** 純淨無負擔，素食者也能輕鬆保養。\n\n從今天起，告別模糊世界，迎接清晰「晨光」！\n\n👉 點擊連結，立即體驗 NT$890/盒：[連結]\n#葉黃素 #游離型葉黃素 #3C族必備 #晨光研選 #晶亮有感\n\n---\n\n#### 👨‍👩‍👧 文案 B：針對家庭與長輩 (強調關愛與安心)\n\n**【給父母最好的愛，就是讓他們看得更清楚】**\n\n您有多久沒有好好關心父母的「視」界了？\n\n隨著年齡增長，晶亮保養更需加強！晨光研選葉黃素，是您送給長輩最貼心的禮物。\n\n🏆 **安心承諾：** 台灣製造，嚴選專利金盞花萃取。\n🏆 **高規格配方：** 30mg 游離型葉黃素，分子小好吸收。\n🏆 **每日一粒：** 簡單方便，輕鬆維持長久清晰。\n\n讓愛不再模糊，讓他們享受清晰的每一天。\n\n🎁 **限時優惠：** 買三盒享優惠價，全家一起守護晶亮！\n🛒 立即選購：[連結]\n#銀髮族保養 #保健食品 #家庭必備 #葉黃素推薦 #晨光研選", "input_tokens": 125, "output_tokens": 1667}, "5a7719dccb158b9af90e": {"text": "這是一份針對「青研 CHINGYEN 胺基酸溫和洗面乳」設計的電商文案，強調其溫和性、pH值平衡以及對敏感肌的友好。\n\n---\n\n## 【青研 CHINGYEN】胺基酸溫和洗面乳 120ml\n**pH5.5 弱酸配方 | 敏感肌安心首選 | 溫和清潔不緊繃**\n\n**💰 售價：NT$420**\n**📦 分類：美妝保養 / 清潔**\n\n---\n\n## 💎 文案主標：告別緊繃乾澀！洗臉，是保養的第一步。\n\n您是否受夠了洗完臉後，肌膚乾澀緊繃，甚至泛紅不適？\n\n傳統的鹼性洗面乳雖然清潔力強，卻會破壞肌膚天然的弱酸性保護膜（酸性屏障），導致水分流失、肌膚屏障受損。\n\n**【青研 CHINGYEN 胺基酸溫和洗面乳】**專為追求溫和高效清潔的您設計。我們堅持使用最親膚的胺基酸界面活性劑，搭配肌膚最愛的 pH5.5 弱酸值，讓您在徹底洗淨髒污的同時，完整保留肌膚原有的水潤與防禦力。\n\n---\n\n## ✨ 核心賣點：溫和潔淨，從 pH5.5 開始呵護\n\n### 1. 弱酸 pH5.5：守護肌膚天然屏障\n我們的配方與肌膚的天然 pH 值完美吻合。確保在清潔過程中，不會過度帶走皮脂膜，有效維持肌膚的健康平衡，降低洗後乾燥、泛紅或敏感的機率。\n\n### 2. 高純度胺基酸：親膚界面的清潔王者\n採用高級胺基酸界面活性劑，能溫和有效帶走毛孔內的油脂與髒污，同時避免過度刺激。即使是脆弱敏感肌、乾燥肌或術後肌，也能安心使用。\n\n### 3. 綿密豐盈泡沫：零摩擦感潔顏體驗\n輕輕搓揉即可產生如雲朵般細緻綿密的泡沫，泡沫細膩且富有彈性，能減少手部與臉部肌膚的摩擦，深入毛孔帶走污垢，讓洗臉成為一種享受。\n\n### 4. 洋甘菊萃取：舒緩鎮靜，穩定膚況\n特別添加洋甘菊萃取精華，具有優異的舒緩與安撫功效，幫助鎮靜肌膚，緩解因環境或清潔帶來的微小不適，洗後肌膚柔軟、舒適、不刺激。\n\n---\n\n## 📋 產品規格與承諾\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **容量** | 120ml |\n| **主要成分** | 胺基酸界面活性劑、洋甘菊萃取 |\n| **質地** | 綿密泡沫 |\n| **pH值** | 弱酸性 pH5.5 (最接近肌膚的酸鹼值) |\n| **適用膚質** | 任何膚質適用，特別推薦敏感肌、乾燥肌、混合肌 |\n| **產地** | 台灣 (品質嚴格把關) |\n| **承諾** | 無添加皂鹼、無酒精、無人工色素 |\n\n---\n\n## 💡 使用建議\n\n取適量（約1-2公分）於掌心，加水輕輕搓揉起泡，將綿密泡沫塗抹於臉部，以指腹輕柔畫圓按摩清潔，最後用清水徹底洗淨。\n\n**👉 立即體驗青研胺基酸溫和洗面乳，讓您的肌膚從清潔開始，感受前所未有的水潤與舒適！**\n\n**🛒 點擊加入購物車，開啟您的溫和潔顏之旅！**", "input_tokens": 119, "output_tokens": 873}, "16017cf3b7b7839a6b03": {"text": "這是一份針對「青研 CHINGYEN 積雪草舒緩精華液」所撰寫的電商文案，包含主標題、痛點分析、產品特色、成分解析與購買資訊。\n\n---\n\n## 【青研 CHINGYEN】積雪草舒緩精華液 30ml\n\n### 💎 主標題：5%高濃度積雪草 | 韓國專業級修護！告別泛紅敏感，重啟肌膚穩定模式\n\n### 💡 痛點分析：你的肌膚正在發出求救訊號嗎？\n\n面對現代環境壓力、換季不適、或是長期佩戴口罩造成的悶熱與摩擦，肌膚屏障極易受損，導致泛紅、乾癢、緊繃等不穩定狀態。\n\n**你需要的是高效、快速、且溫和的「穩定力量」。**\n\n---\n\n## ✨ 產品核心優勢 (The Power Trio)\n\n青研 CHINGYEN 積雪草舒緩精華液，專為敏弱肌與不穩定肌膚設計，注入韓國專業修護配方，一滴到位，迅速鎮靜舒緩。\n\n| 特色 | 說明 |\n| :--- | :--- |\n| **高濃度修護** | 業界少見 **5% 積雪草萃取**，迅速舒緩肌膚不適，有效退紅，強化肌膚耐受性。 |\n| **水感零負擔** | 輕盈的**水感精華質地**，秒速吸收，不黏膩、不悶痘，油肌與混合肌也能安心使用。 |\n| **屏障守護神** | 結合泛醇與神經醯胺，從根本修護受損屏障，提升肌膚鎖水力與防禦力。 |\n| **敏弱肌適用** | 配方溫和，不含刺激性成分，**所有膚質適用**，特別推薦給正在經歷敏感、換季不適的肌膚。 |\n\n---\n\n## 🔬 關鍵成分深度解析：黃金修護配方\n\n我們嚴選三大核心成分，為肌膚提供最完整的修護與滋養。\n\n#### 🌿 1. 積雪草萃取 (Cica) — 5% 高濃度添加\n\n*   **功效：** 強效鎮靜、舒緩泛紅、加速修護。\n*   **優勢：** 幫助肌膚恢復健康狀態，是穩定膚況的明星成分。\n\n#### 💧 2. 泛醇 (Panthenol / 維他命原B5) — 2% 添加\n\n*   **功效：** 深度保濕、軟化角質、修護肌膚黏膜。\n*   **優勢：** 提升肌膚水潤度，減少乾燥引起的緊繃與不適。\n\n#### 🛡️ 3. 神經醯胺 (Ceramide)\n\n*   **功效：** 鞏固肌膚屏障、防止水分流失。\n*   **優勢：** 像肌膚的天然水泥，將細胞緊密結合，有效抵禦外界刺激。\n\n---\n\n## 🛒 購買資訊與規格\n\n**【青研 CHINGYEN】積雪草舒緩精華液**\n\n| 項目 | 規格說明 |\n| :--- | :--- |\n| **品牌** | 青研 CHINGYEN |\n| **分類** | 美妝保養 / 精華液 |\n| **容量** | 30ml |\n| **售價** | **NT$780** (專業級修護，親民價格) |\n| **主成分** | 積雪草萃取 5%、泛醇 2%、神經醯胺 |\n| **質地** | 水感精華 (清爽易吸收) |\n| **適用膚質** | 所有膚質，特別推薦敏弱肌、痘痘肌、不穩定肌膚。 |\n| **產地** | 韓國 (原裝進口，品質保證) |\n\n### 💖 立即行動\n\n**別讓泛紅與敏感成為你的日常！**\n\n現在就將【青研 CHINGYEN 積雪草舒緩精華液】加入購物車，給予肌膚最溫柔且強效的修護，重拾健康、穩定的透亮肌！\n\n**👉 點擊購買，體驗韓國專業級的肌膚穩定術！**", "input_tokens": 124, "output_tokens": 972}, "ec200bec8cbbaf859ebc": {"text": "這是一份針對「溯源焙所 台灣紅烏龍茶包」設計的電商文案，強調其頂級風味、產地優勢與便利性。\n\n---\n\n# 【溯源焙所】台東鹿野紅烏龍 | 蜜香熟果，便利不減頂級風味\n\n**售價：NT$380**\n**規格：20入 / 盒 (每包 3g)**\n\n## 💎 文案標題建議 (擇一使用)\n\n1. **頂級茶包的革命：台東鹿野紅烏龍，重發酵的蜜香熟果饗宴。**\n2. **【溯源焙所】忙碌生活中的療癒儀式：90秒沖泡，享受台灣精品紅烏龍。**\n3. **打破茶包迷思！三角立體設計，讓您隨時品嚐「會回甘」的台東好茶。**\n\n---\n\n## 🍵 產品核心賣點 (KSP)\n\n### 1. 產地溯源：來自黃金茶區的醇厚底蘊\n\n我們的紅烏龍，堅持選用**台東鹿野**優質茶葉。鹿野地區日夜溫差大，土壤肥沃，是孕育獨特茶韻的黃金產區。\n\n*   **重發酵工藝：** 採用高發酵度的烏龍製程，結合紅茶的醇厚與烏龍的層次，茶湯紅亮，口感溫潤。\n*   **風味獨特：** 完美融合了烏龍的清雅與紅茶的甘甜，成就了紅烏龍無可取代的魅力。\n\n### 2. 風味描繪：入口即是豐收的熟果香\n\n這不是一般的茶包，這是您舌尖上的頂級體驗。\n\n| 風味層次 | 描述 |\n| :--- | :--- |\n| **前調** | 撲鼻而來的天然**蜜香**，溫暖而誘人。 |\n| **中調** | 醇厚飽滿的**熟果韻**，如同陳年的葡萄酒般圓潤。 |\n| **尾韻** | 茶湯入喉後，舌根湧現清新的**回甘**，持久不散。 |\n\n### 3. 立體茶包：便利與品質不再是取捨\n\n我們深知，忙碌的生活不該犧牲品味。\n\n*   **三角立體設計：** 採用食品級耐熱材質，提供足夠空間讓 3g 的完整茶葉充分舒展，釋放完整風味。\n*   **原葉封裝：** 內含的不是茶末，而是經過精選、切碎適中的**紅烏龍原葉**，確保每一次沖泡都能達到手沖級的口感。\n\n---\n\n## 🏷️ 產品規格一覽\n\n| 項目 | 內容 |\n| :--- | :--- |\n| **品牌** | 溯源焙所 |\n| **商品名稱** | 台灣紅烏龍茶包 (三角立體) |\n| **淨含量** | 20 包 / 盒 (每包 3g) |\n| **產區** | 台東鹿野 |\n| **發酵度** | 重發酵 |\n| **特色** | 蜜香、熟果、尾韻回甘 |\n| **售價** | NT$380 |\n\n---\n\n## 💡 最佳沖泡指南：90秒的完美釋放\n\n即使是茶包，也值得用最好的方式對待。請遵循以下步驟，喚醒紅烏龍的極致風味：\n\n1.  **水溫準備：** 建議使用 **95°C** 的熱水。\n2.  **浸泡時間：** 將茶包放入杯中，注入熱水後，精準浸泡 **90 秒**。\n3.  **享受時刻：** 90秒後取出茶包，即可享受茶湯的醇厚與甘甜。\n    *   *小提醒：* 紅烏龍耐泡度高，可回沖 2-3 次，每次延長 30 秒，風味依然迷人。\n\n---\n\n## ✨ 誰適合這款紅烏龍？\n\n*   追求高品質，但沒有時間泡散茶的辦公室人士。\n*   喜歡醇厚、溫潤口感，偏愛蜜香與熟果韻的茶飲愛好者。\n*   尋找台灣特色伴手禮，希望送禮送出品味與心意的人。\n\n**【溯源焙所】承諾：讓您用最便利的方式，品嚐到最純正的台灣好茶。**\n\n**立即下單，將這份來自台東的溫暖與甘甜帶回家！**", "input_tokens": 131, "output_tokens": 1058}, "e0959e529484f7961738": {"text": "這是一份針對「Volta 手持無線吸塵器 Lite」所撰寫的電商文案，強調其輕巧、強勁吸力與高性價比。\n\n---\n\n# ⚡ Volta 手持無線吸塵器 Lite：輕巧，但絕不妥協！\n\n**NT$4,680**\n\n## 💡 標語：告別笨重線材，迎接高效輕生活！\n\n您是否厭倦了傳統吸塵器的笨重機身和糾結的電線？Volta 聽見了您的心聲！\n**Volta Lite** 專為追求效率與生活品質的您設計。它不僅輕巧到難以置信，更搭載了足以應對日常挑戰的強勁吸力，讓清潔從此成為一件輕鬆愉快的事。\n\n---\n\n## 🔥 三大核心賣點，讓您愛上清潔\n\n### 1. 輕量化革命：1.4kg 單手無負擔\n\n我們將機身重量控制在極致的 **1.4 公斤**，比四瓶礦泉水還要輕！\n*   **輕鬆舉高：** 清潔高處的窗簾、冷氣出風口或天花板角落，不再需要費力抬舉。\n*   **持久操作：** 即使長時間清潔，手腕也不易感到痠痛，適合全家人使用。\n\n### 2. 強勁吸力：18000Pa 深度潔淨\n\n別被「Lite」的名字騙了！Volta Lite 搭載高效能馬達，提供驚人的 **18000Pa 澎湃吸力**。\n無論是地板上的毛髮、地毯深處的塵蟎，或是縫隙中的細小灰塵，都能瞬間吸入，確保您的居家環境徹底潔淨。\n\n### 3. 35分鐘續航：從容應對全屋清潔\n\n標準模式下提供長達 **35 分鐘** 的穩定續航力。\n足夠您一次性完成小家庭或公寓的深度清潔，無需頻繁充電，讓清潔工作一氣呵成。\n\n---\n\n## ✨ 多功能配件，清潔無死角\n\nVolta Lite 隨附三款專業吸頭，讓您輕鬆應對家中各種複雜環境。\n\n| 配件名稱 | 主要用途 | 清潔效益 |\n| :--- | :--- | :--- |\n| **軟絨滾筒** | 硬質地板、磁磚、短毛地毯 | 有效吸附微塵與毛髮，同時保護地板不被刮傷。 |\n| **床墊吸頭** | 床鋪、沙發、布藝家具 | 專門拍打並吸除深藏於織物中的塵蟎、皮屑與過敏原。 |\n| **縫隙吸頭** | 牆角、門窗軌道、車內空間 | 深入狹窄難以觸及的縫隙，徹底清除積塵。 |\n\n---\n\n## 🧼 簡便維護，省時又環保\n\n*   **0.5L 可水洗集塵盒：** 一鍵傾倒垃圾，無需接觸髒污。集塵盒與濾網皆可直接水洗，晾乾後重複使用，省去耗材費用，更加環保衛生。\n*   **壁掛收納：** 隨附壁掛架，輕鬆收納不佔空間，隨取隨用。\n\n---\n\n## 🛒 產品規格一覽\n\n| 項目 | 規格細節 |\n| :--- | :--- |\n| **商品名稱** | 手持無線吸塵器 Lite |\n| **品牌** | Volta |\n| **分類** | 家電 / 清潔 |\n| **售價** | **NT$4,680** |\n| **吸力** | 18000Pa |\n| **重量** | 1.4kg (主機) |\n| **續航力** | 標準模式 35 分鐘 |\n| **集塵盒容量** | 0.5L (可水洗) |\n| **配件** | 軟絨滾筒、縫隙吸頭、床墊吸頭 |\n\n---\n\n## 🎁 立即入手 Volta Lite，開啟您的輕盈潔淨生活！\n\n**NT$4,680** 的價格，讓您擁有超越同級的吸力與輕巧體驗。\n\n**點擊下方按鈕，告別清潔煩惱，享受 Volta 帶來的極致便利！**\n\n[🛒 立即購買]", "input_tokens": 115, "output_tokens": 996}, "45389cadc8f28b0f2942": {"text": "這是一份針對 Volta 65W 氮化鎵三孔快充充電器的電商文案，強調其體積、速度與多功能性。\n\n---\n\n## 【Volta 65W GaN 快充】\n**體積革命，速度巔峰！一顆搞定筆電、手機、平板**\n\n### 💡 標題：告別笨重！Volta 65W 氮化鎵三孔快充 — 輕巧如魔術方塊，能量超越想像！\n\n---\n\n### 【前言：為什麼你需要升級充電器？】\n\n厭倦了出門要帶三顆充電頭，佔滿插座又塞爆包包？\n傳統 65W 充電器又大又重，讓您的旅行負擔沉重？\n\n**Volta 65W 氮化鎵三孔快充** 專為追求極致效率與輕巧的您而生。我們將強勁的 65W 輸出，濃縮進僅 112g 的口袋尺寸中，讓您輕鬆實現「一充多用」的極簡生活。\n\n---\n\n### 【核心優勢：GaN 科技帶來的充電革命】\n\n#### 1. 🚀 65W 全速輸出，筆電瞬間滿血\n搭載強勁的 65W 總輸出功率，足以應付絕大多數主流筆記型電腦，包括 MacBook Pro 13、Dell XPS、或高階平板。\n*   **效率翻倍：** 讓您的筆電在午休時間快速回血，告別漫長等待。\n*   **智慧分流：** 即使同時為手機和筆電充電，也能自動分配最佳電流。\n\n#### 2. ✨ 第三代 GaN 氮化鎵技術 — 體積縮小 50%\n採用尖端的第三代氮化鎵（GaN）半導體材料，取代傳統矽材料。\n*   **極致輕巧：** 尺寸僅 **5.2 x 5.2 x 2.9 cm**，比信用卡還小，重量僅 **112g**。輕鬆放入口袋或最小的隨身包。\n*   **低溫高效：** GaN 技術有效減少熱量產生，提高充電效率，讓快充更穩定。\n\n#### 3. 🔌 三孔配置，同時滿足所有需求 (2C1A)\n不再需要為了手機、平板、耳機分別攜帶充電頭。Volta 提供最實用的三孔配置：\n*   **USB-C (PD 快充) × 2：** 支援主流筆電與手機的 PD 快速充電。\n*   **USB-A (QC 快充) × 1：** 兼容舊設備或為穿戴裝置充電。\n*   **完美情境：** 一邊充筆電工作，一邊為手機和平板補電，辦公室或咖啡廳，一個插座就夠！\n\n#### 4. 🛡️ 全方位防護，充電更安心\nVolta 內建多重安全保護機制，確保您的設備在快速充電的同時，擁有最可靠的保護：\n*   過壓保護 (OVP)\n*   過流保護 (OCP)\n*   過熱保護 (OTP)\n\n---\n\n### 【產品規格一覽】\n\n| 項目 | 規格細節 |\n| :--- | :--- |\n| **品牌** | Volta |\n| **商品名稱** | 65W 氮化鎵三孔快充充電器 |\n| **分類** | 3C / 充電 |\n| **技術** | 第三代氮化鎵 GaN |\n| **總輸出功率** | 65W Max |\n| **輸出接口** | USB-C × 2 + USB-A × 1 |\n| **尺寸** | 5.2 × 5.2 × 2.9 cm (極致迷你) |\n| **重量** | 112g |\n| **安全保護** | 過壓、過流、過熱保護 |\n| **建議售價** | **NT$990** |\n\n---\n\n### 【行動呼籲】\n\n**效率不該犧牲便攜性！**\n\n現在就用 **NT$990** 的超值價格，升級您的充電裝備。\n告別笨重與混亂，體驗 Volta 65W GaN 快充帶來的極速、極簡生活！\n\n**👉 立即點擊購買，享受輕量級的強大能量！**", "input_tokens": 130, "output_tokens": 1004}, "65b5ae8b0206866497fe": {"text": "這是一份針對「晨光研選 小分子魚膠原蛋白粉 蔓越莓風味」的電商文案，強調其高吸收率、美味口感與複合配方。\n\n---\n\n## 【晨光研選】小分子魚膠原蛋白粉 蔓越莓風味 (30入/盒)\n**NT$1580**\n\n### 💎 逆轉時光，啟動肌膚內建的澎潤開關！\n**妳的肌膚，值得最好的呵護。** 隨著年齡增長，膠原蛋白流失速度加快，細紋、乾燥、失去彈性成為日常困擾嗎？「晨光研選」專為追求高效保養的妳，打造這款**高濃度、高吸收**的美容聖品，讓美麗由內而外自然綻放。\n\n---\n\n### ✨ 為什麼選擇「晨光研選」小分子魚膠原蛋白？\n\n#### 1. 頂級原料：深海魚膠原蛋白胜肽 2500mg\n我們採用來自深海的優質魚膠原蛋白，並透過先進技術水解成**極小分子胜肽**。分子量越小，越能有效穿透吸收，直達美麗核心。\n*   **高濃度：** 每包足量 2500mg，高效補充每日所需。\n*   **高吸收：** 小分子結構，吸收率遠勝一般大分子膠原蛋白。\n\n#### 2. 黃金助攻配方：維生素C 80mg\n維生素C 是合成膠原蛋白不可或缺的「催化劑」！我們添加了 80mg 的維生素C，確保膠原蛋白能被身體有效利用，達到最佳協同作用。\n*   **功效加乘：** 促進膠原蛋白形成，同時提升肌膚光澤與防護力。\n\n#### 3. 告別腥味：酸甜蔓越莓風味\n許多人對膠原蛋白望之卻步，就是因為難以忍受的魚腥味。「晨光研選」獨家調配**清爽酸甜的蔓越莓風味**，完美遮蓋腥味，讓妳的每日保養不再是負擔，而是一種享受！\n\n#### 4. 台灣製造，安心保證\n全程台灣在地生產製造，嚴格把關品質，讓妳吃得安心、美得放心。\n\n---\n\n### 💖 產品特色一覽\n\n| 特色項目 | 規格說明 | 帶給妳的效益 |\n| :--- | :--- | :--- |\n| **主成分劑量** | 膠原蛋白胜肽 2500mg | 高效補充，快速有感 |\n| **輔助成分** | 維生素C 80mg | 促進合成，美白提亮 |\n| **口味** | 蔓越莓風味 | 零腥味，口感極佳 |\n| **劑型** | 粉末隨身包 (30包/盒) | 輕巧便攜，隨時補充 |\n| **產地** | 台灣製造 | 品質安心可靠 |\n\n---\n\n### 💡 建議食用方式 (讓美麗在睡夢中升級)\n\n**【睡前黃金保養法】**\n\n*   **每日用量：** 建議每日 1 包。\n*   **最佳時機：** 建議於**睡前 1 小時**，搭配溫水沖泡或直接食用。\n    *   *註：夜間是肌膚修復的黃金時段，此時補充能達到最佳效益。*\n*   **食用方式：** 粉末細緻，可直接倒入口中配水食用，或加入牛奶、果汁中飲用。\n\n---\n\n### 🛒 立即入手，重拾妳的彈潤光采！\n\n**晨光研選 小分子魚膠原蛋白粉 蔓越莓風味**\n*   **售價：** NT$1580\n*   **規格：** 30包 / 盒 (一個月份)\n\n**不再讓歲月偷走妳的美麗，從今天起，讓晨光研選為妳留住青春光采！**\n**[立即加入購物車]**", "input_tokens": 144, "output_tokens": 939}, "e3303903c09206c4a0ad": {"text": "這是一份針對「五益菌複方益生菌粉」設計的電商文案，強調其專業性、高劑量和生活便利性。\n\n---\n\n## 晨光研選 | 五益菌複方益生菌粉 (30入)\n\n### 🌟 標題：【告別負擔，迎接輕盈】啟動內在淨化力！晨光研選 五益菌複方益生菌粉\n\n**售價：NT$1180**\n**分類：保健食品 / 消化保健**\n\n---\n\n### 💡 前言：您的健康，從「腸」計議\n\n現代生活壓力大、飲食不規律，是否讓您經常感到腹部脹氣、排便不順，甚至精神不濟？\n\n腸道被譽為人體的「第二個大腦」，擁有超過 70% 的免疫細胞。當好菌不足，壞菌滋生，不僅影響消化吸收，更會拖垮您的整體健康與活力。\n\n**晨光研選【五益菌複方益生菌粉】**，專為追求高效、有感健康的您設計。每日一包，精準補充，為您的消化系統重新注入活力，找回由內而外的輕盈與舒適！\n\n---\n\n### 💪 核心優勢：五大保證，有感守護\n\n我們不只提供益生菌，我們提供的是經過科學驗證的「腸道精準營養管理」。\n\n#### 1. 【專利級配方】五株菁英，協同作用\n嚴選五株國際認證的**專利益生菌**，非單一菌種，而是經過精心設計的複方組合。每一株菌種都肩負不同使命，共同作用，達到最佳的定殖與平衡效果。\n\n#### 2. 【超高劑量】每包 100 億 CFU 活性菌\n足夠的菌數是益生菌發揮作用的關鍵。我們承諾，每包 2g 的粉末中，含有高達 **100 億 CFU** 的高活性益生菌。高劑量，讓好菌快速佔領地盤，有效調整體質。\n\n#### 3. 【吸收效率最大化】粉末劑型，直達腸道\n採用細緻粉末劑型，無需配水即可食用，口感極佳。粉末比膠囊更容易被身體吸收，讓益生菌快速通過胃酸考驗，直達腸道發揮作用。\n\n#### 4. 【安心承諾】台灣製造，品質嚴選\n堅持台灣在地生產，嚴格把關製程與原料，確保產品的純淨與穩定性。無添加人工色素、香料，讓您食得安心。\n\n#### 5. 【方便攜帶】獨立包裝，新鮮鎖住\n每盒 30 包，採用獨立鋁箔隨身包設計。有效隔絕濕氣與光線，確保每一包益生菌都保持最高活性。無論是辦公室、旅行或外出，隨時隨地都能輕鬆補充。\n\n---\n\n### 🗓️ 建議食用方式：簡單一步，啟動健康循環\n\n**規格：** 30包 / 盒 (每包 2g)\n**建議用量：** 每日 1 包。\n\n**最佳時機：**\n*   **空腹：** 早上起床空腹食用，有助於益生菌快速通過胃部。\n*   **睡前：** 睡前一小時食用，在夜間黃金修復期持續作用。\n\n**食用方式：** 直接撕開包裝，倒入口中食用。\n\n---\n\n### 🛒 晨光研選 邀請您體驗：\n\n別讓消化問題成為您生活的絆腳石。\n投資您的腸道健康，就是投資您的整體活力與免疫力。\n\n**NT$1180，30 天份的輕盈承諾。**\n\n**立即選購【五益菌複方益生菌粉】，從今天開始，感受由內而外的清爽與舒適！**", "input_tokens": 133, "output_tokens": 898}, "6700fa330f06999ce65d": {"text": "{\n  \"title\": \"青研 CHINGYEN 多重玻尿酸保濕面膜 7片 1入 敏弱肌適用 水感清爽\",\n  \"bullets\": [\n    \"核心成分：高效多重玻尿酸，提供肌膚水分。\",\n    \"水感輕盈質地，敷後清爽不黏膩。\",\n    \"專為敏弱肌設計，配方溫和單純。\",\n    \"美國製造，成分透明，理性保養首選。\"\n  ],\n  \"seo_description\": \"青研 CHINGYEN 多重玻尿酸保濕面膜，專為敏弱肌設計。採用水感輕盈質地，提供肌膚所需水分。內含高效多重玻尿酸，幫助維持肌膚水潤平衡。一盒 7片 1入，是成分黨日常保濕護理的最佳選擇。\",\n  \"hashtags\": [\n    \"青研\",\n    \"敏弱肌保養\",\n    \"玻尿酸面膜\",\n    \"成分透明\",\n    \"保濕面膜\"\n  ]\n}", "input_tokens": 917, "output_tokens": 243}, "747b0cddea66a94d6248": {"text": "這是一份為您的「Volta 無線降噪藍牙耳機 Pro 2」設計的電商文案。我們將提供三種不同長度與風格的版本，以適應不同的行銷平台需求。\n\n---\n\n## 方案一：【主打標題與短文案】（適用於廣告投放、Banner 圖文）\n\n### 標題：\n**啟動降噪，世界瞬間靜音。Volta Pro 2：深度 42dB，只聽你想聽的。**\n\n### 核心文案：\n厭倦了通勤的喧囂和辦公室的雜音嗎？\nVolta Pro 2 搭載業界領先的 **42dB 深度主動降噪**技術，為您打造一個專屬的聲音結界。輕盈的 4.8g 機身，搭配長達 32 小時的超強續航，讓您從早到晚，沉浸在純粹的音樂與專注中。\n\n**NT$3290。現在就升級您的聽覺體驗。**\n\n---\n\n## 方案二：【詳細電商頁面文案】（適用於商品詳情頁、官網）\n\n### Volta 無線降噪藍牙耳機 Pro 2：定義你的專注時刻\n\n### 🔊 核心賣點 1：極致降噪，隔絕喧囂 (The Silence)\n\n**42dB 深度降噪，一鍵進入純淨世界。**\n\nVolta Pro 2 採用高階降噪晶片，提供最高 **42dB** 的主動降噪深度。無論您身處擁擠的捷運、吵雜的咖啡廳，或是繁忙的辦公室，只需輕觸，外界噪音瞬間消弭。\n\n*   ✅ **通勤救星：** 有效消除低頻引擎聲、人聲雜訊。\n*   ✅ **專注模式：** 讓您在工作或學習時，心無旁騖。\n*   ✅ **通透模式：** 隨時感知環境，安全溝通不需摘下耳機。\n\n### 🔋 核心賣點 2：電力無憂，續航突破極限 (The Power)\n\n**單次 8 小時，總續航高達 32 小時。**\n\n告別電力焦慮！Volta Pro 2 擁有驚人的續航能力，即使是長途旅行或一整週的通勤，也無需頻繁充電。\n\n*   **單次使用：** 8 小時不間斷播放。\n*   **搭配充電盒：** 總電力長達 32 小時。\n*   **快充技術：** 短暫充電，迅速回血。\n\n### ⚙️ 核心賣點 3：智能連線，無縫切換 (The Connection)\n\n**藍牙 5.4 + 雙裝置切換，效率翻倍。**\n\n搭載最新的藍牙 5.4 技術，連線更穩定、延遲更低。更支援**雙裝置智慧切換**功能，讓您在手機與筆電間無縫銜接，工作娛樂兩不誤。\n\n*   **工作模式：** 筆電聽音樂，手機來電自動切換接聽。\n*   **極速連線：** 開蓋即連，告別漫長等待。\n\n### 💎 核心賣點 4：輕盈舒適，無感佩戴 (The Comfort)\n\n**單耳僅 4.8g，IPX4 防水等級。**\n\n我們知道舒適度是長時間佩戴的關鍵。Volta Pro 2 單耳僅重 4.8 克，輕若無物，長時間使用也毫無負擔。IPX4 防水等級，讓您無懼汗水與小雨。\n\n*   **極致輕量：** 4.8g，提供穩固且舒適的佩戴感。\n*   **運動無憂：** IPX4 防潑水防汗，是您健身房的最佳夥伴。\n\n---\n\n### 【規格總覽】\n\n| 特性 | Volta Pro 2 規格 |\n| :--- | :--- |\n| **建議售價** | **NT$3290** |\n| **降噪深度** | 最高 **42dB** 主動降噪 |\n| **總續航力** | 32 小時 (含充電盒) |\n| **單次續航** | 8 小時 |\n| **連線技術** | 藍牙 5.4 |\n| **特色功能** | 支援雙裝置切換 |\n| **防水等級** | IPX4 (防汗、防潑水) |\n| **單耳重量** | 4.8g |\n\n---\n\n### 🛒 立即行動 (Call to Action)\n\n**NT$3290。** 投資您的專注力與聽覺體驗。\n\n**點擊購買 Volta Pro 2，從今天起，由您定義世界的聲音。**\n\n---\n\n## 方案三：【社群媒體文案】（適用於 Instagram/Facebook 貼文）\n\n### 貼文主圖建議：\n一張展現耳機設計感，並用視覺效果強調「安靜」的圖片（例如：耳機周圍有柔和的光暈，背景模糊或呈現靜音符號）。\n\n### 貼文內容：\n\n**標題：#42dB的安靜力量 🤫 你與世界的距離，只差一副 Volta Pro 2。**\n\n你是否受夠了噪音的轟炸？\n\nVolta Pro 2 帶來革命性的聽覺升級！我們將主動降噪推向極致，提供高達 **42dB** 的深度降噪效果，讓你在喧囂中也能擁有自己的私人空間。\n\n🔋 **超長續航不間斷：** 總電力 32 小時，從週一到週五，電力焦慮完全消失！\n💻 **雙裝置無縫切換：** 藍牙 5.4 支援，手機、電腦一秒切換，效率滿點。\n☁️ **輕若無物：** 單耳僅 4.8g，戴一整天也舒服到忘記它的存在。\n\n無論是專注工作、沉浸音樂、還是享受運動，Volta Pro 2 都是你的最佳靜音夥伴。\n\n✨ **限時體驗價：NT$3290**\n\n點擊連結，立即擁有你的專屬靜音艙！\n🔗 [您的購買連結]\n\n#Volta #Pro2 #無線降噪 #藍牙耳機 #降噪耳機 #3C新品 #通勤必備 #深度降噪 #藍牙5_4", "input_tokens": 127, "output_tokens": 1481}, "e22e855eb77efdcb424c": {"text": "好的，這是一份為您的「Volta 循環扇 8段」設計的電商文案。\n\n---\n\n## 🌬️ Volta 循環扇 8段：小巧身軀，極致循環！\n\n**【限時優惠價：NT$490】**\n\n### 💡 為什麼你需要 Volta 循環扇？\n\n在炎熱的季節或密閉空間中，空氣不流通不僅悶熱，更影響效率和舒適度。Volta 循環扇不僅是風扇，更是您家中的「空氣調節師」，幫助您快速平衡室內溫度，節省空調電費！\n\n### ✨ 核心亮點，不容錯過！\n\n#### 1. 🇯🇵 日本精工，極致靜音與效率\n源自日本的嚴謹工藝，確保風力強勁且運轉平穩。無論是辦公、睡眠或閱讀，都能享受高品質的靜謐循環。\n\n#### 2. 🌪️ 8段風速，隨心所欲的空氣掌控\n告別傳統的「強、中、弱」三段式！Volta 提供**多達 8 段**的精細風速調整，從輕柔微風到強力循環，滿足您在不同情境下的所有需求。\n\n#### 3. 🛡️ 食品級不鏽鋼機身，耐用與美學兼具\n採用**食品級不鏽鋼**材質，不僅外觀時尚、質感出眾，更具備優異的耐用性與抗腐蝕性。清潔簡單，長久使用依然亮麗如新。\n\n#### 4. 💰 驚喜價格，高CP值的入門首選\n只需 **NT$490**，即可擁有日本品質、8段風速的高效循環扇！是您改善居家空氣品質、搭配空調使用的最佳入門選擇。\n\n### 🏠 多場景應用：\n\n* **搭配空調：** 快速將冷氣均勻送達房間每個角落，節省 20% 以上的電力。\n* **梅雨季節：** 加速室內空氣流動，幫助衣物、地板保持乾燥。\n* **辦公室/書房：** 輕巧不佔空間，提供個人化的舒適微風。\n* **換氣通風：** 對流室內外空氣，迅速排出異味或悶熱感。\n\n### 📋 產品規格一覽：\n\n| 項目 | 規格內容 |\n| :--- | :--- |\n| **品牌** | Volta |\n| **商品名稱** | Volta 循環扇 8段 |\n| **售價** | NT$490 |\n| **分類** | 家電 / 空調 |\n| **風速段數** | 8段 |\n| **機身型態** | 食品級不鏽鋼 |\n| **產地** | 日本 |\n| **保固** | 原廠一年保固 |\n\n---\n\n### 🛒 立即行動！\n\n**NT$490 即可享受日本品質的極致循環體驗！** 讓 Volta 循環扇為您的生活帶來清新、舒適的空氣。\n\n**[點擊此處，立即搶購！]**\n\n---", "input_tokens": 89, "output_tokens": 704}, "24817b3992320e04303e": {"text": "好的，這是一份為「青研 CHINGYEN 多重玻尿酸保濕面膜」設計的電商文案。\n\n---\n\n# 💧 青研 CHINGYEN 多重玻尿酸保濕面膜 💧\n\n**【德國頂級工藝，給肌膚最純粹的保濕饗宴】**\n\n## 🌟 抓住水嫩的秘密！商品介紹 🌟\n\n厭倦了保濕無感的面膜嗎？來自德國的 **青研 CHINGYEN 多重玻尿酸保濕面膜**，專為亞洲肌膚設計，結合頂尖科技與溫和配方，讓您的肌膚從此告別乾燥、重現水潤光澤！\n\n### 💎 為什麼選擇青研 CHINGYEN？\n\n* **商品名稱：** 青研 CHINGYEN 保濕面膜 (多重玻尿酸配方)\n* **規格組合：** 5片/盒，共 3 入組 (總計 15 片)\n* **超值售價：** 僅需 **NT$390**\n* **分類：** 美妝保養 / 面膜\n* **產地保證：** 🇩🇪 **德國原裝進口**，品質值得信賴。\n\n---\n\n## ✨ 核心賣點：讓肌膚喝飽水！ ✨\n\n### 1. 【多重玻尿酸科技】深層補水，長效鎖水\n\n我們採用獨家「多重玻尿酸配方」，能滲透至肌膚不同層次：\n* **小分子玻尿酸：** 迅速滲透肌底，提供深層滋潤。\n* **大分子玻尿酸：** 在肌膚表層形成鎖水網，防止水分流失。\n* **效果：** 立即感受肌膚彈潤，保濕效果持久不間斷！\n\n### 2. 【凝露質地精華】清爽不黏膩\n\n告別厚重黏膩感！面膜精華採用清透的 **凝露質地**，輕盈好吸收，敷後肌膚清爽無負擔，即使夏天使用也舒適。\n\n### 3. 【敏弱肌救星】溫和配方，安心呵護\n\n青研深知敏弱肌的需求。本產品經過嚴格測試，配方極致溫和，**敏弱肌膚也能安心使用**。有效舒緩乾燥泛紅，穩定膚況，讓保養不再是負擔。\n\n### 4. 【德國頂級工藝】品質保證\n\n源自德國的嚴謹製造工藝，確保每一片面膜都達到最高標準，給您歐洲沙龍級的護膚體驗。\n\n---\n\n## 💡 適用情境與使用建議 💡\n\n* **日常保濕：** 每週使用 2-3 次，維持肌膚水油平衡。\n* **急救補水：** 妝前使用 15 分鐘，讓底妝更服貼、持久。\n* **曬後修護：** 鎮靜舒緩肌膚，補充流失的水分。\n\n**使用方法：** 潔面後，取出面膜敷於臉部約 15-20 分鐘，取下後輕拍臉部，讓剩餘精華完全吸收。無需沖洗。\n\n---\n\n## 🛒 超值組合，立即擁有水光肌！\n\n**青研 CHINGYEN 多重玻尿酸保濕面膜**\n* **原價：** (市場參考價)\n* **限時優惠價：** **NT$390** (5片 x 3入組，共 15 片)\n\n**現在就下單，讓您的肌膚體驗德國頂級的保濕力量！**\n\n👉 **點擊購買，開啟您的水潤之旅！**\n\n---\n*(可搭配圖片：產品外觀圖、面膜精華質地特寫、敷臉後肌膚水潤對比圖)*", "input_tokens": 115, "output_tokens": 870}, "56304119289938a180b1": {"text": "## 青研 CHINGYEN 德國原裝 高效保濕面膜\n\n---\n\n### 商品標題\n\n**德國青研 CHINGYEN 敏弱肌專用！高濃度多重玻尿酸爆水面膜 5片x3入組 (共15片)**\n\n---\n\n### 賣點\n\n1.  德國原裝進口，成分黨安心首選。\n2.  獨家多重玻尿酸，深層補水鎖水。\n3.  清爽凝露質地，敏弱肌零負擔。\n4.  密集修護乾燥，打造水潤光澤肌。\n\n---\n\n### SEO 描述\n\n專為敏弱肌設計的保濕救星！青研 CHINGYEN 德國原裝進口，採用高效**多重玻尿酸**配方，快速舒緩乾燥泛紅。清爽凝露質地，成分黨也能安心使用。超值組合內含 **3入**，每入 **5片**，讓肌膚隨時保持水嫩飽滿。\n\n---\n\n### 標籤\n\n\\#敏弱肌保濕 \\#多重玻尿酸 \\#德國面膜 \\#成分黨推薦 \\#爆水面膜", "input_tokens": 254, "output_tokens": 262}, "706ba3606a53d7ad0fcb": {"text": "{\n  \"title\": \"青研 CHINGYEN 神經醯胺修護乳液 80ml 7入組 敏弱肌適用 水感保濕\",\n  \"bullets\": [\n    \"核心成分神經醯胺，提升肌膚保水屏障。\",\n    \"80ml水感清爽質地，塗抹後迅速吸收不黏膩。\",\n    \"溫和透明配方，專為敏弱肌膚設計。\",\n    \"輕巧80ml容量，7入組滿足長期穩定修護需求。\"\n  ],\n  \"seo_description\": \"專為敏弱肌設計的青研修護乳液。採用神經醯胺配方，提供長效保濕，質地輕盈水感，告別厚重黏膩。單瓶80ml，一次7入組，讓您持續維持肌膚穩定與水潤。日本製造，溫和透明的成分，是您日常保養的理想選擇。\",\n  \"hashtags\": [\n    \"青研CHINGYEN\",\n    \"神經醯胺\",\n    \"敏弱肌保養\",\n    \"水感乳液\",\n    \"80ml\"\n  ]\n}", "input_tokens": 921, "output_tokens": 268}, "2c426cc13fd222266f4a": {"text": "{\n  \"title\": \"本木 BENMU 螢幕掛燈 95Ra 高顯色｜霧面鋁合金，為遠距辦公打造的專業光線。\",\n  \"bullets\": [\n    \"專業級顯色指數高達 95Ra，色彩真實呈現，工作不失真。\",\n    \"台灣製造，採用霧面鋁合金，結構堅固耐用，體現樸實工法。\",\n    \"非對稱光學設計，不佔桌面空間，專注照明螢幕與鍵盤區域。\",\n    \"終身技術支援保固，提供長久安心使用體驗，適合長時間辦公。\"\n  ],\n  \"seo_description\": \"長時間面對螢幕，你需要本木 BENMU 螢幕掛燈。專業級 95Ra 高顯色，提供舒適且真實的照明，減輕視覺疲勞。霧面鋁合金材質，台灣製造，耐用可靠。是通勤族與遠距工作者提升效率的理想選擇。\",\n  \"hashtags\": [\n    \"螢幕掛燈\",\n    \"遠距辦公\",\n    \"護眼燈\",\n    \"95Ra顯色\",\n    \"本木BENMU\"\n  ]\n}", "input_tokens": 611, "output_tokens": 271}, "34b5fa7ca82e66deb9ff": {"text": "## 本木 BENMU 螢幕掛燈 95Ra\n\n---\n\n### 商品標題 (39 字)\n\n**本木 BENMU 螢幕掛燈 95Ra 高顯色護眼，霧面鋁合金質感，遠距工作者必備**\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1. **專業級 95Ra 顯色，色彩真實不失真，適合設計與長時間工作。**\n2. **非對稱光源設計，有效防眩光，提供舒適無壓力的閱讀環境。**\n3. **霧面鋁合金質感，不佔桌面空間，為您的工作區增添俐落美學。**\n4. **台灣製造，品質嚴格把關，享終身技術支援，安心有保障。**\n\n---\n\n### SEO 描述 (89 字)\n\n長時間盯著電腦螢幕讓你眼睛疲勞嗎？本木 BENMU 螢幕掛燈是通勤族與遠距工作者的最佳選擇。採用專業級 **95Ra** 高**顯色**指數，讓畫面色彩真實呈現。非對稱光源設計，有效消除螢幕反光，提供舒適的閱讀與工作環境。霧面鋁合金材質，質感升級，台灣製造品質保證。\n\n---\n\n### 標籤 (5 個)\n\n#螢幕掛燈\n#護眼燈\n#遠距工作\n#95Ra\n#電腦週邊", "input_tokens": 222, "output_tokens": 323}, "d4f7c18761e838532136": {"text": "## 青研 CHINGYEN 保濕面膜\n\n### 商品標題\n青研 CHINGYEN｜極致水感修護面膜 (7片/1入)。高濃度多重玻尿酸，敏弱肌速效安瓶級保濕。\n\n### 賣點\n1. 獨家「多重玻尿酸」配方，直達肌底。\n2. 敏弱肌適用，溫和修護零負擔。\n3. 水感質地，極致服貼快速滲透。\n4. 美國研發製造，高規格安心保證。\n\n### SEO 描述\n青研 CHINGYEN 專為成分黨與敏弱肌設計的頂級保濕面膜。內含高濃度多重玻尿酸，提供安瓶級深層補水修護。一盒 1入 共 7片，美國原裝進口水感質地。立即體驗，告別乾燥泛紅，重現水潤透亮。\n\n### 標籤\n#敏弱肌保濕\n#多重玻尿酸\n#成分黨推薦\n#美國面膜\n#水感修護", "input_tokens": 255, "output_tokens": 250}, "0bf3135d4dece4ba7e28": {"text": "## 白日光 SUNDAY 唇部修護膏 文案\n\n---\n\n### 商品標題\n\n白日光 SUNDAY 爆水修護唇膏 15g 3入組｜熬夜唇紋救星！**蜂蠟與植物油**敏弱肌適用\n\n### 賣點\n\n1.  3入超值囤貨組，隨時隨地補水。\n2.  **蜂蠟與植物油**配方，深層修護不黏膩。\n3.  敏弱唇適用，告別乾燥死皮。\n4.  **15g** 大容量，睡前厚敷當唇膜。\n\n### SEO 描述\n\n學生黨必備！白日光 SUNDAY 唇部修護膏，採用天然**蜂蠟與植物油**，溫和滋潤敏弱唇肌。超大容量**15g**，一罐多用，可當日常護唇膏或夜間唇膜。現在購買超划算**3入**組，平均下來更便宜！美國原裝進口，油狀質地秒吸收，告別唇紋，輕鬆擁有水潤嘟嘟唇。\n\n### 標籤\n\n護唇膏、唇膜、敏弱肌、蜂蠟、學生必備", "input_tokens": 257, "output_tokens": 272}, "f39752f84403fb80e4bd": {"text": "{\n  \"title\": \"Volta 鑄鐵平底鍋 20cm。高效均溫，美國製造，適用電磁爐\",\n  \"bullets\": [\n    \"強悍鑄鐵結構，提供優異熱傳導與穩定性\",\n    \"20cm 精巧尺寸，單人料理或快速出餐適用\",\n    \"終身技術支援，確保產品長期使用壽命\",\n    \"適用多種爐具，包含電磁爐與烤箱\"\n  ],\n  \"seo_description\": \"Volta 鑄鐵平底鍋 20cm 專為追求效率與耐用度的使用者設計。美國製造，提供終身技術支援。鑄鐵材質具備絕佳蓄熱能力，適合快速烹調。體積精巧，是專業人士或多裝置使用者廚房的理想選擇。\",\n  \"hashtags\": [\n    \"鑄鐵鍋\",\n    \"20cm\",\n    \"Volta\",\n    \"美國製造\",\n    \"高效廚具\"\n  ]\n}", "input_tokens": 748, "output_tokens": 231}, "7d4270de73c363229711": {"text": "{\n  \"title\": \"Volta 循環扇 8段 風速精準控制 日本不鏽鋼 小坪數換氣專用\",\n  \"bullets\": [\n    \"8段風速設定，精準控制氣流範圍。\",\n    \"食品級不鏽鋼扇葉，適用寵物家庭。\",\n    \"強勁送風，快速平衡小坪數空間溫差。\",\n    \"日本製造，結構穩固，原廠一年保固。\"\n  ],\n  \"seo_description\": \"Volta 8段循環扇專為小坪數及寵物家庭設計。提供8段風速選擇，實現精準的空氣對流。食品級不鏽鋼材質易於清潔，有效改善室內空氣品質。日本製造，提供高效能換氣。\",\n  \"hashtags\": [\n    \"循環扇\",\n    \"Volta\",\n    \"小坪數家電\",\n    \"寵物家庭\",\n    \"8段風速\"\n  ]\n}", "input_tokens": 747, "output_tokens": 220}, "d6420bf600bc7afbea7e": {"text": "{\n  \"title\": \"青研 CHINGYEN 多重玻尿酸保濕面膜 5片x3入 德國凝露質地 溫和水潤\",\n  \"bullets\": [\n    \"採用多重玻尿酸配方，為肌膚注入所需水分。\",\n    \"德國凝露精華，質地清爽不黏膩，快速滲透。\",\n    \"溫和配方設計，敏弱肌也能安心使用。\",\n    \"每組 5片x3入，持續水嫩保濕不間斷。\"\n  ],\n  \"seo_description\": \"青研 CHINGYEN 多重玻尿酸保濕面膜，專為敏弱肌設計。德國凝露精華，提供肌膚所需水分，清爽不負擔。內含多重玻尿酸，高效水潤。規格為 5片x3入，讓您持續水嫩。成分透明，理性保養首選。\",\n  \"hashtags\": [\n    \"敏弱肌保養\",\n    \"玻尿酸面膜\",\n    \"成分黨\",\n    \"青研CHINGYEN\",\n    \"保濕面膜\"\n  ]\n}", "input_tokens": 916, "output_tokens": 261}, "6af330d3a4acaba09019": {"text": "## 青研 CHINGYEN 修護乳液 80ml 7入\n\n---\n\n### 商品標題 (不超過 60 個字)\n\n**青研 CHINGYEN | 日本水感修護乳液 80ml 7入組 | 高濃度神經醯胺，深層保濕，敏弱肌專用**\n\n---\n\n### 賣點 (4 條，每條不超過 30 個字)\n\n1.  高濃度神經醯胺，有效修護肌膚屏障。\n2.  輕盈水感質地，快速吸收清爽不黏膩。\n3.  溫和配方設計，敏弱肌也能安心使用。\n4.  日本原裝進口，80ml 7入超值囤貨組。\n\n---\n\n### SEO 描述 (不超過 120 個字)\n\n專為30-45歲重視溫和保濕的族群設計。青研 CHINGYEN 修護乳液採用日本頂級配方，添加高濃度**神經醯胺**，有效強化肌膚屏障。輕盈水感質地，敏弱肌也能安心使用。超值組合內含 **80ml 7入**，提供您長效且不間斷的深層修護。是換季保養與日常保濕的最佳選擇。\n\n---\n\n### 標籤 (3-5 個)\n\n\\#神經醯胺 \\#敏弱肌適用 \\#保濕乳液 \\#水感修護 \\#日本製造", "input_tokens": 259, "output_tokens": 337}, "3dad6a54fff081b1f95e": {"text": "{\n  \"title\": \"本木 BENMU 台灣製 鑄鐵平底鍋 24cm，傳熱均勻，耐用廚具首選\",\n  \"bullets\": [\n    \"經典鑄鐵材質，蓄熱性佳，使食材受熱均勻。\",\n    \"直徑 24cm 實用尺寸，滿足日常煎炒料理需求。\",\n    \"台灣在地生產製造，品質工藝值得信賴。\",\n    \"採用陽極處理工法，提升鍋體結構耐用性。\"\n  ],\n  \"seo_description\": \"本木 BENMU 鑄鐵平底鍋，直徑 24cm，是追求耐用與美味的廚房必備品。鑄鐵材質蓄熱性佳，適合煎炒。台灣製造，陽極處理，提供原廠一年保固，為您的烹飪生活帶來可靠的體驗。\",\n  \"hashtags\": [\n    \"鑄鐵鍋\",\n    \"平底鍋\",\n    \"24cm\",\n    \"台灣製造\",\n    \"本木BENMU\"\n  ]\n}", "input_tokens": 611, "output_tokens": 242}, "539f33f5e677b8ef1860": {"text": "## 晨光研選 緩釋B群錠\n\n### 商品標題\n\n【晨光研選】日本專利緩釋B群錠｜告別疲勞｜高劑量維生素B群複方 75mg (59字)\n\n---\n\n### 賣點\n\n1.  日本專利緩釋技術，長效穩定釋放，續航力加倍。\n2.  高劑量**維生素B群複方**，幫助代謝，效率有感提升。\n3.  單顆獨立包裝設計，隨身攜帶，衛生方便不變質。\n4.  建議睡前食用，夜間修復，隔日精神飽滿不透支。\n\n---\n\n### SEO 描述\n\n專為長時間使用螢幕的上班族設計。晨光研選緩釋B群錠，採用日本專利技術，確保**維生素B群複方**能長效穩定釋放。每粒高達**75mg**，有效對抗疲勞、提升專注力。睡前一顆，幫助您在夜間完成修復。獨立包裝，一盒**20粒**，輕鬆補充每日所需活力。告別精神不濟，找回高效生活。 (118字)\n\n---\n\n### 標籤\n\nB群 / 緩釋B群 / 上班族必備 / 精神體力 / 螢幕疲勞\n", "input_tokens": 263, "output_tokens": 309}, "bac6e66c11fe4ed8982c": {"text": "## Volta 循環扇 8段\n\n---\n\n### 商品標題\n\n**Volta 日本食品級不鏽鋼循環扇 8段風速 小坪數/寵物家庭清淨首選**\n\n---\n\n### 賣點\n\n1. **8段風速**精準控溫，節能省電。\n2. 循環對流快速除濕，有效驅散異味。\n3. 食品級不鏽鋼，耐用易清潔，不卡毛髮。\n4. 日本原裝輕巧設計，小空間完美適用。\n\n---\n\n### SEO 描述\n\n告別悶熱潮濕！Volta循環扇專為小坪數家庭設計，提供**8段**細膩**風速**選擇，無論是梅雨季除濕、衣物速乾，或是寵物異味淨化，都能輕鬆解決。食品級不鏽鋼材質，清潔無死角，讓您輕鬆維持居家衛生。日本原廠一年保固，享受清淨舒適的居家環境。\n\n---\n\n### 標籤\n\n#循環扇\n#小坪數\n#寵物家電\n#8段風速\n#不鏽鋼", "input_tokens": 218, "output_tokens": 252}, "ef07995e0f6f3d328055": {"text": "**商品標題**\nVolta 8段風速循環扇 日本原裝 不鏽鋼機身 適用小坪數與寵物家庭\n\n**賣點**\n1. 8段精準風速控制，滿足細膩換氣需求。\n2. 食品級不鏽鋼機身，清潔快速耐用。\n3. 強化對流效率，加速室內空氣循環。\n4. NT$490 高性價比，原廠提供一年保固。\n\n**SEO 描述**\nVolta 循環扇，專為小坪數家庭與寵物空間設計。提供 8段 精準 風速 調整，快速驅散異味與濕氣。食品級不鏽鋼機身，耐用且易於清潔維護。NT$490 入手高效能換氣解決方案，日本原裝品質保證。\n\n**標籤**\n*   循環扇\n*   小坪數家電\n*   寵物換氣\n*   8段風速\n*   Volta", "input_tokens": 640, "output_tokens": 226}, "ec1e01236d02747bdbb4": {"text": "這是一份針對「青研 CHINGYEN 多重玻尿酸保濕面膜」所撰寫的電商文案，包含標題、痛點描述、產品特色、使用情境、以及購買呼籲等部分，適用於各大電商平台。\n\n---\n\n# 💧 青研 CHINGYEN 頂級保濕面膜 電商文案\n\n## 💎 主標題 (吸引目光，強調價值與功效)\n\n**【青研 CHINGYEN】美國原裝進口🇺🇸 多重玻尿酸保濕面膜 (7片/盒) | NT$1580 | 敏弱肌救星，15分鐘打造水光肌！**\n\n---\n\n## 🎯 副標題 / 痛點切入 (引起共鳴，指出問題)\n\n**你的肌膚，是不是正在發出「缺水警報」？**\n\n* 換季時節，臉頰總是乾癢、脫屑、泛紅？\n* 擦再多乳液，隔天早上依舊黯沉無光？\n* 尋找一款真正溫和、高效，連敏弱肌都能安心使用的頂級保濕面膜？\n\n**停止無效保養！你需要的是「精準補水」與「長效鎖水」的完美結合！**\n\n---\n\n## ✨ 產品核心賣點 (Highlight 產品優勢)\n\n### 【青研 CHINGYEN】多重玻尿酸保濕面膜 — 專為敏弱肌設計的高效水潤方案\n\n| 特點 | 說明 | 效益 |\n| :--- | :--- | :--- |\n| **1. 🇺🇸 美國原裝進口** | 堅持最高規格原料與製程，品質保證。 | 享受國際級專業護膚體驗。 |\n| **2. 💧 多重玻尿酸配方** | 結合大分子、中分子、小分子玻尿酸，層層滲透。 | 從肌底深層補水，表層形成鎖水網，持久水潤不間斷。 |\n| **3. 🌿 敏弱肌適用** | 配方極度溫和，無添加刺激性成分。 | 舒緩乾燥不適，泛紅肌膚也能安心使用。 |\n| **4. 🌊 水感輕盈質地** | 精華液清爽不黏膩，快速吸收。 | 敷後肌膚清爽無負擔，不悶痘、不致敏。 |\n| **5. 💰 奢華高效保養** | NT$1580/7片，平均單片高規格享受。 | 輕鬆將醫美級保濕導入日常保養程序。 |\n\n---\n\n## 🔬 功效深度解析 (為什麼它有效？)\n\n### 【水潤金三角：深度、持久、溫和】\n\n**✅ 深度補水：** 採用獨家「多分子玻尿酸協同作用」，大分子在肌膚表層形成水膜，減少水分蒸發；小分子則深入角質層，從根本提升肌膚含水量。\n\n**✅ 長效鎖水：** 精華液中蘊含高效保濕因子，敷完後能持續釋放水潤能量，讓肌膚一整天都維持飽滿彈性。\n\n**✅ 溫和修護：** 專為敏弱肌膚研發，配方純淨，有效舒緩因乾燥引起的緊繃與不適，讓肌膚重回健康穩定狀態。\n\n---\n\n## 🧖‍♀️ 最佳使用情境 (引導消費者想像)\n\n**無論你是哪種膚況，青研面膜都能成為你的肌膚救星：**\n\n* **急救補水：** 上妝前敷一片，讓底妝服貼一整天，告別卡粉浮粉困擾。\n* **曬後修護：** 戶外活動後，冰鎮面膜敷上，迅速鎮定舒緩肌膚，補充流失水分。\n* **夜間奢養：** 睡前敷用，將高效精華鎖入肌底，隔天醒來擁有令人稱羨的水光肌。\n* **換季穩定：** 肌膚不穩定時，使用溫和配方，幫助肌膚度過敏感期。\n\n---\n\n## 🛒 立即行動 (Call to Action)\n\n**別讓乾燥，偷走你的光采！**\n\n青研 CHINGYEN 多重玻尿酸保濕面膜，是您投資美麗、穩定膚況的最佳選擇。\n\n**原價 NT$1580 (7片/盒)，現在立即體驗，感受美國頂級保濕科技帶來的驚人水潤力！**\n\n**👉 立即加入購物車，開啟您的水光肌旅程！**\n\n---\n\n### 📋 產品資訊一覽\n\n* **商品名稱：** 青研 CHINGYEN 多重玻尿酸保濕面膜\n* **規格：** 7片 / 1入 / 盒\n* **主要成分：** 多重玻尿酸\n* **質地：** 水感清爽精華\n* **適用膚質：** 敏弱肌、乾燥肌、所有膚質適用\n* **產地：** 美國 (原裝進口)\n* **售價：** NT$1580", "input_tokens": 116, "output_tokens": 1168}, "59b640d54d8278b5a1cd": {"text": "## 晨光研選 凍乾果乾 (軟膠囊) 文案\n\n---\n\n### 商品標題\n\n**晨光研選｜夜間修護凍乾水果膠囊 50g｜外食族必備，20粒整顆營養輕鬆補**\n\n---\n\n### 賣點\n\n1. 美國專利技術，完整保留**整顆凍乾水果**精華。\n2. 睡前一粒軟膠囊，輕鬆補充缺乏的蔬果營養。\n3. 每盒 **20粒**，濃縮 **50g** 凍乾果實，高效率吸收。\n4. 告別疲憊，夜間黃金修護期，隔日煥發晨光。\n\n---\n\n### SEO 描述\n\n專為作息不規律的**上班族**設計。晨光研選夜間修護膠囊，採用美國技術將**整顆凍乾水果**精華濃縮，每盒**20粒**，內含**50g**高濃度營養。睡前輕鬆補充，告別**外食**營養失衡，讓您在夜間完成深度修護，迎接活力早晨。\n\n---\n\n### 標籤\n\n#凍乾水果 #夜間修護 #上班族保健 #外食族必備 #軟膠囊", "input_tokens": 256, "output_tokens": 290}, "5c9914514f44fecbff82": {"text": "好的，這是一份為「溯源焙所 綜合維他命錠 (25種，30粒)」設計的電商文案。\n\n文案結構包含：標題、痛點切入、產品介紹（核心優勢）、產品細節、購買資訊與行動呼籲。\n\n---\n\n## 🛒 溯源焙所 綜合維他命錠 (25種，30粒) 電商文案\n\n### 🌟 主標題 (吸引目光)\n\n**【告別疲憊，活力升級！】溯源焙所 25種全方位綜合維他命軟膠囊 — 每日一顆，補足一日所需！**\n\n---\n\n### 🎯 痛點切入 (為什麼你需要它？)\n\n**現代生活，你是否也面臨這些困擾？**\n\n* ❌ **外食族困境：** 三餐不定，營養攝取嚴重失衡？\n* ❌ **工作壓力大：** 經常感到精神不濟、體力透支？\n* ❌ **飲食不均衡：** 蔬果攝取不足，擔心缺乏關鍵營養素？\n* ❌ **想維持健康：** 卻不知道該從何補充起？\n\n**別讓「缺乏」成為健康的絆腳石！** 身體需要的，不該是零散的補充，而是一站式的完整守護。\n\n---\n\n### ✨ 核心優勢與產品介紹 (為什麼選擇溯源焙所？)\n\n#### **【溯源焙所】全方位綜合維他命：你的每日健康基石！**\n\n我們深知現代人對效率與品質的追求。溯源焙所綜合維他命，以台灣在地研發的專業配方，為您提供最便捷、最全面的營養支持。\n\n**🚀 三大關鍵優勢，啟動你的健康循環：**\n\n1. **超級全面！25種關鍵營養素一網打盡！**\n   * 包含 **23種維生素與礦物質**，從維生素A、B群、C、D、E到鈣、鎂、鋅等重要微量元素，一次補足身體所需的基礎能量，無需瓶瓶罐罐！\n\n2. **高效吸收！軟膠囊劑型，溫和好吞嚥！**\n   * 採用優質**軟膠囊**型態，相較於錠劑更容易分解吸收，有效成分能更快進入體內，發揮最大效益。\n\n3. **安心溯源！台灣製造，品質保證！**\n   * 堅持**台灣在地生產**，嚴格把關原料與製程，讓您吃得安心、補得放心。\n\n---\n\n### 🔍 產品細節與規格\n\n| 項目 | 規格說明 |\n| :--- | :--- |\n| **商品名稱** | 溯源焙所 綜合維他命錠 (25種) |\n| **主要成分** | 23種維生素及礦物質，共25種營養素 |\n| **型態** | 軟膠囊 (高效吸收) |\n| **產地** | 台灣 (安心製造) |\n| **包裝容量** | 30粒 / 盒 (約一個月份量) |\n| **建議售價** | **NT$390** |\n| **建議食用** | 每日1份 (1粒)，隨餐食用，效果更佳。 |\n\n---\n\n### 💡 適用族群\n\n* 忙碌的上班族、學生族\n* 外食比例高、飲食不均衡者\n* 經常感到疲勞、精神不振者\n* 追求基礎健康打底、維持體力者\n\n---\n\n### 🛒 立即行動 (Call to Action)\n\n**健康，從每日一粒開始！**\n\n別再讓營養缺口影響你的生活品質。給自己一個充滿活力的機會！\n\n**【溯源焙所 綜合維他命】**\n\n**原價 NT$490**\n**限時優惠價：NT$390**\n\n**👉 立即加入購物車，開啟你的全方位健康防護網！**\n\n---\n*【溫馨提醒】本產品為保健食品，不可取代正常飲食。如有特殊疾病或正在服用藥物者，請諮詢專業醫師意見。*", "input_tokens": 120, "output_tokens": 944}, "88b823112171088cbda8": {"text": "## 青研 CHINGYEN 修護乳液 80ml 7入\n\n---\n\n### 商品標題\n\n**青研 CHINGYEN 神經醯胺水感修護乳液 80ml 7入組 敏弱肌適用**\n\n---\n\n### 賣點\n\n1.  **核心神經醯胺，維持肌膚水潤**\n2.  **輕盈水感質地，清爽不黏膩**\n3.  **溫和親膚配方，敏弱肌適用**\n4.  **日本製造，大容量 80ml 7入組**\n\n---\n\n### SEO 描述\n\n專為重視溫和保濕的族群設計。青研 CHINGYEN 修護乳液，添加核心神經醯胺，幫助肌膚鎖住水分。日本製造，提供輕盈水感質地，不給肌膚負擔。超值 80ml 7入組，滿足日常保濕需求，敏弱肌適用。\n\n---\n\n### 標籤\n\n*   神經醯胺\n*   敏弱肌\n*   水感乳液\n*   保濕\n*   日本製造", "input_tokens": 814, "output_tokens": 257}, "ee0893df46f5acedfed6": {"text": "**【Volta 鑄鐵平底鍋 20cm】**\n\n---\n\n### 商品標題\n\n**Volta 美國製 20cm 鑄鐵平底鍋：差旅族的快速料理神器，終身保固** (29 字)\n\n---\n\n### 賣點 (4 條)\n\n1.  **20cm輕巧尺寸，單人份料理不佔空間。** (15 字)\n2.  **頂級鑄鐵材質，導熱均勻，耐用傳家。** (16 字)\n3.  **快速上菜，忙碌差旅生活也能享受美味。** (18 字)\n4.  **美國原廠製造，提供終身技術支援。** (15 字)\n\n---\n\n### SEO 描述\n\n專為追求效率的差旅族與多裝置使用者設計。Volta 20cm 鑄鐵平底鍋，美國製造，提供終身保固。小巧尺寸，快速完成單人份餐點，是您移動廚房的最佳選擇。耐用鑄鐵材質，導熱快，讓您在忙碌中也能享受高品質生活。現正優惠 NT$890。 (109 字)\n\n---\n\n### 標籤\n\n#鑄鐵鍋 #20cm平底鍋 #差旅生活 #單身廚房 #Volta", "input_tokens": 219, "output_tokens": 300}, "456afe14520c9e80586e": {"text": "{\n  \"title\": \"白日光 SUNDAY 唇部修護膏：天然蜂蠟與植物油，水潤嘟嘟唇養成術！15g 大容量 3入組\",\n  \"bullets\": [\n    \"溫和蜂蠟與植物油配方，敏弱肌也適用。\",\n    \"油狀質地輕盈不黏膩，隨時補水保濕。\",\n    \"15g 大容量，一罐抵多罐，CP 值超高。\",\n    \"3入分享組，臥室、包包、辦公室各放一罐。\"\n  ],\n  \"seo_description\": \"學生黨和社群控必備！白日光 SUNDAY 唇部修護膏，給你一整天的水潤好氣色。採用天然蜂蠟與植物油，質地滋潤卻不厚重，輕鬆打造水光感嘟嘟唇。15g 大容量，一次擁有 3入分享組，敏弱肌也能安心使用。\",\n  \"hashtags\": [\n    \"唇部保養\",\n    \"嘟嘟唇\",\n    \"白日光SUNDAY\",\n    \"平價好物\",\n    \"學生必備\"\n  ]\n}", "input_tokens": 780, "output_tokens": 268}, "65ebc3bc59309dfc3082": {"text": "**【本木 BENMU 螢幕掛燈 95Ra】**\n\n---\n\n### 商品標題 (50 字)\n本木 BENMU 螢幕掛燈 | 95Ra 高顯色，霧面鋁合金，為長時間工作的你提供專業照明。\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1.  專業級 **95Ra 顯色**，色彩精準，視覺體驗更舒適。\n2.  台灣製造，**霧面鋁合金**機身，堅固耐用，質感樸實。\n3.  不佔桌面空間，專為通勤與遠距工作者設計。\n4.  終身技術支援，本木 BENMU 提供長久使用保障。\n\n---\n\n### SEO 描述 (79 字)\n本木 BENMU 螢幕掛燈是通勤族與遠距工作者的專業選擇。採用高標準 **95Ra 顯色**指數，提供精準色彩照明。霧面鋁合金材質堅固耐用，台灣製造並提供終身技術支援。\n\n---\n\n### 標籤 (5 個)\n#螢幕掛燈\n#遠距工作\n#95Ra\n#BENMU\n#專業照明", "input_tokens": 504, "output_tokens": 275}, "f353dff9b4d96a95fda1": {"text": "## 溯源焙所 低溫烘焙綜合堅果\n\n### 商品標題\n**【晚安養護】溯源焙所 低溫烘焙綜合堅果 350g。外食族/上班族專屬，20粒獨立包裝，睡前輕鬆補給。** (59字)\n\n---\n\n### 賣點\n1. 20粒獨立包裝，每日一包，隨身攜帶不沾手。\n2. 堅持低溫烘焙，完整保留堅果天然營養與香氣。\n3. 專為作息不規律設計，睡前補充優質油脂與膳食纖維。\n4. 高規格綜合堅果 350g，頂級品質，安心溯源。\n\n---\n\n### SEO 描述\n忙碌的上班族與外食族必備！溯源焙所推出頂級低溫烘焙綜合堅果，提供您最方便的健康補給。每盒內含 **350g**，共 **20粒** 獨立包裝，每日睡前一包，輕鬆攝取優質營養。告別不規律的飲食習慣，從睡前開始養護健康。\n\n---\n\n### 標籤\n#綜合堅果 #外食族 #低溫烘焙 #睡前保養 #健康零食", "input_tokens": 256, "output_tokens": 294}, "d02e798fbd7b2319037b": {"text": "## 本木 BENMU 台灣製 24cm 鑄鐵平底鍋 | 陽極處理耐用升級 | 專業級均勻導熱\n\n---\n\n### 🎯 賣點 (4 條)\n\n1.  24cm 鑄鐵鍋體，導熱均勻穩定。\n2.  陽極處理技術，耐磨抗刮易清潔。\n3.  台灣在地製造，品質安心有保障。\n4.  專業級廚具入門款，NT$890 高CP值。\n\n---\n\n### 📝 SEO 描述 (不超過 120 字)\n\n本木 BENMU 24cm 鑄鐵平底鍋，台灣製造，陽極處理強化耐用度。無論是忙碌的出差族或追求效率的多裝置使用者，都能輕鬆享受烹飪樂趣。鑄鐵材質提供絕佳導熱性，煎煮炒炸一鍋搞定，原廠一年保固，安心選購。\n\n---\n\n### 🏷️ 標籤 (3-5 個)\n\n鑄鐵鍋, 24cm, 台灣製, 平底鍋, BENMU", "input_tokens": 222, "output_tokens": 249}, "81688b6028856557e962": {"text": "**Volta 20cm 鑄鐵平底鍋：差旅高效能料理模組，極簡生活配置。**\n\n---\n\n### 賣點\n\n1.  20cm 直徑，適合單人份快速料理配置。\n2.  鑄鐵本體，提供均勻熱傳導與穩定蓄熱。\n3.  PC+ABS 複合手柄，結構穩定且耐高溫。\n4.  無塗層設計，支援多種爐具與高強度烹調。\n\n---\n\n### SEO 描述\n\nVolta 20cm 鑄鐵平底鍋，專為追求高效與極簡配置的出差族設計。堅固鑄鐵材質提供穩定熱源，20cm 尺寸滿足單人份快速料理需求。PC+ABS 複合結構，提供終身技術支援。適合多裝置使用者在有限空間內打造專業級廚房。\n\n---\n\n### 標籤\n\n#Volta\n#鑄鐵鍋\n#20cm\n#差旅廚具\n#極簡生活", "input_tokens": 641, "output_tokens": 228}, "dd57c6d6136299fce955": {"text": "{\n  \"title\": \"溯源焙所 職人嚴選 低溫烘焙綜合堅果 350g 輕巧分裝 20粒裝\",\n  \"bullets\": [\n    \"低溫烘焙技術，保留堅果原始風味與優質口感。\",\n    \"嚴選多種綜合堅果，提供豐富層次與均衡營養。\",\n    \"獨立分裝 20粒，方便攜帶，隨時為忙碌生活充電。\",\n    \"台灣在地職人監製，安心溯源，為品質嚴格把關。\"\n  ],\n  \"seo_description\": \"溯源焙所低溫烘焙綜合堅果，專為作息不規律的上班族與外食族設計。每盒 20粒 獨立包裝，總重 350g，方便攜帶，隨時補充優質營養。採用低溫烘焙，保留堅果自然香氣與口感，是您追求健康生活，輕鬆享受美味的理想選擇。\",\n  \"hashtags\": [\n    \"綜合堅果\",\n    \"低溫烘焙\",\n    \"健康零食\",\n    \"溯源焙所\",\n    \"隨身包\"\n  ]\n}", "input_tokens": 907, "output_tokens": 273}, "c53e2c78679b73563113": {"text": "**商品標題**\n溯源焙所 職人低溫烘焙綜合堅果 350g 獨立膠囊包裝 20粒\n\n**賣點**\n1. 低溫慢火焙製，保留堅果原始完整風味。\n2. 獨立植物膠囊設計，每日定額，方便攜帶。\n3. 嚴選多國溯源堅果，提供豐富營養基礎。\n4. 專為夜間設計，睡前輕食無負擔。\n\n**SEO 描述**\n忙碌外食族與作息不規律上班族的夜間輕補給。溯源焙所職人嚴選高品質**綜合堅果**，採用低溫烘焙工法，保留堅果營養與香氣。每盒**350g**，內含**20粒**獨立植物膠囊，每日睡前一包，輕鬆實踐健康管理。\n\n**標籤**\n#綜合堅果 #低溫烘焙 #外食族必備 #夜間輕食 #獨立包裝", "input_tokens": 800, "output_tokens": 231}, "69d5f81d1d648af4fa6b": {"text": "{\n  \"title\": \"晨光研選 凍乾果乾 50g 20粒。整顆凍乾水果精華，外食族日常補給首選。\",\n  \"bullets\": [\n    \"嚴選整顆凍乾水果製成，保留天然營養素。\",\n    \"專為外食族設計，睡前一顆輕鬆補充。\",\n    \"美國原裝進口軟膠囊，每日持續補充無負擔。\",\n    \"每盒內含 20粒 凍乾果乾，每粒精華濃縮。\"\n  ],\n  \"seo_description\": \"晨光研選凍乾果乾，專為忙碌的上班族及外食族設計。採用整顆凍乾水果精華，每盒 20粒 軟膠囊，規格 50g。建議每日睡前補充，維持日常均衡。溫和配方，輕鬆融入您的生活節奏。\",\n  \"hashtags\": [\n    \"晨光研選\",\n    \"凍乾果乾\",\n    \"外食族補給\",\n    \"睡前保養\",\n    \"軟膠囊\"\n  ]\n}", "input_tokens": 900, "output_tokens": 258}, "206dc00f1c3e1312ac62": {"text": "## 青研 CHINGYEN 多重玻尿酸水感保濕面膜\n\n| 項目 | 內容 |\n| :--- | :--- |\n| **商品標題** | **青研 CHINGYEN 多重玻尿酸水感保濕面膜 7片 1入 敏弱肌適用** (40 字) |\n| **賣點** (4 條，每條不超過 30 字) | 1. 注入多重玻尿酸，深層補水，維持肌膚水潤。 |\n| | 2. 輕盈水感質地，敷後清爽，不造成肌膚負擔。 |\n| | 3. 美國製造，高標準製程，溫和配方敏弱肌友善。 |\n| | 4. 一盒 7片 1入，適合成分黨日常密集保濕使用。 |\n| **SEO 描述** (不超過 120 字) | 專為敏弱肌設計的青研保濕面膜 (NT$1580)。採用高純度多重玻尿酸配方，提供肌膚所需水分。輕盈水感質地，不造成肌膚負擔。規格為 7片 1入/盒，適合成分黨日常密集補水使用。美國原裝進口，安心有感。 |\n| **標籤** (3-5 個) | #敏弱肌 #玻尿酸面膜 #保濕 #成分黨 #青研 |", "input_tokens": 810, "output_tokens": 331}, "97def545ac0dbcd6d658": {"text": "## 溯源焙所 綜合維他命錠\n\n### 商品標題\n**溯源焙所 青春光采綜合維他命錠｜25種全方位營養，由內而外散發自信美麗**\n\n---\n\n### 賣點\n\n1.  嚴選**23種維生素礦物質**，完整基礎補給。\n2.  **25種**關鍵營養素，全方位補足每日所需。\n3.  台灣製造軟膠囊，每日一粒，輕鬆維持美麗。\n4.  告別疲憊暗沉，重現年輕紅潤好氣色。\n\n---\n\n### SEO 描述\n溯源焙所專為30-50歲注重保養的女性打造的綜合維他命錠。提供**25種**關鍵營養，包含**23種維生素礦物質**，幫助您維持青春活力與好氣色。每日只需1粒，一盒**30粒**，輕鬆啟動由內而外的美麗保養。台灣製造，品質安心。\n\n---\n\n### 標籤\n綜合維他命、美容保養、熟齡女性、活力元氣、基礎補充", "input_tokens": 263, "output_tokens": 255}, "3a837a9d1f1b94253d4d": {"text": "好的，這是一份針對「白日光 SUNDAY 唇部修護膏 15g 3入」的電商文案，涵蓋了標題、痛點、產品特色、成分優勢、使用情境與行動呼籲等部分。\n\n---\n\n## ☀️ 白日光 SUNDAY 唇部修護膏 (3入組) ☀️\n\n**【NT$290 限時優惠】美國原裝進口・敏弱肌適用・告別乾裂死皮！**\n\n---\n\n### 💡 前導標題 (抓住目光)\n\n**你的嘴唇，多久沒有好好休息了？**\n\n厭倦了頻繁補擦卻越擦越乾的唇膏嗎？讓美國專業品牌「白日光 SUNDAY」為你帶來一整週的滋潤修護，從此擺脫唇紋、乾裂與暗沉！\n\n---\n\n### 💔 痛點分析 (為什麼你需要它？)\n\n| ❌ 你是否也遇到這些困擾？ | ✅ 白日光 SUNDAY 幫你解決！ |\n| :---------------------- | :-------------------------- |\n| 唇紋深、死皮多，口紅上不去 | **深層修護**，撫平唇紋，妝容更服貼 |\n| 嘴唇容易敏感、刺痛、發炎 | **敏弱肌專用配方**，溫和不刺激 |\n| 睡前擦厚厚一層，早上卻沒效果 | **長效鎖水**，夜間黃金修護不間斷 |\n| 質地黏膩厚重，使用感不佳 | **輕盈油狀質地**，滋潤卻不黏膩 |\n\n---\n\n### ✨ 產品核心特色 (為什麼選擇我們？)\n\n#### 1. 美國原裝進口，專業級修護力\n\n來自美國的專業配方，嚴選高品質蜂蠟與天然植物油，為你的雙唇提供最純淨、最有效的滋養。\n\n#### 2. 敏弱肌救星，溫和零負擔\n\n專為敏感脆弱的唇部肌膚設計，不含刺激性化學成分，即使是極度乾燥或容易過敏的膚質也能安心使用。\n\n#### 3. 輕盈油狀質地，一抹化水\n\n告別傳統唇膏的厚重感！創新的**油狀質地**，接觸唇溫後迅速融化，形成一層輕薄透氣的保護膜，深層滋潤，卻絲毫不黏膩。\n\n#### 4. 超值囤貨組！日夜修護不間斷 (15g x 3入)\n\n大容量 15g，搭配超划算的 3 入組合，無論是辦公室、床頭櫃還是隨身包，都能隨時隨地為雙唇補水，讓你的嘴唇隨時保持最佳狀態！\n\n---\n\n### 🌿 主要成分與功效\n\n| 成分 | 特色 | 功效 |\n| :--- | :--- | :--- |\n| **天然蜂蠟** | 天然屏障 | 形成鎖水膜，防止水分流失，長效保濕。 |\n| **優質植物油** | 深層滋養 | 富含維生素與不飽和脂肪酸，修護受損肌膚，淡化唇紋。 |\n| **溫和配方** | 敏弱適用 | 舒緩乾燥不適，修護紅腫脫皮現象。 |\n\n---\n\n### 💄 多功能使用情境 (一罐多用！)\n\n* **日間打底：** 在塗抹霧面口紅前使用，能使唇妝更平滑、顯色，避免卡紋。\n* **夜間修護：** 睡前厚敷一層，作為「唇膜」使用，隔天醒來死皮輕鬆去除，雙唇水潤飽滿。\n* **急救舒緩：** 換季或乾燥時，隨時補擦，立即緩解乾裂緊繃感。\n\n---\n\n### 💰 價格與規格資訊\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **商品名稱** | 白日光 SUNDAY 唇部修護膏 |\n| **規格** | 15g / 罐 (共 3 入) |\n| **售價** | **NT$290** (平均一罐不到 $100！) |\n| **產地** | 美國 |\n| **適用膚質** | 所有膚質，特別推薦敏弱肌、乾燥唇。 |\n\n---\n\n### 🛒 立即行動 (Call to Action)\n\n**別讓乾燥的嘴唇，洩露你的疲憊！**\n\n現在就入手【白日光 SUNDAY 唇部修護膏 3入組】，給予雙唇最頂級的呵護。\n\n**NT$290 限時優惠，數量有限，立即搶購！**\n\n👉 **[點擊此處，告別乾裂唇！]**\n\n---", "input_tokens": 117, "output_tokens": 1097}, "5ba9a99d25df95dd08cb": {"text": "**品牌：白日光 SUNDAY**\n**分類：美妝保養 / 唇部**\n**售價：NT$290**\n\n---\n\n## 商品文案\n\n### 商品標題\n\n**白日光 SUNDAY 唇部修護膏 15g 大容量 3入組！蜂蠟植物油，隨時保持水潤感**\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1.  15g大容量，一罐抵多條，高CP值囤貨。\n2.  蜂蠟與植物油配方，溫和親膚。\n3.  油狀質地，輕盈好吸收，不黏膩。\n4.  3入組，辦公室、床頭、包包各放一罐。\n\n### SEO 描述 (不超過 120 字)\n\n學生黨、社群控必備！白日光 SUNDAY 唇部修護膏，採用天然**蜂蠟與植物油**配方，質地輕盈，敏弱肌也適用。超值**15g**大容量，一次入手**3入**組，隨時隨地維持唇部舒適感。高CP值囤貨首選！\n\n### 標籤 (3-5 個)\n\n#唇部保養\n#學生必備\n#大容量\n#敏弱肌適用\n#蜂蠟護唇", "input_tokens": 673, "output_tokens": 309}, "4eaa4af4b14ad2653ea4": {"text": "## 晨光研選 緩釋B群錠 專業文案\n\n### 商品標題\n晨光研選 緩釋B群錠｜長效釋放 維生素B群複方 75mg｜20粒\n\n### 賣點\n1. 獨特緩釋技術，穩定支持代謝機能。\n2. 高效**維生素B群複方 75mg**，元氣補給。\n3. 睡前輕鬆一粒，維持整日好狀態。\n4. 日本原裝進口，獨立包裝**20粒**。\n\n### SEO 描述\n專為長時間使用螢幕的忙碌上班族設計。晨光研選緩釋B群錠採用日本技術，提供高單位**維生素B群複方 75mg**。獨特緩釋機制，睡前補充，穩定支持整日精神與代謝運作。每盒**20粒**獨立包裝，方便持續補充，是您維持體力的專業選擇。\n\n### 標籤\nB群、緩釋B群、上班族、精神體力、維生素B", "input_tokens": 800, "output_tokens": 243}, "d4b9da9e192181814bad": {"text": "好的，這是一個台灣製造、高CP值的鑄鐵平底鍋。以下為您撰寫三種不同風格的電商文案，您可以根據您的行銷平台和目標受眾選擇最適合的版本。\n\n---\n\n## 本木 BENMU 鑄鐵平底鍋 24cm 電商文案\n\n### 版本一：【經典實用型】（適合官網、Momo、PChome等綜合電商平台）\n\n**標題：** 台灣製造🇹🇼 本木 BENMU 24cm 鑄鐵平底鍋 | 鎖住美味，傳承經典 | 煎炒燉煮一鍋到位！\n\n**主文案：**\n\n**🍳 台灣職人精神，打造廚房傳家寶！**\n\n厭倦了頻繁更換鍋具嗎？本木 BENMU 24cm 鑄鐵平底鍋，是您廚房中最值得投資的選擇。鑄鐵鍋具擁有卓越的蓄熱能力，能均勻導熱，讓食材受熱完整，輕鬆煎出完美焦糖化外層，鎖住肉汁與鮮甜。\n\n**✨ 為什麼選擇本木鑄鐵鍋？**\n\n* **【極致蓄熱】**：鑄鐵材質能長時間保持高溫，無論是煎牛排、香酥鍋貼，都能輕鬆達到餐廳級的酥脆口感。\n* **【健康無塗層】**：採用陽極處理，無化學塗層剝落疑慮，越用越好用，是真正能傳承給下一代的鍋具。\n* **【台灣製造 品質保證】**：堅持台灣在地生產，品質嚴格把關，給您最安心的烹飪體驗。\n* **【多功能應用】**：直徑 24cm 適合小家庭或單身貴族，煎蛋、炒菜、甚至放入烤箱烘烤都沒問題！\n\n**💰 超值入手價：** NT$890\n**✅ 規格：** 直徑 24cm / 台灣製造 / 原廠一年保固\n**🛒 立即擁有，開啟您的鑄鐵料理生活！**\n\n---\n\n### 版本二：【社群推廣型】（適合 Facebook、Instagram、Line 團購等強調視覺與互動的平台）\n\n**標題：** 😱 煎牛排不再失敗！NT$890 台灣製鑄鐵鍋，讓你在家變身米其林大廚！\n\n**(搭配圖片/影片：煎得金黃酥脆的牛排或太陽蛋)**\n\n**內文：**\n\n**🔥 你的廚房，需要一個「真」鍋！**\n\n你是不是也遇過：鍋子溫度不夠，牛排煎出來像水煮？或是鍋子一下就冷掉，食材口感大打折扣？\n\n**【本木 BENMU 24cm 鑄鐵平底鍋】** 完美解決你的痛點！\n\n這款鍋的秘密武器就是 **「超強蓄熱力」**！\n\n當你把食材放下去，鍋子溫度依然穩定，能瞬間鎖住肉汁，創造出迷人的「梅納反應」焦香外殼！\n\n**✅ 小資族必買清單：**\n\n1. **CP值爆表：** 台灣製造，只要 NT$890，輕鬆入手鑄鐵鍋！\n2. **健康無毒：** 陽極處理，用得安心。\n3. **尺寸剛好：** 24cm，一個人或兩人份料理剛剛好，不佔空間。\n\n**💡 鑄鐵鍋新手也不怕！** 只要簡單開鍋養護，它就會成為你廚房裡最忠實的夥伴！\n\n**👉 點擊連結，用 NT$890 升級你的料理境界！**\n#本木BENMU #鑄鐵鍋 #台灣製造 #小資廚房 #煎牛排神器 #健康料理\n\n---\n\n### 版本三：【簡潔促銷型】（適合限時特賣、Banner 廣告或電商活動頁面）\n\n**標題：** 【限時搶購】台灣製 本木 BENMU 24cm 鑄鐵平底鍋 | 只要 $890！\n\n**核心賣點（條列式）：**\n\n* **🔥 經典鑄鐵：** 卓越蓄熱，煎出完美焦脆口感。\n* **🇹🇼 台灣製造：** 品質保證，原廠一年保固。\n* **✅ 健康無塗層：** 陽極處理，越用越不沾。\n* **💰 超級甜甜價：** NT$890 輕鬆入手，高CP值首選！\n* **🎁 尺寸：** 24cm，小家庭/單身族必備。\n\n**行動呼籲：**\n\n**不要錯過！數量有限，立即加入購物車！**\n\n---", "input_tokens": 95, "output_tokens": 1068}, "e5a92bb56c64f7fafb4a": {"text": "{\n  \"title\": \"溯源焙所 綜合維他命錠 25種營養素 補充23種維生素礦物質 30粒裝\",\n  \"bullets\": [\n    \"嚴選 25種關鍵營養素完整配方\",\n    \"提供每日所需 23種維生素礦物質\",\n    \"台灣在地生產製造，品質安心\",\n    \"每日一粒，輕鬆維持基礎能量補給\"\n  ],\n  \"seo_description\": \"溯源焙所綜合維他命錠，專為忙碌的現代女性設計。一粒補充 25種關鍵營養素，包含 23種維生素礦物質，提供日常所需基礎能量。台灣製造，每盒 30粒裝，輕鬆開啟每日保養。\",\n  \"hashtags\": [\n    \"綜合維他命\",\n    \"基礎保養\",\n    \"女性保健\",\n    \"23種維生素礦物質\",\n    \"溯源焙所\"\n  ]\n}", "input_tokens": 914, "output_tokens": 226}, "5a896d2c9289660bf310": {"text": "## 晨光研選 凍乾水果精華膠囊 50g 美國原裝 整顆凍乾水果濃縮 20粒 睡前營養補給\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1. **整顆凍乾水果製成，保留完整營養**\n2. **每日睡前一粒，輕鬆完成營養補充**\n3. **每盒含 20粒，內含 50g 高濃度精華**\n4. **美國原裝進口，為忙碌生活提供穩定支持**\n\n---\n\n### SEO 描述 (不超過 120 字)\n\n專為作息不規律的上班族與外食族設計。晨光研選凍乾水果精華，採用美國技術將**整顆凍乾水果**濃縮成膠囊。每盒 **20粒**，內含 **50g** 精華。睡前補充，維持日常活力與穩定性。天然營養，輕鬆攝取無負擔。\n\n---\n\n### 標籤 (3-5 個)\n\n#凍乾水果 #外食族必備 #營養補給 #睡前保養 #晨光研選", "input_tokens": 793, "output_tokens": 269}, "d2dccbbaa07279aff813": {"text": "這是一個很棒的商品！以下為您量身打造的電商文案，包含標題、痛點描述、產品介紹、核心賣點、以及行動呼籲，您可以根據不同平台（如官網、蝦皮、FB/IG）調整長度與格式。\n\n---\n\n## ☀️ 白日光 SUNDAY 護手霜 5% 5入組 電商文案\n\n### 一、 標題與關鍵字（用於搜尋和廣告）\n\n| 平台/目的 | 標題建議 (精簡有力) | 關鍵字/標籤 (Hashtags) |\n| :--- | :--- | :--- |\n| **主標題 (SEO)** | 【白日光 SUNDAY】5% 尿素護手霜 (5入組)｜深層保濕、告別乾裂、乾燥缺水肌救星！ | #護手霜 #尿素護手霜 #手部保養 #乾燥肌 #保濕 #台灣製造 #白日光SUNDAY |\n| **促銷標題** | 爆款回饋！白日光SUNDAY 5%尿素護手霜 5入組 $490｜隨身攜帶、隨時滋潤！ | |\n\n---\n\n### 二、 痛點與情境描述（引起共鳴）\n\n**你是否也有這些困擾？**\n\n* **洗手次數增加，雙手乾澀緊繃，甚至出現細紋和脫皮？**\n* **長時間待在冷氣房，手部肌膚像缺水的沙漠，擦什麼都覺得不夠力？**\n* **護手霜擦了半天，卻只有油膩感，無法真正滲透吸收？**\n\n**別讓粗糙的雙手，洩漏了你的年齡！** 雙手是女人的第二張臉，需要最專業、最有效的呵護。\n\n---\n\n### 三、 產品核心介紹與賣點 (FAB 法則)\n\n#### 🌟 【白日光 SUNDAY】5% 尿素護手霜 (5入組)\n\n**售價：NT$490**\n\n#### 💡 產品特色：專業級保濕配方，乾燥肌的急救包！\n\n| 特點 (Feature) | 優勢 (Advantage) | 效益 (Benefit) |\n| :--- | :--- | :--- |\n| **【F】黃金比例 5% 尿素 (Urea)** | **【A】高效鎖水、軟化角質** | **【B】深層滲透，有效改善手部粗糙、乾裂、脫皮現象，讓肌膚恢復柔嫩細緻。** |\n| **【F】清爽乳霜質地** | **【A】不黏膩、好吸收** | **【B】告別厚重油膩感！快速吸收，滋潤不沾手，擦完立即工作或滑手機也沒問題。** |\n| **【F】輕巧隨身包裝 (5入/盒)** | **【A】方便攜帶，隨時補擦** | **【B】辦公室、包包、床頭櫃、車上各放一條，隨時為雙手補充水分，維持一整天的水潤感。** |\n| **【F】台灣製造 (Made in Taiwan)** | **【A】品質嚴格把關** | **【B】使用安心，給予肌膚最溫和、最有效的呵護。** |\n\n#### 🎯 適用對象：\n\n* **乾燥缺水肌、季節性乾癢者**\n* **經常洗手、接觸清潔劑者**\n* **長時間待在空調環境者**\n* **追求清爽高效保濕者**\n\n---\n\n### 四、 規格與購買資訊\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **商品名稱** | 白日光 SUNDAY 護手霜 5% (5入組) |\n| **主要規格** | 5% 尿素 (Urea) |\n| **型態** | 清爽乳霜質地 |\n| **產地** | 台灣 |\n| **包裝** | 5入 / 盒 (每條容量請自行填寫，若無則省略) |\n| **售價** | **NT$490** |\n\n---\n\n### 五、 行動呼籲與結尾\n\n**🔥 買一盒，全家受惠！**\n\n這不只是一條護手霜，更是你對雙手細心呵護的承諾。\n\n現在購買【白日光 SUNDAY 5% 尿素護手霜 5入組】，平均一條不到百元，就能享受專業級的滋潤修護！\n\n**👉 立即加入購物車，告別乾裂，迎接柔嫩細緻的「白日光」美手！**\n\n**(點擊下方連結，手刀搶購！)**", "input_tokens": 109, "output_tokens": 1087}, "8bdc66200f1b174db34c": {"text": "## 白日光 SUNDAY 5% 尿素修護護手霜 5入組｜深度保濕，告別乾燥粗糙\n\n---\n\n### 賣點\n\n1.  專業級**尿素 5%**，有效軟化角質，深層鎖水。\n2.  輕盈乳霜質地，吸收快速，清爽不黏膩。\n3.  溫和配方設計，乾燥缺水肌膚適用，舒緩不適。\n4.  超值**5入**組，辦公室、隨身包、居家隨處可放。\n\n---\n\n### SEO 描述\n\n專為30-45歲重視溫和保濕的您設計。白日光 SUNDAY 護手霜添加高效**尿素 5%**，能深層滋潤乾燥缺水肌，有效修護粗糙手部肌膚，達到長效鎖水。台灣製造，品質安心。超值**5入**組，讓您隨時隨地維持柔嫩雙手，告別乾裂困擾，輕鬆擁有細緻膚觸。\n\n---\n\n### 標籤\n\n保濕 | 護手霜 | 尿素 | 溫和配方 | 台灣製造", "input_tokens": 247, "output_tokens": 268}, "f84f4b0e04e9a53de8b3": {"text": "好的，這是一個非常吸引人的產品組合！以下為您量身打造的電商文案，針對不同平台和宣傳重點，提供三種風格選擇：\n\n---\n\n## 💡 電商文案組合包 (青研 CHINGYEN 修護乳液 80ml 7入)\n\n### 方案一：【主打修護、高CP值】 (適用於 FB 廣告、官方商城首頁)\n\n**標題：**\n**【爆水囤貨組！7入只要$890】日本青研 CHINGYEN 神經醯胺修護乳液 | 敏弱肌救星，一抹爆水、長效鎖潤！**\n\n---\n\n**內文：**\n\n**📢 敏弱肌請注意！年度最強囤貨組駕到！**\n\n乾燥、泛紅、脫屑、換季不適... 您的肌膚正在發出警訊！\n來自日本的 **青研 CHINGYEN 修護乳液**，專為脆弱肌膚設計，用最純粹的成分，給予最深層的呵護。\n\n✨ **為什麼要選擇青研修護乳液？**\n\n1. **【日本製造．安心保證】**：原裝進口，品質嚴格把關。\n2. **【高濃度神經醯胺】**：直擊肌膚乾燥根源，強化肌膚屏障，鎖住水分不流失。\n3. **【水感輕盈質地】**：告別黏膩厚重！一抹化水，迅速吸收，清爽無負擔，油肌、混合肌也能安心使用。\n4. **【敏弱肌專用配方】**：溫和不刺激，舒緩泛紅不適，讓肌膚重回穩定健康狀態。\n\n**🔥 史上最划算！青研修護乳液 (80ml) 7入組！**\n\n* **原價：** (請自行填寫，若無原價可省略)\n* **限時優惠價：NT$890** (平均一瓶不到$130！比開架更超值！)\n\n👉 **囤貨趁現在！給肌膚一整年的穩定呵護！**\n\n🛒 **立即搶購：[您的購買連結]**\n#青研 #CHINGYEN #神經醯胺 #修護乳液 #敏弱肌 #保濕 #日本製造\n\n---\n\n### 方案二：【強調成分、輕盈感】 (適用於 IG 貼文、美妝部落格導購)\n\n**標題：**\n**夏日保濕不黏膩！💧 日本青研 CHINGYEN 神經醯胺水感乳液 | 敏感肌的清爽續命水！**\n\n---\n\n**內文：**\n\n**【水感爆棚！乳液不再是負擔！】**\n\n你是不是也害怕夏天擦乳液的黏膩感？\n日本 **青研 CHINGYEN** 聽到你的心聲了！\n\n這款修護乳液主打 **「水感質地」**，輕盈到像精華液，一抹瞬間被肌膚喝光光，只留下滿滿的保濕度，完全不悶痘、不黏枕頭！\n\n🔬 **核心成分解密：**\n\n* **神經醯胺 (Ceramide)：** 肌膚屏障的關鍵！有效修復受損角質層，提升肌膚防禦力。\n* **80ml 精巧包裝：** 方便攜帶，旅行、健身房隨時補水。\n\n**✨ 適用膚質：** 所有膚質，特別推薦 **敏弱肌、油性肌** 和 **換季不穩定肌**。\n\n**🎁 超狂組合：青研修護乳液 7入組！**\n\n* **價格：NT$890**\n* **產地：日本**\n* **數量有限，售完不補！**\n\n**💬 用戶好評：** 「擦完臉很嫩，隔天起床臉還是很水潤，而且這個價格買7瓶真的太佛心了！」\n\n點擊主頁連結，讓肌膚體驗日系清爽修護力！\n\n#青研CHINGYEN #乳液推薦 #水感乳液 #神經醯胺保濕 #日本保養 #清爽不黏膩 #高CP值囤貨\n\n---\n\n### 方案三：【短文案、電商平台促銷】 (適用於蝦皮/Momo 內頁、限時動態)\n\n**【青研 CHINGYEN】日本神經醯胺修護乳液 80ml (7入組)**\n\n**💰 佛心價：NT$890 (平均一瓶 $127！) 敏弱肌必囤！**\n\n---\n\n**商品特色速覽：**\n\n✅ **【日本原裝】**：品質保證，安心使用。\n✅ **【神經醯胺】**：深度修護，強化肌膚屏障。\n✅ **【水感質地】**：清爽不黏膩，快速吸收。\n✅ **【敏弱肌適用】**：溫和配方，舒緩乾燥泛紅。\n✅ **【超值 7入】**：臉部、身體、全家大小都能用！\n\n**👉 換季肌膚不穩定？選它就對了！**\n\n**立即加入購物車！**\n\n---", "input_tokens": 117, "output_tokens": 1200}, "ff36fadb563d76960fdd": {"text": "{\n  \"title\": \"白日光 SUNDAY 5% 尿素保濕護手霜 5入組 輕盈乳霜質地 告別乾裂粗糙\",\n  \"bullets\": [\n    \"關鍵成分「尿素 5%」，溫和深層補水滋潤。\",\n    \"輕盈乳霜質地，快速吸收不黏膩無負擔。\",\n    \"專為乾燥缺水肌設計，維持手部柔嫩細緻。\",\n    \"貼心 5入 分享包裝，隨時隨地滋潤雙手。\"\n  ],\n  \"seo_description\": \"白日光 SUNDAY 5% 尿素護手霜，專為乾燥肌設計。高濃度 5% 尿素配方，提供手部肌膚長效保濕與滋潤。輕盈乳霜質地，快速吸收不厚重。一盒 5入 隨身包裝，讓您隨時維持雙手柔嫩細緻。適合注重溫和保濕的你。\",\n  \"hashtags\": [\n    \"護手霜\",\n    \"尿素保濕\",\n    \"乾燥肌救星\",\n    \"白日光SUNDAY\",\n    \"輕盈不黏膩\"\n  ]\n}", "input_tokens": 770, "output_tokens": 280}, "53e5c9a32cc99f3d2a7b": {"text": "{\n  \"title\": \"晨光研選 緩釋B群錠 維生素B群複方 75mg 20粒 支援能量代謝 溫和持續釋放\",\n  \"bullets\": [\n    \"嚴選高效能維生素B群複方，完整補給。\",\n    \"每錠含足量 75mg B群，滿足日常所需。\",\n    \"獨特緩釋技術，營養素溫和持續釋放。\",\n    \"建議睡前食用，全素可食，日本製造。\"\n  ],\n  \"seo_description\": \"晨光研選緩釋B群錠，專為長時間使用螢幕的上班族設計。採用維生素B群複方，每份 75mg，提供持續穩定的營養支持。獨家緩釋配方，幫助維持神經系統健康與正常能量代謝。一盒 20粒 獨立包裝，方便每日持續補充，維持工作效率。\",\n  \"hashtags\": [\n    \"B群\",\n    \"緩釋B群\",\n    \"維生素B群複方\",\n    \"上班族必備\",\n    \"晨光研選\"\n  ]\n}", "input_tokens": 907, "output_tokens": 262}, "d4110e6475f0adae7cba": {"text": "**商品標題**\n本木 BENMU 台灣製 24cm 鑄鐵平底鍋 | 陽極處理，傳熱均勻，享受樸實耐用的料理時光 (42字)\n\n---\n\n**賣點**\n1. 鑄鐵材質蓄熱佳，料理受熱更均勻。\n2. 實用 24cm 直徑，煎炒燉煮一鍋搞定。\n3. 台灣在地製造，陽極處理工法更耐用。\n4. 厚實鍋體設計，經久使用不易變形。\n\n---\n\n**SEO 描述**\n本木 BENMU 24cm 鑄鐵平底鍋，專為追求實用與耐用的您設計。台灣在地製造，採用陽極處理工法，鍋體厚實蓄熱優異，無論是煎牛排或家常菜都能輕鬆掌握火候。樸實設計，是您廚房中最可靠的幫手。 (78字)\n\n---\n\n**標籤**\n* 鑄鐵鍋\n* 24cm平底鍋\n* BENMU\n* 台灣製造\n* 耐用廚具", "input_tokens": 504, "output_tokens": 253}, "03df0610cbfb890f6e5e": {"text": "## 溯源焙所 綜合維他命錠 25種完整營養 30粒軟膠囊\n\n---\n\n### 賣點\n1. 單顆濃縮 25種多元營養素。\n2. 涵蓋 23種維生素礦物質。\n3. 每日一粒，軟膠囊型態好吞嚥。\n4. 補充日常飲食不足的基礎能量。\n\n---\n\n### SEO 描述\n溯源焙所嚴選綜合維他命錠，提供 25種 關鍵營養素，包含 23種維生素礦物質。每日僅需一顆 30粒 軟膠囊，輕鬆補足日常所需。為忙碌的妳，由內而外打底，維持活力與美麗。\n\n---\n\n### 標籤\n* 綜合維他命\n* 女性保養\n* 基礎營養\n* 維生素礦物質", "input_tokens": 807, "output_tokens": 198}, "58f48a1b6d05d20b0a3d": {"text": "## Volta 陶瓷刀具組 3件\n\n---\n\n### 商品標題\n\n**Volta 陶瓷刀具組 3件 | 輕奢極簡風，小家庭/寵物友善，鋒利耐用不殘味**\n\n---\n\n### 賣點\n\n1.  陶瓷刀刃，鋒利持久不需磨。\n2.  抗菌不殘味，毛孩家庭安心選。\n3.  極簡設計，3件組收納不佔位。\n4.  韓國製造，享終身技術支援。\n\n---\n\n### SEO 描述\n\n專為小坪數家庭設計的 Volta **陶瓷**刀具組 **3件**。刀刃採用高硬度陶瓷，搭配食品級不鏽鋼刀柄，輕巧、抗菌、不生鏽。NT$3290，韓國製造，是寵物家庭分切食材的最佳選擇，終身技術支援讓您安心。\n\n---\n\n### 標籤\n\n#陶瓷刀\n#小家庭廚房\n#寵物用品分切\n#不鏽鋼刀具\n#3件組", "input_tokens": 218, "output_tokens": 238}, "a4fa10a8ce9db37d5954": {"text": "好的，這是一個非常經典且耐用的商品。針對 **Volta 鑄鐵平底鍋 20cm** (售價 NT$890)，我將為您撰寫三種不同風格的電商文案，以適應不同的行銷情境：\n\n---\n\n## Volta 鑄鐵平底鍋 20cm 電商文案\n\n### 策略分析\n\n* **核心優勢 (USP):** 鑄鐵鍋的耐用性、優異的蓄熱能力、美國製造品質、以及最關鍵的「終身技術支援」。\n* **價格定位:** NT$890 對於一個終身保固的鑄鐵鍋來說，是極具吸引力的入門價格。\n* **目標客群:** 單身貴族、小家庭、注重烹飪品質的初學者、追求高CP值的廚具愛好者。\n\n---\n\n## 文案風格一：經典耐用型 (強調品質與傳承)\n\n**適用情境：** 品牌官網、產品詳情頁、追求穩定與信任感的消費者。\n\n### 標題：【Volta 鑄鐵經典】從新手到大廚，只差這一只 20cm 傳家鍋\n\n**內文：**\n\n**🍳 烹飪的起點，永恆的承諾**\n\n您是否厭倦了頻繁更換不沾鍋？是時候投資一只能夠傳承下去的廚具了。Volta 鑄鐵平底鍋 20cm，以其卓越的蓄熱能力和堅固耐用的特性，為您的日常烹飪帶來革命性的提升。\n\n**✨ 為什麼選擇 Volta 鑄鐵鍋？**\n\n1. **完美蓄熱，鎖住美味：** 鑄鐵材質能均勻且長時間地保持高溫，無論是煎牛排的焦脆外殼，還是烘烤蔬菜的鮮甜，都能完美呈現。\n2. **20cm 萬用尺寸：** 專為單人或小家庭設計，煎一顆蛋、烤一份鬆餅、炒一道菜，輕巧實用，不佔空間。\n3. **美國製造，品質保證：** 堅持美國原產地製造，確保每一只鍋具都符合 Volta 嚴苛的品質標準。\n\n**🛡️ 終身相伴，無後顧之憂**\n\nVolta 不僅提供一只鍋，更提供一個承諾。我們深知鑄鐵鍋的價值在於長久使用，因此提供**「終身技術支援」**。無論您在使用上遇到任何問題，我們都是您最堅實的後盾。\n\n**💰 NT$890 即可擁有，開啟您的鑄鐵傳奇。**\n\n👉 **立即入手，體驗真正的烹飪樂趣！**\n\n---\n\n## 文案風格二：高 CP 值入門型 (強調價格與實用性)\n\n**適用情境：** 促銷活動、社群媒體廣告、電商平台首頁推薦 (如蝦皮、Momo)。\n\n### 標題：【NT$890 終身保固】Volta 20cm 鑄鐵鍋：人生第一只鑄鐵鍋，就是它！\n\n**內文：**\n\n**🔥 告別廚房小白！用最少的預算，享受大廚級的烹飪體驗！**\n\n想嘗試鑄鐵鍋的魔力，卻擔心價格高昂、保養複雜？Volta 20cm 鑄鐵平底鍋，是您踏入鑄鐵世界的最佳起點！\n\n**✅ 入門無負擔！NT$890 震撼價！**\n\n我們將高品質的美國製造鑄鐵鍋，以最親民的價格帶給您。這不僅是價格上的優惠，更是對您烹飪熱情的支持。\n\n**💡 20cm，小巧卻強大！**\n\n* **單身貴族的救星：** 快速加熱，輕鬆完成一人份餐點。\n* **鑄鐵的秘密：** 煎、炒、烤、炸，一鍋多用，讓食物風味更上一層樓。\n* **健康升級：** 鑄鐵在使用過程中會釋放微量鐵質，讓您吃得更安心。\n\n**🎁 買的不只是鍋，是終身服務！**\n\n別忘了，Volta 提供業界罕見的**「終身技術支援」**！只要您擁有它，我們就永遠為您服務。\n\n**限量搶購中！讓 NT$890 成為您廚房最值得的投資！**\n\n🛒 **點擊購買，馬上開始您的鑄鐵料理之旅！**\n\n---\n\n## 文案風格三：細節規格型 (強調材質與專業度)\n\n**適用情境：** 產品規格頁、專業廚具論壇、對材質細節有要求的消費者。\n\n### 標題：Volta 20cm 鑄鐵平底鍋：美國原產，複合手柄設計，專業級蓄熱表現。\n\n**內文：**\n\n**🔬 專注細節，成就卓越**\n\nVolta 20cm 鑄鐵平底鍋，專為追求高品質與細節的您而生。我們不僅注重鑄鐵本身的性能，更優化了使用體驗，讓傳統工藝與現代設計完美結合。\n\n**🛠️ 核心規格解析：**\n\n| 規格項目 | 詳細說明 | 專業優勢 |\n| :--- | :--- | :--- |\n| **直徑** | 20cm | 輕量化設計，適合快速加熱與精準控溫。 |\n| **材質** | 鑄鐵本體 | 極佳的熱傳導與蓄熱性能，耐高溫、耐刮磨。 |\n| **手柄** | PC+ABS 複合材質 | 有效隔熱，提供舒適且安全的握感，易於操作。 |\n| **產地** | 美國製造 (U.S.A) | 嚴格的品管標準，確保鑄鐵密度與表面處理的穩定性。 |\n\n**🔄 複合手柄設計：安全與舒適的平衡**\n\n我們採用 PC+ABS 複合材質設計手柄，解決了傳統鑄鐵鍋手柄過熱的問題，讓您在烹飪過程中操作更安全、更順手。\n\n**🏆 Volta 承諾：終身技術支援**\n\n鑄鐵鍋越用越好用，我們承諾提供終身的保養、開鍋、除鏽等技術諮詢服務，確保您的 Volta 鍋具能陪伴您數十年。\n\n**NT$890，專業品質，值得信賴。**\n\n---", "input_tokens": 92, "output_tokens": 1450}, "a5f21fceda0d743df7c9": {"text": "## 青研 CHINGYEN 多重玻尿酸保濕面膜 5片x3入組 凝露質地 敏弱肌適用\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1.  多重玻尿酸配方，提供肌膚長效水潤。\n2.  清爽凝露質地，敷感輕盈不悶膚。\n3.  德國研發配方，敏弱肌膚友善測試。\n4.  每盒 5片，共 3入，密集保濕不間斷。\n\n---\n\n### SEO 描述 (不超過 120 字)\n\n青研 CHINGYEN 專為敏弱肌設計的保濕面膜。成分黨必備，採用高效多重玻尿酸，將水分注入肌底，維持肌膚水嫩。清爽凝露質地，不造成肌膚負擔。規格為 5片/組，一次購買 3入，讓您持續水潤不間斷。德國溫和配方，安心有感。\n\n---\n\n### 標籤 (3-5 個)\n\n#青研\n#保濕面膜\n#多重玻尿酸\n#敏弱肌\n#成分黨", "input_tokens": 809, "output_tokens": 277}, "a01322e99e25d4d8fbef": {"text": "## 白日光 SUNDAY 護手霜 (NT$490)\n\n---\n\n### 商品標題\n白日光 SUNDAY 5%尿素高保濕護手霜 5入組｜日常滋潤，溫和舒緩乾癢 (50字)\n\n---\n\n### 賣點\n1. 專業級 5% 尿素，深層補水鎖水。\n2. 輕盈乳霜質地，清爽不黏膩。\n3. 貼心 5入組，隨時隨地呵護雙手。\n4. 台灣製造，溫和配方，乾燥肌適用。\n\n---\n\n### SEO 描述\n30-45歲的你，雙手需要更溫和的滋潤。白日光 SUNDAY 護手霜，添加高效保濕的 **5% 尿素**成分，能為乾燥缺水肌帶來長效水潤。輕盈質地好吸收，不影響工作。超值 **5入** 組合，讓您辦公室、居家、外出都能輕鬆保養。台灣製造，安心可靠。 (99字)\n\n---\n\n### 標籤\n#白日光SUNDAY #尿素護手霜 #5%尿素 #保濕 #乾燥肌", "input_tokens": 663, "output_tokens": 276}, "ad943eac08efcea137ee": {"text": "這是一份針對「本木 BENMU 螢幕掛燈 95Ra」設計的電商文案，強調其專業級的顯色能力、台灣製造的品質，以及終身保固的承諾。\n\n---\n\n## 💡 【本木 BENMU】螢幕掛燈 95Ra：點亮真實色彩，告別視覺疲勞\n\n**售價：NT$890**\n**分類：3C / 週邊**\n\n---\n\n## 🌟 文案標題建議\n\n**主標題：** 專業級色彩！本木 BENMU 螢幕掛燈 — 95Ra 高顯色，台灣製造，終身技術支援。\n**副標題：** 釋放桌面空間，享受最接近自然光的舒適照明。設計師、工程師、內容創作者必備！\n\n---\n\n## 💎 五大必買理由 (KSP)\n\n### 1. 專業級 95Ra 高顯色指數 (CRI)\n**不再讓色彩失真！** 一般螢幕掛燈的顯色指數約在 80Ra 左右，而 BENMU 達到驚人的 **95Ra**，極度接近自然光（100Ra）。這意味著：\n*   **設計師與攝影師：** 確保您在螢幕前看到的顏色，就是作品的真實顏色。\n*   **長時間使用者：** 高顯色能有效減少眼睛辨識疲勞，讓視覺更舒適。\n\n### 2. 頂級霧面鋁合金工藝\n採用高質感**霧面鋁合金**材質，不僅外觀簡約時尚，完美融入任何桌面佈置，更具備優異的散熱性與耐用度。精緻的質感，讓您的工作空間瞬間升級。\n\n### 3. 台灣製造，品質保證\n堅持 **Made in Taiwan** 的高品質製造標準。從設計到生產，嚴格把關每個細節，為您提供穩定、可靠、耐用的照明產品。\n\n### 4. 終身技術支援，安心無虞\n本木 BENMU 承諾提供**終身技術支援**。我們對產品的信心，就是您最堅實的後盾。購買後無須擔心產品問題，讓您買得安心，用得放心。\n\n### 5. 零佔用桌面空間\n螢幕掛燈設計，無需底座或支架，完美懸掛於螢幕上方。徹底解放您的桌面空間，讓工作區更整潔、更有效率。\n\n---\n\n## 📋 產品細節與規格\n\n| 規格項目 | 詳細說明 | 亮點強調 |\n| :--- | :--- | :--- |\n| **品牌** | 本木 BENMU | 台灣在地品牌 |\n| **顯色指數** | **95Ra** | 專業級色彩還原，保護視力 |\n| **型態** | 霧面鋁合金 | 高質感、耐用、散熱佳 |\n| **產地** | 台灣 | 品質與信賴的保證 |\n| **保固** | 終身技術支援 | 業界罕見承諾，長期使用無憂 |\n| **售價** | NT$890 | 高性價比的專業級選擇 |\n\n---\n\n## 🛒 誰需要這盞 BENMU 螢幕掛燈？\n\n*   **💻 長時間電腦工作者：** 減少螢幕眩光反射，降低眼睛疲勞。\n*   **🎨 視覺設計師/修圖師：** 對色彩準確度有極高要求，95Ra 讓您不再有色差困擾。\n*   **📚 學生/研究人員：** 提供充足且舒適的照明，提升專注力。\n*   **✨ 追求質感美學的您：** 簡約鋁合金設計，輕鬆打造高階桌面環境。\n\n---\n\n## 立即行動！\n\n**NT$890** 的價格，即可擁有 **95Ra 專業級顯色**、**台灣製造** 的頂級螢幕掛燈，以及 **終身技術支援** 的服務。\n\n**別再讓低品質的光線傷害您的雙眼！**\n\n**點擊下方按鈕，立即將【本木 BENMU 螢幕掛燈】帶回家，享受最舒適、最真實的桌面光線體驗！**", "input_tokens": 95, "output_tokens": 960}, "bccd5a7ab5f0373f3b2b": {"text": "## Volta 陶瓷刀具組 3件\n\n### 商品標題\nVolta 陶瓷刀具組 3件 食品級不鏽鋼柄 輕量化設計 寵物家庭適用\n\n### 賣點\n1. 陶瓷刀刃，高硬度耐用，維持鋒利度。\n2. 零金屬異味，不殘留食材氣味。\n3. 3件組合，滿足日常備料需求。\n4. 食品級不鏽鋼握柄，結構穩定。\n\n### SEO 描述\n專為小坪數家庭設計的Volta 陶瓷刀具組 3件。陶瓷刀刃提供優異的切割性能，且不與食材產生化學反應。食品級不鏽鋼結構，確保高衛生標準。輕量化設計，收納不佔空間。適合注重廚房清潔的寵物族群。\n\n### 標籤\n#陶瓷刀\n#小家庭廚具\n#寵物家庭\n#輕量化\n#Volta", "input_tokens": 640, "output_tokens": 218}, "1def1b3364b9913ad349": {"text": "好的，這是一份針對您的「Volta 陶瓷刀具組 3件」所撰寫的電商文案。\n\n這份文案包含了標題、賣點、痛點解決、產品特色、規格介紹以及行動呼籲，適用於商品頁面或廣告投放。\n\n---\n\n# 🔪 Volta 陶瓷刀具組 3件：頂級韓製工藝，讓您的料理體驗全面升級！\n\n## 🌟 標題與核心賣點\n\n**主標題：** 【Volta】韓國原裝進口陶瓷刀具組 (3件)：極致鋒利，永不生鏽！NT$3290 享終身技術支援。\n\n**副標題：** 告別鈍刀與金屬味！體驗米其林主廚等級的精準切割，讓食材保持最純粹的鮮甜。\n\n---\n\n## 🎯 痛點分析與解決方案 (為什麼您需要 Volta？)\n\n| 您的痛點 | Volta 陶瓷刀具組的解決方案 |\n| :--- | :--- |\n| **刀具容易鈍化？** | 採用高密度奈米陶瓷，鋒利度是傳統鋼刀的數倍，持久耐用，大幅減少磨刀頻率。 |\n| **切完食材有金屬味？** | 陶瓷材質不含金屬離子，不會與食材產生化學反應，完美保留蔬果、魚肉的原始風味。 |\n| **刀具容易生鏽、滋生細菌？** | 陶瓷具備極佳的抗氧化和抗腐蝕特性，不生鏽、易清潔，有效抑制細菌附著。 |\n| **擔心產品品質與售後服務？** | 韓國原裝進口，品質保證，並提供業界罕見的**終身技術支援**，讓您買得安心、用得放心。 |\n\n---\n\n## ✨ 產品特色：Volta 的四大頂級優勢\n\n### 1. 💎 奈米陶瓷技術：極致鋒利與輕盈手感\nVolta 採用韓國尖端陶瓷燒結技術，刀刃硬度僅次於鑽石。它不僅擁有令人驚豔的切割力，同時重量極輕，長時間使用也不易感到疲勞。\n\n### 2. 🛡️ 食材原味守護者：零金屬殘留\n無論是切水果、生魚片或嬰兒副食品，Volta 陶瓷刀都能確保食材的純淨度。告別傳統鋼刀帶來的鐵鏽味或金屬異味，讓您的味蕾享受最真實的美味。\n\n### 3. 💧 抗菌防污：清潔保養超簡單\n陶瓷表面光滑細密，不滲透、不吸水，只需清水沖洗即可輕鬆去除油污和異味。其抗腐蝕特性，讓您無需擔心酸性食物（如檸檬、番茄）造成的損害。\n\n### 4. 🇰🇷 韓國製造，品質保證與終身支援\n堅持韓國原廠製造，確保每一把刀具都符合最高標準。Volta 承諾提供**終身技術支援**，展現我們對產品品質的絕對信心與對客戶的長久承諾。\n\n---\n\n## 📦 規格詳情：一組滿足所有需求\n\n本套組包含三把最常用刀型，滿足您日常料理的各種需求：\n\n| 刀具名稱 | 適用情境 |\n| :--- | :--- |\n| **主廚刀 (Chef Knife)** | 切割肉類、大塊蔬果、剁碎香料 |\n| **萬用刀 (Utility Knife)** | 處理中型食材、去皮、切片、切丁 |\n| **水果刀 (Paring Knife)** | 精細處理、削皮、去核、切割小型水果 |\n\n| 產品資訊 | 說明 |\n| :--- | :--- |\n| **商品名稱** | Volta 陶瓷刀具組 3件 |\n| **品牌** | Volta |\n| **分類** | 生活 / 廚具 |\n| **主要規格** | 3件套組 |\n| **型態** | 食品級不鏽鋼 (刀柄) + 高密度奈米陶瓷 (刀刃) |\n| **產地** | 韓國 (Made in Korea) |\n| **售價** | NT$3290 |\n| **保固** | 終身技術支援 |\n\n---\n\n## 🛒 立即行動：升級您的廚房裝備！\n\n**別再忍受鈍刀帶來的料理挫折！**\n\n現在入手 Volta 陶瓷刀具組，您投資的不僅是一套頂級廚具，更是一種高效、健康、愉悅的料理生活方式。\n\n**限時優惠！NT$3290 即可將韓國頂級工藝帶回家。**\n\n**👉 點擊「立即購買」，開啟您的精準料理新時代！**", "input_tokens": 90, "output_tokens": 1089}, "ce112daab24006612d8e": {"text": "{\n  \"title\": \"Volta 陶瓷刀具組 3件 輕量化高硬度刀刃 適用小家庭精準備料\",\n  \"bullets\": [\n    \"高密度陶瓷刀刃，提供持久鋒利度與精準切割。\",\n    \"陶瓷材質不殘留異味，生熟食分切衛生，適合寵物鮮食。\",\n    \"3件式精簡配置，有效節省小坪數廚房收納空間。\",\n    \"食品級不鏽鋼刀柄，抗鏽蝕，韓國製造品質保證。\"\n  ],\n  \"seo_description\": \"Volta 陶瓷刀具組 3件，專為小坪數家庭與寵物族群設計。高硬度陶瓷刀刃輕巧易握，精準切割食材，不殘留氣味。韓國製造，提供終身技術支援。\",\n  \"hashtags\": [\n    \"陶瓷刀\",\n    \"小坪數廚房\",\n    \"寵物鮮食\",\n    \"Volta\",\n    \"3件組\"\n  ]\n}", "input_tokens": 747, "output_tokens": 240}, "f3f7617a4ab5f193d3b3": {"text": "## 溯源焙所 | 頂級深海魚油軟膠囊 | 高純度 Omega-3 800mg | 輕巧20粒裝 | 打造紅潤好氣色\n\n---\n\n### 賣點\n\n1.  **高效有感：** 單顆提供高達 **Omega-3 800mg**，高純度吸收。\n2.  **循環順暢：** 深入調理體質，告別疲憊，維持青春美麗。\n3.  **輕巧包裝：** 韓國原裝進口，獨立密封，**20粒**確保新鮮度。\n4.  **夜間保養：** 建議睡前食用，把握黃金修護期，由內而外散發光采。\n\n---\n\n### SEO 描述\n\n專為追求內外兼修的熟齡女性設計。溯源焙所深海魚油，每份提供高達 **Omega-3 800mg**，幫助您維持思緒清晰與健康美麗。採用獨立包裝，輕巧一盒**20粒**，方便攜帶，確保魚油新鮮度。每日睡前一顆，輕鬆守護心血管健康，讓氣色紅潤，由內而外散發自信光采。\n\n---\n\n### 標籤\n\n魚油、Omega3、循環保健、美容保養、溯源焙所", "input_tokens": 259, "output_tokens": 305}, "2664a499dc74b671ae87": {"text": "**商品標題**\n【青研CHINGYEN】成分黨必收！高純度氧化鋅礦物防曬乳 50ml x 3入組 敏弱肌安心防護\n\n**賣點**\n\n1.  高純度氧化鋅，物理防曬零負擔。\n2.  敏弱肌專用配方，溫和不刺激。\n3.  超值 50ml x 3入組，全身防護不心疼。\n4.  韓國製造技術，清爽服貼不泛白。\n\n**SEO 描述**\n青研CHINGYEN 礦物防曬乳，專為敏弱肌與成分黨打造。採用高規格氧化鋅，提供溫和高效的物理性防護。無添加刺激物，質地清爽不黏膩。超值 50ml 3入組，讓您安心度過每個季節。韓國原裝進口，NT$390 即可擁有安心防曬。\n\n**標籤**\n#氧化鋅 #礦物防曬 #敏弱肌 #成分黨 #物理防曬", "input_tokens": 254, "output_tokens": 244}, "de765e6f88c1dd6ff860": {"text": "{\n  \"title\": \"溯源焙所 深海魚油軟膠囊 800mg高濃度Omega-3 韓國原裝20粒\",\n  \"bullets\": [\n    \"每粒含高濃度Omega-3 800mg。\",\n    \"韓國原裝進口，深海魚油純淨來源。\",\n    \"獨立封裝20粒，方便攜帶與每日食用。\",\n    \"睡前補充，幫助維持順暢循環與健康基底。\"\n  ],\n  \"seo_description\": \"溯源焙所深海魚油軟膠囊，專為注重外在保養的女性設計。每粒提供800mg高濃度Omega-3，來自韓國純淨深海。獨立20粒包裝，方便睡前補充，幫助您維持健康基底，由內而外散發光采。\",\n  \"hashtags\": [\n    \"魚油\",\n    \"Omega3\",\n    \"循環保健\",\n    \"溯源焙所\",\n    \"美容保養\"\n  ]\n}", "input_tokens": 910, "output_tokens": 237}, "16c998bb72362a688302": {"text": "## 溯源焙所 深海魚油軟膠囊 文案\n\n---\n\n### 商品標題\n\n**溯源焙所 高濃度深海魚油軟膠囊 800mg Omega-3 獨立包裝 20粒**\n\n### 賣點 (4 條，每條不超過 30 個字)\n\n1.  高規格 Omega-3，每份濃度達 **800mg**。\n2.  嚴選深海魚油，純淨無負擔。\n3.  獨立 PTP 包裝，保鮮易攜帶。\n4.  睡前補充，維持內在循環順暢。\n\n### SEO 描述 (不超過 120 個字)\n\n溯源焙所為注重內在保養的您，提供高規格深海魚油。嚴選純淨 **Omega-3**，每份濃度達 **800mg**。韓國製造，獨立 PTP 包裝，一盒 **20粒**。每日睡前輕鬆補充，協助您維持健康基礎，展現自然光采。\n\n### 標籤 (3-5 個)\n\n魚油, Omega3, 循環保健, 溯源焙所, 高濃度", "input_tokens": 803, "output_tokens": 266}, "75ef73acb98f23449290": {"text": "這是一份針對「溯源焙所 低溫烘焙堅果」的電商文案，強調其高品質、獨特烘焙技術、以及方便的「睡前滋養」定位。\n\n---\n\n# 溯源焙所 低溫烘焙堅果 (350g | 20粒裝)\n\n## 💎 主標題：【夜間修護級堅果】告別傳統零食！睡前黃金一小時，啟動深層滋養，迎接高效晨光。\n\n---\n\n## 🌟 前言：獻給追求極致健康的您\n\n您是否在尋找一種既能滿足口腹之慾，又能真正為身體帶來益處的「機能性零食」？\n\n「溯源焙所」耗時研發，將頂級綜合堅果以獨家**低溫烘焙**技術鎖住天然營養，並採用**獨立植物膠囊包裝**，為忙碌的現代人提供最純粹、最方便的夜間滋養方案。這不僅是堅果，更是您投資健康、提升睡眠品質的奢華選擇。\n\n---\n\n## ✨ 三大核心賣點：為什麼選擇溯源焙所？\n\n### 1. 【極致營養鎖定】低溫烘焙的堅持\n\n傳統高溫烘烤會破壞堅果中珍貴的不飽和脂肪酸和維生素。我們堅持採用**耗時費工的低溫烘焙技術**，溫和喚醒堅果的生命力，完整保留其天然酵素與營養價值。\n\n*   **口感升級：** 保留堅果的清脆與原始香氣，無油耗味。\n*   **營養最大化：** 維持Omega-3、維生素E、膳食纖維等活性成分。\n\n### 2. 【機能性定位】睡前一粒，夜間修護\n\n我們建議在**睡前食用一粒**。堅果提供的優質脂肪和微量元素，能在您進入睡眠狀態時，為身體提供穩定且緩慢釋放的能量與營養，幫助身體在夜間進行高效修護。\n\n*   **告別空虛感：** 睡前輕盈補充，避免夜間飢餓影響睡眠。\n*   **優化體內環境：** 讓身體在休息時也能持續獲得高品質的滋養。\n\n### 3. 【頂級便利設計】獨立植物膠囊包裝 (20粒/盒)\n\n告別大包裝堅果受潮、氧化、難以控制份量的困擾。\n\n每一粒「溯源焙所」堅果都採用**獨立密封的植物膠囊包裝**，確保您吃到的每一口都如同剛出爐般新鮮。\n\n*   **精準份量控制：** 每日一粒，不多不少，輕鬆達成健康目標。\n*   **隨身攜帶：** 辦公室、健身房、旅行，隨時隨地享受頂級堅果。\n*   **環保安心：** 採用植物性包材，對環境更友善。\n\n---\n\n## 🌰 嚴選規格與安心承諾\n\n「溯源焙所」嚴格追溯原料產地，確保每一顆堅果都符合最高品質標準。\n\n*   **品名：** 溯源焙所 低溫烘焙綜合堅果\n*   **規格：** 350g (共20粒獨立膠囊包裝)\n*   **產地：** 台灣 (專業低溫烘焙與包裝)\n*   **成分：** 嚴選頂級綜合堅果（含杏仁、核桃、腰果等）\n*   **建議食用：** 每日 1 份（1粒），建議於**睡前 30-60 分鐘**食用。\n\n---\n\n## 🛒 奢華投資，值得擁有\n\n健康是無法妥協的奢侈品。\n\n「溯源焙所 低溫烘焙堅果」以其獨特的工藝、頂級的品質和精緻的包裝，重新定義了健康零食的標準。\n\n**原價：NT$2,290**\n\n**現在就將這份極致的夜間滋養，帶入您的健康生活儀式中。**\n\n**👉 立即下單，體驗從內而外的深層滋養！**", "input_tokens": 118, "output_tokens": 955}, "3fff81d49aa2cedae490": {"text": "這是一份針對高階市場、強調「夜間修復」與「緩釋科技」的電商文案。\n\n---\n\n## 【夜間修復，晨間煥活】\n**晨光研選 緩釋B群錠 75mg：從睡夢中啟動你的高效能模式！**\n\n**售價：NT$2,290**\n\n---\n\n### 💡 你是否正深陷「疲勞循環」？\n\n在高壓生活中，你是否經常面臨以下困境？\n\n*   早上鬧鐘響起，身體卻像沒充飽電一樣沉重？\n*   午餐後精神崩潰，必須靠咖啡或甜食硬撐？\n*   即使睡滿八小時，隔天依然感到疲憊，無法深度修復？\n\n傳統B群追求「立即有感」，但高劑量往往快速代謝，導致下午電力耗盡。**晨光研選** 顛覆傳統，將B群的功效延伸至夜間，讓你在睡眠中完成深層修復，醒來迎接持續一整天的穩定能量。\n\n---\n\n### 🌟 晨光研選：緩釋科技的四大革新優勢\n\n我們深知，高效能人士需要的不是短暫的刺激，而是穩定的續航力。晨光研選緩釋B群，正是為此而生。\n\n#### 1. 【夜間修復，緩釋啟動】\n**顛覆傳統：睡前食用，效果更持久。**\n本產品採用先進的「緩釋科技」，讓高達 **75mg** 的維生素B群複方，在您入睡後緩慢、穩定地釋放。B群在夜間參與身體的代謝與修復過程，確保細胞在休息時獲得充足養分，真正達到「睡飽」的效果。\n\n#### 2. 【高規複方，穩定續航】\n高劑量 75mg 複方設計，確保身體在緩釋過程中能持續吸收足夠的活性成分。告別傳統B群帶來的短暫興奮感，提供從早到晚的穩定精神與體力支持。\n\n#### 3. 【日本製造，品質保證】\n源自對細節與品質的極致追求。本產品於 **日本** 嚴格製造，採用頂級原料，為您提供最高標準的保健承諾。\n\n#### 4. 【獨立包裝，新鮮鎖定】\n每一份 B群皆採獨立鋁箔包裝，有效隔絕濕氣與光線，確保 B群活性不流失。方便攜帶，無論出差或旅行，隨時隨地維持最佳狀態。\n\n---\n\n### 🔬 為什麼選擇「睡前」食用？\n\n維生素B群除了提供能量外，也參與神經系統的穩定與修復。晨光研選的緩釋設計，讓B群在夜間持續作用：\n\n*   **優化代謝：** 協助身體在休息時更有效率地代謝壓力物質。\n*   **深層滋養：** 穩定神經系統，幫助您獲得更高品質的睡眠。\n*   **預防崩潰：** 確保隔天早晨醒來時，體內B群濃度已達高峰，有效預防下午的能量低谷。\n\n---\n\n### 🛒 立即投資你的高效人生\n\n**晨光研選 緩釋B群錠** 不只是一瓶維他命，更是您對抗高壓生活、追求卓越表現的頂級投資。\n\n**建議食用方式：** 每日1份 (1粒)，**睡前** 搭配開水食用。\n\n**規格資訊：**\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **商品名稱** | 晨光研選 緩釋B群錠 |\n| **分類** | 保健食品 / 精神體力 |\n| **主要規格** | 維生素B群複方 75mg |\n| **產地** | 日本 |\n| **型態** | 獨立包裝錠劑 |\n| **包裝** | 20粒 / 盒 (約20天份) |\n| **售價** | NT$2,290 |\n\n**【告別疲憊，迎接晨光】立即體驗日本高規緩釋B群，讓你的每一天都從深度修復開始。**", "input_tokens": 121, "output_tokens": 965}, "6f85fa6308e7af4f7d8a": {"text": "## 青研 CHINGYEN 礦物防曬乳 50ml 3入\n\n### 商品標題\n青研 CHINGYEN 礦物防曬 溫和純淨配方 氧化鋅物理防曬 敏弱肌適用 50ml 3入組\n\n### 賣點\n1.  採用高純度**氧化鋅**，給予肌膚溫和保護。\n2.  **油狀質地**輕盈好推，膚感清爽不黏膩。\n3.  專為**敏弱肌**設計，成分單純透明。\n4.  囤貨必備**50ml 3入**組，持續防護更划算。\n\n### SEO 描述\n青研 CHINGYEN 礦物防曬乳，專為敏弱肌設計的溫和物理防曬。核心成分高純度**氧化鋅**，提供穩定且全面的防護。**油狀質地**輕盈不厚重，不造成肌膚負擔。韓國製造，安心可靠。超值**50ml 3入**組，讓成分黨安心囤貨，享受清爽無負擔的日常防護。\n\n### 標籤\n物理防曬、氧化鋅、敏弱肌、成分黨、韓國美妝", "input_tokens": 809, "output_tokens": 282}, "142329a387eca8decda5": {"text": "{\n  \"title\": \"Volta USB-C 編織傳輸線 240W：高功率快充，差旅多裝置供電首選。\",\n  \"bullets\": [\n    \"支援 USB-C PD 3.1 240W 高功率輸出。\",\n    \"食品級不鏽鋼接頭，抗插拔耐用。\",\n    \"差旅必備，一線滿足多裝置快充。\",\n    \"高密度編織線材，享終身技術支援。\"\n  ],\n  \"seo_description\": \"Volta USB-C 240W 編織線，專為差旅與多裝置使用者設計。支援 240W 高速電力傳輸，快速滿足筆電、平板等設備的快充需求。食品級不鏽鋼接頭與高密度編織，確保線材耐用可靠，提供終身技術支援。\",\n  \"hashtags\": [\n    \"USBC\",\n    \"240W快充\",\n    \"差旅必備\",\n    \"編織線\",\n    \"PD31\"\n  ]\n}", "input_tokens": 756, "output_tokens": 249}, "a850edfe0fa5aa6c3eec": {"text": "好的，這是一個高規格的 USB-C 傳輸線，文案需要強調其極致的充電速度、耐用性以及品牌提供的優質服務。\n\n以下為您準備的電商文案，包含標題、痛點解決、產品特色、規格說明與行動呼籲。\n\n---\n\n## ⚡ Volta USB-C 編織傳輸線 240W ⚡\n\n### 標題：【極速充電新標準】Volta 240W USB-C 編織線 — 告別龜速，迎接全速高效能！\n\n---\n\n### 一、痛點與解決方案（開場白）\n\n**您是否厭倦了？**\n\n* ❌ 手機、筆電充電慢如蝸牛，等待時間漫長？\n* ❌ 買了高功率充電頭，卻被低規格線材限制了速度？\n* ❌ 線材用沒多久就斷裂、破損，頻繁更換浪費錢？\n\n**Volta 240W USB-C 編織傳輸線，為您終結所有充電煩惱！**\n\n我們不僅提供「快」，更提供「耐用」與「安心」。這不只是一條線，更是您高效工作與娛樂的強力後盾。\n\n---\n\n### 二、核心賣點與產品特色\n\n#### 🥇 突破極限：240W 超高功率輸出\n\n* **超越傳統 100W 限制：** 支援最新的 USB Power Delivery 3.1 規範，提供驚人的 **240W (48V/5A)** 電力傳輸能力。\n* **全速支援：** 無論是 MacBook Pro 16 吋、高效能電競筆電，或是平板、手機，都能以最快速度充滿電，大幅縮短您的等待時間。\n\n#### 💎 頂級材質：食品級不鏽鋼接頭，日本精工\n\n* **極致耐用：** 接頭採用高成本的**食品級不鏽鋼**材質，抗腐蝕、抗氧化，插拔數萬次依然穩固如新。\n* **日本製造工藝：** 堅持日本產地製造，確保每一條線材都具備最高標準的品質與穩定性。\n\n#### 💪 強韌結構：高密度編織設計\n\n* **抗拉扯、防纏繞：** 採用高密度尼龍編織外被，有效抵抗日常使用中的彎折、拉扯，告別線材內部斷裂的困擾。\n* **美觀與實用兼具：** 優雅的編織紋理，手感舒適，不易纏繞打結，讓桌面保持整潔。\n\n#### 🛡️ 終身保障：Volta 的承諾\n\n* **終身技術支援：** Volta 對產品品質深具信心。我們提供**終身技術支援**，讓您買得安心，用得放心，真正實現一次購買，長久使用。\n\n---\n\n### 三、商品規格一覽\n\n| 規格項目 | 詳細說明 |\n| :--- | :--- |\n| **商品名稱** | Volta USB-C 編織傳輸線 240W |\n| **分類** | 3C / 線材 |\n| **主要規格** | **240W (48V/5A) 超高功率傳輸** |\n| **接頭材質** | 食品級不鏽鋼 |\n| **線材外被** | 高密度耐磨編織 |\n| **產地** | 日本 (Made in Japan) |\n| **保固服務** | 終身技術支援 |\n| **建議售價** | **NT$290** |\n\n---\n\n### 四、結語與行動呼籲\n\n**別再讓低規格線材拖垮您的效率！**\n\nVolta 240W USB-C 編織傳輸線，以 NT$290 的親民價格，為您帶來旗艦級的充電體驗、日本製造的頂級工藝，以及終身技術支援的安心保障。\n\n**立即升級您的充電裝備，享受全速高效生活！**\n\n👉 **[點擊此處，立即搶購 Volta 240W 極速傳輸線！]**", "input_tokens": 98, "output_tokens": 937}, "8b579bfe7bb15fca327e": {"text": "{\n\"title\": \"青研 CHINGYEN 礦物防曬乳 50ml 3入組 溫和氧化鋅配方 敏弱肌適用\",\n\"bullets\": [\n\"採用溫和氧化鋅，提供物理性防護。\",\n\"油狀質地，延展性佳，清爽不厚重。\",\n\"專為敏弱肌設計，成分單純透明。\",\n\"50ml 大容量 3入組，高CP值囤貨首選。\"\n],\n\"seo_description\": \"青研 CHINGYEN 礦物防曬乳，專為敏弱肌與成分黨設計。使用溫和氧化鋅配方，提供肌膚物理性防護。油狀質地輕盈好推開，不造成肌膚負擔。現在購買 50ml 3入組，享受高CP值清爽防護。\",\n\"hashtags\": [\n\"敏弱肌\",\n\"礦物防曬\",\n\"氧化鋅\",\n\"成分黨\",\n\"物理性防曬\"\n]\n}", "input_tokens": 916, "output_tokens": 235}, "3105d24312fd3152a446": {"text": "## Volta USB-C 編織傳輸線 240W\n\n---\n\n### 商品標題\n\n**Volta 240W 極速快充線：日本不鏽鋼編織 USB-C 傳輸線，筆電手機一線搞定**\n\n---\n\n### 賣點\n\n1. 業界頂規 **240W** 瓦數，極速滿電不等待。\n2. 日本食品級不鏽鋼編織，堅固耐用抗磨損。\n3. **USB-C** 萬用規格，筆電手機平板通用。\n4. 終身技術支援保固，差旅使用最安心。\n\n---\n\n### SEO 描述\n\n專為出差族設計的 Volta **USB-C** 編織傳輸線。支援 PD 3.1 協議，提供業界頂尖 **240W** 極速快充，輕鬆應對高耗能筆電與多裝置充電需求。採用日本食品級不鏽鋼材質，抗拉扯、高耐用，終身保固讓您差旅無憂。\n\n---\n\n### 標籤\n\n快充線, 240W, USB-C, 出差必備, 終身保固", "input_tokens": 227, "output_tokens": 271}, "369a130f5b3a7b563882": {"text": "這是一個高單價、高濃縮的保健食品文案。由於價格高昂（NT$1180/20粒），文案必須著重於「科研」、「濃縮精華」和「睡前黃金修護」來合理化其價值，而非單純的「零食」。\n\n以下提供三種不同風格的電商文案：\n\n---\n\n## 💎 頂級科研版 (主文案)\n\n### 標題：【晨光研選】夜間黃金修護｜凍乾果乾精華膠囊 50g (20粒) – 喚醒由內而外的晨光肌\n\n### 導言：告別無效保養，從睡眠開始逆轉時光\n\n您是否投入大量時間在外部保養，卻忽略了身體夜間的黃金修護期？\n「晨光研選」深知，真正的光采來自於細胞深層的滋養。我們不只是提供果乾，而是將整顆頂級水果的精華，透過尖端科技濃縮成一顆高效能的「夜間修護膠囊」。\n\nNT$1180，投資的是您每天醒來都能感受到的，由內而外的透亮與活力。\n\n---\n\n### 🌟 三大核心價值，定義頂級修護\n\n#### 1. 【極致濃縮：整顆凍乾精華】\n我們採用最先進的 **真空凍乾技術 (Freeze-Drying)**，在極低溫下瞬間鎖住水果的完整營養結構。與傳統高溫烘乾相比，凍乾技術能保留高達98%的活性成分，確保您攝取到的每一份都是水果的巔峰能量。50g 的精華，超越數倍新鮮水果的營養價值。\n\n#### 2. 【美國科研標準：純淨與安心】\n本產品源自美國頂級原料供應鏈，從選材到製程，皆遵循嚴苛的國際食品安全標準。我們將高純度的凍乾果乾精華，精準封存於軟膠囊中，杜絕氧化，確保每一粒都擁有穩定的高效能。\n\n#### 3. 【睡前儀式：啟動夜間黃金修護】\n為什麼選擇睡前食用？夜晚是細胞修復與代謝的關鍵時刻。\n「晨光研選 凍乾果乾精華」專為夜間設計，在您進入深層睡眠時，高效成分開始作用，幫助身體進行深層淨化與滋養，讓您告別疲憊暗沉，一覺醒來，迎接充滿「晨光」般的透亮與活力。\n\n---\n\n### 💡 誰需要「晨光研選」？\n\n*   追求高效、極簡保養的忙碌人士。\n*   希望改善氣色，從內部調理的愛美族群。\n*   重視睡眠品質與夜間修護的健康主義者。\n*   尋求高純度、高規格天然營養補充品者。\n\n### 📋 產品規格與食用建議\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **品牌** | 晨光研選 |\n| **商品名稱** | 凍乾果乾精華膠囊 |\n| **規格** | 50g / 20粒 / 盒 |\n| **型態** | 軟膠囊 (高濃縮型態) |\n| **產地** | 美國 (頂級原料) |\n| **售價** | NT$1180 |\n| **建議食用** | 每日1份 (1粒)，**睡前30分鐘** 搭配溫水食用。 |\n\n---\n\n## 🚀 簡潔高效版 (社群/廣告用)\n\n### 標題：【一顆抵十顆】睡前吃的「晨光膠囊」！高濃度凍乾果乾精華，醒來透亮。\n\n**你還在吃零食果乾？我們已經升級到「精華膠囊」！**\n\n「晨光研選 凍乾果乾精華」不是零食，是專為現代人設計的 **夜間高效能營養補給**。\n\n✅ **超高濃度：** 採用美國頂級凍乾技術，將整顆水果精華濃縮至一顆軟膠囊，營養保留度高達98%。\n✅ **夜間修護：** 建議睡前食用，利用睡眠黃金期，啟動深層代謝與滋養。\n✅ **極簡高效：** 每日只需一顆，輕鬆完成頂級保養。\n\n**NT$1180 頂級修護，20天見證改變。**\n👉 立即體驗，讓您的肌膚每天都像被晨光喚醒！\n\n---\n\n## 📝 產品特色列表 (電商頁面資訊欄)\n\n### 產品名稱：晨光研選 凍乾果乾精華膠囊 (50g/20粒)\n\n| 特色分類 | 詳細說明 |\n| :--- | :--- |\n| **產品定位** | 頂級濃縮型態的夜間修護營養補充品。 |\n| **核心技術** | 美國頂級真空凍乾技術 (Freeze-Drying)，最大化保留水果活性營養。 |\n| **成分純度** | 50g 高純度整顆凍乾水果精華，無多餘添加。 |\n| **型態優勢** | 軟膠囊封存，隔絕光線與空氣，確保成分穩定與吸收效率。 |\n| **食用時機** | **建議睡前食用**，配合身體夜間代謝，達到最佳修護效果。 |\n| **適用對象** | 追求高效抗氧化、希望改善睡眠品質與氣色者。 |\n| **產地保證** | 美國原裝進口原料，品質嚴格把關。 |\n| **包裝規格** | 20粒/盒 (約20天份)。 |\n| **售價** | NT$1180。 |", "input_tokens": 117, "output_tokens": 1344}, "7fafb9fdba64102f51a3": {"text": "這是一份針對「溯源焙所 深海魚油軟膠囊」設計的電商文案，強調其高濃度、韓國製造的品質，以及方便攜帶的特性。\n\n---\n\n# 【溯源焙所】高濃度深海魚油軟膠囊 (800mg Omega-3)\n\n**淨化循環 | 思緒清晰 | 韓國製造 | 獨立保鮮包裝**\n\n💰 **限時體驗價：NT$390** (20粒/盒)\n\n---\n\n## 💡 您的身體，需要頂級的「好油」來啟動！\n\n您是否經常外食、飲食不均衡，或感到思緒卡頓、循環不順暢？現代人生活節奏快，體內壞油堆積，更需要高純度的Omega-3來幫助身體進行大掃除。\n\n**「溯源焙所 深海魚油軟膠囊」** 專為追求高效能、高品質的您設計，提供每日所需的黃金比例，讓您輕鬆維持健康循環，重啟清晰思緒。\n\n---\n\n## ✨ 為什麼選擇「溯源焙所」高濃度魚油？三大核心優勢\n\n### 1. 頂級濃度，高效有感 (Omega-3 800mg)\n\n我們堅持高標準，每份軟膠囊提供 **高達800mg的Omega-3**，其中包含關鍵的EPA和DHA。高濃度意味著您不需要吞食大量膠囊，就能達到有效的保健劑量，效率加倍，負擔更少。\n\n### 2. 韓國製造，品質溯源保證\n\n魚油的純淨度是關鍵！本產品堅持在保健品製造業標準嚴苛的 **韓國原裝生產**。從深海魚油的採集到製程，層層把關，確保魚油的純淨度與新鮮度，讓您吃得安心，真正達到「溯源」的承諾。\n\n### 3. 獨立包裝，新鮮度完美鎖定\n\n魚油最怕氧化！我們採用 **獨立鋁箔包裝 (20粒/盒)**，有效隔絕空氣、濕氣與光線，確保每一粒魚油都保持在最佳的新鮮狀態。無論是辦公室、外出旅行，隨身攜帶都極為方便。\n\n---\n\n## 🎯 誰最需要「溯源焙所」深海魚油？\n\n*   **循環保健者：** 關注心血管健康，希望維持順暢循環。\n*   **長時間用腦者：** 學生、上班族，需要提升專注力與記憶力。\n*   **外食族/高油脂飲食者：** 飲食不均衡，想補充優質的Omega-3脂肪酸。\n*   **銀髮族群：** 幫助維持行動力與整體健康。\n\n---\n\n## 🌙 建議食用方式 (睡前黃金時段)\n\n**【每日1份，睡前食用】**\n\n建議在睡前搭配開水食用。利用夜間休息時間，讓身體更好地吸收與運用Omega-3，為隔天的活力做好準備。\n\n---\n\n## 📋 商品規格與承諾\n\n| 項目 | 規格說明 |\n| :--- | :--- |\n| **商品名稱** | 溯源焙所 深海魚油軟膠囊 |\n| **主要規格** | Omega-3 800mg (高濃度) |\n| **包裝規格** | 20粒 / 盒 (獨立包裝) |\n| **產地** | 韓國 (原裝進口) |\n| **分類** | 保健食品 / 循環保健 |\n| **售價** | NT$390 |\n\n### 🛒 立即體驗高純淨、高效率的深海魚油！\n\n**別讓壞油阻礙您的健康之路。現在就將【溯源焙所】深海魚油加入購物車，給自己一個更順暢、更清晰的明天！**\n\n**👉 點擊購買，開啟您的健康循環之旅！**", "input_tokens": 119, "output_tokens": 895}, "03aa4dd4f874711e1f97": {"text": "## 晨光研選 複方益生菌粉\n\n### 商品標題 (不超過 60 個字)\n\n**晨光研選 專利益生菌 熟齡順暢配方 100億CFU 植物膠囊 60粒**\n\n---\n\n### 賣點 (4 條，每條不超過 30 個字)\n\n1.  嚴選高活性**專利益生菌**，調整體質有感。\n2.  足量添加，每份提供高達**100億CFU**。\n3.  睡前一顆溫和保養，幫助消化道順暢。\n4.  全素植物膠囊設計，一盒**60粒**，輕鬆維持。\n\n---\n\n### SEO 描述 (不超過 120 個字)\n\n熟齡保養首選！晨光研選複方益生菌，採用高活性**專利益生菌**，每份提供**100億CFU**，幫助您調整體質、維持消化道機能。全素植物膠囊，溫和不刺激。建議睡前食用，一盒**60粒**，讓您每天輕盈有感，告別卡關人生。\n\n---\n\n### 標籤 (3-5 個)\n\n#益生菌 #熟齡保養 #消化道順暢 #專利益生菌 #植物膠囊", "input_tokens": 266, "output_tokens": 308}, "800005fe2272e71496e6": {"text": "## 青研 CHINGYEN 礦物防曬乳\n\n---\n\n### 商品標題\n**青研 CHINGYEN 德國原裝 礦物防曬乳 60ml | 高效氧化鋅物理防護 | 敏弱肌專用**\n\n---\n\n### 賣點\n1. 高效氧化鋅，物理性溫和阻隔。\n2. 敏弱肌適用，無負擔零刺激配方。\n3. 輕盈油狀質地，保濕不黏膩。\n4. 德國原裝進口，規格 60ml 1入。\n\n---\n\n### SEO 描述\n專為30-45歲重視溫和保濕的您設計。青研 CHINGYEN 德國礦物防曬乳，採用高純度**氧化鋅**，提供高效物理防護，不刺激肌膚。輕盈油狀質地，長效鎖水。規格為 **60ml 1入**，是敏弱肌日常防曬的最佳選擇。\n\n---\n\n### 標籤\n#礦物防曬\n#氧化鋅\n#敏弱肌\n#溫和保濕\n#德國製造", "input_tokens": 257, "output_tokens": 257}, "4fc175692458f60d4e14": {"text": "{\n  \"title\": \"Volta 不鏽鋼保溫瓶 350ml 台灣製造 差旅通勤高效保溫解決方案\",\n  \"bullets\": [\n    \"容量 350ml，輕量化設計，不佔用行李空間。\",\n    \"高效能不鏽鋼內膽，提供穩定保溫效果。\",\n    \"台灣製造，品質可靠，原廠提供一年保固。\",\n    \"PC+ABS 複合材質外殼，結構堅固耐用。\"\n  ],\n  \"seo_description\": \"Volta 350ml 不鏽鋼保溫瓶專為商務人士與高效率使用者設計。輕巧容量，隨身攜帶無負擔。採用優異保溫技術，確保飲品溫度持久穩定。台灣製造，複合材質結構，耐用可靠。是您差旅、辦公、通勤時的最佳保溫夥伴。\",\n  \"hashtags\": [\n    \"Volta\",\n    \"保溫瓶\",\n    \"350ml\",\n    \"出差必備\",\n    \"台灣製造\"\n  ]\n}", "input_tokens": 750, "output_tokens": 245}, "b74d62acf2fb1dbba67d": {"text": "## 本木 BENMU 不鏽鋼保溫瓶 750ml\n\n### 商品標題\n**本木 BENMU 德國工藝 750ml 不鏽鋼保溫瓶｜食品級材質，長效保溫，耐用設計，小家庭與寵物外出首選。**\n\n---\n\n### 賣點\n1. 德國食品級不鏽鋼，材質堅固耐用。\n2. 恰到好處的 **750ml**，滿足全日飲水需求。\n3. 高效真空層結構，提供穩定**保溫**效果。\n4. 德國嚴謹工藝，提供終身技術支援。\n\n---\n\n### SEO 描述\n本木 BENMU 750ml 保溫瓶，採用德國食品級不鏽鋼，結構紮實耐用。對於空間有限的小坪數家庭，或需要頻繁外出的養寵物族群，提供穩定長效的保溫性能。750ml 容量適中，輕鬆攜帶，是您追求樸實高品質生活的理想選擇。\n\n---\n\n### 標籤\n#本木BENMU\n#保溫瓶\n#750ml\n#小家庭必備\n#寵物外出", "input_tokens": 509, "output_tokens": 267}, "afb947b9cbdb74c8578e": {"text": "{\n  \"title\": \"晨光研選 專利益生菌 植物膠囊 100億CFU 60粒 熟齡日常保養首選\",\n  \"bullets\": [\n    \"足量添加 100億CFU 專利益生菌\",\n    \"每日睡前一顆 輕鬆維持消化道機能\",\n    \"適合中壯年族群 溫和持續補充\",\n    \"嚴選植物膠囊 60粒裝方便食用\"\n  ],\n  \"seo_description\": \"晨光研選專為熟齡族群設計的日常保健品。每份提供足量 100億CFU 專利益生菌，幫助維持消化道健康平衡。採用植物膠囊，一盒 60粒，方便每日睡前定時補充。溫和配方，支持您持續的健康基礎。\",\n  \"hashtags\": [\n    \"益生菌\",\n    \"晨光研選\",\n    \"消化道保健\",\n    \"熟齡保養\",\n    \"專利益生菌\"\n  ]\n}", "input_tokens": 910, "output_tokens": 243}, "e329b173d79fe4661218": {"text": "{\n  \"title\": \"青研 CHINGYEN 礦物防曬乳 60ml 1入 溫和配方 敏弱肌適用 含氧化鋅\",\n  \"bullets\": [\n    \"高純度氧化鋅，提供溫和物理性防護。\",\n    \"專為敏弱肌設計，配方單純不易刺激。\",\n    \"油狀質地好推勻，肌膚感受輕盈不厚重。\",\n    \"德國原裝進口 60ml 規格，安心防曬必備。\"\n  ],\n  \"seo_description\": \"青研 CHINGYEN 礦物防曬乳 60ml 1入，採用高規格氧化鋅，為肌膚提供穩定防護。溫和配方專為敏弱肌膚設計，質地清爽好延展。無添加爭議成分，讓您在日常防護中也能維持肌膚舒適與水潤感。\",\n  \"hashtags\": [\n    \"青研CHINGYEN\",\n    \"礦物防曬\",\n    \"氧化鋅防曬\",\n    \"敏弱肌適用\",\n    \"溫和配方\"\n  ]\n}", "input_tokens": 919, "output_tokens": 263}, "157b7c36d8fcea53b5a5": {"text": "{\n  \"title\": \"白日光 SUNDAY 舒緩精華液 10% 積雪草萃取 | 輕盈凝露質地 1入 | 混合肌上班族必備\",\n  \"bullets\": [\n    \"高濃度 10% 積雪草萃取，溫和穩定肌膚。\",\n    \"清爽凝露質地，秒吸收不黏膩，混合肌最愛。\",\n    \"告別肌膚小情緒，維持透亮好狀態。\",\n    \"輕巧 1入 包裝，日常或旅行保養都方便。\"\n  ],\n  \"seo_description\": \"混合肌上班族救星！白日光 SUNDAY 舒緩精華液，專為敏弱肌設計。蘊含高達 10% 積雪草萃取，輕盈凝露質地，快速滲透，不給肌膚負擔。單入包裝 (1入) 方便攜帶，隨時穩定膚況，輕鬆擁有透亮好氣色。\",\n  \"hashtags\": [\n    \"積雪草\",\n    \"混合肌保養\",\n    \"舒緩精華\",\n    \"輕盈質地\",\n    \"上班族保養\"\n  ]\n}", "input_tokens": 780, "output_tokens": 275}, "18c0dbf73b6cc1520961": {"text": "## 晨光研選 複方益生菌粉\n\n---\n\n### 商品標題\n\n**晨光研選 專利益生菌 植物膠囊 每日補充 100億CFU 60粒**\n\n---\n\n### 賣點\n\n1.  嚴選專利益生菌配方，穩定性高。\n2.  每份足量添加 100億CFU 活性菌。\n3.  幫助維持消化道機能平衡。\n4.  植物膠囊，溫和不刺激，睡前食用。\n\n---\n\n### SEO 描述\n\n40歲以上族群的日常保健首選。晨光研選複方益生菌，採用專利益生菌配方，每份提供100億CFU高活性菌數，幫助維持消化道健康。植物膠囊，一盒60粒，溫和穩定，適合長期持續補充，為您的健康打底。\n\n---\n\n### 標籤\n\n益生菌、消化保健、日常保養、專利配方、40歲以上", "input_tokens": 803, "output_tokens": 232}, "bb3eab381547c2e6f72d": {"text": "## Volta 不鏽鋼保溫瓶 350ml\n\n### 商品標題\n**Volta 輕量不鏽鋼保溫瓶 350ml | 商務差旅、多工辦公必備隨身保溫杯**\n\n---\n\n### 賣點 (4 條)\n1.  350ml 精巧容量，輕鬆收納不佔位。\n2.  長效保溫鎖溫，隨時享受最佳飲品溫度。\n3.  台灣製造，複合材質輕量設計，堅固耐用。\n4.  多裝置使用者適用，隨行咖啡、茶飲的最佳選擇。\n\n---\n\n### SEO 描述\n專為商務人士與多裝置使用者設計的 Volta 不鏽鋼保溫瓶，擁有輕巧的 **350ml** 容量，輕鬆放入公事包。採用 PC+ABS 複合材質，堅固耐用。提供長效**保溫**效果，讓您在忙碌的出差或辦公中，隨時享用溫熱飲品。台灣製造，品質保證。\n\n---\n\n### 標籤 (5 個)\n保溫瓶, 350ml, 出差必備, 輕量, 隨行杯", "input_tokens": 221, "output_tokens": 270}, "ee341551b6ad34a5c2dd": {"text": "## 白日光 SUNDAY 舒緩精華液\n\n---\n\n### 商品標題\n\n**白日光 SUNDAY 積雪草萃取 10% 舒緩精華：清爽凝露質地，告別辦公室疲憊肌** (50 字)\n\n---\n\n### 賣點\n\n1.  高濃度積雪草萃取 10%，高效舒緩穩定膚況。 (24 字)\n2.  輕盈凝露質地，混合肌秒吸收不黏膩、不悶痘。 (21 字)\n3.  強化肌膚屏障，對抗環境壓力與敏弱不適。 (18 字)\n4.  美國原裝進口，單盒 1入 體驗價 NT$290。 (21 字)\n\n---\n\n### SEO 描述\n\n專為台灣混合肌設計！白日光 SUNDAY 舒緩精華液，添加高達 **10% 積雪草萃取**，有效鎮定修護，緩解壓力肌困擾。清爽凝露質地，快速吸收不悶痘。美國原裝進口，單盒 **1入** 體驗價，讓您輕鬆告別敏弱不適，重現健康光澤。 (109 字)\n\n---\n\n### 標籤\n\n\\#積雪草精華 \\#混合肌救星 \\#清爽不黏膩 \\#敏弱肌適用 \\#辦公室保養", "input_tokens": 257, "output_tokens": 316}, "45cbe5ef463d76e89564": {"text": "{\n  \"title\": \"本木 BENMU 德國工藝食品級不鏽鋼保溫瓶 750ml | 樸實耐用，小坪數家庭與寵物族群的安心選擇。\",\n  \"bullets\": [\n    \"採用食品級不鏽鋼，堅固耐用，不怕寵物碰撞。\",\n    \"實用 750ml 容量，滿足日常飲水需求，輕巧不佔空間。\",\n    \"德國工藝長效保溫，隨時享受溫熱或冰涼飲品。\",\n    \"樸實設計，終身技術支援，體現對品質的承諾。\"\n  ],\n  \"seo_description\": \"本木 BENMU 不鏽鋼保溫瓶 750ml，專為小坪數家庭與寵物族群設計。採用食品級不鏽鋼，耐用抗摔，清潔容易。德國工藝提供可靠的保溫效果，確保飲品溫度持久。樸實設計，實用主義者的最佳選擇。\",\n  \"hashtags\": [\n    \"本木BENMU\",\n    \"不鏽鋼保溫瓶\",\n    \"750ml\",\n    \"小坪數生活\",\n    \"耐用設計\"\n  ]\n}", "input_tokens": 616, "output_tokens": 275}, "a70430fc5bda7e46ed4d": {"text": "## 青研 CHINGYEN 礦物防曬乳 60ml 敏弱肌適用 溫和配方\n\n---\n\n### 賣點\n1. 採用高純度**氧化鋅**，提供溫和防護。\n2. 質地滋潤服貼，為肌膚帶來日常保濕。\n3. 德國原裝進口，專為敏弱肌膚設計。\n4. 標準規格**60ml**，單盒包裝為**1入**。\n\n---\n\n### SEO 描述\n30-45歲重視保濕與溫和配方的族群首選。青研CHINGYEN礦物防曬乳，採用高純度**氧化鋅**，提供物理性防護。雖然是油狀質地，但膚感滋潤不厚重，適合敏弱肌日常使用。德國原裝進口，規格為**60ml**，每盒**1入**。給予肌膚透明且安心的防曬體驗。\n\n---\n\n### 標籤\n#礦物防曬\n#氧化鋅\n#敏弱肌適用\n#溫和保濕\n#物理性防曬", "input_tokens": 812, "output_tokens": 256}, "29fad8c0c36fcdee8745": {"text": "好的，這是一個非常具有潛力的商品！NT$290 的價格搭配美國原裝進口、10% 積雪草萃取，非常吸引人。\n\n以下為您量身打造的電商文案，包含標題、痛點描述、產品特色、使用情境與行動呼籲，您可以直接應用於蝦皮、Momo 或官網。\n\n---\n\n## 🛒 白日光 SUNDAY 舒緩精華液 10% 電商文案\n\n### 💎 主標題（吸引點與高價值感）\n\n**【美國原裝🇺🇸 敏弱肌救星】白日光 SUNDAY 舒緩精華液 | 10% 高濃度積雪草 | 舒緩泛紅、穩定膚況 | NT$290超值入手！**\n\n---\n\n### 📝 正文案：內容結構\n\n#### Part 1：痛點與情境導入（抓住敏感肌的焦慮）\n\n**👉 你是否也有這些困擾？**\n\n* 換季時臉頰總是**泛紅、乾癢**，怎麼擦都不對？\n* 戴口罩悶出**小顆粒、痘痘**，肌膚狀態極不穩定？\n* 嘗試了各種保養品，但高濃度成分總是**刺激、刺痛**？\n* 想要溫和修護，但市面上的舒緩產品**濃度太低、無感**？\n\n**別擔心！白日光 SUNDAY 舒緩精華液，就是為敏弱肌量身打造的「肌膚穩定器」！**\n\n---\n\n#### Part 2：產品核心特色與賣點（強調高濃度與產地優勢）\n\n**✨ 為什麼要選擇【白日光 SUNDAY 舒緩精華液】？**\n\n**✅ 1. 10% 高濃度積雪草萃取 (Cica)：**\n市面上少見的黃金比例！積雪草是公認的肌膚修護專家，能有效鎮定不安定的膚況，快速舒緩泛紅、乾燥，讓肌膚重回健康平衡。\n\n**✅ 2. 美國原裝進口，品質保證：**\n堅持美國製造與包裝，嚴格把關成分純度與製程，讓您用得安心、有感。\n\n**✅ 3. 專為敏弱肌設計的溫和配方：**\n無酒精、無香精、無色素添加。配方單純，能溫和修護受損屏障，即使是醫美後或極度敏感的肌膚也能安心使用。\n\n**✅ 4. 清爽「凝露質地」，不黏膩：**\n打破高濃度精華液的厚重印象！輕盈水感的凝露質地，一抹化水，快速吸收，不悶痘、不給肌膚造成負擔，油性肌或夏天使用也超舒適。\n\n**✅ 5. 超高CP值，小資族必備：**\n高濃度積雪草精華，美國進口品質，現在只要 NT$290！用開架的價格，享受專櫃級的舒緩修護力。\n\n---\n\n#### Part 3：使用建議與適用對象\n\n**💡 誰需要這瓶「白日光 SUNDAY 舒緩精華液」？**\n\n* **口罩肌受害者：** 舒緩悶熱造成的泛紅與不適。\n* **換季敏感者：** 強化肌膚屏障，對抗環境刺激。\n* **醫美後修護者：** 溫和鎮定，加速肌膚穩定。\n* **追求極簡保養者：** 一瓶搞定舒緩、保濕與修護。\n\n**📌 建議使用方式：**\n早晚清潔、使用化妝水後，取適量精華液塗抹於全臉或局部泛紅處，輕拍至吸收即可。\n\n---\n\n#### Part 4：價格與行動呼籲（強調限時優惠與急迫性）\n\n**🔥 穩定膚況，從今天開始！**\n\n* **原價：** NT$XXX\n* **限時體驗價：NT$290** (一入/盒)\n\n**肌膚不穩定，保養全白費！**\n現在就將這瓶高CP值的【白日光 SUNDAY 舒緩精華液】加入購物車，給您的肌膚一個喘息的機會！\n\n**👉 點擊立即購買，告別泛紅敏感，迎接白日光般的健康美肌！**\n\n---\n\n### 🏷️ 關鍵字標籤（Keywords / Hashtags）\n\n```\n#白日光SUNDAY #積雪草精華 #Cica精華 #舒緩精華液 #敏感肌適用 #敏弱肌救星 #泛紅 #醫美修護 #美國原裝 #高濃度精華 #凝露質地 #保濕修護 #NT290 #超值CP #換季保養\n```", "input_tokens": 117, "output_tokens": 1098}, "0c2f785814e9bbaa2483": {"text": "## 白日光 SUNDAY 舒緩精華液 10%\n\n---\n\n### 商品標題\n\n**白日光 SUNDAY 積雪草舒緩精華液 10% 清爽凝露 混合肌夏日穩定對策**\n\n---\n\n### 賣點\n\n1.  清爽凝露質地，混合肌零負擔\n2.  高濃度積雪草萃取 10% 溫和穩定\n3.  舒緩肌膚不適，維持健康狀態\n4.  單盒 1入 輕巧包裝，輕鬆入門體驗\n\n---\n\n### SEO 描述\n\n專為 25-40 歲上班族混合肌設計！白日光 SUNDAY 舒緩精華液，添加高達 **10%** 的**積雪草萃取**，凝露**質地**輕盈好吸收，告別黏膩悶熱感。單盒**1入** NT$290 體驗價，讓您輕鬆應對環境變化，維持肌膚舒適穩定。\n\n---\n\n### 標籤\n\n*   積雪草精華\n*   混合肌保養\n*   清爽凝露\n*   舒緩\n*   夏日對策", "input_tokens": 673, "output_tokens": 268}, "b49ea16e66ffff1c58e6": {"text": "## 本木 BENMU 德國工藝 食品級不鏽鋼保溫瓶 750ml｜終身保固，小家庭/毛孩族戶外神隊友\n\n---\n\n### 賣點\n\n1.  強效**保溫**：德國雙層真空技術，鎖住溫度，長效維持冷熱。\n2.  完美容量：**750ml** 滿足全日飲水或毛孩備品（溫水）需求。\n3.  食品級不鏽鋼：安全無毒，清洗容易，不殘留異味。\n4.  終身技術支援：NT$690 享受德國工藝與品牌承諾，安心耐用。\n\n---\n\n### SEO 描述\n\n德國 BENMU 本木 **750ml** 不鏽鋼**保溫**瓶，專為小坪數家庭與養寵物族群設計。無論是居家辦公或帶毛孩外出散步，都能提供長效**保溫**。食品級材質，安全耐用，NT$690 即可擁有終身保固的安心承諾，是您投資高品質生活用品的最佳選擇。\n\n---\n\n### 標籤\n\n\\#保溫瓶\n\\#德國工藝\n\\#750ml\n\\#養寵物必備\n\\#小坪數生活", "input_tokens": 227, "output_tokens": 288}, "a0228741e6acc0197875": {"text": "## 本木 BENMU 空氣清淨機 15坪\n\n---\n\n### 商品標題\n**本木 BENMU 日本頂級工藝 15坪 空氣清淨機｜通勤族/遠距工作者 居家辦公室必備**\n\n---\n\n### 賣點\n1. 適用 **15坪** 空間，小套房/書房高效淨化首選。\n2. 日本原裝進口，食品級不鏽鋼機身，質感耐用。\n3. 專為久待室內設計，高效濾淨，健康深呼吸。\n4. 輕巧不佔空間，原廠提供一年安心保固。\n\n---\n\n### SEO 描述\n本木 BENMU 空氣清淨機是通勤族與遠距工作者的健康首選。日本製造，採用食品級不鏽鋼，品質可靠。高效淨化適用 **坪數** 達 **15坪**，無論是臥室或居家辦公室都能享受清新空氣。輕巧設計，NT$890超值入手，讓您在長時間工作後也能擁有潔淨的呼吸環境。\n\n---\n\n### 標籤\n#空氣清淨機 #15坪 #居家辦公 #BENMU #日本家電", "input_tokens": 223, "output_tokens": 274}, "b1f2b34fec2a759802d7": {"text": "這是一份針對「青研 CHINGYEN 礦物防曬乳 50ml 3入」的電商文案，強調其溫和性、物理防護力以及超值價格。\n\n---\n\n## 💎 青研 CHINGYEN 礦物防曬乳 (50ml x 3入) 💎\n\n**【敏弱肌專屬防護】溫和不刺激，每日防曬零負擔！**\n\n**💰 超值組合價：NT$390 (平均每瓶不到 $130！) 💰**\n\n---\n\n### 🌟 文案主標題：為敏感肌量身打造的「安心盾牌」！\n\n**厭倦了化學防曬帶來的泛紅與刺激嗎？**\n\n青研 CHINGYEN 礦物防曬乳，專為亞洲敏弱肌設計。我們捨棄了複雜的化學成分，採用最純淨、最溫和的**氧化鋅（Zinc Oxide）**作為核心防護，在肌膚表面形成物理性保護層，有效反射紫外線，給您全天候的安心守護。\n\n---\n\n### ✨ 產品四大核心優勢 (Why You Need This)\n\n#### 1. 【純淨物理防護】敏弱肌的救星\n\n本產品以**氧化鋅**為主要成分，屬於物理性防曬。它不滲入肌膚，而是停留在肌膚表面，大幅降低刺激性，特別適合：\n*   容易泛紅、過敏的肌膚。\n*   雷射術後或醫美保養後的脆弱肌膚。\n*   追求成分單純、溫和防曬的族群。\n\n#### 2. 【滋潤型油狀質地】告別乾燥卡粉\n\n有別於傳統物理防曬容易乾燥或泛白，青研防曬乳採用特殊的**油狀質地**，確保極佳的延展性與保濕度。\n*   **高延展性：** 輕輕一抹即可均勻塗開，不拉扯肌膚。\n*   **保濕滋潤：** 鎖住水分，讓敏感肌在防曬同時也能維持水潤舒適，告別乾澀緊繃感。\n\n#### 3. 【韓國製造 品質保證】\n\n堅持高品質標準，本產品於韓國製造生產，從原料到成品都經過嚴格把關，讓您用得安心，擦得放心。\n\n#### 4. 【超值囤貨組】每日使用不心疼！\n\n防曬是每日保養的最後一步，絕不能省！我們提供超划算的**三入組合包**，讓您輕鬆實現「臉部、頸部、身體」全面防護。\n*   **原價單瓶高昂？** 現在三瓶只要 **NT$390**！\n*   **CP值爆表：** 讓您用最親民的價格，享受專櫃級的溫和防護。\n\n---\n\n### 📋 商品規格一覽\n\n| 項目 | 詳細說明 |\n| :--- | :--- |\n| **品牌** | 青研 CHINGYEN |\n| **商品名稱** | 礦物防曬乳 |\n| **主要規格** | 氧化鋅 (Zinc Oxide) |\n| **容量** | 50ml / 瓶 (共 3 入) |\n| **售價** | **NT$390 (3入組)** |\n| **適用膚質** | 敏弱肌、乾燥肌、一般膚質皆適用 |\n| **型態** | 油狀質地 (滋潤型) |\n| **產地** | 韓國 |\n\n---\n\n### 🛒 立即搶購！\n\n**防曬是最好的抗老保養！**\n\n別再讓化學防曬成為肌膚的負擔。選擇青研 CHINGYEN，給您的敏感肌最溫柔、最有效的防護。\n\n**限量超值三入組，售完不補！**\n\n**👉 立即加入購物車，開啟您的安心防曬之旅！**", "input_tokens": 117, "output_tokens": 897}, "fa7fe0b25d9896e68afb": {"text": "## 晨光研選 甘胺酸亞鐵錠 20mg 90粒\n\n### 商品標題\n晨光研選 甘胺酸亞鐵錠 20mg 90粒 | 溫和高吸收率 專為女性與上班族設計 (49 字)\n\n---\n\n### 賣點\n1.  選用甘胺酸亞鐵，吸收率佳。 (15 字)\n2.  溫和配方，空腹食用也舒適。 (14 字)\n3.  日本原裝進口，每日補充 20mg。 (18 字)\n4.  適合長時間用眼與氣色調理。 (15 字)\n\n---\n\n### SEO 描述\n專為長時間使用螢幕的上班族與女性設計。晨光研選甘胺酸亞鐵錠，每份提供足量 **20mg** 的高吸收率鐵質，溫和不刺激。日本原裝進口軟膠囊，一盒 **90粒** 裝，是您維持紅潤氣色的日常補給首選。堅持每日補充 **甘胺酸亞鐵**，讓您輕鬆面對忙碌生活。 (109 字)\n\n---\n\n### 標籤\n*   補鐵\n*   甘胺酸亞鐵\n*   女性保健\n*   上班族\n*   日本進口", "input_tokens": 796, "output_tokens": 299}, "c381ace6be1ae25e1999": {"text": "**商品標題：**\n【青研 CHINGYEN】奢潤淨膚｜頂級胺基酸洗面乳 150ml 1入｜敏弱肌適用，洗後水嫩透亮\n\n**賣點：**\n1. 高規格胺基酸界面活性劑，溫和潔淨不刺激。\n2. 凝露質地，洗淨不帶走水分，長效保濕。\n3. 專為敏弱肌設計，有效穩定膚況，舒緩不適。\n4. NT$2290 奢華淨膚體驗，台灣製造，安心可靠。\n\n**SEO 描述：**\n30-45歲的您，肌膚保濕是關鍵！青研 CHINGYEN 奢潤淨膚洗面乳，採用頂級**胺基酸界面活性劑**，溫和洗淨不刺激。**150ml** 大容量 **1入**裝，凝露質地，洗後肌膚水嫩透亮，敏弱肌也能安心使用。高規格溫和配方，給您最頂級的清潔體驗。\n\n**標籤：**\n胺基酸洗面乳, 敏弱肌, 保濕, 溫和清潔, CHINGYEN", "input_tokens": 269, "output_tokens": 277}, "a4c024779425b5a5b39f": {"text": "## 晨光研選 | 日本高效吸收 甘胺酸亞鐵 20mg 軟膠囊 | 90粒裝 | 擊退疲勞，綻放紅潤好氣色\n\n---\n\n### 賣點\n\n1.  高效吸收率的**甘胺酸亞鐵**，溫和不刺激。\n2.  每份足量 **20mg**，輕鬆補足一日所需。\n3.  日本原裝進口，嚴選品質，安心有保障。\n4.  大容量 **90粒** 軟膠囊，三個月份量超值。\n\n---\n\n### SEO 描述\n\n長時間使用螢幕的上班族必備！晨光研選甘胺酸亞鐵錠，採用高吸收率的**甘胺酸亞鐵**，溫和不刺激腸胃。每份足量**20mg**，高效補鐵，幫助維持紅潤好氣色。日本原裝進口，**90粒**大包裝，輕鬆對抗辦公室疲勞，是女性保健的首選。\n\n---\n\n### 標籤\n\n補鐵 / 甘胺酸亞鐵 / 女性保健 / 上班族 / 日本進口", "input_tokens": 259, "output_tokens": 259}, "2b85162f497c6242c128": {"text": "好的，這是一份為您的商品「Volta 不鏽鋼保溫瓶 350ml」設計的電商文案。\n\n文案將分為**標題與口號**、**主文案（痛點與優勢）**、**產品特色（規格化呈現）**、**情境應用**與**結尾呼籲**五個部分，以達到最佳的銷售效果。\n\n---\n\n## ⚡ Volta 不鏽鋼保溫瓶 350ml ⚡\n\n### 一、標題與口號（吸引點擊與目光）\n\n**主標題：** 【Volta】輕巧隨行！350ml 台灣製不鏽鋼保溫瓶 — 你的質感日常，從此開始。\n\n**副標題/口號：** 輕量化設計，長效保溫。通勤、辦公、短程外出，剛剛好的容量美學。\n\n---\n\n### 二、主文案：痛點與優勢（說服消費者）\n\n#### 💡 你是否也遇到這些困擾？\n\n* **大容量太重：** 700ml、1000ml 的保溫瓶雖然實用，但日常通勤或短時間外出攜帶，總覺得笨重佔空間？\n* **保溫效果不佳：** 早上裝的熱飲，中午就涼了？下午的冰咖啡，很快就融化稀釋？\n* **材質不安心：** 擔心來路不明的保溫瓶材質，影響飲品風味或健康？\n\n#### 💡 選擇 Volta 350ml，解決你的日常煩惱！\n\n**【剛剛好的輕巧哲學】**\nVolta 350ml 保溫瓶，專為追求輕量化生活的你而設計。它不是最大，但絕對是最「剛好」的容量。無論是放進小包包、公事包，或隨手拿著，都毫無負擔。\n\n**【台灣製造，品質保證】**\n我們堅持台灣在地生產製造，從原料選擇到製程控管，都經過嚴格把關。搭配耐用 PC+ABS 複合材質瓶蓋，堅固耐用，讓您用得更安心。\n\n**【長效鎖溫，冷熱隨心】**\n採用優質不鏽鋼內膽，提供卓越的保溫與保冷效果。無論是冬日的暖茶，還是夏日的冰水，Volta 都能為您鎖住最佳溫度，隨時享受飲品的最佳風味。\n\n---\n\n### 三、產品特色（規格化呈現）\n\n| 特色項目 | 詳細說明 | 你的優勢 |\n| :--- | :--- | :--- |\n| **容量設計** | 350ml 輕巧容量 | 輕量隨行，不佔空間，適合單人飲用需求。 |\n| **頂級材質** | 優質不鏽鋼內膽 | 安全無毒，不殘留異味，易於清洗。 |\n| **複合瓶蓋** | PC+ABS 複合材質 | 堅固耐用，密封性佳，有效防漏。 |\n| **產地保證** | 台灣製造 (Made in Taiwan) | 品質嚴格把關，支持在地優良工藝。 |\n| **安心保障** | 原廠一年保固 | 售後服務完善，讓您購買無後顧之憂。 |\n| **超值入手** | NT$290 銅板價 | 高品質、台灣製，極具性價比的日常好物。 |\n\n---\n\n### 四、情境應用（激發購買慾望）\n\n**1. 辦公室夥伴：** 上班族必備！裝一杯咖啡或熱水，放在桌上隨時補充水分，小巧不佔用寶貴的桌面空間。\n**2. 通勤族首選：** 輕巧放入隨身包，搭捷運、公車時隨時補充水分，不增加通勤負擔。\n**3. 學生族輕便：** 課間休息時裝一杯飲品，350ml 剛剛好，輕便帶去圖書館或自習室。\n**4. 短程外出：** 遛狗、散步、短時間購物，裝上飲用水，輕鬆出門無負擔。\n\n---\n\n### 五、結尾呼籲與購買資訊\n\n**【Volta 承諾 — 質感生活，唾手可得】**\n\n別再讓笨重的保溫瓶成為你的負擔！Volta 350ml 不鏽鋼保溫瓶，以極簡設計、優異性能和親民價格，成為你日常生活中最可靠的飲水夥伴。\n\n**NT$290，投資你的每日好心情！**\n\n👉 **立即點擊購買，享受輕盈保溫新生活！**\n（原廠一年保固，讓您買得安心，用得放心。）", "input_tokens": 93, "output_tokens": 1084}, "d8fcf1742476dee26d4d": {"text": "**商品標題**\nVolta 240W USB-C 編織傳輸線：高瓦數快充，差旅多裝置供電穩定首選 (34字)\n\n**賣點**\n1. 240W 高瓦數供電，滿足筆電全速充電。\n2. 編織線身搭配食品級不鏽鋼接頭。\n3. 穩定傳輸，多裝置同時供電不降速。\n4. 終身技術支援，差旅使用無後顧之憂。\n\n**SEO 描述**\nVolta 240W USB-C 編織線專為高階筆電與多裝置用戶設計。支援 PD 3.1 規範，提供穩定且極速的電力傳輸。食品級不鏽鋼接頭，耐用抗彎折，是商務差旅人士的可靠選擇。\n\n**標籤**\n#240W快充\n#USBC線材\n#筆電充電\n#差旅必備\n#Volta", "input_tokens": 649, "output_tokens": 228}, "dfac76223965fcc2a80f": {"text": "{\n  \"title\": \"溯源焙所 德國分離乳清蛋白粉 30g 隨身包 60粒 熟齡日常營養補給\",\n  \"bullets\": [\n    \"嚴選德國進口 分離乳清蛋白 (WPI)\",\n    \"每份足量 30g 高純度蛋白質\",\n    \"獨立隨身包裝 共 60 份 方便攜帶\",\n    \"適合中年族群 輕鬆維持日常活力\"\n  ],\n  \"seo_description\": \"溯源焙所嚴選德國高品質分離乳清蛋白粉 (WPI)，每份提供 30g 純淨蛋白質。專為注重日常保養的熟齡族群設計，60粒獨立隨身包裝，輕鬆補充營養，維持健康基礎。\",\n  \"hashtags\": [\n    \"分離乳清蛋白\",\n    \"熟齡保養\",\n    \"蛋白質補充\",\n    \"運動營養\",\n    \"溯源焙所\"\n  ]\n}", "input_tokens": 906, "output_tokens": 226}, "9a112bea3d3bf716b68a": {"text": "好的，這是一份為「晨光研選 甘胺酸亞鐵錠 20mg 90粒」設計的電商文案。\n\n文案結構包含：標題、痛點切入、產品介紹（核心優勢）、成分強調、使用情境、購買呼籲與產品資訊。\n\n---\n\n## ☀️ 晨光研選 甘胺酸亞鐵錠 20mg 90粒 - 電商文案\n\n### 💎 主標題 (吸引目光)\n\n**【告別蠟黃疲憊！日本原裝🇯🇵 晨光研選 甘胺酸亞鐵】高效吸收 20mg，給妳由內而外的紅潤好氣色！**\n\n---\n\n### 💡 副標題 (痛點切入與承諾)\n\n**妳是否經常感到：臉色蒼白、精神不濟、爬樓梯就喘？**\n**缺鐵不再是小事！選擇「對」的鐵，讓妳每天都像充滿電！**\n\n---\n\n### 💖 內文第一段：痛點切入與產品定位\n\n親愛的妳，每天忙碌於工作與家庭，卻發現氣色越來越差？化妝品只能遮蓋表面的疲憊，真正的美麗需要從「內」開始滋養！\n\n**晨光研選** 專為現代女性打造的【甘胺酸亞鐵錠】，是妳維持紅潤好氣色的秘密武器。我們深知傳統鐵劑的吸收率與副作用困擾，因此我們選擇了**吸收率高、溫和不刺激**的頂級原料，讓妳輕鬆補鐵，重拾活力光采！\n\n---\n\n### ✨ 內文第二段：核心優勢強調 (為什麼選晨光研選？)\n\n#### 【三大理由，讓妳非選不可！】\n\n**1. 🥇 最高效吸收率：專利甘胺酸亞鐵 (Ferrous Bisglycinate)**\n有別於傳統的硫酸亞鐵，我們採用「胺基酸螯合技術」的甘胺酸亞鐵。這種結構能模擬食物中的鐵質，**吸收率高達90%以上**，且不會與其他食物成分競爭，確保鐵質直達身體所需！\n\n**2. 💊 溫和不刺激：告別傳統鐵劑副作用**\n傳統鐵劑常引起腸胃不適、噁心、甚至便秘。甘胺酸亞鐵結構穩定，**對腸胃極度溫和**，即使空腹食用也不易造成負擔，讓妳補鐵無壓力。\n\n**3. 🇯🇵 日本原裝進口，品質保證**\n本產品堅持**日本製造、原裝進口**，採用嚴謹的製程與最高規格的原料把關。每粒含足量 **20mg** 甘胺酸亞鐵，高濃度、高純度，給妳最安心的選擇。\n\n---\n\n### 👩‍🦰 內文第三段：適用對象與使用情境\n\n**誰最需要「晨光研選 甘胺酸亞鐵錠」？**\n\n* **經期女性：** 規律補充，維持好氣色與好體力。\n* **素食者/飲食不均衡者：** 輕鬆補足日常飲食中缺乏的鐵質。\n* **孕前/產後媽咪：** 滋補強身，維持健康狀態。\n* **上班族/學生族：** 告別精神不濟，提升專注力與工作效率。\n\n**【建議食用方式】**\n每日1份（1粒），建議隨餐食用，搭配開水吞服。持續補充，效果更佳！\n\n---\n\n### 🛒 結尾呼籲與產品資訊\n\n**美麗，從來不是靠化妝品堆砌，而是由內而外散發的健康光芒！**\n\n給自己一個重拾活力的機會，選擇高效、溫和、日本品質的【晨光研選 甘胺酸亞鐵錠】。現在就開始妳的紅潤計畫！\n\n**立即下單，讓妳的每一天都充滿「晨光」般的活力！**\n\n---\n\n### 📋 產品資訊一覽\n\n| 項目 | 內容 |\n| :--- | :--- |\n| **商品名稱** | 晨光研選 甘胺酸亞鐵錠 |\n| **分類** | 保健食品 / 女性保健 |\n| **主要規格** | 甘胺酸亞鐵 20mg |\n| **型態** | 軟膠囊 (易吞食) |\n| **產地** | 日本 (原裝進口) |\n| **包裝規格** | 90粒 / 盒 (約三個月份) |\n| **建議售價** | NT$1580 |\n| **食用建議** | 每日1粒，隨餐食用 |\n| **保存方式** | 請置於陰涼乾燥處，避免陽光直射。 |", "input_tokens": 118, "output_tokens": 1088}, "d32fdcb2dbff47bfab39": {"text": "{\n  \"title\": \"青研 CHINGYEN 唇部修護膏 8g 7入 敏弱肌適用 日本製造 水感輕盈\",\n  \"bullets\": [\n    \"採用蜂蠟與植物油配方，溫和滋潤唇部肌膚。\",\n    \"輕盈水感質地，不黏膩好吸收，告別厚重感。\",\n    \"8g 獨立小包裝，一盒 7入設計，方便分裝與攜帶。\",\n    \"適用於睡前厚敷保養或作為妝前唇部打底使用。\"\n  ],\n  \"seo_description\": \"青研 CHINGYEN 唇部修護膏，專為敏弱肌設計。內含蜂蠟與植物油，提供唇部溫和滋潤。水感輕盈不厚重，夜間修護或日常打底皆宜。一盒 7入，每份 8g 獨立包裝，隨身攜帶超方便。\",\n  \"hashtags\": [\n    \"唇部保養\",\n    \"敏弱肌友善\",\n    \"CHINGYEN\",\n    \"睡前厚敷\",\n    \"唇膏打底\"\n  ]\n}", "input_tokens": 921, "output_tokens": 274}, "1b6f62bda8ad31728d3c": {"text": "{\n  \"title\": \"晨光研選 甘胺酸亞鐵錠 20mg 90粒 日本專利原料 溫和補給 每日活力應援\",\n  \"bullets\": [\n    \"選用高吸收率的甘胺酸亞鐵形式。\",\n    \"每份足量提供亞鐵 20mg 溫和不刺激。\",\n    \"日本專利原料，每日持續補充好氣色。\",\n    \"90粒大容量包裝，上班族活力應援。\"\n  ],\n  \"seo_description\": \"晨光研選 甘胺酸亞鐵錠，專為長時間使用螢幕的上班族設計。每份含足量 20mg 甘胺酸亞鐵，高吸收率且溫和不刺激。日本專利原料，90粒大容量，幫助您維持紅潤氣色與日常活力，輕鬆應對工作挑戰。\",\n  \"hashtags\": [\n    \"甘胺酸亞鐵\",\n    \"補鐵\",\n    \"上班族\",\n    \"活力應援\",\n    \"晨光研選\"\n  ]\n}", "input_tokens": 903, "output_tokens": 247}, "41943ea540e091da1aad": {"text": "## 溯源焙所 德國高純度【分離乳清蛋白】30g 隨身包 60粒裝｜熟齡族群日常活力補給\n\n---\n\n### 賣點 (4條，每條不超過 30 字)\n\n1.  高純度**分離乳清蛋白**，乳糖低，好消化無負擔。\n2.  每份足量 **30g** 蛋白質，高效補充日常所需活力。\n3.  德國原裝進口，嚴格品質把關，安心有保障。\n4.  獨立隨身包設計，方便攜帶，隨時隨地輕鬆補充。\n\n---\n\n### SEO 描述 (不超過 120 字)\n\n溯源焙所專為注重日常保養的熟齡族群設計。採用高成本的【**分離乳清蛋白**】，乳糖含量低，好消化不脹氣。每包提供足量【**30g**】優質蛋白，維持體力與活力。德國進口，品質保證。獨立【**60粒**】隨身包裝，方便每日隨餐食用，輕鬆完成日常保養。\n\n---\n\n### 標籤 (3-5 個)\n\n熟齡保養 | 分離乳清 | 蛋白質補充 | 運動營養 | 隨身包", "input_tokens": 255, "output_tokens": 293}, "aa3e9f36bc1cfa1efb7f": {"text": "{\n  \"title\": \"Volta 不鏽鋼保溫瓶 750ml。長效保溫，PC+ABS複合材質，台灣製造。\",\n  \"bullets\": [\n    \"750ml 大容量，滿足全日飲水需求。\",\n    \"高效真空保溫結構，鎖住飲品溫度。\",\n    \"採用 PC+ABS 複合材質，抗衝擊耐用。\",\n    \"台灣製造，原廠提供一年保固服務。\"\n  ],\n  \"seo_description\": \"Volta 不鏽鋼保溫瓶 750ml，專為高頻率使用者設計。長效保溫技術，無論熱飲或冰水皆能維持理想溫度。PC+ABS複合結構，輕量且堅固耐用，適合商務出差或辦公室使用。台灣製造，品質可靠。\",\n  \"hashtags\": [\n    \"Volta\",\n    \"保溫瓶\",\n    \"750ml\",\n    \"出差必備\",\n    \"台灣製造\"\n  ]\n}", "input_tokens": 750, "output_tokens": 233}, "ceaae4c655671b46c1ec": {"text": "## 台灣電商平台資深商品文案\n\n### 商品標題 (不超過 60 個字)\n\n**溯源焙所 德國分離乳清蛋白粉 熟齡日常保養 30g 隨身包 60粒**\n\n---\n\n### 賣點 (4 條，每條不超過 30 個字)\n\n1.  高純度**分離乳清蛋白**，好吸收無負擔。\n2.  每份足量**30g**蛋白質，補足日常所需。\n3.  獨立**60粒**隨身包裝，方便攜帶與沖泡。\n4.  嚴選德國進口原料，品質安心有保障。\n\n---\n\n### SEO 描述 (不超過 120 個字)\n\n溯源焙所專為熟齡族群設計的日常營養補充。採用高純度**分離乳清蛋白**，每包足量**30g**，低乳糖好消化。**60粒**獨立包裝，隨時隨地補充優質蛋白質，維持活力與體力。德國原料，安心可靠。\n\n---\n\n### 標籤 (3-5 個)\n\n熟齡保養, 分離乳清, 蛋白質補充, 運動營養, 溯源焙所", "input_tokens": 799, "output_tokens": 285}, "2f6e7400c2b287e0d149": {"text": "**商品標題**\n【青研 CHINGYEN】日系爆水感唇膜修護組 8g*7入 | 蜂蠟植萃，告別死皮，素顏也超嫩！\n\n**賣點**\n1. 日系水感質地，一抹爆水不黏膩。\n2. 蜂蠟與植物油，深度修護敏弱肌。\n3. 8g 大容量 X 7入，CP值爆表！\n4. 夜間厚敷，隔天唇色嫩到發光。\n\n**SEO 描述**\n學生黨必囤！青研 CHINGYEN 唇部修護膏，日本原裝進口，專為敏弱肌設計。採用天然蜂蠟與植物油配方，水感質地清爽不厚重。超值大容量 8g 7入組，讓你一週七天，隨時擁有嘟嘟嫩唇，告別乾燥死皮，唇紋瞬間隱形！\n\n**標籤**\n#唇膜 #日系保養 #敏弱肌 #蜂蠟 #囤貨必備", "input_tokens": 259, "output_tokens": 245}, "287a61abd8ddc3a662ca": {"text": "{\n  \"title\": \"溯源焙所 韓國進口 分離乳清蛋白粉 30g 20粒 睡前保養專用\",\n  \"bullets\": [\n    \"高純度分離乳清蛋白，輕盈無負擔。\",\n    \"每份足量 30g，獨立隨身包共 20粒。\",\n    \"睡前補充，為夜間美麗打底。\",\n    \"韓國優質乳源，職人嚴選把關。\"\n  ],\n  \"seo_description\": \"專為熟齡女性設計的夜間保養配方。採用韓國進口高純度分離乳清蛋白，每份提供紮實的30g蛋白質。獨立20粒隨身包裝，方便每日睡前補充，維持優雅體態與美麗光采。\",\n  \"hashtags\": [\n    \"溯源焙所\",\n    \"分離乳清\",\n    \"睡前保養\",\n    \"熟齡女性\",\n    \"運動營養\"\n  ]\n}", "input_tokens": 908, "output_tokens": 233}, "d036bdb3ab0ee5976fd3": {"text": "**【Volta 不鏽鋼保溫瓶】**\n\n---\n\n### 商品標題\n**Volta PC複合材質 350ml 不鏽鋼保溫瓶 差旅隨行 輕巧保溫**\n\n### 賣點\n1. **350ml 精巧容量，差旅收納無負擔。**\n2. **PC+ABS 複合材質，結構抗衝擊。**\n3. **高效能真空層，長效保溫保冷。**\n4. **台灣製造，原廠提供一年保固。**\n\n### SEO 描述\n專為出差族設計。Volta 350ml 不鏽鋼保溫瓶，PC+ABS 複合材質結構，輕巧耐用。提供長效保溫功能，滿足多裝置使用者在移動中的飲水需求。台灣製造，原廠一年保固。\n\n### 標籤\n保溫瓶, 350ml, 出差必備, 隨行杯, Volta", "input_tokens": 643, "output_tokens": 217}, "6be1dbb9dc3357f96d08": {"text": "## 青研 CHINGYEN 唇部修護膏 8g 7入\n\n---\n\n### 商品標題\n\n**青研 CHINGYEN 唇部修護膏 | 蜂蠟與植物油 8g x 7入 | 水感輕盈 敏弱肌適用** (48 字)\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1. 水感質地輕盈，告別厚重黏膩感。\n2. 蜂蠟與植物油，溫和潤澤雙唇。\n3. 8g 隨身容量，7入分裝超實用。\n4. 日本製造，敏弱肌適用配方。\n\n---\n\n### SEO 描述 (不超過 120 字)\n\n青研 CHINGYEN 專為敏弱肌設計的唇部修護膏，採用溫和的蜂蠟與植物油配方，提供雙唇水感滋潤。單支 8g 容量，一盒 7入，方便學生族群隨身攜帶，隨時保持水潤狀態。輕盈質地，是妝前打底或夜間保養的首選。\n\n---\n\n### 標籤 (3-5 個)\n\n#唇部保養 #敏弱肌 #水感唇膏 #蜂蠟 #學生愛用", "input_tokens": 814, "output_tokens": 297}, "8c0c3328e331bea61d70": {"text": "## 本木 BENMU 空氣清淨機 15坪\n\n### 商品標題\n**本木 BENMU 日本不鏽鋼空氣清淨機 適用15坪 靜音耐用設計 打造居家清新工作空間**\n\n---\n\n### 賣點\n1. 食品級不鏽鋼機身，堅固耐用，體現樸實工藝。\n2. 適用坪數達15坪，高效淨化，提供全室清新空氣。\n3. 低噪運轉設計，不干擾居家辦公或深夜休憩。\n4. 日本工藝製造，原廠一年保固，品質穩定可靠。\n\n---\n\n### SEO 描述\n本木 BENMU 空氣清淨機，專為長時間待在家的遠距工作者與通勤族設計。食品級不鏽鋼機身，體現樸實耐用工藝。適用坪數達15坪，提供穩定清淨的室內空氣。低噪音運轉，讓您專注工作不受打擾。日本製造，原廠保固一年，是您打造健康居家環境的理想選擇。\n\n---\n\n### 標籤\n#空氣清淨機\n#遠距工作\n#15坪\n#不鏽鋼\n#靜音家電", "input_tokens": 505, "output_tokens": 273}, "5e541fd0c2755d729002": {"text": "這是一個為您的「Volta 不鏽鋼保溫瓶 750ml」設計的電商文案，包含標題、痛點解決、產品特色、規格亮點與行動呼籲。\n\n---\n\n## ⚡ Volta 不鏽鋼保溫瓶 750ml ⚡\n\n### 【主標題】\n**大容量・長效保冰溫！台灣製造 Volta 750ml 不鏽鋼保溫瓶 — 你的日常補水神隊友，環保又時尚！**\n\n---\n\n### 【前言 / 痛點解決】\n**還在忍受這些不便嗎？**\n\n* ❌ **容量不夠用？** 早上裝滿，中午就見底，頻繁跑茶水間好麻煩！\n* ❌ **保溫效果差？** 剛泡好的熱飲，不到兩小時就變溫水？\n* ❌ **材質不安心？** 擔心來路不明的保溫瓶，影響健康？\n\n**Volta 750ml 保溫瓶，一次解決你的所有煩惱！**\n\n---\n\n### 【產品核心亮點與特色】\n\n#### 1. 💧 超越日常的大容量：750ml 滿足你一整天所需！\n無論是辦公室久坐、戶外運動，還是長時間通勤，750ml 的黃金容量設計，讓你輕鬆達到每日建議飲水量。減少重複裝水的次數，專注於你的重要事務。\n\n#### 2. 🛡️ 台灣製造，品質保證！\n我們深知您對健康的重視。Volta 保溫瓶堅持**台灣在地生產**，採用高品質不鏽鋼材質，搭配 PC+ABS 複合型態，結構堅固耐用，讓您用得安心、喝得放心。\n\n#### 3. 🌡️ 長效鎖溫技術：冷熱皆宜，四季通用！\n採用高效能真空隔熱技術，無論是炎炎夏日想喝冰涼的飲料，還是寒冷冬天想來一杯暖胃的熱茶，Volta 都能長時間維持最佳溫度，讓你的每一口都恰到好處。\n\n#### 4. ✨ 時尚設計，輕巧便攜\n簡約俐落的外型設計，搭配多種經典配色，不僅是實用的水壺，更是你提升生活品味的時尚配件。雖然容量大，但設計輕巧，輕鬆放入背包或車內杯架。\n\n#### 5. 💚 環保生活，從 Volta 開始\n選擇可重複使用的保溫瓶，減少一次性塑膠瓶的使用，為地球盡一份心力。NT$490 的親民價格，讓您輕鬆加入環保行列，享受高品質生活。\n\n---\n\n### 【規格速覽與保障】\n\n| 項目 | 規格說明 | 亮點強調 |\n| :--- | :--- | :--- |\n| **商品名稱** | Volta 不鏽鋼保溫瓶 | |\n| **容量** | **750ml** | 補水無憂，滿足全日需求！ |\n| **售價** | **NT$490** | 高 CP 值，輕鬆入手！ |\n| **型態** | PC+ABS 複合 | 結構堅固，耐用性高。 |\n| **產地** | **台灣** | 🇹🇼 品質保證，安心使用。 |\n| **分類** | 生活 / 水壺 | 日常必備良品。 |\n| **保固** | **原廠一年保固** | 完善售後服務，購買更放心！ |\n\n---\n\n### 【適用情境】\n\n* 💼 **上班族：** 辦公桌上的最佳伴侶，減少起身裝水次數。\n* 🏋️ **運動愛好者：** 運動前後快速補水，保持飲品冰涼。\n* 🚌 **通勤族/學生：** 隨身攜帶，隨時享受溫熱或冰涼飲品。\n* 🌳 **戶外活動：** 野餐、露營、健行，長效保溫保冰。\n\n---\n\n### 【行動呼籲 Call to Action】\n\n**別再讓劣質水壺影響你的生活品質！**\n\n現在就入手 **Volta 750ml 不鏽鋼保溫瓶**，享受台灣製造的安心品質、長效的保溫保冰效果，以及 NT$490 的超值價格！\n\n**🛒 立即點擊購買，開啟你的高品質補水生活！**\n\n---", "input_tokens": 93, "output_tokens": 989}, "497850526f851523e6ff": {"text": "## 溯源焙所 高純度分離乳清蛋白 晚安修護配方 30g 隨身包 20粒\n\n---\n\n### 賣點\n\n1.  高純度**分離乳清蛋白**，輕盈無負擔好吸收。\n2.  嚴選韓國優質乳源，風味純淨，職人級選品。\n3.  每份足量**30g**，獨立**20粒**隨身包裝，方便攜帶。\n4.  睡前黃金時段補給，為美麗與活力打下基礎。\n\n---\n\n### SEO 描述\n\n專為 30-50 歲注重內在保養的女性設計。溯源焙所嚴選韓國高純度**分離乳清蛋白**，每包足量**30g**，提供夜間所需基礎滋養。獨立**20粒**隨身包，輕鬆融入日常保養流程。享受頂級純淨的蛋白質補給，維持青春光采。\n\n---\n\n### 標籤\n\n女性保養、分離乳清蛋白、晚安蛋白、高純度、隨身包", "input_tokens": 801, "output_tokens": 255}, "c17fcaea2c63cc78187f": {"text": "## Volta 不鏽鋼保溫瓶 750ml\n\n### 商品標題\n**Volta 750ml 不鏽鋼保溫瓶：商務差旅必備，長效保溫，輕巧耐用，台灣製造。** (36 字)\n\n---\n\n### 賣點 (4 條)\n\n1.  **750ml** 大容量設計，滿足全日飲水需求。\n2.  長效真空**保溫**技術，冷熱飲溫度持久。\n3.  不鏽鋼材質堅固耐用，差旅攜帶無壓力。\n4.  PC+ABS 複合瓶蓋，輕盈易開，單手操作便利。\n\n---\n\n### SEO 描述\n專為商務人士設計的 Volta 不鏽鋼保溫瓶，擁有 **750ml** 大容量，長效**保溫**保冷。堅固耐用，適合經常出差、需要多裝置切換的您。台灣製造，品質保證，讓您隨時隨地享受最佳飲品溫度。 (78 字)\n\n---\n\n### 標籤\n#保溫瓶 #750ml #出差必備 #商務旅行 #台灣製造", "input_tokens": 221, "output_tokens": 267}, "4118a2ba005e8576ad96": {"text": "{\n  \"title\": \"本木 BENMU 15坪適用 空氣清淨機。食品級不鏽鋼機身，日系簡約設計，打造純淨工作環境。\",\n  \"bullets\": [\n    \"適用 15坪 空間，滿足都會小家庭或個人工作室需求。\",\n    \"採用食品級不鏽鋼機身，結構紮實，經久耐用。\",\n    \"日本製造工藝，細節講究，提供穩定可靠的空氣淨化。\",\n    \"靜音設計，為遠距工作者與通勤族提供專注呼吸環境。\"\n  ],\n  \"seo_description\": \"本木 BENMU 空氣清淨機，專為追求質感與耐用的您設計。適用坪數達 15坪，無論是居家辦公室或臥室都能輕鬆淨化。食品級不鏽鋼機身，結合日系簡約設計，提供安靜且穩定的空氣品質。\",\n  \"hashtags\": [\n    \"BENMU\",\n    \"空氣清淨機\",\n    \"15坪\",\n    \"遠距辦公\",\n    \"不鏽鋼家電\"\n  ]\n}", "input_tokens": 612, "output_tokens": 258}, "9f1e776b6fd3c26bccce": {"text": "這是一份針對「晨光研選 複方益生菌粉」設計的電商文案，涵蓋了標題、痛點解決、產品亮點和行動呼籲，適用於商品頁面或廣告投放。\n\n---\n\n## 💡 版本一：詳細商品頁面文案 (強調專業與夜間修護)\n\n### 標題：告別卡關人生！【晨光研選】夜間黃金修護，100億專利益生菌，讓您一覺到天亮，順暢到天明！\n\n---\n\n### 🌟 前言：您是否也有這些困擾？\n\n現代生活壓力大、飲食不均衡，讓您的腸道常常鬧脾氣？\n*   ❌ 早上醒來總是感到腹部悶脹？\n*   ❌ ❌ 排便時間不固定，經常「卡關」？\n*   ❌ ❌ ❌ 換季時，保護力總是差人一截？\n\n腸道健康是全身健康的基石！【晨光研選】專為亞洲人體質設計，提供您最有效率的夜間腸道保養方案。\n\n---\n\n### 🔬 產品亮點：為什麼選擇晨光研選複方益生菌？\n\n#### 1. 【高規格】100億CFU 專利複方，有效直達\n我們不只追求數量，更注重品質！每份提供高達 **100億CFU** 的活性益生菌，採用多種專利菌株，確保益生菌能通過胃酸考驗，順利抵達並定殖於腸道，發揮最大效益。\n\n#### 2. 【獨家】睡前黃金修護，效率加倍！\n不同於一般益生菌，我們建議在**睡前食用**。夜間是腸道進行修復和蠕動的黃金時段。當您進入夢鄉時，益生菌正在您的體內默默工作，隔天早上輕鬆迎接順暢人生！\n\n#### 3. 【安心】台灣製造，植物膠囊更友善\n*   **台灣在地研發製造**：品質嚴格把關，安心有保障。\n*   **植物膠囊**：素食者可食，溫和不刺激，無多餘化學添加。\n\n#### 4. 【高CP值】健康投資，輕盈無負擔\nNT$490 即可擁有 60 粒裝（足足兩個月份量），平均每日花費不到 $9 元，就能為您的消化道建立強大防線。\n\n---\n\n### 📋 產品資訊與食用建議\n\n| 項目 | 規格說明 |\n| :--- | :--- |\n| **商品名稱** | 晨光研選 複方益生菌粉 |\n| **主要規格** | 100億CFU 專利益生菌 |\n| **型態** | 植物膠囊 (60粒/盒) |\n| **產地** | 台灣 (品質保證) |\n| **售價** | **NT$490** |\n| **建議食用** | 每日1份（1粒），**睡前搭配溫水食用**。 |\n\n---\n\n### 🛒 立即行動：\n\n別讓不順暢影響您的好心情！現在就加入購物車，體驗【晨光研選】帶來的夜間黃金修護力！\n\n**👉 點擊購買，開始您的順暢輕盈計畫！**\n\n---\n---\n\n## 💡 版本二：社群媒體/短版廣告文案 (強調痛點與價格)\n\n### 【晨光研選】夜間益生菌｜NT$490 搶購中！\n\n**主題標籤：** #益生菌 #腸道健康 #消化保健 #順暢人生 #晨光研選 #睡前保養\n\n**文案內容：**\n\n您知道嗎？腸道在夜間的活動力更強！\n\n🔥 **【晨光研選 複方益生菌】** 抓住夜間黃金修復期，讓您睡一覺起來，輕鬆告別腹脹、卡關的困擾！\n\n✨ **我們的承諾：**\n1. **超有感！** 專利複方，每份高達 **100億CFU** 活性益生菌。\n2. **超方便！** 睡前一顆，輕鬆搞定，無需擔心飯前飯後。\n3. **超安心！** 台灣製造，植物膠囊，素食者友善。\n\n💰 **超值優惠：** 60粒裝（2個月份），只要 **NT$490**！\n\n**👉 睡前吃，早上順！立即點擊連結，讓您的腸道煥然一新！**", "input_tokens": 124, "output_tokens": 1061}, "bbebc64d91c456f05de4": {"text": "## 青研 CHINGYEN 胺基酸溫和潔顏露 150ml 敏弱肌適用 凝露質地 清爽保濕 1入\n\n---\n\n### 賣點\n\n1.  採用**胺基酸界面活性劑**，溫和潔淨不刺激\n2.  洗後不乾澀緊繃，維持肌膚水潤感\n3.  透明凝露質地，泡沫細緻好沖洗\n4.  敏弱肌適用配方，成分單純透明\n\n---\n\n### SEO 描述\n\n青研CHINGYEN專為30-45歲重視溫和保濕的族群設計。本款潔顏露採用親膚的**胺基酸界面活性劑**，有效清潔同時維持肌膚天然屏障。**150ml** 大容量，**1入** 裝。凝露質地，洗後清爽不緊繃，是敏弱肌日常保濕清潔的理想選擇。\n\n---\n\n### 標籤\n\n*   胺基酸洗面乳\n*   敏弱肌\n*   溫和清潔\n*   保濕\n*   青研", "input_tokens": 824, "output_tokens": 253}, "4ffcd4e6ee7838b61685": {"text": "**商品標題**\nVolta 不鏽鋼複合保溫瓶 750ml 高效溫控 台灣製造 出差差旅適用\n\n**賣點**\n1. 複合式溫控結構，有效鎖住溫度。\n2. 標準 750ml 容量，減少重複裝填。\n3. PC+ABS 複合外殼，耐衝擊設計。\n4. 台灣製造，原廠提供一年保固。\n\n**SEO 描述**\nVolta 不鏽鋼保溫瓶 750ml，專為高頻率出差及多裝置使用者設計。採用高效保溫技術，確保飲品長效維持溫度。PC+ABS 複合材質，堅固耐用，是您商務旅行的最佳飲水解決方案。台灣製造，品質可靠。\n\n**標籤**\n#Volta\n#保溫瓶\n#750ml\n#出差必備\n#不鏽鋼", "input_tokens": 643, "output_tokens": 206}, "bac4f986b535c24a7141": {"text": "{\n  \"title\": \"青研 CHINGYEN 溫和胺基酸洗面乳 150ml 1入 敏弱肌適用\",\n  \"bullets\": [\n    \"採用溫和胺基酸界面活性劑，親膚潔淨。\",\n    \"清透凝露質地，洗後水潤不緊繃。\",\n    \"配方單純，溫和低刺激，敏弱肌適用。\",\n    \"標準規格 150ml，台灣製造，每盒 1入。\"\n  ],\n  \"seo_description\": \"青研 CHINGYEN 溫和胺基酸洗面乳，專為重視保濕與溫和清潔的您設計。採用親膚的胺基酸界面活性劑，凝露質地，能溫和帶走髒污，洗後肌膚感到水潤舒適，不乾澀緊繃。規格為 150ml，台灣製造，每盒 1入，是您日常清潔的理性選擇。\",\n  \"hashtags\": [\n    \"青研CHINGYEN\",\n    \"胺基酸洗面乳\",\n    \"溫和清潔\",\n    \"敏弱肌\",\n    \"保濕洗臉\"\n  ]\n}", "input_tokens": 931, "output_tokens": 274}, "87fe67ca8324cb624cc1": {"text": "## 溯源焙所 晚安美妍乳清 | 高純度分離乳清蛋白 30g | 睡前黃金修護 20粒 輕奢組\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1.  高規格**分離乳清蛋白**，純淨無負擔，好吸收。\n2.  單包足量**30g**蛋白質，高效補給，維持優雅體態。\n3.  韓國原裝進口，**20粒**隨身包設計，輕巧方便攜帶。\n4.  睡前黃金修護期飲用，啟動夜間代謝與美妍力。\n\n---\n\n### SEO 描述 (不超過 120 字)\n\n專為注重內外保養的熟齡女性設計。溯源焙所採用高純度**分離乳清蛋白**，單份提供足量**30g**蛋白質，促進代謝與美妍修護。韓國原裝進口**20粒**隨身包，方便攜帶。建議睡前飲用，輕鬆維持優雅體態，讓美麗從睡眠中甦醒。\n\n---\n\n### 標籤 (3-5 個)\n\n#分離乳清 #美妍保養 #睡前修護 #蛋白質30g #溯源焙所", "input_tokens": 257, "output_tokens": 301}, "56a7269a6e30cdbec6d9": {"text": "{\n  \"title\": \"白日光 SUNDAY 神經醯胺修護乳液 50ml 1入 | 溫和舒緩，敏弱肌日常保濕首選。\",\n  \"bullets\": [\n    \"核心成分神經醯胺，幫助肌膚維持水潤與穩定。\",\n    \"輕盈乳霜質地，清爽不黏膩，快速滲透好吸收。\",\n    \"溫和配方設計，舒緩乾燥不適，適合敏弱膚質。\",\n    \"美國原裝進口，50ml 輕巧包裝，每日保養攜帶方便。\"\n  ],\n  \"seo_description\": \"白日光 SUNDAY 修護乳液專為敏弱肌設計。蘊含關鍵神經醯胺成分，能為肌膚提供所需的水分與滋潤。50ml 輕巧容量，美國原裝進口 1入。質地清爽好吸收，是成分黨日常保濕的安心選擇。\",\n  \"hashtags\": [\n    \"神經醯胺\",\n    \"敏弱肌保養\",\n    \"成分黨\",\n    \"日常保濕\",\n    \"乳液50ml\"\n  ]\n}", "input_tokens": 774, "output_tokens": 267}, "34244fabe0a2eabeb788": {"text": "## 晨光研選 即溶燕麥飲 商品文案\n\n---\n\n### 商品標題\n\n**晨光研選 即溶燕麥飲 德國原裝 全穀燕麥 25g 獨立包裝 30粒 睡前輕食首選**\n\n---\n\n### 賣點\n\n1.  嚴選德國進口，優質**全穀燕麥**來源。\n2.  單包**25g**定量，獨立包裝共**30粒**。\n3.  睡前輕盈補充，提供穩定飽足感。\n4.  快速沖泡即飲，適合健身與家庭常備。\n\n---\n\n### SEO 描述\n\n晨光研選即溶燕麥飲，專為注重健康的家庭與健身族群設計。嚴選德國優質**全穀燕麥**，單包**25g**輕巧份量，提供優質碳水化合物。獨立包裝共**30粒**，方便攜帶，隨時享受。建議睡前沖泡，作為輕盈的夜間補充，穩定持續日常營養，是原型食物愛好者的理想選擇。\n\n---\n\n### 標籤\n\n燕麥飲、全穀物、原型食物、健身飲食、睡前輕食", "input_tokens": 788, "output_tokens": 275}, "dc1d38c6bf2f11dd06ca": {"text": "## 晨光研選 德國進口【原型全穀燕麥】即溶飲 25g 高纖飽足 30粒裝｜家庭健身首選\n\n---\n\n### 賣點\n\n1.  德國嚴選：純正**全穀燕麥**，保留完整營養價值。\n2.  **25g**黃金份量，高纖飽足，輕盈無負擔。\n3.  **30粒**獨立包裝，隨沖即飲，忙碌生活神隊友。\n4.  無添加精製糖，符合原型食物標準，健身族群安心選。\n\n---\n\n### SEO 描述\n\n晨光研選即溶燕麥飲，嚴選德國進口**全穀燕麥**，堅持原型食物理念。每份**25g**獨立包裝，一盒**30粒**，方便計算熱量，是健身族群與忙碌家庭的優質早餐或宵夜選擇。高纖維提供長效飽足感，輕鬆補充能量，健康無負擔。\n\n---\n\n### 標籤\n\n#原型食物 #全穀燕麥 #即溶沖泡 #健身飲食 #德國進口", "input_tokens": 251, "output_tokens": 260}, "e6bc7440a2db8dca22c2": {"text": "這是一份針對「溯源焙所 分離乳清蛋白粉」的電商文案，強調其高含量、德國品質與極致便利性。\n\n---\n\n# 🚀 溯源焙所 頂級分離乳清蛋白粉 (WPI) 60天份超值組\n\n## 【主標題】\n**打破效率天花板！德國頂級WPI，單包直衝 30g 高效蛋白！**\n**60天份隨身包裝，NT$390 限時搶購！**\n\n---\n\n## 🎯 抓住目光！您是否面臨這些挑戰？\n\n*   ❌ 健身後需要快速補充，但傳統乳清吸收慢、容易脹氣？\n*   ❌ 忙碌工作或外出，無法精準測量份量，攜帶不便？\n*   ❌ 追求高品質、低負擔的純淨蛋白來源？\n\n**溯源焙所，為您提供最純粹、最高效的解答！**\n\n---\n\n## ✨ 產品核心優勢：為什麼選擇溯源焙所 WPI？\n\n### 1. 🥇 德國頂級分離乳清 (WPI)：純淨的黃金標準\n我們堅持採用**德國進口**的頂級分離乳清蛋白（Whey Protein Isolate, WPI）。WPI 經過特殊工藝處理，有效去除多餘的脂肪和乳糖，提供極高的蛋白質純度，適合乳糖不耐者或追求極致低負擔的您。\n\n### 2. 💪 業界震撼！單份高達 30g 蛋白質\n不再需要兩勺！每份隨身包提供**足足 30 克**的純淨蛋白質。無論是重訓後的肌肉修復，或是作為高蛋白代餐增加飽足感，一包到位，效率翻倍！\n\n### 3. 🎒 隨身包設計：極致便利，精準份量\n告別笨重的桶裝和混亂的量勺！60 份獨立隨身包設計，讓您輕鬆放入健身包、辦公室抽屜或隨身口袋。隨時隨地，撕開即飲，精準掌握每日攝取量。\n\n### 4. 🗓️ 超值大容量：60 天份，高 CP 值首選\n一盒即享 **60 份**的長效供應！以極具競爭力的價格，提供長達兩個月的頂級營養補充，是您持續健身、維持健康的最佳夥伴。\n\n---\n\n## 🔍 產品規格一覽\n\n| 規格項目 | 內容說明 |\n| :--- | :--- |\n| **商品名稱** | 溯源焙所 分離乳清蛋白粉 |\n| **主要成分** | 分離乳清蛋白 (WPI) |\n| **單份含量** | **30g 蛋白質** |\n| **型態** | 粉末隨身包 (單包裝) |\n| **產地** | 德國 (高品質溯源保證) |\n| **包裝規格** | 60 粒 / 盒 (60 天份) |\n| **建議食用** | 每日 1 份，可隨餐或運動後 30 分鐘內食用。 |\n| **售價** | **NT$390** |\n\n---\n\n## 🛒 立即行動！給身體最高效的燃料！\n\n**無論您是專業運動員、健身愛好者，還是追求健康飲食的忙碌上班族，溯源焙所 WPI 都是您不可或缺的選擇。**\n\n**限時優惠！60 天份頂級分離乳清，只要 NT$390！**\n\n**點擊下方按鈕，即刻入手，開啟您的純淨高效蛋白之旅！**\n\n👉 **[立即購買]** 👈", "input_tokens": 116, "output_tokens": 848}, "5f484454173fa583d046": {"text": "這是一份針對「青研 CHINGYEN 礦物防曬乳」設計的電商文案，強調其德國品質、物理防曬特性以及對敏弱肌的溫和呵護。\n\n---\n\n## 💎 青研 CHINGYEN 礦物防曬乳 (60ml)\n\n### 標題：【德國純淨防護力】青研 CHINGYEN 礦物防曬乳 — 敏弱肌專屬！溫和氧化鋅配方，享受陽光零負擔。\n\n---\n\n### 💡 前言：告別刺激，迎接純淨防護\n\n您是否正為尋找一款高效防曬，卻又擔心化學成分對肌膚造成刺激？\n\n青研 CHINGYEN 專為極度敏感及脆弱肌膚設計，將德國精準的配方工藝，注入最溫和的物理性防曬核心。我們相信，防曬不該是肌膚的負擔，而是最輕柔的呵護。\n\n### ✨ 三大核心優勢：敏弱肌的安心首選\n\n#### 1. 100% 物理性防曬：溫和氧化鋅配方\n\n本產品採用高純度**氧化鋅 (Zinc Oxide)** 作為主要防曬成分。它在肌膚表面形成一層保護屏障，反射紫外線，而非被肌膚吸收，從根本上杜絕化學性防曬可能引起的刺激和過敏反應。\n\n*   **零刺激：** 專為敏弱肌、術後肌、甚至兒童設計。\n*   **廣譜防護：** 有效阻擋 UVA 及 UVB，提供全面的陽光保護。\n\n#### 2. 德國原裝進口：精準與品質的保證\n\n青研 CHINGYEN 礦物防曬乳堅持**德國原產地製造**。嚴謹的生產標準和對原料的極致要求，確保每一瓶防曬乳都擁有最高品質和穩定性。\n\n*   **國際級標準：** 享受歐洲頂級美妝保養的安心承諾。\n*   **配方穩定：** 確保防曬效果持久且可靠。\n\n#### 3. 創新油狀質地：服貼不泛白\n\n許多物理性防曬產品常有厚重、難推勻或嚴重泛白的問題。青研 CHINGYEN 採用獨特的**油狀質地**，徹底顛覆您對礦物防曬的刻板印象：\n\n*   **高延展性：** 輕盈好推，快速服貼於肌膚，不留下白色殘留。\n*   **親膚滋養：** 油性基底有助於鎖住水分，提供額外的保濕效果，讓肌膚在防曬同時保持柔嫩。\n\n---\n\n### 📋 產品規格一覽\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **商品名稱** | 青研 CHINGYEN 礦物防曬乳 |\n| **分類** | 美妝保養 / 物理防曬 |\n| **主要成分** | 氧化鋅 (Zinc Oxide) |\n| **容量** | 60ml / 1入 |\n| **型態** | 油狀質地 (高延展性) |\n| **適用膚質** | **敏弱肌、乾燥肌、一般肌** |\n| **產地** | 德國 (原裝進口) |\n| **售價** | **NT$890** |\n\n---\n\n### 🛒 立即體驗：給肌膚最純淨的愛！\n\n別再讓刺激的防曬成分傷害您嬌嫩的肌膚。\n\n選擇青研 CHINGYEN 礦物防曬乳，您選擇的不僅是高效防護，更是對肌膚健康的長久投資。\n\n**👉 立即下單，享受來自德國的溫和防護力！**", "input_tokens": 117, "output_tokens": 860}, "438ed294a75a242b9c1f": {"text": "**品牌：** 白日光 SUNDAY\n**分類：** 美妝保養 / 乳液\n**售價：** NT$890\n\n---\n\n## 商品文案\n\n### 商品標題\n\n白日光 SUNDAY 高濃度神經醯胺修護乳液 50ml 1入 | 專為敏弱肌設計，強化肌膚屏障。\n\n### 賣點\n\n1.  **成分黨首選：** 高純度神經醯胺，深度修護乾燥受損。\n2.  **屏障強化：** 迅速穩定泛紅，告別乾燥脫屑困擾。\n3.  **輕盈乳霜：** 零負擔感乳霜質地，清爽不黏膩。\n4.  **美國原裝：** 溫和無添加，敏弱肌安心使用。\n\n### SEO 描述\n\n白日光 SUNDAY 修護乳液專為敏弱肌設計，核心添加高濃度神經醯胺，有效強化肌膚屏障。這款 50ml 規格，單盒 1入 包裝，能深度滋潤乾燥，穩定膚況。成分單純，美國製造，是追求高效保養的成分黨安心首選。\n\n### 標籤\n\n#神經醯胺 #敏弱肌保養 #屏障修護 #成分黨必備 #美國製造", "input_tokens": 251, "output_tokens": 297}, "df00b171fef34ac39dec": {"text": "## 本木 BENMU 機械式鍵盤 104鍵 專業文案\n\n---\n\n### 商品標題 (不超過 60 個字)\n\n**本木 BENMU｜美國原裝 專業級機械式鍵盤 104鍵。霧面鋁合金，打造高效靜謐工作空間。** (54字)\n\n---\n\n### 賣點 (4 條，每條不超過 30 個字)\n\n1.  專為長時間輸入設計，**機械式**軸體提供舒適回饋。 (18字)\n2.  頂級霧面鋁合金機身，穩固耐用，質感全面升級。 (17字)\n3.  標準**104鍵**配置，數字區讓資料輸入更有效率。 (19字)\n4.  美國原廠進口，享一年保固，品質安心有保障。 (17字)\n\n---\n\n### SEO 描述 (不超過 120 個字)\n\n遠距工作者與通勤族必備！本木 BENMU 專業級**機械式**鍵盤，採用堅固的霧面鋁合金設計，提供極致穩定的輸入體驗。標準**104鍵**配置，滿足所有文書處理與數字輸入需求。美國原裝進口，高效能、高質感，讓您的辦公效率全面提升。 (98字)\n\n---\n\n### 標籤 (3-5 個)\n\n#機械鍵盤\n#遠距辦公\n#104鍵\n#鋁合金\n#高效率", "input_tokens": 226, "output_tokens": 344}, "debad8670dc6c065e2e6": {"text": "好的，這是一份針對「青研 CHINGYEN 溫和洗面乳」的電商文案，強調其高端定位、胺基酸配方以及對敏弱肌的呵護。\n\n---\n\n## 💎 青研 CHINGYEN 胺基酸溫和洗面乳 150ml 💎\n**NT$2290 | 敏弱肌專屬的「高訂」淨膚儀式**\n\n### 【商品介紹】\n您是否厭倦了洗後乾澀、緊繃，甚至泛紅的清潔產品？\n青研 CHINGYEN 溫和洗面乳，專為極度重視成分與膚感、以及敏弱肌膚設計。我們相信，頂級的保養始於最溫和的清潔。這款凝露質地的洗面乳，以高規格胺基酸配方為核心，在有效帶走髒污的同時，完整保留肌膚所需的天然皮脂膜，讓您的肌膚在清潔後依然保持水潤、柔軟、零負擔。\n\n---\n\n### ✨ 核心賣點：奢華淨膚，溫和不妥協\n\n#### 1. 胺基酸黃金配方：親膚性極高\n本產品採用高純度**胺基酸界面活性劑**，這是公認最溫和、最親膚的清潔成分。它能精準捕捉毛孔內的髒污與多餘油脂，卻不會過度清潔，徹底告別傳統皂基或刺激性界面活性劑帶來的乾澀感。\n\n#### 2. 專為敏弱肌設計：屏障守護者\n肌膚敏感的根源往往在於清潔時屏障受損。青研洗面乳經嚴格測試，**適用於所有膚質，特別是敏弱肌、乾燥肌**。無添加酒精、人工香精、Paraben防腐劑等刺激性成分，給予肌膚最純淨的呵護。\n\n#### 3. 療癒凝露質地：奢華洗臉體驗\n透明凝露質地，輕盈滑順，遇水即能產生細緻綿密的泡沫。泡沫極易沖洗，不殘留，洗臉過程舒適療癒，讓每日的清潔步驟昇華為一場頂級的護膚享受。\n\n#### 4. 頂級定位，台灣製造\nNT$2290 的價格，代表我們對原料品質、配方穩定度以及製造工藝的極致追求。堅持台灣在地研發與生產，為您提供安心可靠的高規格保養品。\n\n---\n\n### 💡 誰需要這瓶洗面乳？\n\n*   ✅ 正在尋找一款真正溫和、不刺激的洗面乳。\n*   ✅ 敏弱肌、酒糟肌或正在經歷肌膚不穩定期的使用者。\n*   ✅ 追求高品質、願意投資頂級保養品的消費者。\n*   ✅ 希望洗後肌膚柔軟、水潤，不緊繃乾澀者。\n\n---\n\n### 【商品規格一覽】\n\n| 項目 | 規格說明 |\n| :--- | :--- |\n| **商品名稱** | 青研 CHINGYEN 溫和洗面乳 |\n| **容量** | 150ml (1入/盒) |\n| **主要成分** | 胺基酸界面活性劑 |\n| **適用膚質** | 敏弱肌、乾燥肌、所有膚質適用 |\n| **型態** | 凝露質地 |\n| **產地** | 台灣 (Made in Taiwan) |\n| **售價** | NT$2290 |\n| **分類** | 美妝保養 / 清潔 |\n\n---\n\n### 🛒 立即體驗：給予肌膚最溫柔的奢華呵護。\n**選擇青研 CHINGYEN，讓清潔成為修護的起點。**", "input_tokens": 124, "output_tokens": 859}, "80eaa5cee61f699cea2e": {"text": "這是一份針對「本木 BENMU 不鏽鋼保溫瓶 750ml」設計的電商文案，涵蓋了標題、痛點解決、核心賣點與行動呼籲。\n\n---\n\n## 💡 電商文案：本木 BENMU 不鏽鋼保溫瓶 750ml\n\n### 💎 主標題 (Headline)\n\n**德國工藝，終身承諾。本木 BENMU 750ml 不鏽鋼保溫瓶：您值得擁有的「一輩子」好水壺。**\n\n---\n\n### 🎯 痛點與解決方案 (Interest & Problem Solving)\n\n**您是否厭倦了...**\n\n*   ❌ 便宜水壺保溫效果差，早上裝的熱水，中午就變涼？\n*   ❌ 擔心塑膠材質釋放有害物質，影響飲水健康？\n*   ❌ 水壺容量太小，一天需要頻繁裝水，打斷工作節奏？\n\n**【本木 BENMU】為您帶來德國級的飲水體驗！**\n\n我們不只提供一個水壺，我們提供一個**終身可靠的飲水夥伴**。結合頂級食品級不鏽鋼與嚴謹的德國製造標準，確保您的每一口飲水都純淨、適溫、安心。\n\n---\n\n### ✨ 核心賣點深度解析 (Desire & Features)\n\n#### 1. 德國製造，品質保證 (The German Advantage)\n\n*   **產地：德國 (Made in Germany)**：告別粗製濫造。本木 BENMU 堅持德國精密工藝，從結構設計到真空技術，確保卓越的保溫性能和耐用性。\n*   **終身技術支援**：我們對產品的信心，來自於對品質的堅持。購買本木，即享終身技術支援，讓您買得安心，用得放心。\n\n#### 2. 750ml 巨量容量，滿足全日需求 (Capacity & Convenience)\n\n*   **黃金容量 750ml**：無論是長時間會議、健身訓練，或是一整天的戶外活動，750ml 容量能有效減少裝水次數，輕鬆達成每日飲水目標。\n*   **輕巧設計**：大容量卻不笨重，完美適應汽車杯架和背包側袋。\n\n#### 3. 食品級不鏽鋼，健康無憂 (Safety & Material)\n\n*   **型態：食品級不鏽鋼**：採用最高標準的食品級不鏽鋼材質，不含 BPA，不殘留異味，不影響飲品風味。讓您喝到的永遠是水或飲品最原始的純淨味道。\n*   **超強保溫保冷**：高效真空隔熱層，無論是炎炎夏日的冰飲，還是寒冷冬日的熱茶，都能長效維持最佳溫度。\n\n---\n\n### 💰 價格與價值 (Value Proposition)\n\n**本木 BENMU 不鏽鋼保溫瓶 750ml**\n*   **售價：NT$690**\n\n投資一個 NT$690 的水壺，卻換來德國製造的頂級品質，以及品牌承諾的終身技術支援。這不僅是購買一個水壺，這是對健康與永續生活的一次高價值投資。\n\n---\n\n### 🛒 行動呼籲 (Call to Action)\n\n**別再讓劣質水壺影響您的生活品質！**\n\n立即將【本木 BENMU 750ml 不鏽鋼保溫瓶】加入購物車，體驗德國工藝帶來的純淨與長效保溫。\n\n**本木 BENMU：讓飲水，成為一種享受。**\n\n---\n\n### 🖼️ 建議搭配圖片/視覺呈現\n\n1.  **情境圖**：專業人士在辦公桌上使用，旁邊搭配筆電和文件，強調專業感。\n2.  **細節圖**：特寫瓶身底部或標籤，清晰標示 **\"MADE IN GERMANY\"**。\n3.  **對比圖**：展示 750ml 容量與一般水瓶的差異，強調大容量優勢。\n4.  **保溫測試圖**：視覺化呈現冰塊在瓶內數小時後仍未融化的情境。", "input_tokens": 96, "output_tokens": 951}, "1facc3cf7fbbab16aec3": {"text": "## 白日光 SUNDAY 修護乳液\n\n---\n\n### 商品標題\n\n**敏弱肌的週日儀式感！白日光 SUNDAY 神經醯胺舒緩保濕乳液 50ml 1入**\n\n---\n\n### 賣點\n\n1. 核心成分神經醯胺，提升肌膚防護力。\n2. 輕盈乳霜質地，清爽保濕不黏膩。\n3. 專為敏弱肌設計，溫和日常呵護。\n4. 美國原裝進口，規格 50ml 1入。\n\n---\n\n### SEO 描述\n\n成分黨必收！白日光 SUNDAY 修護乳液專為敏弱肌打造，添加高效神經醯胺，幫助肌膚維持水潤。輕盈乳霜質地，提供舒適的日常保濕。美國原裝進口，規格 50ml 1入，讓你的肌膚享受溫和的週日呵護。\n\n---\n\n### 標籤\n\n#神經醯胺 #敏弱肌保濕 #成分黨 #修護乳液 #美國保養", "input_tokens": 667, "output_tokens": 245}, "36812956d6a2153d0d23": {"text": "{\n  \"title\": \"晨光研選 即溶全穀燕麥飲 德國原裝進口 25g 獨立包裝 30粒\",\n  \"bullets\": [\n    \"嚴選德國進口優質全穀燕麥，保留完整營養。\",\n    \"獨立小包裝，每份 25g，方便快速沖泡攜帶。\",\n    \"提供原型食物的單純營養，適合家庭與健身族群。\",\n    \"一盒 30粒，滿足每日持續補充的飲食需求。\"\n  ],\n  \"seo_description\": \"晨光研選即溶全穀燕麥飲，嚴選德國優質燕麥，提供原型食物的單純營養。每份 25g 獨立包裝，一盒 30粒，方便快速補充。適合注重飲食均衡的家庭或運動後補充。\",\n  \"hashtags\": [\n    \"原型食物\",\n    \"全穀燕麥\",\n    \"即溶燕麥\",\n    \"健身飲食\",\n    \"晨光研選\"\n  ]\n}", "input_tokens": 895, "output_tokens": 240}, "31b83ea2f422252f3ca4": {"text": "這是一份針對「青研 CHINGYEN 唇部修護膏 8g 7入」的電商文案，涵蓋了產品特色、成分優勢、使用體驗及購買資訊。\n\n---\n\n## 【青研 CHINGYEN】日本水感修護唇膏 7日密集修護組 (8g x 7入)\n**NT$1180**\n\n### 🌟 主標題：告別乾燥脫皮！日本青研 CHINGYEN 7日密集修護，為敏弱唇肌打造的「水感隱形唇膜」\n\n---\n\n### 💧 產品痛點解決：您還在忍受厚重黏膩的護唇膏嗎？\n\n乾燥、脫皮、唇紋深重，是許多人長期的困擾。市面上常見的護唇膏往往為了強調滋潤度而犧牲了舒適感，讓您在白天難以使用，夜晚又擔心黏膩沾染。\n\n**青研 CHINGYEN 唇部修護膏，專為亞洲敏弱肌設計，徹底顛覆您對「滋潤」的想像！**\n\n---\n\n### ✨ 核心特色：為什麼選擇青研 CHINGEN？\n\n#### 1. 獨家「水感」質地，清爽零負擔\n告別傳統油膏的厚重感！青研採用日本頂尖技術，將滋潤成分轉化為**水感輕盈質地**。一抹瞬間化開，迅速被唇部吸收，形成一層隱形保護膜，不黏膩、不悶厚，即使在炎熱氣候下使用也極度舒適。\n\n#### 2. 敏弱肌救星：溫和修護，深層滋養\n我們深知敏弱唇肌的脆弱。本產品嚴選天然成分，不含刺激性物質，讓您安心使用：\n*   **天然蜂蠟與植物油精萃：** 提供長效鎖水屏障，深層修護乾燥受損的唇部肌膚。\n*   **適用膚質：** 專為敏弱肌設計，溫和舒緩，有效改善唇部泛紅與乾癢。\n\n#### 3. 日本製造，品質保證\n堅持日本原裝進口，從原料篩選到生產製造，皆符合嚴格的品質標準，給您的雙唇最安心、最高規格的呵護。\n\n---\n\n### 🎁 獨家規格：7入獨立包裝，打造一週密集修護療程\n\n青研 CHINGYEN 採用獨特的 **7入獨立小包裝 (每入 8g)** 設計，讓修護效果達到最佳化：\n\n*   **衛生與新鮮度提升：** 每次使用都是最新鮮的狀態，避免大罐裝反覆開蓋造成的污染與氧化。\n*   **密集修護循環：** 建議以一週為週期，進行高強度修護。無論是作為夜間厚敷唇膜，或隨身攜帶補給，都極為便利。\n*   **隨身攜帶零死角：** 一盒七入，您可以將它放置在辦公室、床頭、化妝包、甚至車上，隨時隨地為雙唇補水。\n\n---\n\n### 📋 產品規格一覽\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **商品名稱** | 青研 CHINGYEN 唇部修護膏 7入組 |\n| **售價** | NT$1180 |\n| **主要成分** | 天然蜂蠟、多種植物油精萃 |\n| **質地** | 水感輕盈質地 (不黏膩) |\n| **規格** | 8g x 7入 / 盒 |\n| **產地** | 日本 (Made in Japan) |\n| **適用膚質** | 所有膚質，特別推薦敏弱肌、長期乾燥脫皮者 |\n\n---\n\n### 🛒 立即行動：給您的雙唇一個重生的機會！\n\n不再讓乾燥唇紋影響您的笑容！現在入手【青研 CHINGYEN 7日密集修護唇膏】，享受日本頂級水感修護體驗。\n\n**限量發售中，立即加入購物車，開啟您的水嫩櫻唇之旅！**", "input_tokens": 120, "output_tokens": 950}, "dde2f94b7e123c69063b": {"text": "{\n  \"title\": \"本木 BENMU 經典機械式鍵盤 104鍵。霧面鋁合金，專為長時間打字設計。\",\n  \"bullets\": [\n    \"經典機械式手感，提供精準回饋與穩定輸入。\",\n    \"標準 104鍵配置，滿足專業工作者高效需求。\",\n    \"堅固霧面鋁合金機身，結構紮實，經久耐用。\",\n    \"樸實耐用取向，適合通勤族與遠距工作者使用。\"\n  ],\n  \"seo_description\": \"本木 BENMU 104鍵機械式鍵盤，採用堅固霧面鋁合金打造，提供穩定且舒適的打字體驗。無論是遠距工作或辦公通勤，其經典機械式軸體設計，都能確保長時間輸入的效率與手感。樸實耐用，是專業人士的首選裝備。\",\n  \"hashtags\": [\n    \"機械式鍵盤\",\n    \"104鍵\",\n    \"遠距工作\",\n    \"鋁合金\",\n    \"辦公室周邊\"\n  ]\n}", "input_tokens": 615, "output_tokens": 252}, "7b8013074e5456d10d2e": {"text": "好的，這是一份為您的「白日光 SUNDAY 修護乳液」量身打造的電商文案，包含標題、痛點切入、產品介紹、核心賣點、使用建議及結尾呼籲，適用於各大電商平台。\n\n---\n\n## ☀️ 白日光 SUNDAY 修護乳液 (50ml) ☀️\n\n### 💎 標題建議 (可擇一使用或組合)\n\n**短標題 (吸睛型)：**\n【白日光 SUNDAY】神經醯胺修護乳液｜美國原裝｜告別乾燥敏弱肌，重現水潤光澤！\n\n**長標題 (SEO 友好型)：**\n白日光 SUNDAY 修護乳液 50ml (神經醯胺配方)｜深層保濕、舒緩修護、乳霜質地｜所有膚質適用｜NT$890\n\n---\n\n### 💧 正文文案\n\n#### 💡 第一段：痛點切入與情境營造\n\n**【你是否也有這些肌膚困擾？】**\n\n換季時肌膚總是乾癢泛紅？冷氣房待久了，臉部緊繃不適？保養品擦了好多層，卻感覺水分還是留不住？\n\n現代生活中的環境壓力、空調、紫外線，都在無形中損耗著肌膚屏障。當肌膚的「城牆」受損，水分流失加速，乾燥、粗糙、敏弱等問題就會接踵而來。\n\n**現在，是時候讓肌膚享受週日般的寧靜與修護了！**\n\n#### ✨ 第二段：產品介紹與核心成分\n\n**【美國原裝進口🇺🇸 白日光 SUNDAY 修護乳液】**\n\n我們為您帶來這款來自美國、專為修護肌膚屏障而設計的頂級乳液——**白日光 SUNDAY 修護乳液**。\n\n這款乳液的核心秘密，就在於高濃度的**「神經醯胺 (Ceramide)」**。神經醯胺是肌膚角質層中最重要的脂質成分，它能有效填補細胞間隙，如同天然的黏合劑，強健肌膚屏障。\n\n**一瓶 50ml，給予肌膚最精準、最有效的深度滋養。**\n\n#### 🔬 第三段：四大核心賣點 (為什麼選擇 SUNDAY？)\n\n| 🌟 **賣點** | **詳細說明** |\n| :--- | :--- |\n| **1. 深度修護屏障 (神經醯胺)** | 補充肌膚流失的神經醯胺，從根本強化肌膚防禦力，有效舒緩乾燥、泛紅等敏弱狀況，讓肌膚更穩定、更健康。 |\n| **2. 鎖水保濕長效 (乳霜質地)** | 雖然是乳霜質地，但輕盈好推開，不黏膩、不悶厚。在肌膚表面形成保護膜，將水分與養分牢牢鎖住，實現長效保濕。 |\n| **3. 溫和配方 (所有膚質適用)** | 堅持溫和配方，不含刺激性成分，通過嚴格測試。無論是乾性肌、油性肌、混合肌，甚至是敏感肌，都能安心使用。 |\n| **4. 美國原裝品質保證** | 堅持美國原產地製造與包裝，確保產品品質與活性成分的穩定性，讓您用得更放心。 |\n\n#### 📝 第四段：使用建議與規格資訊\n\n**【使用方式】**\n\n早晚清潔臉部及使用化妝水、精華液後，取適量（約一顆紅豆大小）均勻塗抹於臉部及頸部肌膚，輕柔按摩至吸收即可。特別乾燥部位可加強塗抹。\n\n| 📋 **商品規格一覽** | |\n| :--- | :--- |\n| **商品名稱** | 白日光 SUNDAY 修護乳液 |\n| **主要規格** | 神經醯胺 |\n| **容量** | 50ml / 1入 |\n| **型態** | 乳霜質地 (輕盈不厚重) |\n| **適用膚質** | 所有膚質 (包含敏感肌) |\n| **產地** | 美國 🇺🇸 |\n| **售價** | NT$890 |\n\n#### 💖 第五段：結尾呼籲與行動指令\n\n**【給肌膚一個週日般的休息日！】**\n\n保養，不該是負擔，而是一種享受。\n\n選擇「白日光 SUNDAY 修護乳液」，就是選擇給予肌膚最溫柔、最有效的呵護。讓您的肌膚在忙碌的生活中，也能重拾飽滿、水潤、透亮的健康光采。\n\n**🛒 立即加入購物車，體驗來自美國的深度修護力量！**\n\n---\n**(可搭配圖片展示：乳液質地特寫、產品包裝圖、使用前後對比圖)**", "input_tokens": 112, "output_tokens": 1120}, "5c61916201da8eeb587d": {"text": "這是一份針對「本木 BENMU 空氣清淨機 15坪」的電商文案，強調其獨特的材質、日本品質與極具競爭力的價格。\n\n---\n\n## 【本木 BENMU】空氣清淨機 15坪適用 | 日本原裝 | 食品級不鏽鋼機身 | 質感淨化首選\n\n### 💎 產品標語：\n**日本工藝，不鏽鋼淨化。用最安心的材質，給您最純淨的空氣。**\n\n---\n\n### 🔥 必買重點 (Why You Need This)\n\n*   **【獨家材質】** 業界罕見！採用**食品級不鏽鋼**機身，耐用、易潔、更衛生。\n*   **【高效覆蓋】** 適用坪數高達 **15 坪**，客廳、主臥室淨化無死角。\n*   **【品質保證】** **日本原裝**進口，精密工藝，穩定可靠。\n*   **【超值入手】** 頂級規格，驚喜價格 **NT$890**，輕鬆升級居家空氣品質。\n\n---\n\n### 🌿 告別塑膠時代：食品級不鏽鋼的淨化革命\n\n您是否擔心清淨機外殼材質的潛在疑慮？本木 BENMU 帶來革命性的解決方案！\n\n我們堅持採用**食品級不鏽鋼**打造機身，這不僅賦予清淨機極致的耐用性與高雅的現代質感，更重要的是：\n\n1.  **衛生保障：** 不鏽鋼材質不易吸附灰塵與異味，清潔保養更輕鬆。\n2.  **極簡美學：** 俐落的日式設計，完美融入任何居家風格，不再是突兀的家電。\n3.  **長效穩定：** 堅固耐用，確保您的清淨機長期維持高效運作。\n\n### 💨 15坪大空間，清新無所不在\n\n無需擔心清淨機效能不足！BENMU 空氣清淨機專為中大型空間設計，適用面積高達 **15 坪**。\n\n無論是開放式客廳、寬敞的主臥室，或是需要高度空氣品質的嬰兒房，BENMU 都能快速循環室內空氣，有效淨化，讓您和家人隨時享受如森林般的清新感受。\n\n### 🇯🇵 日本原裝進口，信賴的品質承諾\n\nBENMU 核心技術與零件源自**日本**，代表著對細節的堅持與對品質的承諾。每一台清淨機都經過嚴格的品管測試，確保為您提供最穩定、最可靠的空氣淨化體驗。\n\n我們提供**原廠一年保固**，讓您買得安心，用得放心。\n\n---\n\n### 🛒 規格一覽\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **商品名稱** | 本木 BENMU 空氣清淨機 15坪 |\n| **分類** | 家電 / 空氣清淨 |\n| **適用坪數** | 15 坪 |\n| **機身型態** | 食品級不鏽鋼 (高質感、易清潔) |\n| **產地** | 日本 (原裝進口) |\n| **保固** | 原廠一年保固 |\n| **超值售價** | **NT$890** (限時優惠，數量有限！) |\n\n---\n\n### 🎁 立即行動：為您的居家環境投資最好的空氣！\n\n頂級的材質、日本的工藝，現在以超乎想像的價格擁有！\n\n**NT$890**，讓您將高品質的 BENMU 不鏽鋼空氣清淨機帶回家。\n\n**限量供應，立即下單，享受健康呼吸！**", "input_tokens": 96, "output_tokens": 855}, "005098899c8017ce0ddc": {"text": "這是一款針對追求極致效率和高品質恢復的運動者設計的頂級乳清蛋白。由於價格較高且強調「分離乳清」和「睡前食用」，文案需要著重於**純粹性、高效吸收**以及**夜間黃金修復**。\n\n---\n\n## 溯源焙所 | 頂級分離乳清蛋白粉 (WPI)\n**30g 黃金修復力，將睡眠轉化為肌肉成長的黃金期**\n\n### 💎 標題一：【極致純粹，高效吸收】夜間專屬！溯源焙所 30g 分離乳清蛋白隨身包\n\n---\n\n### 🌟 商品核心訴求 (開場白)\n\n您是否厭倦了低效的補給，渴望在每一次訓練後都能獲得最純粹、最快速的修復？\n\n**溯源焙所** 專為追求卓越的您打造。我們將韓國頂級分離乳清蛋白（WPI）濃縮於輕巧的隨身包中，每份提供驚人的 **30 克高純度蛋白質**。更重要的是，我們建議您在睡前飲用，啟動您夜間的黃金修復機制，讓您在夢中也能持續成長！\n\n---\n\n### 🔬 三大頂級優勢，定義高效補給\n\n#### 1. 【30g 極致純粹 WPI】高濃度、零負擔\n我們選用最高規格的「分離乳清蛋白 (WPI)」，經過精密過濾，大幅降低乳糖、脂肪和碳水化合物含量。\n*   **高達 30g 蛋白質 / 份：** 遠超市場平均，為重訓者提供足夠的修復基石。\n*   **快速吸收：** WPI 特性確保胺基酸能迅速進入血液，即時支援肌肉修復。\n*   **韓國原裝進口：** 嚴選高品質原料與製程，確保每一口都是最高標準。\n\n#### 2. 【夜間黃金修復期】睡夢中持續成長\n為什麼建議睡前食用？夜間是人體分泌生長激素、進行肌肉合成與修復的關鍵時刻。\n*   **持續供能：** 睡前補充高純度蛋白質，能確保您在長達 6-8 小時的睡眠中，肌肉組織持續獲得胺基酸的穩定供應。\n*   **優化修復效率：** 讓您的休息時間不再只是休息，而是高效的肌肉重建工程。\n\n#### 3. 【頂級便利性】告別量匙與結塊\n我們深知專業人士對效率的要求。\n*   **獨立隨身包裝 (20粒/盒)：** 精準定量 30g，無需量匙，輕鬆撕開即可沖泡。\n*   **極致便攜：** 無論是出差、健身房訓練或辦公室補給，隨手一包，隨時補充。\n*   **新鮮保證：** 獨立包裝有效隔絕濕氣與空氣，確保粉末始終保持最佳狀態。\n\n---\n\n### 💡 誰需要「溯源焙所 分離乳清蛋白」？\n\n*   追求極致肌肉線條的專業健美人士。\n*   需要高強度修復的耐力型運動員。\n*   對乳清品質要求極高，追求低脂、低碳水補給者。\n*   重視效率與便利性，不願浪費時間測量粉末的忙碌商務人士。\n\n---\n\n### 📋 產品規格與食用建議\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **品牌** | 溯源焙所 |\n| **分類** | 運動營養 / 高階補給 |\n| **主要規格** | 分離乳清蛋白 (WPI) 30g / 份 |\n| **型態** | 粉末隨身包 (獨立包裝) |\n| **產地** | 韓國 (高品質原料與製程) |\n| **包裝** | 20 粒 / 盒 |\n| **售價** | NT$4680 (頂級修復，值得投資) |\n| **建議食用** | **每日 1 份，建議於睡前 30-60 分鐘食用。** |\n| **沖泡方式** | 搭配 250-300cc 冷水或牛奶，搖勻後立即飲用。 |\n\n---\n\n### 🛒 立即投資您的修復品質！\n\n**NT$4680** 不僅是購買一盒乳清，更是購買 **20 個高效修復的夜晚**。\n\n停止使用低效的補給品，選擇 **溯源焙所**，讓您的每一次睡眠都成為肌肉成長的加速器。\n\n**限量供應，立即下單，體驗頂級韓國 WPI 的純粹力量！**", "input_tokens": 117, "output_tokens": 1098}, "2da0e4397d725ee43283": {"text": "## 本木 BENMU 機械式鍵盤 104鍵\n\n### 商品標題\n本木 BENMU 104鍵 機械式鍵盤｜霧面鋁合金機身｜打字手感紮實，為專業工作者打造\n\n---\n\n### 賣點\n1. 霧面鋁合金機身，結構穩固，經久耐用。\n2. 經典**機械式**軸體，回饋清晰，輸入精準可靠。\n3. 標準**104鍵**配置，滿足高頻率資料輸入需求。\n4. 美國原廠製造，提供一年保固，品質有保障。\n\n---\n\n### SEO 描述\n本木 BENMU 專為長時間工作設計的**機械式**鍵盤。採用堅固的霧面鋁合金機身，提供紮實的打字手感。標準**104鍵**配置，讓遠距工作者與通勤族都能享受高效率的輸入體驗。強調材質與工法，耐用取向，是您桌面上的可靠夥伴。\n\n---\n\n### 標籤\n#機械鍵盤 #遠距辦公 #104鍵 #鋁合金 #BENMU", "input_tokens": 508, "output_tokens": 256}, "0460662091d3cb8545a2": {"text": "## 青研 CHINGYEN 純淨卸妝油 100ml 1入\n\n---\n\n### 商品標題\n\n**青研 CHINGYEN | 純淨植萃卸妝油 100ml 敏弱肌適用 | 溫和乳化 不熏眼 零負擔**\n\n---\n\n### 賣點\n\n1.  高純度植物油脂配方，溫和溶解彩妝。\n2.  專為敏弱肌設計，卸妝過程不刺激。\n3.  深層淨化毛孔，洗後清爽零殘留感。\n4.  台灣製造，成分單純，成分黨安心首選。\n\n---\n\n### SEO 描述\n\n專為敏弱肌設計的青研純淨卸妝油，以高純度**植物油脂**為基底，溫和卸除彩妝與髒污。質地輕盈，快速乳化不殘留，不堵塞毛孔，給予肌膚最純淨的呵護。規格為**100ml**，**1入**包裝，成分黨安心選購，讓卸妝成為享受。\n\n---\n\n### 標籤\n\n*   敏弱肌卸妝\n*   植物油\n*   成分黨\n*   純淨保養\n*   卸妝油100ml", "input_tokens": 257, "output_tokens": 290}, "c0c35973024b7029887a": {"text": "{\n  \"title\": \"晨光研選 專利益生菌錠 50億CFU 30粒 調整體質 輕鬆補給\",\n  \"bullets\": [\n    \"每粒含高達 50億CFU 專利益生菌。\",\n    \"幫助維持消化道菌叢平衡，調整體質。\",\n    \"嚴選日本製造，品質安心有保障。\",\n    \"輕巧錠劑設計，方便每日持續食用。\"\n  ],\n  \"seo_description\": \"長時間久坐、飲食不規律的上班族必備。晨光研選複方專利益生菌，每粒提供 50億CFU 活性菌，幫助您維持消化道機能。日本原裝進口，30粒小包裝，輕鬆納入每日保健習慣。\",\n  \"hashtags\": [\n    \"益生菌\",\n    \"消化道保健\",\n    \"專利益生菌\",\n    \"上班族補給\",\n    \"每日保養\"\n  ]\n}", "input_tokens": 906, "output_tokens": 229}, "ab1c69a93a8b1a1cfd8d": {"text": "這是一份針對「晨光研選 即溶燕麥飲」的電商文案，強調其高品質、便利性，以及獨特的「睡前食用」定位。\n\n---\n\n## 【晨光研選】德國全穀燕麥飲：告別宵夜罪惡感，睡前暖胃輕享受！\n\n### 🌙 晚安燕麥新哲學：不再為宵夜煩惱，健康從夜晚開始\n\n您是否常在深夜感到飢餓，卻又擔心攝取過多熱量，影響睡眠和體態？\n\n「晨光研選 即溶燕麥飲」打破傳統燕麥飲的早餐印象，專為您的夜晚時光設計。我們嚴選**德國優質全穀燕麥**，提供最純粹、無負擔的溫暖與飽足感，讓您在睡前也能享受一份安心的輕食。\n\n---\n\n### ✨ 核心賣點：為什麼選擇晨光研選？\n\n#### 1. 睡前輕食首選 (Nighttime Ritual)\n根據建議，本產品適合在**睡前食用**。溫熱的燕麥飲能輕柔地安撫腸胃，提供恰到好處的飽足感，避免空腹影響睡眠品質，同時又不會造成消化負擔。\n\n#### 2. 德國嚴選，全穀營養 (Premium Quality)\n我們堅持使用來自**德國產地**的優質全穀燕麥。每一口都富含膳食纖維與完整營養，為您的健康打下堅實基礎。25g 的黃金份量，營養足夠，熱量輕盈。\n\n#### 3. 極致便利，即沖即飲 (Instant & Convenient)\n採用獨立小包裝設計（30粒/盒），無論是辦公室加班、旅行，或是在家準備睡前飲品，只需倒入熱水即可享用。無需測量、無需等待，隨時隨地，輕鬆補充能量。\n\n---\n\n### 💡 產品規格一覽\n\n| 特點 | 說明 |\n| :--- | :--- |\n| **商品名稱** | 晨光研選 即溶燕麥飲 |\n| **主要規格** | 德國全穀燕麥 25g / 包 |\n| **產品型態** | 獨立包裝，即溶沖泡 |\n| **產地來源** | 德國 (嚴選優質燕麥) |\n| **包裝數量** | 30粒 / 盒 (約一個月份量) |\n| **建議售價** | **NT$490** |\n| **建議食用** | 每日1份，**睡前食用** |\n\n---\n\n### 💖 誰適合這款「晚安燕麥飲」？\n\n*   **夜貓族與加班族：** 尋找健康、無負擔的深夜點心。\n*   **健康管理人士：** 需要高纖、低負擔的輕食替代品。\n*   **重視儀式感者：** 追求睡前一杯溫暖、療癒飲品。\n*   **追求效率者：** 喜歡即沖即飲的極致便利性。\n\n### 🛒 立即行動！\n\n告別高熱量宵夜，從今晚開始，用一杯溫暖的德國全穀燕麥飲，為您的身體和心靈畫下完美的句點。\n\n**【晨光研選】即溶燕麥飲 (30粒裝) 只要 NT$490**\n\n**點擊下方按鈕，將夜晚的健康與溫暖帶回家！**", "input_tokens": 116, "output_tokens": 789}, "6fbdca536d0aee1cb457": {"text": "## 青研 CHINGYEN 純淨卸妝油 100ml 1入\n\n### 商品標題\n青研 CHINGYEN 純淨卸妝油 100ml 1入 | 植萃溫和配方 敏弱肌友善 好沖洗不殘留 (55字)\n\n---\n\n### 賣點\n1.  嚴選多種親膚**植物油脂**基底。 (13字)\n2.  輕盈乳霜質地，快速乳化不油膩。 (15字)\n3.  無添加酒精香料，降低肌膚負擔。 (15字)\n4.  輕巧**100ml**容量，方便攜帶，**1入**組。 (18字)\n\n---\n\n### SEO 描述\n專為敏弱肌設計的青研純淨卸妝油。採用溫和**植物油脂**配方，質地輕盈好沖洗，不造成肌膚負擔。無添加爭議成分，成分黨也能安心使用。輕巧**100ml**規格，**1入**盒裝，台灣製造，讓卸妝成為溫柔的日常儀式。 (109字)\n\n---\n\n### 標籤\n#敏弱肌卸妝 #植萃卸妝油 #成分黨 #溫和清潔 #CHINGYEN", "input_tokens": 812, "output_tokens": 301}, "d949a60f0425536c0ca4": {"text": "{\n  \"title\": \"青研 CHINGYEN 純淨卸妝油 100ml 敏弱肌適用 植物油脂溫和配方 1入\",\n  \"bullets\": [\n    \"採用溫和植物油脂基底，輕盈不悶痘。\",\n    \"遇水快速乳化，沖洗後肌膚不殘留油感。\",\n    \"專為敏弱肌設計，卸妝過程溫和不刺激。\",\n    \"台灣製造，規格 100ml，單盒包裝 1入。\"\n  ],\n  \"seo_description\": \"青研 CHINGYEN 純淨卸妝油，專為敏弱肌與成分黨設計。嚴選溫和植物油脂，質地輕盈，能快速溶解彩妝髒污。100ml 規格，台灣製造，單盒 1入。快速乳化不致粉刺，給予肌膚清爽無負擔的卸妝體驗。\",\n  \"hashtags\": [\n    \"青研CHINGYEN\",\n    \"卸妝油\",\n    \"敏弱肌\",\n    \"植物油脂\",\n    \"溫和卸妝\"\n  ]\n}", "input_tokens": 919, "output_tokens": 261}, "3ebc6c7811460e4bfbd8": {"text": "## 晨光研選 複方益生菌粉 商品文案\n\n### 商品標題\n晨光研選 日本原裝 專利益生菌 50億CFU 30粒 | 告別久坐卡關，打造高效體質。\n\n### 賣點\n1. 嚴選日本專利菌株，高品質**專利益生菌**配方。\n2. 每份足量添加**50億CFU**，高活性直達，有感順暢。\n3. 精緻**30粒**包裝，每日一顆，輕鬆啟動腸道防護力。\n4. 專為久坐族設計，幫助維持消化道機能，告別悶脹。\n\n### SEO 描述\n25-45歲長時間使用螢幕的上班族必備！晨光研選推出日本原裝**專利益生菌**，每份提供高達**50億CFU**的活性菌群，幫助您維持消化道健康。專為忙碌生活設計，一盒**30粒**，輕鬆補充好菌，告別體內卡關，讓您由內而外散發光采，提升工作效率與保護力。\n\n### 標籤\n益生菌、消化保健、上班族必備、腸道健康、日本原裝", "input_tokens": 262, "output_tokens": 289}, "36c02a6b651cca61c0a4": {"text": "## 晨光研選 複方益生菌 專為久坐上班族打造的日常順暢支援\n\n---\n\n### 商品標題 (不超過 60 個字)\n\n**晨光研選 專利益生菌 50億CFU 30粒 日本原裝 每日順暢調整體質**\n\n---\n\n### 賣點 (4 條，每條不超過 30 個字)\n\n1.  嚴選日本原裝進口，品質安心可靠。\n2.  每份含 **50億CFU 專利益生菌**，高效定殖。\n3.  幫助維持消化道機能，輕鬆體內環保。\n4.  **30粒** 輕巧錠劑包裝，方便每日持續補充。\n\n---\n\n### SEO 描述 (不超過 120 個字)\n\n專為長時間久坐、飲食不規律的上班族設計。晨光研選複方益生菌，日本原裝進口，每份提供高達 **50億CFU 專利益生菌**，幫助您調整體質，維持消化道健康。每日僅需補充 **30粒** 錠劑，輕鬆打造舒適的體內環境，是您維持日常順暢的基礎保養。\n\n---\n\n### 標籤 (3-5 個)\n\n#益生菌 #消化道保健 #日本保健品 #上班族必備 #調整體質", "input_tokens": 799, "output_tokens": 318}, "5661ce62322401bedc1d": {"text": "這是一份針對「本木 BENMU 機械式鍵盤」的電商文案，強調其頂級手感、金屬質感與卓越性能。\n\n---\n\n## 💻 本木 BENMU 機械式鍵盤 104鍵：重塑您的指尖體驗\n\n**售價：NT$3290**\n\n### 🌟 標題：告別軟綿，迎接精準！職人級「本木 BENMU」鋁合金機械鍵盤，為效率而生。\n\n---\n\n### 【前言：為什麼您需要一把機械鍵盤？】\n\n您是否厭倦了普通鍵盤的遲鈍與模糊手感？對於長時間工作、程式編碼或追求極致遊戲體驗的您而言，鍵盤不只是輸入工具，更是生產力的核心。\n\n**本木 BENMU 104鍵機械式鍵盤**，結合了頂級的材質工藝與最純粹的機械軸體，為您的每一次敲擊帶來清脆、精準、且無與倫比的滿足感。這不僅是鍵盤的升級，更是您工作空間的質感升級。\n\n---\n\n### 【核心賣點深度解析】\n\n#### 💎 1. 頂級工藝：霧面鋁合金機身\n\nBENMU 鍵盤採用高規格的**霧面鋁合金**外殼，提供遠超塑膠材質的沉穩與耐用性。\n\n*   **視覺饗宴：** 低調奢華的霧面質感，完美融入任何現代辦公或電競環境。\n*   **觸感升級：** 冰涼堅固的金屬手感，有效減少鍵盤移動，確保您在高速輸入時也能保持穩定。\n*   **職人精神：** 經久耐用，體現對品質的極致追求。\n\n#### ⚙️ 2. 機械軸心：極致精準的輸入回饋\n\n104鍵全尺寸配置，無論是數字輸入、文書處理或複雜指令，都能一氣呵成。\n\n*   **清脆回饋：** 機械軸體提供明確的觸覺與聽覺反饋，讓您清楚知道每一次按鍵是否成功觸發，有效減少誤按。\n*   **提升效率：** 快速的觸發速度與穩定的鍵程，讓您的打字速度與準確度同步提升。\n\n#### 🛡️ 3. 美國製造，品質保證\n\nBENMU 鍵盤堅持美國原產地製造，確保產品從設計到組裝都符合最高標準。\n\n*   **信任基石：** 卓越的製造工藝，讓您買得安心，用得放心。\n*   **原廠承諾：** 提供**原廠一年保固**，我們為產品的耐用性與可靠性背書。\n\n---\n\n### 【商品規格一覽】\n\n| 項目 | 規格細節 |\n| :--- | :--- |\n| **品牌** | 本木 BENMU |\n| **商品名稱** | 機械式鍵盤 104鍵 |\n| **售價** | **NT$3290** |\n| **按鍵數** | 104鍵 (全尺寸) |\n| **型態** | 霧面鋁合金機身 |\n| **產地** | 美國 (USA) |\n| **保固** | 原廠一年保固 |\n| **適用對象** | 專業工作者、程式設計師、重度文字輸入者、追求質感品味的用戶 |\n\n---\n\n### 🛒 【立即行動】\n\n別讓遲鈍的鍵盤成為您效率的絆腳石。\n\n**本木 BENMU 機械式鍵盤**，是您桌面上最值得投資的生產力工具。\n\n**點擊下方連結，即刻體驗指尖上的頂級工藝！**\n\n👉 [立即購買]\n👉 [了解更多 BENMU 品牌故事]", "input_tokens": 98, "output_tokens": 860}, "b0214c6aa41b1f52d52a": {"text": "這是一份針對「青研 CHINGYEN 純淨卸妝油」設計的電商文案，強調其溫和性、獨特質地以及對敏弱肌的適用性。\n\n---\n\n## 【青研 CHINGYEN】純淨卸妝油 100ml\n**NT$390**\n\n### 💎 主標題：卸妝新革命！敏弱肌專屬的「奶油感」純淨卸妝油，溫和淨透，零負擔。\n\n---\n\n### ✨ 為什麼你需要這瓶「青研 純淨卸妝油」？\n\n許多卸妝油雖然卸妝力強，卻常讓敏弱肌感到油膩、刺激或緊繃。青研 CHINGYEN 專為亞洲敏弱肌設計，結合植物油的強效淨化力與乳霜般的溫和觸感，讓卸妝成為一種享受。\n\n**我們承諾：有效卸妝，同時呵護肌膚屏障。**\n\n---\n\n### 🌿 三大核心特色，給予肌膚最純淨的呵護\n\n#### 1. 【獨家奶油感質地】打破卸妝油的油膩迷思\n這不是一般的液態油！我們採用獨特的技術，將純淨植物油脂轉化為**輕盈的「乳霜狀」質地**。\n*   **觸感升級：** 輕柔滑順，減少肌膚拉扯。\n*   **清爽不殘留：** 遇水瞬間完美乳化，快速帶走彩妝與髒污，洗後肌膚清爽、不緊繃、不致痘。\n\n#### 2. 【敏弱肌專用配方】溫和淨化，不刺激\n專為脆弱膚質設計，嚴選親膚的天然植物油脂。\n*   **適用膚質：** 敏弱肌、乾燥肌、混合肌皆適用。\n*   **純淨保證：** 不含礦物油、酒精、人工香精、Paraben 類防腐劑，給予肌膚最純粹的卸妝體驗。\n\n#### 3. 【強效卸妝力】防水彩妝輕鬆瓦解\n雖然溫和，但卸妝力絕不打折！\n*   能有效溶解臉部底妝、防水睫毛膏、持久型唇彩等頑固彩妝。\n*   卸妝同時滋潤肌膚，洗後肌膚柔軟水潤，告別乾澀緊繃感。\n\n---\n\n### 📋 產品規格一覽\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **商品名稱** | 青研 CHINGYEN 純淨卸妝油 |\n| **容量** | 100ml |\n| **售價** | NT$390 |\n| **主要成分** | 植物油脂 |\n| **型態** | 乳霜質地 (非傳統液態油) |\n| **適用膚質** | **敏弱肌適用** |\n| **產地** | 台灣 |\n| **分類** | 美妝保養 / 卸妝 |\n\n---\n\n### 💖 卸妝步驟建議\n\n1.  **取量：** 保持手部乾燥，按壓適量（約 2-3 泵）於掌心。\n2.  **按摩：** 輕柔塗抹於乾燥的臉部，以畫圓方式按摩約 30 秒，讓卸妝油充分溶解彩妝。\n3.  **乳化：** 沾取少量清水，持續按摩臉部，直到卸妝油完全轉變為乳白色液體（此步驟是清潔關鍵）。\n4.  **沖洗：** 最後以大量清水徹底洗淨即可。\n\n---\n\n### 🛒 立即體驗！\n\n**卸妝，是保養的第一步。**\n\n給您的肌膚一次溫和、高效、純淨的卸妝體驗。\n**青研 CHINGYEN 純淨卸妝油，讓敏弱肌也能安心卸淨！**\n\n**👉 點擊加入購物車，享受奶油般的卸妝療癒時光！**", "input_tokens": 119, "output_tokens": 910}, "85aebfb01c72ddaab32d": {"text": "這是一份針對「晨光研選 複方益生菌粉」的高端電商文案，強調其日本製造、專利技術與頂級品質，以合理化 NT$4680 的高價位。\n\n---\n\n# 晨光研選 頂級專利複方益生菌\n\n## 【日本研製．極致順暢】50億CFU 專利益生菌 | 打造黃金消化道，由內而外煥發光采\n\n**💰 售價：NT$4680 (30粒/盒)**\n**🌍 產地：日本**\n\n---\n\n## 💎 產品定位：獻給追求極致效率與頂級品質的您\n\n您是否厭倦了市面上效果不彰、需要大量吞食的益生菌？\n\n「晨光研選」秉持著對品質的極致追求，從源頭開始，嚴選日本專利菌株，以高規格製程打造這款頂級複方益生菌。我們相信，真正的體內環保，不需要複雜的步驟，只需要一顆高效、穩定的「研選級」益生菌。\n\n**這不只是一般益生菌，這是您投資健康、提升生活品質的頂級選擇。**\n\n---\n\n## ✨ 為什麼選擇晨光研選？三大核心優勢，超越傳統益生菌\n\n### 1. 【日本專利技術】頂級菌株，穩定高效\n\n我們的益生菌源自對品質要求極高的**日本**，採用獨家專利技術培育。雖然單顆含量為 50 億 CFU，但我們更注重菌株的**活性、穩定性與定殖率**。確保每一顆錠劑都能有效通過胃酸考驗，直達消化道，發揮最大功效。\n\n### 2. 【複方協同作用】全面調理，不只求順暢\n\n本產品為**複方配方**，結合多種優質益生菌與益生質（Prebiotics），形成強大的協同作用。不僅能幫助消化道順暢，更能調整體質，平衡菌叢生態，讓您感受由內而外的輕盈與舒適。\n\n### 3. 【研選哲學】高規格錠劑，方便穩定\n\n告別傳統粉末易受潮、易變質的困擾。我們採用高穩定性的**錠劑型態**，精準控制每日攝取量。小巧包裝，方便攜帶，無論您身在何處，都能輕鬆維持每日體內環保。\n\n---\n\n## 🎯 產品規格一覽\n\n| 項目 | 規格細節 | 晨光研選的承諾 |\n| :--- | :--- | :--- |\n| **主要成分** | 專利益生菌 50億 CFU | 高活性、高定殖率，確保有效性。 |\n| **型態** | 錠劑 (Tablet) | 穩定性高，方便吞食，無需冷藏。 |\n| **包裝** | 30粒 / 盒 (約一個月份) | 精緻包裝，體現頂級保健品價值。 |\n| **產地** | 日本 (Japan) | 嚴謹製程，品質安心保證。 |\n| **建議用量** | 每日 1-2 粒 | 飯前或飯後搭配溫水食用。 |\n\n---\n\n## 🙋‍♀️ 誰需要「晨光研選」頂級複方益生菌？\n\n如果您符合以下任一狀況，這款日本研製的頂級益生菌將是您最佳的投資：\n\n*   **追求效率的商務人士：** 經常外食、生活壓力大，需要快速恢復消化道機能。\n*   **注重體態管理族群：** 希望透過調整體內環境，達到更好的新陳代謝。\n*   **久坐辦公室族群：** 經常感到腹部悶脹、排便不規律者。\n*   **換季敏感者：** 希望透過調整體質，提升自我保護力。\n\n---\n\n## 🛒 立即投資您的黃金消化道\n\n健康是最高級的奢侈品。「晨光研選 複方益生菌」以 NT$4680 的價格，為您帶來日本頂級的研發成果與高效的體內調理體驗。\n\n**選擇晨光研選，讓順暢不再是偶爾的幸運，而是每日的常態。**\n\n**立即加入購物車，開啟您的輕盈有感人生！**", "input_tokens": 120, "output_tokens": 1005}, "d62311eb2efcb476ea5b": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出『積雪草』能舒緩肌膚不適？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及『泛醇』或『神經醯胺』有助於保濕或修護？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否描述產品質地為『水感』或『清爽不黏膩』？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用『一秒』、『立即見效』等誇大不實的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否強調產品『敏弱肌適用』或專為敏弱肌設計？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將『5% 積雪草萃取』這個規格，轉化為對消費者的具體好處（例如：高濃度舒緩）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用表情符號（emoji）或驚嘆號，以維持理性的品牌調性？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 998, "output_tokens": 448}, "e5613a298665ca4e607e": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「多重玻尿酸」的規格，轉譯為「深層補水」或類似的消費者利益點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否明確點出此面膜適合「敏弱肌」使用？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「凝露質地」帶來「清爽不黏膩」或類似的膚觸感受？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「立即見效」、「史上最強」等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"為符合理性溫柔的語調，文案是否完全未使用驚嘆號（!）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否避免使用「我們的」、「為您帶來」等不必要的冗詞贅字？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"標題或賣點是否清楚說明這是一款「保濕」用途的面膜？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    }\n  ]\n}", "input_tokens": 991, "output_tokens": 435}, "fcd3e106e988462b6fb6": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出『多重玻尿酸』能為肌膚帶來深層或長效的保濕效果？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否明確提及產品『敏弱肌適用』或專為敏弱肌設計？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否描述了『水感質地』清爽、不黏膩的使用感受？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用『秒速』、『奇蹟』、『終結』等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用會造成焦慮或壓力的詞語（如：『再不保養就晚了』、『肌膚問題拉警報』）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"標題是否包含核心成分『玻尿酸』與主要功效『保濕』，讓消費者能一眼看懂商品用途？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否一句話只溝通一個重點，避免在單一賣點中塞入過多資訊？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 991, "output_tokens": 464}, "d1b89a4979eec98e173b": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「3件」的規格，轉化為適合小坪數家庭或簡約廚房的優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及陶瓷刀不易殘留氣味的特性，以連結寵物飼主的需求？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否點出陶瓷材質比傳統鋼刀更持久鋒利的優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「極致」、「完美」等主觀且誇大的形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否提及「終身技術支援」，以塑造品牌專業與可信賴的形象？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案的四條賣點中，是否避免重複提及品牌「Volta」或產品名「陶瓷刀」？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"SEO 描述是否包含至少一個核心優點（如：持久鋒利、不殘留氣味）與一個目標客群（如：小家庭、寵物族）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 956, "output_tokens": 457}, "69aa8f47719455856e5d": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將『50億CFU』的規格連結到『每日足量補充』或類似的益處？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出產品適合『上班族』或『長時間久坐』的族群？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及『每日補充』或『養成習慣』等持續使用的概念？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用『立即見效』、『最強』、『保證』等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全沒有使用驚嘆號？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將『專利益生菌』的優勢轉化為『品質信賴』或『來源安心』等消費者語言？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案的四條賣點是否一句話只講一個好處，沒有將同一件事情重複說明？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 997, "output_tokens": 438}, "ae77193e3c09020edf81": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提及柑橘、莓果或花香等具體風味描述？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否將「耶加雪菲」產區與其經典的明亮果酸或花香風味連結？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否點出「淺焙」是為了保留咖啡豆更豐富的原始花果風味？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「秒殺」、「必買」、「地表最強」等過度促銷或誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否專注於描述風味、香氣與口感，而非提神、醒腦等功能性效果？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否點出在家或辦公室即可輕鬆享受精品咖啡的便利性？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免出現「我們精心挑選」、「特別為您帶來」等無實質資訊的冗餘贅詞？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1007, "output_tokens": 448}, "2c53970691ef5bec9cb3": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「神經醯胺」連結到「修護肌膚屏障」或「穩定膚況」等具體功效？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出產品適合「敏弱肌」使用？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否描述乳霜質地是「好吸收」或「不黏膩」的？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否使用「日常」、「每天」等生活化詞彙，而非僅陳述產品規格？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全避免使用「！」、「～」或網路流行語？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否避免使用「最強」、「終極」、「唯一」等無法證實的最高級形容詞？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否以「你」或「你的」作為主要人稱，直接對消費者溝通？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 983, "output_tokens": 433}, "5b1a844513f43c428709": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「42dB 降噪」連結到「通勤或辦公時的專注情境」？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及「雙裝置切換」功能，並點出其在電腦與手機間無縫切換的便利性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「單耳 4.8g」的重量，並連結到「長時間配戴的舒適感」？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「極致」、「完美」、「無與倫比」等主觀性強烈的形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全不使用表情符號（emoji）或「～」、「！」等情緒性標點符號？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將「IPX4」這個規格，直接轉譯為「防汗、防潑水」等消費者能理解的日常用語？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案中關於「降噪」的描述，是否僅限於「有效降低環境噪音」，而未誇大為「完全隔絕所有聲音」？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    }\n  ]\n}", "input_tokens": 998, "output_tokens": 484}, "ab1f2624ebdc953d7c20": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"標題是否清楚標示「唇部修護」或「護唇膏」等核心功能，讓消費者第一眼就知道產品用途？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否將「油狀質地」轉化為消費者能理解的正面描述（如：潤澤不黏膩、好吸收的精華油質地）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否將「蜂蠟與植物油」的成分與「滋潤修護」或「天然溫和」等好處連結？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否點出產品適合敏弱肌膚使用？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否點出「3入」包裝在生活場景中的好處（如：方便隨身攜帶、在家與辦公室各放一個）？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否提及適合社群使用者或學生的具體使用情境（如：熬夜追劇、上妝前打底）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用「一抹見效」、「終極」等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 988, "output_tokens": 486}, "b5a6c73a68c6998c09d5": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提及「緩釋」技術能幫助一夜好眠或隔天精神飽滿？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出產品適合長時間使用電腦或手機的上班族？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「獨立包裝」方便攜帶或保持產品新鮮？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「最強」、「立即見效」、「有感」等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全沒有使用驚嘆號「！」或問號「？」，以維持專業穩重的語氣？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將「維生素B群複方」連結到具體的益處，例如「維持能量正常代謝」或「增進神經系統的健康」？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用「幫助您」、「為您帶來」等冗餘的引導詞，直接陳述產品特點？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 997, "output_tokens": 451}, "64a0b3bfb3b09e1d2745": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提及 30mg 葉黃素為衛福部建議的每日最高攝取量？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出產品適合長時間使用電腦或手機的族群？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否將「游離型」葉黃素的優點，連結到分子小或好吸收？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「立即見效」、「保證有效」等過度承諾的詞語？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全未使用驚嘆號「！」來製造強烈或促銷的語氣？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否點出「60粒」的包裝為兩個月份量？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否在標題之外，將「金盞花萃取葉黃素」簡稱為「葉黃素」以避免內文冗長？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 999, "output_tokens": 443}, "65d4c20bb44eb6f94e45": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提及茶湯帶有蜜香與熟果風味？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出茶葉產自「台東鹿野」這個知名產區？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否點出「三角立體茶包」能讓茶葉完整舒展的優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「最好喝」、「冠軍」等最高級或誇大不實的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否使用「職人」、「精心」或「焙製」等詞彙來傳達專業手工感？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否避免使用「您不容錯過」、「CP值超高」等無助於描述商品風味或特色的網路流行語？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用問句作為賣點，而是直接陳述產品特色或優點？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1010, "output_tokens": 441}, "03020a6d9faa993ba992": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出「三孔」能同時為筆電、手機、耳機充電？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及「65W」能為 MacBook 等筆記型電腦快充？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否強調「氮化鎵」技術帶來的小體積或輕巧易攜特性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「地表最強」、「無與倫比」等主觀誇飾形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否以規格（如 65W、三孔）作為溝通重點，而非情感訴求？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否避免使用連續的驚嘆號（!!）或問號（??）等強烈情緒標點？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用「的」、「了」、「呢」等非必要的語助詞或贅字？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1001, "output_tokens": 442}, "093fb461176f9c093804": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出 18000Pa 吸力能有效清除寵物毛髮或細小粉塵？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否強調 1.4kg 的輕量設計，適合單手舉起清潔高處？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及附贈的床墊吸頭，能用於清潔沙發或床鋪？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「超強」、「極致」、「完美」等主觀誇大的形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用「溫馨」、「可愛」等與科技感無關的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將「Pa」這個單位與具體吸力效果（如：吸走貓砂、灰塵）連結說明？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否完全沒有使用問句或驚嘆號，維持直接陳述的語氣？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 993, "output_tokens": 448}, "831a41191344a9abf690": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提及低溫烘焙能保留堅果的原始風味或營養？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出「無調味」適合追求原型食物或健身族群？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否強調包含四種堅果能提供豐富口感或多元營養？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否使用描述風味或口感的詞彙（如：香醇、酥脆），而不只是列出成分？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用「最強」、「第一」、「保證」等過度誇大的宣傳詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否提供具體的食用情境建議，例如「搭配優格」或「健身後補充」？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免在同一句話或賣點中，重複說明「無調味」與「無加糖鹽」等相同概念？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1007, "output_tokens": 441}, "a212656e8de3dd809e9b": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出「5%尿素」能幫助軟化乾燥粗糙的雙手？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及「5入」包裝方便在辦公室、包包等不同地方各放一條？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否點出此產品適合「乾燥缺水」的膚質？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否使用生活化情境詞彙（如「洗完手」、「辦公室冷氣房」），而非僅有功能說明？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全避免使用「！、～」等會顯得輕浮的標點符號？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否避免使用「終極」、「最強」、「立即見效」等過度誇大的詞彙？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否將「尿素」這個專有名詞與「保濕」或「潤澤」等具體功效連結說明？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 979, "output_tokens": 455}, "79fa90ad6f5303385933": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將『胺基酸』的特色連結到『溫和清潔、洗後不緊繃』的感受？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出 pH5.5 弱酸性有助於維持肌膚健康屏障？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及『洋甘菊萃取』成分能舒緩肌膚不適？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用『一秒』、『立即見效』等誇大不實的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用表情符號 (emoji) 或『～』、『喔』等非正式的語氣助詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"標題是否以簡潔的短句組成，而非一個完整的長句？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否將『綿密泡沫』的質地特色與『減少肌膚摩擦』的優點連結？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 997, "output_tokens": 446}, "7ac34e7a32f37519a695": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「甘胺酸亞鐵」的優點轉化為「好吸收」或「溫和」的說法？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出產品適合需要維持好氣色或活力的上班族？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及產品為「日本製造」或「日本製」？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「立即」、「最強」、「保證」等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全不使用驚嘆號「！」來維持專業溫和的語氣？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將「90粒」的規格換算成「三個月用量」或「90天份」的具體概念？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否在各個欄位（標題、賣點）中都避免了不必要的贅詞，例如「我們精心挑選」、「一個幫助你...的產品」？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 995, "output_tokens": 459}, "b553262bfaaa68e575bf": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提及五株專利益生菌能幫助調整體質？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出「隨身包」設計方便外食族或上班族攜帶補充？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否將「100億」的菌數連結到「每日足量補充」的概念？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「最強」、「立即見效」等誇大或過度承諾的詞語？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否傳達出「每日規律補充」是維持健康的關鍵，而非強調單次速效？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否避免使用驚嘆號，以維持專業、溫和的語氣？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否用一句話就清楚說明產品如何幫助目標受眾（例如：幫助外食族補充好菌）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    }\n  ]\n}", "input_tokens": 1006, "output_tokens": 439}, "d86225ad854573436043": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出 16L 除濕力適合大坪數或潮濕空間？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否將「陽極處理」的特色連結到機身的耐用或抗鏽蝕？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「終身技術支援」的保固服務？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否完全沒有使用「驚爆」、「史上最強」等誇張的行銷詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否強調產品的材質或結構，而非僅僅功能或外觀？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否避免使用沒有解釋的技術規格縮寫（例如：HEPA、CADR）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案的賣點描述中，是否每一句都只溝通一個核心好處，沒有混雜多個概念？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 956, "output_tokens": 431}, "e0ed0e3572c2ecdcb77d": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「德國產地」的特色，連結到「精良工藝」或「耐用品質」的印象？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及「終身技術支援」的保固承諾？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否點出此容量適合「帶寵物散步」或「小家庭外出」的具體使用情境？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否完全避免使用「最強」、「必買」、「爆款」等誇大或熱銷的宣傳詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否使用「堅固」、「耐用」、「純粹」等詞彙來描述瓶身，以符合品牌調性？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將「食品級不鏽鋼」的特色直接呈現，而未增加不必要的技術原理說明？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"標題是否清楚點出「品牌」、「品類（保溫瓶）」與「容量（750ml）」三個要素？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    }\n  ]\n}", "input_tokens": 964, "output_tokens": 465}, "3fa5e105df3265666377": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出 750ml 容量適合長時間會議或半日外出，無需頻繁補水？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及 PC+ABS 複合材質的耐用或抗摔特性，以對應差旅情境？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「台灣製造」或「一年保固」作為品質保證的賣點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「極致」、「完美」、「頂級」等主觀且空泛的形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否以規格（如容量、材質）作為溝通重點，而非營造溫馨或感性的情境？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否完全沒有使用驚嘆號，以維持專業、直接的語氣？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免出現「提供...的功能」、「擁有...的設計」這類冗餘句型？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 961, "output_tokens": 449}, "5da13c10f36bf6b5713b": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「胺基酸界面活性劑」與「溫和不刺激」或「洗後不緊繃」的感受連結？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出此產品「敏弱肌適用」？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「凝露質地」帶來的清爽膚觸？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「最強」、「終極」、「立即見效」等誇大或絕對性的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全不使用驚嘆號「！」來製造激昂或促銷感？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否使用第二人稱「你」或「您的肌膚」，直接與目標受眾溝通？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用「我們都知道」、「眾所皆知」等無助於傳達產品價值的冗詞贅字？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1005, "output_tokens": 442}, "642ab902732447367c42": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「3000mg」的規格連結到「足量補充」的益處？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否將「胜肽」的特性轉譯為「分子小」或「好吸收」的優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「養顏美容」或「維持彈潤」等與目標受眾直接相關的核心好處？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否提及「韓國」產地，以呼應品牌對來源的重視？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用「立即見效」、「保證有效」等誇大不實的過度承諾詞語？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否完全沒有使用驚嘆號（!），以維持職人品牌的專業沉穩語氣？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否點出「睡前食用」或「獨立包裝方便攜帶」等具體使用情境？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1008, "output_tokens": 455}, "726d1e7cedce6704c1b8": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出「霧面鋁合金」材質帶來的耐用或質感優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出「104鍵」配置對工作者（如：輸入數字）的便利性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「一年保固」提供的安心感或保障？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否以描述材質或工法為主，而非使用華麗的形容詞（如：極致、巔峰、奢華）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用表情符號 (emoji) 或驚嘆號，以維持樸實專業的語調？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否直接陳述產品特點，而沒有使用問句或與消費者過度對話的語氣（例如：「你是否也...？」）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用無法驗證的最高級宣稱（如：「最耐用」、「手感第一」）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    }\n  ]\n}", "input_tokens": 961, "output_tokens": 462}, "9417e0e3aee3213b0541": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提到「整顆凍乾」能保留更完整的營養或風味？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出商品適合外食族或作息不規律者？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否將「每日1份」的用法，連結到規律的日常保養情境？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「最強」、「保證見效」等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用驚嘆號或網路流行語，以維持專業溫和的語氣？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將「20粒」的規格，轉化為「20天份」等對消費者有意義的份量概念？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否用平實口語描述好處，而非使用「賦能」、「締造」等生硬書面語？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 992, "output_tokens": 445}, "dd836201232d1db80390": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出 350ml 的容量適合放入公事包或隨身小包？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及 PC+ABS 複合材質帶來的輕量或耐用特性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否將「台灣製造」作為品質或安心的保證來訴求？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「溫暖的」、「療癒的」、「美好的」等情感形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否以規格（如 350ml）為核心溝通，而非僅用抽象形容詞（如「完美尺寸」）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否有點出適合「出差族」或「通勤族」等具體使用情境？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用「史上最強」、「絕對保溫」等無法證實的誇大宣稱？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 961, "output_tokens": 444}, "336b53646aa0429dbb41": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出『30g 蛋白質』有助於熟齡族群維持肌肉量？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及產品來自『德國』產地？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否點出『分離乳清』純度高、乳糖含量低的特性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用『最強』、『冠軍』、『第一』等極端誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否使用與『風味』或『口感』相關的描述，而非僅強調功能？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否用一句話解釋『分離乳清蛋白』的好處（如：純度高、好吸收）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否以主動句（如：『補充蛋白質』）取代被動句（如：『蛋白質可被補充』）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 993, "output_tokens": 438}, "3139e44a10874366dd49": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「8段風速」與具體使用情境（如：睡眠微風、快速換氣）連結？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出此循環扇適合「有養寵物」的族群？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否點出不鏽鋼材質帶來「耐用」或「好清潔」的優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「舒適」、「涼爽」、「愜意」等主觀感受的形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否以規格或功能（如：8段風速、日本產地）作為溝通重點，而非營造情境故事？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否避免使用「最強」、「唯一」、「終極」等誇大或無法驗證的最高級詞彙？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否直接陳述賣點，避免使用「我們都知道」、「值得一提的是」等冗餘的開頭？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 958, "output_tokens": 460}, "fb4c9c667a666b3403c2": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提及「五重玻尿酸」能為肌膚分層補水或深層保濕？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出「天絲」膜布材質服貼、親膚的特性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否連結「海藻醣」成分與長效鎖水或舒緩乾燥的功效？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「一敷見效」、「立即水潤」等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全不使用驚嘆號「！」來營造激昂語氣？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否明確指出產品適合「乾燥肌」或「敏弱肌」？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用「CP值首選」、「小資必備」等與產品功效無直接關聯的行銷詞彙？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 999, "output_tokens": 441}, "6b1e6b5553b96f3db1f2": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將 Omega-3 800mg 的高含量與「養顏美容」或「維持青春」等女性在意的保養概念連結？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出「獨立包裝」方便攜帶或保持新鮮的優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「韓國產地」，並將其與高品質或純淨的形象連結？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「立即見效」、「最強」、「保證」等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否使用「嚴選」、「精粹」等詞語，來傳達品牌對原料來源的講究（職人感）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將「Omega-3 800mg」這個規格數字，轉化為「高單位」或「高濃度」等讓消費者能直接理解其價值的說法？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免在單一賣點（30字內）或標題中，重複提及同一個概念？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 996, "output_tokens": 482}, "4ae554d1d5a34ee179e3": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出此產品適合長時間使用螢幕的上班族？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及「緩釋」劑型，並連結到「長時間維持活力」的好處？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「睡前食用」，並連結到「為隔日儲備精神」的好處？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「最強」、「有感」、「立即見效」等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全沒有使用驚嘆號「！」？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否用一句話就清楚點出產品的核心功能（例如：補充能量、維持精神旺盛）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否點出「一盒為 20 天份」的實用資訊？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 999, "output_tokens": 425}, "19de2018e5a574261b4d": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出 240W 可為筆記型電腦充電？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及「終身技術支援」或「耐用」等暗示產品壽命的優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否點出適合「出差」或「多裝置」使用者？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"標題是否直接包含「240W」這個關鍵規格？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用「最棒」、「極致」、「完美」等主觀形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否避免使用「...的救星」、「...神器」等過於誇大或陳腔濫調的詞彙？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用問句，而是以直接的陳述句說明功能或優點？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 966, "output_tokens": 427}, "c8202fc4ad29e7acf1c1": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「100億CFU」連結到「每日足量補充」的好處？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出「60粒」包裝為「兩個月份量」的便利性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及此產品適合需要「調整體質」或「維持消化道機能」的族群？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「最強」、「立即見效」等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用「限時搶購」、「別再等了」等促銷感過強的字眼？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將「專利益生菌」這個詞，與「品質保證」或「來源安心」等消費者能理解的價值連結？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否清楚說明「每日1份」以及「睡前食用」的建議用法？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1002, "output_tokens": 451}, "be5f5fd9c88c0e111db9": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出產品專為敏弱肌設計？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否將『植物油脂』的優點連結到溫和不刺激的卸妝體驗？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否點出產品有別於傳統卸妝油的『乳霜質地』？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用『最強』、『第一』等過度誇大的宣傳詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否使用溫和、不具攻擊性的詞彙來描述潔淨效果（例如：使用「帶走」而非「剷除」髒污）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"標題是否直接點出產品的核心功能（卸妝）與主要客群（例如：敏弱肌）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否以一句話就能讓消費者理解一個賣點，沒有出現冗長或繞口的句子？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 993, "output_tokens": 444}, "2f1bcd48e4d150ba758c": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出「以油溶油」能溫和卸除彩妝的特性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提到卸妝後皮膚「不乾澀」或「水潤」的感受？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否強調其「純淨」或「單純」的植物油配方？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否將卸妝描寫為一天結束時的「療癒」或「放鬆」時刻？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用「秒殺」、「終極」、「最強」等過度誇大的行銷詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否以完整的句子溝通，而非僅是名詞或規格的堆疊？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否直接點出適合的對象，例如「乾肌」、「乾燥缺水」或「重視溫和保養」的族群？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 987, "output_tokens": 446}, "f3adaa4ac58339e3c877": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「全穀燕麥 25g」的規格，轉化為對應「一餐所需」或「提供飽足感」的具體好處？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出「獨立包裝」方便攜帶、沖泡或確保新鮮的優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「德國產地」，並連結到「品質安心」或「嚴謹製程」的印象？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「最強」、「唯一」、「保證見效」等過度誇大的最高級或承諾性詞語？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全沒有使用驚嘆號「！」來製造激動或促銷感？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否直接點出適合的族群，例如「健身族」或「注重飲食的家庭」？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案的四條賣點，是否都以名詞或動詞詞組開頭，而非完整的長句？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 987, "output_tokens": 472}, "e37e3bc17c4f345acca2": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出「15坪」適用於客廳或個人工作室等具體空間？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否將「食品級不鏽鋼」機身材質與耐用、安心的價值連結？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「日本產地」並連結到精緻工藝或品質保證的印象？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否完全避免使用「驚爆」、「史上最強」等誇張、煽動性的行銷詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否以描述產品材質或設計本身為主，而非營造短暫的促銷氛圍（如：未使用「限時特價」等字眼）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案的四條賣點是否各自聚焦在一個獨立好處，而沒有在一句話內混雜多個概念？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否每一句話都傳達了具體的產品資訊或使用者益處，沒有空泛的形容詞堆砌（如：極致完美、無與倫比）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 960, "output_tokens": 480}, "ea7a815c281298e9d29c": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出『積雪草萃取 10%』的核心功效是舒緩肌膚？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否描述了『凝露質地』清爽不黏膩的膚觸，以吸引重視質地的客群？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否點出產品『敏弱肌適用』，或可用於膚況不穩定的情境？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否提及具體的日常使用情境（例如：妝前保養、曬後安撫、換季調理）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全避免使用驚嘆號『！』，以維持口語但不誇大的品牌語調？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否避免使用『肌膚的超級食物』、『喚醒原生力』等無法驗證的模糊行銷詞彙？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用『我們都知道』、『眾所周知』等無意義的開場白贅字，直接切入重點？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 988, "output_tokens": 470}, "167ca6a4e01a9706c1a6": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出此為「物理性防曬」？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出「敏弱肌適用」或類似的溫和訴求？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及產品的保濕或潤澤效果？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否未使用「最強」、「必買」、「NO.1」等誇大詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用驚嘆號，以維持理性溫柔的語氣？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案中的每一條賣點，是否皆為能獨立閱讀的完整短句？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否專注於產品優點，而未與其他產品進行負面比較？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 994, "output_tokens": 407}, "c4151df72ae32ea37921": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出「神經醯胺」有助於修護肌膚屏障或提升保濕力？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及產品「水感質地」所帶來的清爽、好吸收或不黏膩等優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否明確指出產品適合「敏弱肌」族群？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用驚嘆號，以維持理性溫柔的溝通風格？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用「奇蹟」、「立即見效」等誇大不實的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將「7入」包裝的特點，與「方便攜帶」或「定量使用」等情境連結？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案的四條賣點中，是否有任何一句話只是在重複品名或規格，而沒有說明它對消費者的好處？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 996, "output_tokens": 454}, "1510cd145f7af9ed7824": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提及鑄鐵材質「均勻導熱」或「蓄熱性佳」的優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出 24cm 尺寸適合單人、小家庭或小空間使用？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否將「陽極處理」或「台灣製造」與鍋具的耐用度或品質連結？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「最強」、「必買」、「神器」等過度誇大的宣傳詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否使用「溫潤」、「紮實」、「工藝」等詞彙來描述鍋具的質感或製程？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否具體提及一種適合用此鍋具烹煮的料理（例如：煎牛排、烘蛋）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"四條賣點文案是否都以一句話完整表達，且沒有使用刪節號（...）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 958, "output_tokens": 456}, "a1630ae5cf1349948ffb": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出「低溫烘焙」能保留堅果的完整營養或風味？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出此商品適合做為「上班族」或「外食族」的營養補充？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及堅果的「產地」或「嚴選」概念，以呼應品牌名稱？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否使用至少一個描述堅果「風味」或「口感」的詞語（例如：香脆、甘甜、溫潤）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用「最強」、「必買」、「爆款」等過度誇大的行銷詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否以具體事實或特色開頭，而非「您是否還在煩惱...」之類的問句？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否清楚說明「20粒」的包裝形式（例如：獨立小包裝、20日份）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    }\n  ]\n}", "input_tokens": 993, "output_tokens": 464}, "f5c23be23c1de56cb623": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將『23種維生素礦物質』連結到養顏美容或維持好氣色的好處？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否強調『25種』配方的完整性或全面性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否點出『每日1份』的便利性，適合忙碌的現代女性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用『爆款』、『必買』、『最強』等過度銷售的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否使用『嚴選』、『專為...設計』等詞語，來傳達產品的專業與用心？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案中的每一條賣點，是否都由一個獨立的短句或詞組構成，而非冗長的複合句？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案標題或賣點是否直接點出目標受眾（如：30+女性）或其痛點（如：氣色不佳、營養失衡）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1000, "output_tokens": 465}, "343870636d7315df62ec": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「95Ra」連結到「色彩真實」或「不失真」的好處？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及「霧面鋁合金」材質來傳達質感或耐用性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否將「終身技術支援」作為一個能讓消費者安心使用的承諾？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「地表最強」、「史上最美」等誇張、最高級的形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否使用樸實、直接的語言，而非「解放你的雙眼」、「點亮你的靈感」等抽象或感性的訴求？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否點出產品適合長時間面對螢幕的族群（例如：遠距工作者、創作者）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案的賣點是否皆為精簡的詞組或短句，而非充滿「的」、「地」等助詞的長句？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 958, "output_tokens": 461}, "6824e421f05c848f72a9": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否強調此產品為「敏弱肌適用」？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出「水感質地」帶來清爽不黏膩的感受？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否將「7入」包裝連結到方便攜帶或隨處使用的優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否完全避免使用驚嘆號，或「終極」、「最強」、「必買」等強烈銷售字眼？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否聚焦於成分與質地本身帶來的益處，而非訴諸無關的感性情境（如：約會小心機）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否沒有使用「賦能」、「肌底」、「黑科技」等消費者不易理解的行銷詞彙？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否將「蜂蠟與植物油」這個組合，用一句話轉換成消費者能懂的好處（如：形成天然保護膜）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    }\n  ]\n}", "input_tokens": 994, "output_tokens": 459}, "2de7c6d4fdbf3ffd26cc": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「小分子」或「胜肽」的特性，轉化為「好吸收」的消費者利益點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否強調膠原蛋白含量為 2500mg，並將其作為一個關鍵優勢？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及添加維生素C能幫助膠原蛋白合成？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「立即見效」、「奇蹟」、「保證」等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否使用溫和、建議的語氣（如「幫助維持」、「日常補充」），而非命令或強迫的語氣（如「你必須擁有」）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否點出蔓越莓風味能提升口感或覆蓋腥味，讓補充更容易持續？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否清楚說明「每日1包」或「睡前食用」等具體的食用建議？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1019, "output_tokens": 459}, "8f39bac3fe5fc4934df3": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出「分離乳清」是高純度、低脂質的蛋白質來源？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否連結「睡前食用」與「夜間美容修護」的需求？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「30g」蛋白質是高單位或足量補充的概念？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否使用「嚴選」、「產地」、「風味」等詞彙，以符合品牌職人感調性？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用驚嘆號、或「必買」、「爆款」等熱情但非專業的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將「分離乳清蛋白」的益處，直接連結到目標客群在意的「體態」或「氣色」？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案的各條賣點是否各自獨立，沒有一句話解釋另一句話的狀況？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 994, "output_tokens": 449}, "375cd43fc2256cb3744c": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出 20cm 尺寸適合單人料理或小空間收納？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及鑄鐵材質具備蓄熱均勻的優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否將「終身技術支援」作為產品耐用度的佐證？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「美味」、「完美」、「極致」等主觀形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全不使用問句或驚嘆號，維持直接陳述的語氣？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將「鑄鐵」的優點（如：導熱、蓄熱）直接寫出，而非僅列出材質名詞？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免「必買」、「唯一選擇」等過度誇大的銷售字眼？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    }\n  ]\n}", "input_tokens": 958, "output_tokens": 432}, "362a0890aaba6a9cd295": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出產品『敏弱肌適用』的特性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否將『氧化鋅』這個成分連結到『物理性防曬』的好處？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否點出『3入』組的價值（例如：適合囤貨、更划算）？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否完全避免使用『最強』、『終極』、『保證』等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全沒有使用驚嘆號『！』，以維持理性、溫和的溝通風格？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將『油狀質地』這個特點，轉化為正向或中性的膚觸描述（例如：絲滑好推、延展性佳）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用會讓消費者困惑的非必要專有名詞（必含的『氧化鋅』除外）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 991, "output_tokens": 460}, "b93946ea42a47dd0b35b": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"足量 3000mg，每日定額補充。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"缺少「分子小」或「好吸收」等關於胜肽特性的說明\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"由內而外維持光采。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"韓國進口原料\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案用詞為「每日補充」、「維持光采」，沒有出現誇大不實的詞語。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案中未使用驚嘆號。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"建議睡前輕鬆食用。\"\n    }\n  ]\n}", "input_tokens": 825, "output_tokens": 290}, "91ff22d87e9abae615f2": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"嚴選小分子魚膠原蛋白胜肽，好吸收\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"每日足量補充 2500mg 膠原蛋白\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及維生素C或其幫助膠原蛋白合成的功能\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"持續為美麗打底\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"輕鬆融入日常保養\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"清新蔓越莓風味，粉末細緻好入口\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"睡前一包\"\n    }\n  ]\n}", "input_tokens": 810, "output_tokens": 283}, "0195c356bc4c8ebfe8a2": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及 30mg 為衛福部建議的每日最高攝取量。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"適合長時間使用電腦、手機的上班族，維持舒適晶亮。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"採用游離型葉黃素，分子小好吸收，適合每日持續補充。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"為您的日常保養打下基礎。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案中完全未使用驚嘆號「！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"每日一粒，一盒 60粒裝，輕鬆持續補充兩個月。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"每粒含足量葉黃素 30mg 及玉米黃素 6mg。\"\n    }\n  ]\n}", "input_tokens": 848, "output_tokens": 319}, "55e11e9bfca733b43406": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"高濃度胺基酸，溫和潔淨不刺激。洗後清爽不緊繃。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"貼近肌膚弱酸性 pH5.5，維持健康屏障。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"添加洋甘菊萃取，洗臉同時舒緩肌膚。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用『一秒』、『立即見效』等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用表情符號 (emoji) 或『～』、『喔』等非正式語氣助詞。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"青研 CHINGYEN 胺基酸溫和洗面乳 120ml 弱酸性 pH5.5 敏弱肌友善\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案提及『綿密細緻泡沫』，但未連結到『減少肌膚摩擦』的優點。\"\n    }\n  ]\n}", "input_tokens": 831, "output_tokens": 363}, "0f5e472197100012d3d1": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"5%高濃度積雪草，快速穩定泛紅膚況。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"搭配泛醇與神經醯胺，深層補水並強化肌膚屏障。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"水感輕盈精華，秒速吸收不黏膩。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了「秒速吸收」，屬於誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌安心適用。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"5%高濃度積雪草，快速穩定泛紅膚況。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案在標題與 SEO 描述中使用了驚嘆號「！」\"\n    }\n  ]\n}", "input_tokens": 849, "output_tokens": 309}, "bd71f867ac15b0bb133b": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"溫和的熱度能最大限度地保留堅果中珍貴的不飽和脂肪酸、維生素E及礦物質，避免高溫破壞營養結構。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"適合生酮飲食、低碳飲食、健身族群及注重健康的全家大小。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"嚴選四種高品質堅果，提供全面均衡的營養\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"清脆、不燥熱的完美口感。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「品質保證」字樣，違反了「避免使用『保證』等過度誇大詞彙」的原則。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"早餐良伴：加入優格、牛奶或燕麥片中，增加飽足感與風味。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"在「純淨無調味」賣點中，同時使用了「完全無調味」與「無添加糖、無添加鹽」，重複說明了相同概念。\"\n    }\n  ]\n}", "input_tokens": 1600, "output_tokens": 377}, "00d0a5db74509243ba23": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"採用天然植物油脂配方，溫和溶解彩妝與髒污。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"卸後水潤不緊繃。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"嚴選天然植物油脂，親膚溫和不刺激。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"享受純淨卸妝儀式感。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"親膚溫和不刺激。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"輕盈油狀質地，快速乳化，清爽無負擔。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"專為乾燥缺水肌設計\"\n    }\n  ]\n}", "input_tokens": 860, "output_tokens": 286}, "d1d6c250b7e2a9662871": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"小分子胜肽形式，好吸收利用\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"深海魚膠原蛋白胜肽，足量 2500mg\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及維生素C及其幫助膠原蛋白合成的功能。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"幫助您維持青春光采\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"幫助您維持青春光采\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"清新蔓越莓風味，順口無腥味\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"睡前一包，輕鬆養成美麗習慣\"\n    }\n  ]\n}", "input_tokens": 830, "output_tokens": 287}, "f41add14705c58f83ff6": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"蜜香熟果重發酵\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"台東鹿野 紅烏龍\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"缺少「三角立體茶包」能讓茶葉完整舒展的說明\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「頂級茶湯」、「最佳選擇」等最高級詞彙\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"溯源焙所\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"台東鹿野紅烏龍，重發酵蜜香熟果。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"三角立體茶包，90秒快速回甘。\"\n    }\n  ]\n}", "input_tokens": 879, "output_tokens": 295}, "ab8599428ca1d5f8babe": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"18000Pa 瞬效吸力，徹底清除寵物毛髮與塵蟎。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案強調了 1.4kg 輕量，但未提及「單手舉起」或「清潔高處」的應用場景。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"附贈床墊吸頭，有效對抗沙發與寢具過敏源。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「超輕量」、「強勁」、「救星」、「極輕量」等主觀誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"搭載 18000Pa 瞬效吸力，輕鬆吸除毛髮與灰塵。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"18000Pa 瞬效吸力，徹底清除寵物毛髮與塵蟎。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"標題與 SEO 描述中使用了驚嘆號「！」。\"\n    }\n  ]\n}", "input_tokens": 908, "output_tokens": 369}, "2b51e72362f8da03f86d": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"厭倦了通勤的喧囂和辦公室的雜音嗎？ Volta Pro 2 搭載業界領先的 42dB 深度主動降噪技術，為您打造一個專屬的聲音結界。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"更支援雙裝置智慧切換功能，讓您在手機與筆電間無縫銜接，工作娛樂兩不誤。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"單耳僅重 4.8 克，輕若無物，長時間使用也毫無負擔。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「極致」一詞，例如「極致降噪」、「極致輕量」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了表情符號（如 🤫、🔋）和情緒性標點符號（！）。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"IPX4 (防汗、防潑水)\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「世界瞬間靜音」、「外界噪音瞬間消弭」等描述，誇大了降噪效果，而非僅限於「有效降低環境噪音」。\"\n    }\n  ]\n}", "input_tokens": 2109, "output_tokens": 404}, "5a224f1bd103a976867f": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專為25-45歲上班族設計的晨光研選緩釋B群錠。長時間使用螢幕...\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"緩釋劑型設計，穩定釋放營養。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"適合睡前食用，為隔日工作做好準備。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「最強」、「有感」、「立即見效」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案完全沒有使用驚嘆號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"支援長時間工作與用眼，維持精神。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「20粒」，但未說明每日食用量，因此無法判斷一盒是幾天份。\"\n    }\n  ]\n}", "input_tokens": 821, "output_tokens": 322}, "d43b41ffbf9f4b2c923d": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"【青研 CHINGYEN】胺基酸溫和洗面乳 120ml | 溫和清潔不緊繃\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"弱酸 pH5.5：守護肌膚天然屏障\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"特別添加洋甘菊萃取精華，具有優異的舒緩與安撫功效，幫助鎮靜肌膚，緩解因環境或清潔帶來的微小不適...\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用『一秒』、『立即見效』等誇大詞彙，整體語氣務實，例如「緩解因環境或清潔帶來的微小不適」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了 💎, ✨, 📋, 💡, 👉, 🛒 等表情符號 (emoji)。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"告別緊繃乾澀！洗臉，是保養的第一步。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"綿密豐盈泡沫：零摩擦感潔顏體驗...能減少手部與臉部肌膚的摩擦...\"\n    }\n  ]\n}", "input_tokens": 1464, "output_tokens": 401}, "ae45d128b37e9a7151b4": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"胜肽級小分子，吸收率極佳，美麗直達。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"足量添加 2500mg 膠原蛋白，高效有感。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"搭配維生素C協同作用。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「立即見效」、「奇蹟」、「保證」等過度誇大的詞彙，使用的詞彙如「高效有感」、「輕鬆為美麗打底」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"輕鬆養成美妍習慣。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"嚴選天然蔓越莓風味，酸甜順口無腥味。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"每日一包，輕鬆養成美妍習慣。\"\n    }\n  ]\n}", "input_tokens": 895, "output_tokens": 319}, "4546259b43a22992b988": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"晨光研選 五益菌複方益生菌粉 30包入 | ... | 調整體質\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"粉末隨身包設計，適合忙碌外食族群。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"每包足量 100億 CFU 益生菌...每日一包...方便持續補充\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「最強」、「立即見效」等誇大詞語，語氣中性客觀。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"每日持續補充，輕鬆做好日常保健。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案中未使用驚嘆號。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"晨光研選五益菌複方益生菌粉，專為外食族及作息不規律者設計。\"\n    }\n  ]\n}", "input_tokens": 852, "output_tokens": 340}, "0bb184b778086456f4c2": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"五重玻尿酸配方，層層補水，維持肌膚水嫩。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用輕薄服貼的天絲蠶絲膜布，親膚透氣。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及「海藻醣」成分。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"提供深層且持久的保濕效果。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"一盒5片裝，幫助肌膚維持健康水循環。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"乾燥敏弱肌適用。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"溫和成分，質地清爽，乾燥敏弱肌適用。\"\n    }\n  ]\n}", "input_tokens": 842, "output_tokens": 300}, "d8cf4f59a77da910b996": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"每日足量 3000mg 深海魚膠原蛋白胜肽\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"胜肽小分子結構，吸收利用率佳\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"輕鬆維持日常光采。職人精神，為您的美麗打底。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"韓國原裝進口\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"輕鬆維持日常光采\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"溯源焙所嚴選高品質深海魚膠原蛋白胜肽，專為30-50歲注重保養的女性設計。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"獨立包裝好攜帶、睡前補充\"\n    }\n  ]\n}", "input_tokens": 813, "output_tokens": 307}, "3095f6ca3e96da6a7778": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"呈現獨特蜜香與熟果韻味。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"嚴選台東鹿野產區茶葉\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"三角立體茶包設計，完整釋放茶葉精華。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"重發酵工藝，呈現獨特蜜香與熟果韻味。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"職人級焙製\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"是茶葉愛好者或商務送禮的優雅選擇。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"重發酵工藝，呈現獨特蜜香與熟果韻味。\"\n    }\n  ]\n}", "input_tokens": 848, "output_tokens": 301}, "507ba2e1c27f0b33657b": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專為長時間螢幕族設計，幫助舒緩緊繃壓力。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"獨特緩釋技術，長效釋放不浪費，持續有感。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"睡前一顆，啟動夜間修復，醒來精神更飽滿。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「有感」一詞，屬於應避免的誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現驚嘆號「！」。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"夜間修護，日間續航，告別螢幕疲勞\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「一盒20粒」與「每日睡前食用」，但未直接整合說明「一盒為 20 天份」。\"\n    }\n  ]\n}", "input_tokens": 851, "output_tokens": 339}, "156b8c92a8b588568fb7": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"柑橘莓果調性，口感清爽無負擔。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"嚴選衣索比亞耶加雪菲，淺焙保留花果香。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"採用淺焙工藝，完整保留柑橘與莓果的細緻花香。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案專注於描述風味（如「柑橘莓果調性」）與品質（如「精品級」），未包含「秒殺」、「必買」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"柑橘莓果調性，口感清爽無負擔。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"讓您在辦公室或家中都能輕鬆製作冰咖啡。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案風格直接，例如「嚴選衣索比亞耶加雪菲，淺焙保留花果香」，有效傳遞資訊，未出現「我們精心挑選」等冗詞。\"\n    }\n  ]\n}", "input_tokens": 917, "output_tokens": 371}, "f00dc47ee9c71282d738": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"擁有 18000Pa 強大吸力，輕鬆吸除毛髮與灰塵。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提及 1.4kg 輕量，但未說明其適合單手舉起清潔高處的用途。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"軟絨滾筒與床墊吸頭，專攻地板與織物。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了「強吸力」、「強勁吸力」、「強大吸力」，屬於主觀誇大的形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"標準模式續航35分鐘，清掃不中斷。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"擁有 18000Pa 強大吸力，輕鬆吸除毛髮與灰塵。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"機身僅 1.4kg，輕巧好操作。\"\n    }\n  ]\n}", "input_tokens": 840, "output_tokens": 350}, "548e0931e1c6d1c16494": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"完整保留堅果的天然風味與營養。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"是健身族群的能量補充、更是家庭常備的原型食物首選。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"缺少說明四種堅果能帶來豐富口感或多元營養。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"缺少描述風味或口感的詞彙（如：香醇、酥脆）。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案中沒有使用「最強」、「第一」、「保證」等詞彙。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"是健身族群的能量補充\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"在賣點「堅持純粹無添加，完全無調味、無糖鹽」中重複了相同概念。\"\n    }\n  ]\n}", "input_tokens": 882, "output_tokens": 318}, "0faaae133648233f12ac": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"小分子結構，吸收率遠勝一般大分子膠原蛋白。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"高濃度：每包足量 2500mg，高效補充每日所需。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"維生素C 是合成膠原蛋白不可或缺的「催化劑」！\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"讓美麗由內而外自然綻放。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"建議每日 1 包。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"獨家調配清爽酸甜的蔓越莓風味，完美遮蓋腥味，讓妳的每日保養不再是負擔，而是一種享受！\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"每日用量：建議每日 1 包。 最佳時機：建議於睡前 1 小時，搭配溫水沖泡或直接食用。\"\n    }\n  ]\n}", "input_tokens": 1542, "output_tokens": 346}, "08d1b4a7cf7ef32c5e09": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"核心科技：五重玻尿酸，層層滲透鎖水\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"輕薄服貼：彷彿第二層肌膚...親膚透氣：溫和不刺激\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"讓肌膚在補水同時，強化自身保濕屏障。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案中沒有出現「一敷見效」、「立即水潤」等過度誇大的詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了驚嘆號，例如：「告別乾荒！」、「缺水，是所有肌膚問題的根源！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"適用膚質 | 特別推薦乾燥肌、缺水肌、換季不適肌\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案內容聚焦於產品功效，未出現「CP值首選」、「小資必備」等與功效無直接關聯的行銷詞彙。\"\n    }\n  ]\n}", "input_tokens": 1433, "output_tokens": 366}, "f0a11b73b9901fb2a4d5": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"以精準烘焙帶出柑橘、莓果與花香的明亮調性。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"衣索比亞 耶加雪菲產區，風味純淨明亮。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"精品淺焙工藝，完整保留柑橘與莓果調性。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"享受頂級精品咖啡的便利選擇。溯源焙所嚴選衣索比亞 耶加雪菲 淺焙 咖啡豆，以精準烘焙帶出柑橘、莓果與花香的明亮調性。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"以精準烘焙帶出柑橘、莓果與花香的明亮調性。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"每盒內含 10包，隨時隨地享受職人級風味。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「嚴選」，屬於無實質資訊的贅詞。\"\n    }\n  ]\n}", "input_tokens": 858, "output_tokens": 372}, "2fb523e9291de273115f": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"Volta Pro 2 專為通勤與遠距工作設計。搭載 42dB 強效降噪技術，在吵雜環境中也能保持專注。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"僅提及「支援雙裝置快速切換」，未點出在電腦與手機間切換的便利性。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"僅提及「單耳 4.8g 輕量設計」，未連結到「長時間配戴的舒適感」。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「極致」、「完美」、「無與倫比」等主觀性強烈的形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案中未使用表情符號或「～」、「！」等情緒性標點符號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"僅提及「IPX4 防水等級」，未轉譯為「防汗、防潑水」等日常用語。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"42dB 主動降噪，有效隔絕環境噪音，保持專注。\"\n    }\n  ]\n}", "input_tokens": 891, "output_tokens": 381}, "c9955af8fa1ad8b2d4b4": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"採用高純度植物油脂配方，能迅速滲透毛孔，溫和溶解頑固彩妝、防曬與空氣髒污。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"【專為乾燥缺水肌設計】洗後水潤不緊繃\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"【100% 植物油脂基底】...採用高純度植物油脂配方...嚴選純淨成分。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"讓卸妝成為一天中最放鬆的時刻！\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「秒殺」、「終極」、「最強」等詞彙，整體語調溫和，例如「乾肌的療癒級卸妝體驗」。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"您是否受夠了卸妝後肌膚的緊繃感？對於乾燥缺水肌而言，傳統卸妝產品往往會帶走肌膚必需的水分與油脂，導致乾燥、泛紅甚至細紋。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"適用對象：特別推薦給乾燥肌、缺水肌，以及追求溫和卸妝體驗的敏感肌膚。\"\n    }\n  ]\n}", "input_tokens": 1505, "output_tokens": 402}, "ea838e0bd732d7cb8168": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"高濃度 5% 積雪草萃取，溫和舒緩肌膚。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"搭配泛醇與神經醯胺，維持肌膚健康屏障。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"水感輕盈精華質地，快速吸收不黏膩。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用『一秒』、『立即見效』等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"高濃度 5% 積雪草萃取，溫和舒緩肌膚。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案中未使用任何表情符號或驚嘆號。\"\n    }\n  ]\n}", "input_tokens": 848, "output_tokens": 313}, "dfd42729c74b64341dcc": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"42dB 深度降噪，隔絕捷運喧囂，為通勤族打造專屬靜音艙。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「雙裝置無縫切換」，但未點出在「電腦與手機」之間切換的具體便利情境。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"單耳僅 4.8g 輕量設計...長時間配戴依然舒適。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「最佳選擇」、「極速」等主觀性強烈的形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案完全不含表情符號（emoji）或「～」、「！」等情緒性標點符號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「IPX4 防水」，未轉譯為「防汗」或「防潑水」等更具體的日常用語。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"有效消除環境雜音，提升專注力。\"\n    }\n  ]\n}", "input_tokens": 977, "output_tokens": 369}, "9e6902fc4655c10fefbb": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"獨特蜜香與熟果韻\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"台東鹿野產地\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"三角立體茶包設計，讓茶葉完整舒展\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「最佳選擇」，屬於最高級詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"溯源焙所\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"獨特蜜香與熟果韻，尾韻回甘\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"三角立體茶包，完整釋放茶葉原味\"\n    }\n  ]\n}", "input_tokens": 819, "output_tokens": 285}, "fcf7c5de4ae66a9141b9": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"足量添加 3000mg，每日補足青春能量。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"嚴選深海魚膠原蛋白胜肽，分子小好吸收。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"重現Q彈緊緻\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"韓國原裝進口\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用「立即見效」、「保證有效」等誇大不實的詞語。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案完全沒有使用驚嘆號。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"建議睡前食用，把握夜間黃金修護期。\"\n    }\n  ]\n}", "input_tokens": 883, "output_tokens": 299}, "fddb325043c76d1aa788": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"低溫慢焙，保留堅果原始風味。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"是健身與家庭常備的純淨原型食物。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案提及有四種堅果，但未說明這能提供豐富口感或多元營養。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"鎖住堅果酥脆口感。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"職人嚴選 低溫烘焙綜合堅果\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案提及適合健身族群與家庭，但未提供如「搭配優格」或「健身後補充」等具體食用情境或方式。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"SEO 描述中同時出現「完全無調味、無添加糖鹽」，概念重複。\"\n    }\n  ]\n}", "input_tokens": 797, "output_tokens": 326}, "f042e5d8d76afb555ec7": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"使用高純度胺基酸配方，溫和清潔不刺激。洗後水潤不緊繃。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"弱酸性 pH5.5，維持肌膚健康屏障。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"添加洋甘菊萃取，舒緩敏弱不適。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇使用「溫和淨膚」、「舒緩敏弱不適」等詞彙，未見誇大宣稱。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"青研 CHINGYEN 專為敏弱肌設計。使用高純度胺基酸配方，溫和清潔不刺激。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"青研 CHINGYEN 胺基酸溫和潔顏乳 120ml | pH5.5弱酸配方 敏弱肌專屬的修護級洗顏\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案提及『豐盈綿密泡沫』，但未連結到『減少肌膚摩擦』的優點。\"\n    }\n  ]\n}", "input_tokens": 851, "output_tokens": 387}, "5b2ac60849d6f410b913": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"18000Pa颶風吸力，深入清潔毛髮與微塵。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"輕量化機身僅1.4kg，單手操作不費力，高處清潔無負擔。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"配備床墊吸頭與軟絨滾筒，有效處理寵物毛髮與皮屑。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"Volta 手持無線吸塵器 Lite 18000Pa強勁吸力 輕量1.4kg 小坪數/寵物家庭適用\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"標準模式續航35分鐘，滿足小坪數空間快速清掃需求。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"18000Pa颶風吸力，深入清潔毛髮與微塵。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"輕量化機身僅1.4kg，單手操作不費力，高處清潔無負擔。\"\n    }\n  ]\n}", "input_tokens": 876, "output_tokens": 372}, "736e78abc30a1a6ee3d3": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"5%積雪草舒緩精華\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提及泛醇與神經醯胺，但未說明其具體功效為保濕或修護，僅提及『提升肌膚舒適度』。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"水感清爽質地，快速吸收不黏膩。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用『一秒』、『立即見效』等誇大詞彙，例如「溫和穩定膚況」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"5%高純度積雪草萃取，溫和穩定膚況。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用表情符號或驚嘆號，維持理性的品牌調性。\"\n    }\n  ]\n}", "input_tokens": 827, "output_tokens": 335}, "e99c234a1ea7e0496f9e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"配置USB-C×2與USB-A×1，可同時為筆電、手機、平板充電。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"65W大功率輸出，滿足筆電、平板等高耗能裝置快充需求。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"採用第三代氮化鎵技術，實現輕量化設計，尺寸僅5.2 × 5.2 × 2.9 cm。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"採用第三代氮化鎵技術，實現輕量化設計，尺寸僅5.2 × 5.2 × 2.9 cm。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"65W大功率輸出，滿足筆電、平板等高耗能裝置快充需求。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"重量僅112g，內建過壓、過流、過熱保護機制，確保充電安全。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"滿足筆電、平板等高耗能裝置快充需求。\"\n    }\n  ]\n}", "input_tokens": 887, "output_tokens": 374}, "76b1ff34956192171d79": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"5% 積雪草萃取，迅速舒緩肌膚不適，有效退紅，強化肌膚耐受性。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"結合泛醇與神經醯胺，從根本修護受損屏障，提升肌膚鎖水力與防禦力。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"輕盈的水感精華質地，秒速吸收，不黏膩、不悶痘\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「秒速吸收」這個誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌與不穩定肌膚設計\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"高濃度修護 | 業界少見 5% 積雪草萃取，迅速舒緩肌膚不適\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了多個表情符號（如💎、✨、💖、👉）和驚嘆號。\"\n    }\n  ]\n}", "input_tokens": 1564, "output_tokens": 362}, "199e6454a8fa89447425": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"五重玻尿酸配方，分層滲透，深度補水。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"高親膚天絲膜布，輕薄服貼\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"添加海藻醣精華，幫助肌膚維持長效水潤感。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"幫助肌膚維持長效水潤感。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌安心使用。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「成分黨必備」，此為與產品功效無直接關聯的行銷詞彙。\"\n    }\n  ]\n}", "input_tokens": 911, "output_tokens": 301}, "d63bcb4d7ccab8c86363": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"採用高純度植物油脂，能徹底溶解防水彩妝與毛孔髒污\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"卸妝後肌膚依然水潤，告別緊繃乾澀感。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"純淨植萃卸妝油\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"讓卸妝成為每日最療癒的保養步驟。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"溫和不刺激，乾燥缺水肌專用\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"專為乾燥缺水肌設計，卸妝後肌膚依然水潤，告別緊繃乾澀感。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"乾燥缺水肌專用\"\n    }\n  ]\n}", "input_tokens": 900, "output_tokens": 309}, "e10a08c538514af6d454": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"不再需要為了手機、平板、耳機分別攜帶充電頭。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"搭載強勁的 65W 總輸出功率，足以應付絕大多數主流筆記型電腦，包括 MacBook Pro 13...\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"第三代 GaN 氮化鎵技術 — 體積縮小 50%\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了「速度巔峰」、「能量超越想像」等主觀誇飾形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案以「65W 全速輸出」、「第三代 GaN 氮化鎵技術」、「三孔配置」等規格作為核心優勢進行溝通。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案中沒有出現連續的驚嘆號（!!）或問號（??）。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案句子簡潔，沒有明顯非必要的「的」、「了」、「呢」等語助詞或贅字。\"\n    }\n  ]\n}", "input_tokens": 1590, "output_tokens": 373}, "59de1a80f11f957a49e6": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案提及了筆電、手機、平板，但未提及耳機\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"65W PD極速快充，筆電電力瞬間回血。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"採用氮化鎵技術，體積縮小50%，輕巧好攜帶。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"具備USB-C×2+USB-A三孔輸出，最高65W PD快充，輕鬆應付筆電、手機、平板。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"【Volta】迷你口袋型 65W 氮化鎵三孔快充頭｜PD/QC 極速充電\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"僅112g超輕量，出差旅行一顆抵三顆。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"筆電手機一次搞定\"\n    }\n  ]\n}", "input_tokens": 872, "output_tokens": 345}, "46745e6ca7db7658a860": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"五重玻尿酸精華，層層注水直達肌底。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"輕薄天絲蠶絲膜布，極致服貼親膚透氣。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"嚴選五重玻尿酸與海藻醣，深層補水鎖水。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"密集修護乾燥缺水\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計的青研 CHINGYEN 多重玻尿酸保濕面膜。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌專屬\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案缺少對「CP值首選」、「小資必備」等詞彙的避免，反而使用了「安心首選」、「超值組」、「成分黨必備」等相似的行銷詞彙。\"\n    }\n  ]\n}", "input_tokens": 863, "output_tokens": 342}, "1c6457240d1289751a35": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"適合長時間用腦、經常感到疲勞的上班族群持續補充。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用緩釋技術，持續釋放營養，幫助維持整日精神。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"睡前輕鬆補充，為您的日常活力打底。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案使用「幫助維持」、「支持正常代謝」等中性表述，未出現誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"晨光研選 緩釋B群錠 維生素B群複方 100mg 20粒 睡前補充\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"採用緩釋技術，持續釋放營養，幫助維持整日精神。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「每盒 20粒」與「每日一錠」，但未直接整合說明「一盒為 20 天份」。\"\n    }\n  ]\n}", "input_tokens": 829, "output_tokens": 355}, "5435772576a2ae29d98e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"搭載 42dB 主動降噪 技術，讓您在通勤或遠距辦公中保持專注。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「雙裝置切換」與「工作娛樂無縫銜接」，但未明確點出是在「電腦與手機」之間切換的便利性。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「單耳4.8g輕量設計」，但未將其連結到「長時間配戴的舒適感」這個具體的好處。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案使用「42dB深度降噪」、「32小時長效續航」等客觀描述，未使用「極致」、「完美」等主觀性強烈的形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"Volta Pro 2 專為高效率人士設計。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"IPX4防水防汗\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"42dB深度降噪，高效隔絕環境雜音\"\n    }\n  ]\n}", "input_tokens": 951, "output_tokens": 377}, "f1b3705233b5b205b9a2": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"缺少將「胺基酸」與「洗後不緊繃」的感受連結，文案是將「洋甘菊萃取」與「洗後不緊繃」連結。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"弱酸性 pH5.5，維持肌膚健康平衡。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"僅提及「膚感溫和」，未點出「洋甘菊萃取」有「舒緩肌膚不適」的功效。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"採用胺基酸界面活性劑，溫和清潔不刺激。弱酸性 pH5.5 配方，貼近肌膚酸鹼值，洗後不乾澀。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計的青研胺基酸溫和洗面乳，容量 120ml。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"青研 CHINGYEN 胺基酸溫和洗面乳 120ml 弱酸性 pH5.5 敏弱肌適用\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"輕盈綿密泡沫，減少摩擦感，好沖洗。\"\n    }\n  ]\n}", "input_tokens": 850, "output_tokens": 403}, "a1901db0ce1c4745d744": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"每日一包，輕鬆補給，調整體質好幫手。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為外食族與作息不規律的上班族設計。…輕巧隨身包\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"足量 100億 活性益生菌，幫助維持戰力。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案使用「幫助維持」、「好幫手」等溫和詞彙，未出現誇大用語。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"持續補充，輕鬆做好體內環保。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案全文未使用驚嘆號，語氣專業溫和。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案有提及目標受眾（外食族與上班族）及產品能提供的幫助（維持消化道機能），但分散在不同句子，未用一句話清楚說明。\"\n    }\n  ]\n}", "input_tokens": 864, "output_tokens": 350}, "26e31fac61a7e2ad6075": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"讓您輕鬆對抗台灣潮濕氣候，不再擔心霉味與衣物曬不乾！\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"陽極處理機身，耐用度大升級。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"享終身技術支援，買得最安心。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了「必搶」、「業界唯一」等較誇張的行銷詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"陽極處理機身，耐用度大升級。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案中沒有使用未經解釋的技術規格縮寫。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"賣點混合了多個概念，例如第一點同時講「除濕力」和「價格」。\"\n    }\n  ]\n}", "input_tokens": 853, "output_tokens": 325}, "0bd6445bb9b1c4f66d97": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及30mg為衛福部建議的每日最高攝取量。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為長時間使用螢幕的上班族設計。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"足量游離型葉黃素 30mg，吸收率佳。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"幫助維持晶亮舒適。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"每盒 60粒，每日一粒，持續補充兩個月。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"每盒 60粒，每日一粒，持續補充兩個月。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"SEO描述開頭仍使用「金盞花萃取葉黃素」全名，未簡稱。\"\n    }\n  ]\n}", "input_tokens": 896, "output_tokens": 316}, "d9ad71f6baa033df172e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"【溯源焙所】台東鹿野紅烏龍 | 蜜香熟果，便利不減頂級風味\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"我們的紅烏龍，堅持選用台東鹿野優質茶葉。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"三角立體設計：採用食品級耐熱材質，提供足夠空間讓 3g 的完整茶葉充分舒展，釋放完整風味。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「頂級」、「極致」、「最純正」、「最便利」等最高級詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"內含的不是茶末，而是經過精選、切碎適中的紅烏龍原葉\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"完美融合了烏龍的清雅與紅茶的甘甜，成就了紅烏龍無可取代的魅力。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"產地溯源：來自黃金茶區的醇厚底蘊\"\n    }\n  ]\n}", "input_tokens": 1643, "output_tokens": 371}, "b010350103b4264406e5": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"採用低溫慢焙工法，完整保留堅果原始風味與營養。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"是注重原型食物的家庭與健身族群最佳選擇。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案僅列出包含四種堅果，未說明這能帶來豐富口感或多元營養。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「原始風味」、「健康美味」，但缺乏更具體的風味或口感描述詞彙（如：酥脆、香甜）。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「最佳選擇」，屬於應避免的誇大宣傳詞彙。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"健身族群與家庭常備，補充優質油脂與膳食纖維\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"在同一個賣點中重複提及「完全無調味、無添加糖鹽」。\"\n    }\n  ]\n}", "input_tokens": 854, "output_tokens": 349}, "1254b23b0957ea128026": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"經典耶加雪菲，柑橘莓果風味。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"經典耶加雪菲，柑橘莓果風味。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"精選淺焙，保留細緻花香。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溯源焙所嚴選衣索比亞耶加雪菲，以專業淺焙技術，展現乾淨明亮的柑橘與莓果調性。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"展現乾淨明亮的柑橘與莓果調性。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"讓您隨時隨地享受高品質冷萃咖啡。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"經典耶加雪菲，柑橘莓果風味。\"\n    }\n  ]\n}", "input_tokens": 825, "output_tokens": 316}, "388821a8e66c8483df6b": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"每日除濕量達 16L，輕鬆應對台灣潮濕季節。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用美國陽極處理工法，機身結構紮實耐用。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"終身技術支援服務，讓您買得安心，用得長久。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"樸實耐用 首購族CP值首選\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"採用美國陽極處理工法，機身結構紮實耐用。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案中使用的規格 (16L) 與詞彙 (CP值) 均為消費者普遍理解的用語，無難懂的技術縮寫。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"第四點賣點「高CP值入門款，功能到位，為居家環境帶來乾爽」混雜了性價比、功能性、基礎效果等多個概念。\"\n    }\n  ]\n}", "input_tokens": 804, "output_tokens": 358}, "ab1f45f576f995bedbfd": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及 30mg 為衛福部建議的每日最高攝取量\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"長時間螢幕族必備\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"未將「游離型」的特性與「分子小」或「好吸收」等優點連結說明\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"輕鬆維持晶亮健康\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"台灣研製，品質安心，是您數位生活必備的眼部保養品。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"一盒 60粒 滿足雙月保養需求\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"晨光研選高濃度游離型葉黃素，每粒足量 30mg\"\n    }\n  ]\n}", "input_tokens": 951, "output_tokens": 322}, "521c2ef3a1658474d41c": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"單包足量 3000mg！\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用深海魚膠原蛋白胜肽，分子極小...身體能快速吸收利用\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"睡前一包，喚醒澎潤光采！\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"韓國原裝進口\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案中沒有出現「立即見效」、「保證有效」等誇大不實的詞語。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中多處使用驚嘆號，例如：「喚醒澎潤光采！」、「美麗無需等待！」等。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"建議「睡前食用」...獨立包裝設計，方便衛生...方便攜帶，無論是辦公室、旅行或外出...\"\n    }\n  ]\n}", "input_tokens": 1515, "output_tokens": 337}, "e1ffd387f0bed956e519": {"text": "{\"results\":[{\"id\":\"R1\",\"passed\":true,\"evidence\":\"經常加班、壓力大的上班族\"},{\"id\":\"R2\",\"passed\":true,\"evidence\":\"我們採用獨家緩釋技術，讓維生素B群不再一次性釋放，而是像「定時鬧鐘」一樣，在您體內緩慢、穩定地釋放長達數小時。\"},{\"id\":\"R3\",\"passed\":true,\"evidence\":\"每天睡前一錠，為隔日電量預先充飽！\"},{\"id\":\"R4\",\"passed\":false,\"evidence\":\"文案中使用了「有感」一詞，例如「持續有感」。\"},{\"id\":\"R5\",\"passed\":false,\"evidence\":\"文案中多處使用了驚嘆號「！」。\"},{\"id\":\"R6\",\"passed\":true,\"evidence\":\"每天睡前一錠，為隔日電量預先充飽！\"},{\"id\":\"R7\",\"passed\":false,\"evidence\":\"文案提供了「20粒/盒」與「每日1錠」的資訊，但沒有直接點出「一盒為 20 天份」。\"}]}", "input_tokens": 1946, "output_tokens": 235}, "983211a4054e5df03173": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"嚴選五株專利益生菌...有效調整體質。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"獨立隨身包，方便攜帶\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"每包足量100億CFU\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"高效有感\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"一盒30包足月補給。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案在標題與SEO描述中使用了驚嘆號（！）。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"專為外食族設計，調整消化道機能，維持健康。\"\n    }\n  ]\n}", "input_tokens": 902, "output_tokens": 288}, "fe2af39e91d28f21093f": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案提及可同時為「筆電、平板、手機」充電，但未提及「耳機」。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"65W高功率輸出 筆電也能快速回血\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"採用第三代氮化鎵 體積縮小50%\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案以規格和功能陳述為主，例如「採用第三代氮化鎵 體積縮小50%」，未見誇飾詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"Volta 65W 氮化鎵快充頭，專為多裝置使用者與出差族設計。提供三孔（雙C一A）輸出...\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"僅112g輕量化設計 差旅無負擔\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"65W高功率輸出 筆電也能快速回血\"\n    }\n  ]\n}", "input_tokens": 844, "output_tokens": 359}, "c0ee4bd1c411a9625d86": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"副標題提及「柑橘莓果花香一應俱全」，內文描述「完整保留咖啡豆中最珍貴的 柑橘、莓果、花香」。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"嚴選咖啡故鄉 衣索比亞 耶加雪菲，得天獨厚的風土孕育出迷人的花果香氣。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"專業淺焙，完整保留咖啡豆中最珍貴的 柑橘、莓果、花香，告別焦苦，只留清甜。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案整體語調專注於風味描述，例如「前調： 清新的柑橘香氣率先引爆味蕾...中調： ...飽滿而甜美的莓果氣息...尾韻： 乾淨優雅的花香緩緩收尾」，未見「秒殺」、「必買」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"口感特色： 乾淨、輕盈、酸甜平衡。冷萃後，咖啡的苦澀感降到最低，酸質轉化為清爽的果酸，即使不加糖奶也風味十足。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"極致便利 | 獨立掛耳包裝（12g/包），免磨豆、免濾紙，隨時隨地都能享受，辦公室或戶外露營的最佳選擇。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「【溯源焙所】聽到你的心聲了！」、「為喜愛明亮酸質與豐富層次的你量身打造」等無實質資訊的冗餘贅詞。\"\n    }\n  ]\n}", "input_tokens": 1789, "output_tokens": 517}, "9f5d73159069501aaf2f": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「國際公認的有效保養劑量」，但未說明 30mg 是衛福部建議的「每日最高攝取量」。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"長時間面對手機、電腦，感到疲勞乾澀？\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"高規格配方：30mg 游離型葉黃素，分子小好吸收。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"每日一粒，輕鬆維持晶亮狀態。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案在多處使用了驚嘆號「！」，例如「數位族必備！」，不符合「完全未使用」的要求。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"一盒 60 粒裝，可食用兩個月，高 CP 值。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"誰最需要晨光研選葉黃素？\"\n    }\n  ]\n}", "input_tokens": 2254, "output_tokens": 345}, "d0d29d9c8005404fe6af": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"高劑量，讓好菌快速佔領地盤，有效調整體質。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"無論是辦公室、旅行或外出，隨時隨地都能輕鬆補充。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"足夠的菌數是益生菌發揮作用的關鍵。我們承諾，每包 2g 的粉末中，含有高達 100 億 CFU 的高活性益生菌。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"我們不只提供益生菌，我們提供的是經過科學驗證的「腸道精準營養管理」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"簡單一步，啟動健康循環\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案在標題、前言結尾、及最終號召處使用了驚嘆號。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"每日一包，精準補充，為您的消化系統重新注入活力，找回由內而外的輕盈與舒適！\"\n    }\n  ]\n}", "input_tokens": 1481, "output_tokens": 372}, "adad9ce8fdfa1c0da03b": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"缺少「以油溶油」或類似的原理說明，僅提及「油狀質地...帶走彩妝髒污」。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"洗後水潤不緊繃。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"白日光 SUNDAY 純淨卸妝油 | 溫和植物油脂配方\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案強調功能性（溫和、不緊繃），未將卸妝過程描寫為「療癒」或「放鬆」的時光。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"溫和卸妝無負擔。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"嚴選親膚植物油脂，溫和卸妝無負擔。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"專為乾燥缺水肌設計\"\n    }\n  ]\n}", "input_tokens": 857, "output_tokens": 328}, "b2ecb0faa06704118a10": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"每日除濕量達 16L，快速改善室內濕度，告別潮濕感。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用陽極處理機身，結構紮實，符合樸實耐用的品牌調性。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"終身技術支援服務，提供長久後勤保障，使用更安心無虞。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「驚爆」、「史上最強」等誇張詞彙，整體語調如「品質穩定可靠，是小資族群的實惠選擇」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"採用陽極處理機身，結構紮實，符合樸實耐用的品牌調性。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用 HEPA、CADR 等未經解釋的技術縮寫。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"第四點賣點「美國製造工藝標準，品質穩定可靠，是小資族群的實惠選擇。」混合了「品質可靠」和「價格實惠」兩個不同的概念。\"\n    }\n  ]\n}", "input_tokens": 846, "output_tokens": 384}, "b50e38951368e5b5fec0": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專業級 95Ra 顯色，色彩精準，視覺體驗更舒適。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"霧面鋁合金機身，堅固耐用，質感樸實。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"終身技術支援，本木 BENMU 提供長久使用保障。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案整體用語樸實，例如：「為長時間工作的你提供專業照明」，未出現最高級形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"為長時間工作的你提供專業照明。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"專為通勤與遠距工作者設計。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"賣點皆為短句，例如：「專業級 95Ra 顯色，色彩精準，視覺體驗更舒適。」\"\n    }\n  ]\n}", "input_tokens": 880, "output_tokens": 329}, "2244c606a2a7f9c6c65a": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"無論是地板上的毛髮、地毯深處的塵蟎，或是縫隙中的細小灰塵，都能瞬間吸入，確保您的居家環境徹底潔淨。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"1.4kg 單手無負擔...輕鬆舉高：清潔高處的窗簾、冷氣出風口或天花板角落，不再需要費力抬舉。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"配件名稱：床墊吸頭, 主要用途：床鋪、沙發、布藝家具\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「極致」一詞，例如「我們將機身重量控制在極致的 1.4 公斤」、「享受 Volta 帶來的極致便利」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"告別笨重線材，迎接高效輕生活！\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"18000Pa 澎湃吸力。無論是地板上的毛髮、地毯深處的塵蟎，或是縫隙中的細小灰塵，都能瞬間吸入\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了問句與驚嘆號，例如「您是否厭倦了傳統吸塵器的笨重機身和糾結的電線？」和「輕巧，但絕不妥協！」\"\n    }\n  ]\n}", "input_tokens": 1588, "output_tokens": 444}, "5ed3cbdf9ef953bdfabc": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"嚴選整顆凍乾水果製成，保留天然營養素。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為外食族設計，睡前一顆輕鬆補充。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"建議每日睡前補充，維持日常均衡。溫和配方，輕鬆融入您的生活節奏。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「最強」、「保證見效」等過度誇大的詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用驚嘆號或網路流行語。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案提到「20粒」和「每日補充」，但未明確轉換為「20天份」的概念。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"溫和配方，輕鬆融入您的生活節奏。\"\n    }\n  ]\n}", "input_tokens": 847, "output_tokens": 326}, "205660be8408c215d654": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"整顆凍乾水果製成，保留完整營養\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為作息不規律的上班族與外食族設計。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"每日睡前一粒，輕鬆完成營養補充\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"為忙碌生活提供穩定支持\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"天然營養，輕鬆攝取無負擔。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案提到「20粒」和「每日一粒」，但未明確換算成「20天份」。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"輕鬆攝取無負擔\"\n    }\n  ]\n}", "input_tokens": 858, "output_tokens": 299}, "1c79ea651623ff520137": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專業級顯色指數高達 95Ra，色彩真實呈現，工作不失真。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用霧面鋁合金，結構堅固耐用，體現樸實工法。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"終身技術支援保固，提供長久安心使用體驗，適合長時間辦公。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"霧面鋁合金材質，台灣製造，耐用可靠。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"非對稱光學設計，不佔桌面空間，專注照明螢幕與鍵盤區域。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"是通勤族與遠距工作者提升效率的理想選擇。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"專業級顯色指數高達 95Ra，色彩真實呈現，工作不失真。\"\n    }\n  ]\n}", "input_tokens": 876, "output_tokens": 337}, "38fad912e22dae2b78cc": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"注入多重玻尿酸，深層補水，維持肌膚水潤。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"輕盈水感質地，敷後清爽，不造成肌膚負擔。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溫和配方敏弱肌友善。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"適合成分黨日常密集保濕使用。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"青研 CHINGYEN 多重玻尿酸水感保濕面膜\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"輕盈水感質地，敷後清爽，不造成肌膚負擔。\"\n    }\n  ]\n}", "input_tokens": 940, "output_tokens": 305}, "70a50dc5fa5a16f36f36": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"提供安瓶級深層補水修護。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用，溫和修護零負擔。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"缺少對水感質地「清爽」或「不黏膩」感受的直接描述。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溫和修護零負擔。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"告別乾燥泛紅，重現水潤透亮。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"高濃度多重玻尿酸，敏弱肌速效安瓶級保濕。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"獨家「多重玻尿酸」配方，直達肌底。\"\n    }\n  ]\n}", "input_tokens": 859, "output_tokens": 310}, "3173ca577ccbac37e7be": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"採用多重玻尿酸配方，為肌膚注入所需水分。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"溫和配方設計，敏弱肌也能安心使用。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"德國凝露精華，質地清爽不黏膩，快速滲透。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案整體用語溫和，如「溫和水潤」、「高效水潤」，未出現誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案完全未使用驚嘆號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「讓您」等可以省略的贅字。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"青研 CHINGYEN 多重玻尿酸保濕面膜\"\n    }\n  ]\n}", "input_tokens": 841, "output_tokens": 318}, "642ef005227ac5112754": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"建議睡前食用，夜間修復，隔日精神飽滿不透支。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為長時間使用螢幕的上班族設計。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"單顆獨立包裝設計，隨身攜帶，衛生方便不變質。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「有感」一詞，屬於應避免的誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案完全沒有使用驚嘆號「！」或問號「？」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"高劑量維生素B群複方，幫助代謝\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「幫助您」的冗餘引導詞。\"\n    }\n  ]\n}", "input_tokens": 903, "output_tokens": 320}, "1d409c12bc43faed5050": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專為應對極端濕度而生\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"機身採用高成本的陽極處理 (Anodized Finish)，這不僅賦予機身極佳的耐腐蝕性和耐磨性\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"本木 BENMU 承諾提供業界罕見的 「終身技術支援」。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"源自美國的設計與技術標準，確保了 BENMU 除濕機的穩定性和耐用性。我們堅持提供最高標準的產品，讓您買得安心，用得放心。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"機身採用高成本的陽極處理 (Anodized Finish)，這不僅賦予機身極佳的耐腐蝕性和耐磨性，更帶來低調奢華的霧面質感，完美融入任何現代裝潢風格。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"無論是客廳、臥室還是儲藏室，BENMU 都能迅速降低濕度。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"賣點描述混合了多個概念，例如「陽極處理外觀」的說明同時提及「質感升級，耐用抗腐蝕，清潔保養更容易」三個好處。\"\n    }\n  ]\n}", "input_tokens": 1408, "output_tokens": 424}, "dec82ca391a3b2162fb8": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"完整保留整顆凍乾水果精華。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為作息不規律的上班族設計。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"睡前一粒軟膠囊，輕鬆補充缺乏的蔬果營養。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"高效率吸收\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用驚嘆號或網路流行語\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「20粒」與「睡前一粒」，未明確換算為「20天份」等份量概念。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"告別疲憊，夜間黃金修護期，隔日煥發晨光。\"\n    }\n  ]\n}", "input_tokens": 879, "output_tokens": 315}, "cee01f6c4031c1cdfd8e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"白日光 SUNDAY 唇部修護膏\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"油狀質地輕盈不黏膩\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"溫和蜂蠟與植物油配方\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌也適用\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"3入分享組，臥室、包包、辦公室各放一罐。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案提及了目標客群（學生黨和社群控），但未說明具體的使用情境（如：熬夜、上妝前）。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案整體用詞平實，沒有使用「一抹見效」、「終極」等誇大詞彙。\"\n    }\n  ]\n}", "input_tokens": 899, "output_tokens": 323}, "aca18348bef38a3f8ec8": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"睡前輕鬆一粒，維持整日好狀態。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為長時間使用螢幕的忙碌上班族設計。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"獨立包裝，方便持續補充\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「最強」、「立即見效」、「有感」等誇大詞彙，例如使用「穩定支持代謝機能」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案完全沒有使用驚嘆號「！」或問號「？」。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"穩定支持代謝機能。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案直接陳述產品特點，例如「獨特緩釋技術，穩定支持代謝機能」。\"\n    }\n  ]\n}", "input_tokens": 838, "output_tokens": 314}, "0d03626dd2af5a5ef599": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"我們堅持採用耗時費工的低溫烘焙技術，溫和喚醒堅果的生命力，完整保留其天然酵素與營養價值。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"隨身攜帶：辦公室、健身房、旅行，隨時隨地享受頂級堅果。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"「溯源焙所」嚴格追溯原料產地，確保每一顆堅果都符合最高品質標準。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"口感升級：保留堅果的清脆與原始香氣，無油耗味。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案使用「頂級」、「極致」、「奢華」等詞彙建立高端定位，但避免了「最強」、「必買」、「爆款」等過度誇大的行銷詞彙。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案前言以問句「您是否在尋找一種既能滿足口腹之慾，又能真正為身體帶來益處的『機能性零食』？」開頭，而非以具體事實或特色開頭。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"每一粒「溯源焙所」堅果都採用獨立密封的植物膠囊包裝\"\n    }\n  ]\n}", "input_tokens": 1563, "output_tokens": 423}, "aed4add6ab2f100e954e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"低溫烘焙技術，保留堅果原始風味與優質口感。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為作息不規律的上班族與外食族設計。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"台灣在地職人監製，安心溯源，為品質嚴格把關。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"缺少具體的風味或口感描述詞，例如：香脆、甘甜、溫潤。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案整體用詞平實，例如「是您追求健康生活，輕鬆享受美味的理想選擇」，沒有使用誇大詞彙。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"低溫烘焙技術，保留堅果原始風味與優質口感。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"獨立分裝 20粒，方便攜帶\"\n    }\n  ]\n}", "input_tokens": 881, "output_tokens": 336}, "145fc7d713b658f3dec8": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"核心神經醯胺，維持肌膚水潤\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"輕盈水感質地，清爽不黏膩\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"專為重視溫和保濕的族群設計。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"幫助肌膚鎖住水分。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"未將「7入」的包裝特點與方便攜帶、定量使用等具體使用情境連結。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"第四點賣點「日本製造，大容量 80ml 7入組」僅列出規格，未說明對消費者的好處。\"\n    }\n  ]\n}", "input_tokens": 855, "output_tokens": 315}, "3d2ceb74f95711c2f462": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案未將『23種維生素礦物質』連結到養顏美容或好氣色，僅提到『提供日常所需基礎能量』。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"嚴選 25種關鍵營養素完整配方\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"專為忙碌的現代女性設計。一粒補充...\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案中未使用『爆款』、『必買』、『最強』等過度銷售詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"嚴選 25種關鍵營養素完整配方\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"賣點皆為獨立短句或詞組，例如：『每日一粒，輕鬆維持基礎能量補給』。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"專為忙碌的現代女性設計\"\n    }\n  ]\n}", "input_tokens": 835, "output_tokens": 337}, "dd9a2450a177054029d9": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"鑄鐵材質蓄熱佳，料理受熱更均勻。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"缺少將 24cm 尺寸與「單人」、「小家庭」或「小空間」等使用情境連結的描述\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"台灣在地製造，陽極處理工法更耐用。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"享受樸實耐用的料理時光\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"厚實鍋體設計\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"無論是煎牛排或家常菜都能輕鬆掌握火候。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"四條賣點文案皆為一句話的完整表達，且未使用刪節號。\"\n    }\n  ]\n}", "input_tokens": 853, "output_tokens": 314}, "e64af53731b9d7b1c8cb": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"20cm輕巧尺寸，單人份料理不佔空間。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「導熱均勻」，但缺少「蓄熱」優點的說明。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案分別提及「耐用」與「終身技術支援」，但未將後者作為前者的佐證。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「美味」一詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"全文皆為直接陳述句，未使用問句或驚嘆號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"頂級鑄鐵材質，導熱均勻，耐用傳家。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「神器」、「最佳選擇」等誇大字眼。\"\n    }\n  ]\n}", "input_tokens": 876, "output_tokens": 327}, "086ee0149cd717a8c707": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"白日光 SUNDAY 爆水修護唇膏\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"油狀質地秒吸收\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"採用天然蜂蠟與植物油，溫和滋潤敏弱唇肌。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"3入超值囤貨組，隨時隨地補水。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"熬夜唇紋救星！\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案使用「溫和滋潤」、「深層修護」等詞彙，未出現「一抹見效」、「終極」等誇大用語。\"\n    }\n  ]\n}", "input_tokens": 903, "output_tokens": 306}, "6123221fc0f05c73329d": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專業級均勻導熱\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"缺少將 24cm 尺寸與單人、小家庭或小空間使用的情境連結\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"陽極處理耐用升級\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"專業級廚具入門款，NT$890 高CP值\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"缺少「溫潤」、「紮實」、「工藝」等描述質感或製程的詞彙\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"僅提及「煎煮炒炸」等烹飪方式，未提及具體的料理名稱\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"24cm 鑄鐵鍋體，導熱均勻穩定。\"\n    }\n  ]\n}", "input_tokens": 849, "output_tokens": 317}, "ee0d0f1e13d92296c394": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"提供 8段 精準 風速 調整，快速驅散異味與濕氣。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"適用小坪數與寵物家庭\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"食品級不鏽鋼機身，清潔快速耐用。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"8段精準風速控制，滿足細膩換氣需求。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"Volta 8段風速循環扇 日本原裝 不鏽鋼機身\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"NT$490 高性價比，原廠提供一年保固。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"8段精準風速控制，滿足細膩換氣需求。\"\n    }\n  ]\n}", "input_tokens": 830, "output_tokens": 315}, "12c88de01e98d0ac3cc0": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"採用高效多重玻尿酸，將水分注入肌底，維持肌膚水嫩。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"清爽凝露質地，敷感輕盈不悶膚。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"德國溫和配方，安心有感。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"提供肌膚長效水潤。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"清爽凝露質地，不造成肌膚負擔。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"多重玻尿酸保濕面膜\"\n    }\n  ]\n}", "input_tokens": 857, "output_tokens": 289}, "62853ccdd614972aa212": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"鑄鐵鍋具擁有卓越的蓄熱能力，能均勻導熱，讓食材受熱完整\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"直徑 24cm 適合小家庭或單身貴族\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"【台灣製造 品質保證】：堅持台灣在地生產，品質嚴格把關\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「必買」、「神器」、「必備」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"台灣職人精神，打造廚房傳家寶！\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"無論是煎牛排、香酥鍋貼，都能輕鬆達到餐廳級的酥脆口感。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"版本一的四條賣點文案皆為一句話且未使用刪節號，例如：「【極致蓄熱】：鑄鐵材質能長時間保持高溫，無論是煎牛排、香酥鍋貼，都能輕鬆達到餐廳級的酥脆口感。」\"\n    }\n  ]\n}", "input_tokens": 1668, "output_tokens": 376}, "733926c2f21af5bde9a9": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"我們採用最先進的真空凍乾技術 (Freeze-Drying)，在極低溫下瞬間鎖住水果的完整營養結構。與傳統高溫烘乾相比，凍乾技術能保留高達98%的活性成分...\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提到適合「忙碌人士」，但未明確點出「外食族」或「作息不規律者」這兩個特定族群。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"【睡前儀式：啟動夜間黃金修護】\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用「最強」、「最有效」、「保證」等過度誇大的詞彙，而是以「頂級」、「高效能」、「保留高達98%」等具體或有條件的詞彙描述產品。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中的「簡潔高效版」使用了多個驚嘆號，例如：「睡前吃的「晨光膠囊」！」、「我們已經升級到「精華膠囊」！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"20粒/盒 (約20天份)。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"讓您告別疲憊暗沉，一覺醒來，迎接充滿「晨光」般的透亮與活力。\"\n    }\n  ]\n}", "input_tokens": 1933, "output_tokens": 440}, "fc1980bf0288a9ae21e6": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"20cm 萬用尺寸：專為單人或小家庭設計，煎一顆蛋、烤一份鬆餅、炒一道菜，輕巧實用，不佔空間。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"鑄鐵材質能均勻且長時間地保持高溫...\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"我們承諾提供終身的保養、開鍋、除鏽等技術諮詢服務，確保您的 Volta 鍋具能陪伴您數十年。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「完美」、「美味」等主觀形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現問句與驚嘆號，例如「您是否厭倦了頻繁更換不沾鍋？」、「立即入手，體驗真正的烹飪樂趣！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"極佳的熱傳導與蓄熱性能，耐高溫、耐刮磨。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案中沒有出現「必買」、「唯一選擇」等過度誇大的字眼。\"\n    }\n  ]\n}", "input_tokens": 2026, "output_tokens": 384}, "4b7b87b6cf585a295875": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"白日光 SUNDAY 5% 尿素保濕護手霜 5入組 輕盈乳霜質地 告別乾裂粗糙\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「5入」與「隨時隨地」，但未舉例說明可分放在辦公室、包包等不同地點的便利性。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"專為乾燥缺水肌設計，維持手部柔嫩細緻。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案僅有功能說明，未提及「洗完手」、「辦公室冷氣房」等生活化情境。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「！、～」等標點符號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案用詞相對客觀，未出現「終極」、「最強」、「立即見效」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"關鍵成分「尿素 5%」，溫和深層補水滋潤。\"\n    }\n  ]\n}", "input_tokens": 879, "output_tokens": 378}, "1df7951a44d90e73b33e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「建議睡前食用」，但未明確說明緩釋技術能幫助「一夜好眠」或「隔天精神飽滿」。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為長時間使用螢幕的上班族設計。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"一盒 20粒 獨立包裝，方便每日持續補充\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"支援能量代謝 溫和持續釋放\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"嚴選高效能維生素B群複方，完整補給。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"獨家緩釋配方，幫助維持神經系統健康與正常能量代謝。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"獨特緩釋技術，營養素溫和持續釋放。\"\n    }\n  ]\n}", "input_tokens": 857, "output_tokens": 324}, "62374a395fc1e0e0e54a": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「提供肌膚水分」及「維持肌膚水潤平衡」，未點出『多重玻尿酸』可帶來深層或長效的保濕效果。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計，配方溫和單純。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"水感輕盈質地，敷後清爽不黏膩。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"理性保養首選。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"幫助維持肌膚水潤平衡。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"多重玻尿酸保濕面膜\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"核心成分：高效多重玻尿酸，提供肌膚水分。\"\n    }\n  ]\n}", "input_tokens": 852, "output_tokens": 316}, "bc8fd2b7af21fd1cc483": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"高濃度神經醯胺，有效修護肌膚屏障。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"輕盈水感質地，快速吸收清爽不黏膩。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌專用\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案中未使用驚嘆號\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案中未使用「奇蹟」、「立即見效」等誇大詞彙\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「超值囤貨組」，未連結到「方便攜帶」或「定量使用」等具體使用情境。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"第四點賣點「日本原裝進口，80ml 7入超值囤貨組」主要在說明規格與產地，其「超值囤貨」較偏向促銷方案而非產品本身帶給消費者的好處。\"\n    }\n  ]\n}", "input_tokens": 935, "output_tokens": 355}, "e5442c2fab30e42b69b2": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"不再讓色彩失真！...確保您在螢幕前看到的顏色，就是作品的真實顏色。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用高質感霧面鋁合金材質...更具備優異的散熱性與耐用度。精緻的質感...\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"終身技術支援，安心無虞...我們對產品的信心，就是您最堅實的後盾...讓您買得安心，用得放心。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"一般螢幕掛燈的顯色指數約在 80Ra 左右，而 BENMU 達到驚人的 95Ra，極度接近自然光（100Ra）。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"點亮真實色彩，告別視覺疲勞...釋放桌面空間...減少螢幕眩光反射，降低眼睛疲勞。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"設計師、工程師、內容創作者必備！...長時間電腦工作者...視覺設計師/修圖師...學生/研究人員...\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"賣點標題為「專業級 95Ra 高顯色指數 (CRI)」、「頂級霧面鋁合金工藝」、「零佔用桌面空間」等精簡詞組或短句。\"\n    }\n  ]\n}", "input_tokens": 1565, "output_tokens": 438}, "875c372056c3c0f69bf4": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"讓你在睡眠中完成深層修復，醒來迎接持續一整天的穩定能量。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及產品適合「上班族」，或與「長時間使用電腦/手機」的情境連結。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"獨立鋁箔包裝，有效隔絕濕氣與光線，確保 B群活性不流失。方便攜帶，無論出差或旅行，隨時隨地維持最佳狀態。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「最強」、「立即見效」等誇大詞彙，並將「有感」一詞用於描述傳統B群以做出區隔。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了驚嘆號與問號，例如：「從睡夢中啟動你的高效能模式！」、「你是否正深陷「疲勞循環」？」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"維生素B群除了提供能量外，也參與神經系統的穩定與修復。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現了「幫助您」、「為您提供」等引導詞，例如：「幫助您獲得更高品質的睡眠」。\"\n    }\n  ]\n}", "input_tokens": 1560, "output_tokens": 412}, "52d61751f3e9384d6aee": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"小坪數換氣專用\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"適用寵物家庭\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"食品級不鏽鋼材質易於清潔\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"日本製造，結構穩固，原廠一年保固。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"8段風速設定，精準控制氣流範圍。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"食品級不鏽鋼扇葉，適用寵物家庭。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"強勁送風，快速平衡小坪數空間溫差。\"\n    }\n  ]\n}", "input_tokens": 824, "output_tokens": 284}, "5b1071883bfca9675d79": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"低溫慢火焙製，保留堅果原始完整風味。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"忙碌外食族與作息不規律上班族的夜間輕補給。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"嚴選多國溯源堅果，提供豐富營養基礎。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"保留堅果營養與香氣。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案在標籤中使用了「#外食族必備」，其中的「必備」屬於應避免的誇大行銷詞彙。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"低溫慢火焙製，保留堅果原始完整風味。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"內含20粒獨立植物膠囊，每日睡前一包\"\n    }\n  ]\n}", "input_tokens": 839, "output_tokens": 324}, "776ebbde5078c7f24a1f": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"【高濃度神經醯胺】：直擊肌膚乾燥根源，強化肌膚屏障，鎖住水分不流失。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"【水感輕盈質地】：告別黏膩厚重！一抹化水，迅速吸收，清爽無負擔...\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"【敏弱肌專用配方】：溫和不刺激，舒緩泛紅不適，讓肌膚重回穩定健康狀態。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中多處使用驚嘆號，例如「年度最強囤貨組駕到！」、「一抹爆水、長效鎖潤！」等，不符合避免使用驚嘆號的原則。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"用最純粹的成分，給予最深層的呵護。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"【超值 7入】：臉部、身體、全家大小都能用！\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"【日本製造．安心保證】：原裝進口，品質嚴格把關。\"\n    }\n  ]\n}", "input_tokens": 1798, "output_tokens": 393}, "5a1a354700e00026eacd": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"20cm 精巧尺寸，單人料理或快速出餐適用\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"鑄鐵材質具備絕佳蓄熱能力，適合快速烹調\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"終身技術支援，確保產品長期使用壽命\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了「優異」、「絕佳」、「理想」等主觀形容詞\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案全為直接陳述句，未使用問句或驚嘆號\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"強悍鑄鐵結構，提供優異熱傳導與穩定性\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「必買」、「唯一選擇」等誇大字眼\"\n    }\n  ]\n}", "input_tokens": 807, "output_tokens": 319}, "1b3e57318224bf9e58f8": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"20cm 直徑，適合單人份快速料理配置。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"鑄鐵本體，提供均勻熱傳導與穩定蓄熱。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「終身技術支援」，但未將其與產品的「耐用度」進行連結或作為佐證。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"PC+ABS 複合手柄，結構穩定且耐高溫。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"Volta 20cm 鑄鐵平底鍋，專為追求高效與極簡配置的出差族設計。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"鑄鐵本體，提供均勻熱傳導與穩定蓄熱。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"無塗層設計，支援多種爐具與高強度烹調。\"\n    }\n  ]\n}", "input_tokens": 804, "output_tokens": 339}, "99100a5009e02df458db": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"堅持低溫烘焙，完整保留堅果天然營養與香氣。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"外食族/上班族專屬\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"安心溯源\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"香氣\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「必備」，屬於較誇大的行銷詞彙。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案以商品標題【晚安養護】溯源焙所 低溫烘焙綜合堅果... 開頭，直接點出商品特色，未使用問句。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案使用「20粒獨立包裝」的說法，單位詞「粒」與「包裝」搭配不當，造成語意混亂，不夠清楚。應為「20包」或「20份」。\"\n    }\n  ]\n}", "input_tokens": 902, "output_tokens": 347}, "f4987794ce71af4596cd": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"白日光 SUNDAY 護手霜添加高效尿素 5%，能深層滋潤乾燥缺水肌，有效修護粗糙手部肌膚\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"超值5入組，辦公室、隨身包、居家隨處可放。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"溫和配方設計，乾燥缺水肌膚適用，舒緩不適。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"超值5入組，辦公室、隨身包、居家隨處可放。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案全篇未使用「！」或「～」等標點符號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案全篇未使用「終極」、「最強」、「立即見效」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"護手霜添加高效尿素 5%，能深層滋潤乾燥缺水肌\"\n    }\n  ]\n}", "input_tokens": 867, "output_tokens": 359}, "ffd8fa48149b49839ab4": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"提供8段細膩風速選擇，無論是梅雨季除濕、衣物速乾，或是寵物異味淨化，都能輕鬆解決。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"寵物家庭清淨首選\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"食品級不鏽鋼，耐用易清潔，不卡毛髮。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現主觀感受詞「舒適」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"8段風速精準控溫，節能省電。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現最高級詞彙「首選」。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"循環對流快速除濕，有效驅散異味。\"\n    }\n  ]\n}", "input_tokens": 856, "output_tokens": 318}, "7d1ddf3f801656e202b1": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專為注重外在保養的女性設計...由內而外散發光采。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"獨立封裝20粒，方便攜帶與每日食用。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"韓國原裝進口，深海魚油純淨來源。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"幫助維持順暢循環與健康基底。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中未出現「嚴選」、「精粹」等傳達職人感的詞語。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"800mg高濃度Omega-3\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"獨立封裝20粒，方便攜帶與每日食用。\"\n    }\n  ]\n}", "input_tokens": 863, "output_tokens": 311}, "bdd12db2c20ad8cffadc": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"唇部修護膏\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"油狀質地，輕盈好吸收，不黏膩。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"蜂蠟與植物油配方，溫和親膚。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌也適用\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"3入組，辦公室、床頭、包包各放一罐。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案雖提及學生與社群使用者，但缺少為他們設計的具體使用情境（例如：熬夜追劇、上妝前打底）。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案整體用詞務實（如「溫和親膚」、「隨時保持水潤感」），未出現「一抹見效」、「終極」等誇大詞彙。\"\n    }\n  ]\n}", "input_tokens": 940, "output_tokens": 341}, "7dc8e6764d2d7f10073b": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"為忙碌的妳，由內而外打底，維持活力與美麗。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"25種完整營養\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"每日僅需一顆...輕鬆補足日常所需。為忙碌的妳...\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"補充日常飲食不足的基礎能量。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"溯源焙所嚴選綜合維他命錠\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"單顆濃縮 25種多元營養素。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"補充日常飲食不足的基礎能量。\"\n    }\n  ]\n}", "input_tokens": 807, "output_tokens": 289}, "9eeda051bfea74717fad": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"核心成分神經醯胺，提升肌膚保水屏障。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"80ml水感清爽質地，塗抹後迅速吸收不黏膩。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌膚設計。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案全篇未使用驚嘆號，例如「溫和透明配方，專為敏弱肌膚設計。」一句，體現了理性的溝通風格。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案用詞平實，未見誇大詞彙，例如「採用神經醯胺配方，提供長效保濕」。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"7入組滿足長期穩定修護需求。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"所有賣點都說明了對消費者的好處，沒有一句只是重複品名或規格，例如：「輕巧80ml容量，7入組滿足長期穩定修護需求。」\"\n    }\n  ]\n}", "input_tokens": 866, "output_tokens": 368}, "1bfe00521d1a14c54510": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"氧化鋅物理防曬\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"囤貨必備50ml 3入組，持續防護更划算。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溫和純淨配方\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案完全沒有使用驚嘆號\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"油狀質地輕盈好推，膚感清爽不黏膩。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"成分單純透明\"\n    }\n  ]\n}", "input_tokens": 886, "output_tokens": 276}, "180ea9935f947defdf62": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"經典鑄鐵材質，蓄熱性佳，使食材受熱均勻。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提及 24cm 是「實用尺寸」，但未點出適合單人、小家庭或小空間。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"採用陽極處理工法，提升鍋體結構耐用性。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「必備品」，屬於較為誇大的宣傳詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"品質工藝值得信賴。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「煎炒」等烹飪方式，未舉出具體的料理名稱（如：煎牛排）。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"經典鑄鐵材質，蓄熱性佳，使食材受熱均勻。\"\n    }\n  ]\n}", "input_tokens": 842, "output_tokens": 338}, "239ead7bd4cb0a091b84": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"【多重玻尿酸科技】深層補水，長效鎖水\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"【敏弱肌救星】溫和配方，安心呵護...敏弱肌膚也能安心使用\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"【凝露質地精華】清爽不黏膩\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「立即感受肌膚彈潤」，屬於對效果的誇大描述。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中多處使用驚嘆號，例如「抓住水嫩的秘密！」、「讓肌膚喝飽水！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「我們採用」、「讓您的」、「給您」等冗詞贅字。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"青研 CHINGYEN 多重玻尿酸保濕面膜\"\n    }\n  ]\n}", "input_tokens": 1450, "output_tokens": 346}, "a4dce410386b84e36e18": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"從輕柔微風到強力循環，滿足您在不同情境下的所有需求...辦公室/書房：...提供個人化的舒適微風...換氣通風：...迅速排出異味或悶熱感。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及適合養寵物的族群。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"食品級不鏽鋼機身，耐用與美學兼具...更具備優異的耐用性與抗腐蝕性。清潔簡單，長久使用依然亮麗如新。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「舒適度」、「舒適微風」、「舒適的空氣」等主觀感受的形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案以「日本精工」、「8段風速」、「食品級不鏽鋼機身」等規格與功能作為核心亮點進行溝通。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「極致循環」、「極致靜音」等誇大或無法驗證的最高級詞彙。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「不容錯過」等冗餘片語。\"\n    }\n  ]\n}", "input_tokens": 1308, "output_tokens": 411}, "6c139603f160f2455a4f": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"獨家多重玻尿酸，深層補水鎖水。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌專用！\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"清爽凝露質地\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"密集修護乾燥，打造水潤光澤肌。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案在標題「敏弱肌專用！」與SEO描述「保濕救星！」中使用了驚嘆號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"德國原裝進口，成分黨安心首選。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"高效保濕面膜\"\n    }\n  ]\n}", "input_tokens": 842, "output_tokens": 293}, "4875eb676cb82c1bfba4": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"【敏弱肌專屬防護】溫和不刺激，每日防曬零負擔！\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用最純淨、最溫和的氧化鋅（Zinc Oxide）作為核心防護，在肌膚表面形成物理性保護層，有效反射紫外線\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"【超值囤貨組】每日使用不心疼！\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「保證」一詞（品質保證）。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中多處使用驚嘆號「！」，例如「每日防曬零負擔！」、「每日使用不心疼！」等。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"採用特殊的油狀質地，確保極佳的延展性與保濕度。高延展性：輕輕一抹即可均勻塗開，不拉扯肌膚。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案除了必要的「氧化鋅」外，沒有使用其他非必要的專有名詞。\"\n    }\n  ]\n}", "input_tokens": 1501, "output_tokens": 383}, "f68157d59fdd9e16ee9b": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專業級 95Ra 顯色，色彩真實不失真\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"霧面鋁合金質感\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"享終身技術支援，安心有保障。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案於 SEO 描述中使用「最佳選擇」，屬於應避免的最高級形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"色彩真實不失真...非對稱光源設計，有效防眩光...台灣製造，品質嚴格把關\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"遠距工作者必備...適合設計與長時間工作。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"台灣製造，品質嚴格把關，享終身技術支援，安心有保障。\"\n    }\n  ]\n}", "input_tokens": 928, "output_tokens": 318}, "c729fc0b480726a91232": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"包含23種維生素礦物質，幫助您維持青春活力與好氣色。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"25種全方位營養\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"每日一粒，輕鬆維持美麗。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"品質安心\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"專為30-50歲注重保養的女性打造\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"告別疲憊暗沉，重現年輕紅潤好氣色。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"告別疲憊暗沉\"\n    }\n  ]\n}", "input_tokens": 864, "output_tokens": 283}, "8105b6ec0eaf50b0e29e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"極簡設計，3件組收納不佔位。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"抗菌不殘味，毛孩家庭安心選。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"陶瓷刀刃，鋒利持久不需磨。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「最佳」這個主觀且誇大的形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"韓國製造，享終身技術支援。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"四條賣點中僅第一條提及「陶瓷刀」，並未重複。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"專為小坪數家庭設計的...是寵物家庭分切食材的最佳選擇...\"\n    }\n  ]\n}", "input_tokens": 839, "output_tokens": 304}, "36313a5d9c8872f0c446": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"協助您維持健康基礎，展現自然光采。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"獨立 PTP 包裝，保鮮易攜帶。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"僅提及「韓國製造」，未將其與高品質或純淨等形象連結。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用「立即見效」、「最強」、「保證」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"嚴選深海魚油，純淨無負擔。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"高濃度深海魚油軟膠囊\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"賣點 1「高規格 Omega-3，每份濃度達 800mg」在 30 字內重複傳達了「高含量」此一概念。\"\n    }\n  ]\n}", "input_tokens": 892, "output_tokens": 333}, "63c3870ec57afc193bde": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"多重玻尿酸配方 | 結合大分子、中分子、小分子玻尿酸，層層滲透。 | 從肌底深層補水，表層形成鎖水網，持久水潤不間斷。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"【青研 CHINGYEN】多重玻尿酸保濕面膜 — 專為敏弱肌設計的高效水潤方案\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"水感輕盈質地 | 精華液清爽不黏膩，快速吸收。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「救星」、「驚人」等較為誇大的詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了「缺水警報」、「別讓乾燥，偷走你的光采！」等引發焦慮的語句。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"【青研 CHINGYEN】美國原裝進口🇺🇸 多重玻尿酸保濕面膜 (7片/盒)\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"主標題在單一句子內塞入了品牌、產地、成分、功效、規格、價格、適用對象、效果等多重資訊，過於擁擠。\"\n    }\n  ]\n}", "input_tokens": 1777, "output_tokens": 420}, "a938c02b14390695e930": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"輕鬆應對高耗能筆電與多裝置充電需求\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"終身技術支援保固\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"專為出差族設計的 Volta USB-C 編織傳輸線\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"Volta 240W 極速快充線\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「最安心」，屬於應避免的主觀形容詞\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"筆電手機平板通用\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"支援 PD 3.1 協議，提供業界頂尖 240W 極速快充\"\n    }\n  ]\n}", "input_tokens": 843, "output_tokens": 304}, "c22adcf521502131c669": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用溫和氧化鋅，提供物理性防護。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"高CP值囤貨首選。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溫和氧化鋅配方\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計，成分單純透明。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"油狀質地，延展性佳，清爽不厚重。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"成分單純透明\"\n    }\n  ]\n}", "input_tokens": 839, "output_tokens": 273}, "87dd69c5cb950aea15df": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"【A】高效鎖水、軟化角質 【B】深層滲透，有效改善手部粗糙、乾裂、脫皮現象，讓肌膚恢復柔嫩細緻。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"【B】辦公室、包包、床頭櫃、車上各放一條，隨時為雙手補充水分，維持一整天的水潤感。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"乾燥缺水肌救星！\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"洗手次數增加，雙手乾澀緊繃...長時間待在冷氣房...擦完立即工作或滑手機也沒問題。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案多處使用「！」驚嘆號，例如「乾燥缺水肌救星！」、「隨時滋潤！」等。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「最專業、最有效的呵護」，「最」為過度誇大的詞彙。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"【F】黃金比例 5% 尿素 (Urea) 【A】高效鎖水、軟化角質\"\n    }\n  ]\n}", "input_tokens": 1686, "output_tokens": 406}, "6fdf0208992792a6f4c0": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"缺少「軟化乾燥粗糙」的說明。文案只提及「高保濕」、「深層補水鎖水」、「長效水潤」、「溫和舒緩乾癢」。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"超值 5入 組合，讓您辦公室、居家、外出都能輕鬆保養。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"乾燥肌適用。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"讓您辦公室、居家、外出都能輕鬆保養。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"白日光 SUNDAY 5%尿素高保濕護手霜 5入組｜日常滋潤，溫和舒緩乾癢\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"日常滋潤，溫和舒緩乾癢\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"5%尿素高保濕護手霜\"\n    }\n  ]\n}", "input_tokens": 875, "output_tokens": 341}, "35a662fe98c202406eef": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"缺少將高含量 Omega-3 與「養顏美容」或「維持青春」等概念的連結。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"我們採用 **獨立鋁箔包裝 (20粒/盒)**，有效隔絕空氣、濕氣與光線，確保每一粒魚油都保持在最佳的新鮮狀態。無論是辦公室、外出旅行，隨身攜帶都極為方便。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"本產品堅持在保健品製造業標準嚴苛的 **韓國原裝生產**。從深海魚油的採集到製程，層層把關，確保魚油的純淨度與新鮮度...\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「保證」一詞（品質溯源保證）。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"缺少「嚴選」、「精粹」等能直接傳達對原料來源講究的詞語。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"【溯源焙所】高濃度深海魚油軟膠囊 (800mg Omega-3)\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案的三大核心優勢（頂級濃度、韓國製造、獨立包裝）各自聚焦於一個獨立概念，沒有在標題或單一賣點中重複。\"\n    }\n  ]\n}", "input_tokens": 1521, "output_tokens": 438}, "ed7e75e30558f04c1dda": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"維持青春美麗\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用獨立包裝，輕巧一盒20粒，方便攜帶，確保魚油新鮮度。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"僅提及「韓國原裝進口」，未將其與高品質或純淨的形象連結。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"深入調理體質，告別疲憊，維持青春美麗。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"頂級深海魚油軟膠囊\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"高純度 Omega-3 800mg\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"頂級深海魚油軟膠囊 | 高純度 Omega-3 800mg | 輕巧20粒裝 | 打造紅潤好氣色\"\n    }\n  ]\n}", "input_tokens": 931, "output_tokens": 330}, "f59ee4d7395599f9a8f2": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"舒緩肌膚不適，維持健康狀態\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"清爽凝露質地，混合肌零負擔\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"讓您輕鬆應對環境變化，維持肌膚舒適穩定\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及妝前保養、曬後安撫等具體的日常使用情境\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案在 SEO 描述中使用了驚嘆號『！』\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"高濃度積雪草萃取 10% 溫和穩定\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"專為 25-40 歲上班族混合肌設計！\"\n    }\n  ]\n}", "input_tokens": 882, "output_tokens": 315}, "6158ec14d893ee2dc10a": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"每份提供足量 100億CFU 專利益生菌，幫助維持消化道健康平衡。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「60粒」與「每日一顆」，但未點出這是「兩個月份量」。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"每日睡前一顆 輕鬆維持消化道機能\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溫和配方，支持您持續的健康基礎。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"晨光研選專為熟齡族群設計的日常保健品。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"僅提及「專利益生菌」，未說明「專利」對消費者的價值（如品質保證、來源安心等）。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"每日睡前一顆\"\n    }\n  ]\n}", "input_tokens": 838, "output_tokens": 330}, "aa0dffc88067344158a2": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"240W 高瓦數供電，滿足筆電全速充電。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"終身技術支援，差旅使用無後顧之憂。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"差旅多裝置供電穩定首選\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"Volta 240W USB-C 編織傳輸線\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「極速」這個主觀形容詞。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"是商務差旅人士的可靠選擇。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"穩定傳輸，多裝置同時供電不降速。\"\n    }\n  ]\n}", "input_tokens": 800, "output_tokens": 302}, "90e9f7e92cf1de462b7c": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌專用配方，溫和不刺激。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"高純度氧化鋅，物理防曬零負擔。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"超值 50ml x 3入組，全身防護不心疼。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"採用高規格氧化鋅，提供溫和高效的物理性防護。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案標題中使用了驚嘆號『！』\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及『油狀質地』此一特點，而是直接描述為『清爽』，因此未執行將『油狀質地』轉化為正向描述的任務。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"質地清爽不黏膩。\"\n    }\n  ]\n}", "input_tokens": 848, "output_tokens": 333}, "b1e0caa5365b0e2df140": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"享受德國工藝與品牌承諾，安心耐用。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"終身技術支援：NT$690 享受德國工藝與品牌承諾，安心耐用。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"專為小坪數家庭與養寵物族群設計。無論是居家辦公或帶毛孩外出散步...\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案在標籤中使用了「#養寵物必備」，與「必買」語氣相似，屬於熱銷宣傳詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"安心耐用\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"食品級不鏽鋼：安全無毒，清洗容易，不殘留異味。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"本木 BENMU 德國工藝 食品級不鏽鋼保溫瓶 750ml\"\n    }\n  ]\n}", "input_tokens": 898, "output_tokens": 349}, "77ea5af0d847405fa596": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案將維生素礦物質連結到『補足身體所需的基礎能量』，但未提及養顏美容或維持好氣色。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"超級全面！25種關鍵營養素一網打盡！\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"每日一顆，補足一日所需！\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案整體語氣專業，以「告別疲憊，活力升級！」、「你的每日健康基石」等方式溝通，未使用「爆款」、「必買」、「最強」等詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"以台灣在地研發的專業配方...堅持台灣在地生產，嚴格把關原料與製程...\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"超級全面！25種關鍵營養素一網打盡！\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"【告別疲憊，活力升級！】\"\n    }\n  ]\n}", "input_tokens": 1553, "output_tokens": 356}, "d41160f78aec3f1d4b2f": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"白日光 SUNDAY 唇部修護膏\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"輕盈油狀質地，滋潤卻不黏膩\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"嚴選高品質蜂蠟與天然植物油，為你的雙唇提供最純淨、最有效的滋養。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"無論是辦公室、床頭櫃還是隨身包，都能隨時隨地為雙唇補水\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"日間打底：在塗抹霧面口紅前使用，能使唇妝更平滑、顯色，避免卡紋。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「最純淨、最有效」、「最頂級」等過度誇大的最高級詞彙。\"\n    }\n  ]\n}", "input_tokens": 1728, "output_tokens": 342}, "46a0a058db5cf3b735d4": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"快速滿足筆電、平板等設備的快充需求。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"享終身技術支援。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"差旅多裝置供電首選。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"Volta USB-C 編織傳輸線 240W\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"支援 USB-C PD 3.1 240W 高功率輸出。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"差旅必備，一線滿足多裝置快充。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"食品級不鏽鋼接頭，抗插拔耐用。\"\n    }\n  ]\n}", "input_tokens": 821, "output_tokens": 297}, "fc327261c2a655979d46": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"德國嚴謹工藝\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"提供終身技術支援。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"小家庭與寵物外出首選。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案在標籤中使用了「必備」一詞（#小家庭必備）。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"材質堅固耐用\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"德國食品級不鏽鋼\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"本木 BENMU ... 750ml 不鏽鋼保溫瓶\"\n    }\n  ]\n}", "input_tokens": 877, "output_tokens": 280}, "2709b263b0e3c462a781": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"足量添加，每份提供高達100億CFU。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案未點明60粒為「兩個月份量」，僅提及「一盒60粒，輕鬆維持」。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"幫助您調整體質、維持消化道機能。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案整體語氣平實，例如「嚴選高活性專利益生菌，調整體質有感」，並無使用「最強」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案專注於產品說明，例如「全素植物膠囊設計，一盒60粒，輕鬆維持」，未包含促銷字眼。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"嚴選高活性專利益生菌\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"睡前一顆溫和保養\"\n    }\n  ]\n}", "input_tokens": 903, "output_tokens": 349}, "4f1fe980bea1fc9c1f82": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"無論是 MacBook Pro 16 吋、高效能電競筆電，或是平板、手機，都能以最快速度充滿電，大幅縮短您的等待時間。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"我們提供終身技術支援，讓您買得安心，用得放心，真正實現一次購買，長久使用。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案僅列舉可支援的裝置類型（筆電、平板、手機），但未點出這對「多裝置使用者」或「出差」情境的具體好處（例如：一線通用、減少攜帶負擔）。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"【極速充電新標準】Volta 240W USB-C 編織線 — 告別龜速，迎接全速高效能！\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「極致耐用」、「頂級材質」、「最高標準」等主觀性強的形容詞。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了「終結所有充電煩惱」、「強力後盾」等較為誇大或陳腔濫調的表述。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案開頭使用「您是否厭倦了？」以及後續三個問句來描述痛點，未遵循避免使用問句的原則。\"\n    }\n  ]\n}", "input_tokens": 1509, "output_tokens": 447}, "79e4f04bcc8158e9c2e5": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專為小坪數家庭設計的Volta 陶瓷刀具組 3件...輕量化設計，收納不佔空間。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"零金屬異味，不殘留食材氣味...適合注重廚房清潔的寵物族群。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"陶瓷刀刃，高硬度耐用，維持鋒利度。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用「極致」、「完美」等誇大形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及「終身技術支援」或相關服務承諾。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"四條賣點中未重複提及品牌「Volta」或產品名「陶瓷刀」。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"專為小坪數家庭設計的...陶瓷刀刃提供優異的切割性能...適合注重廚房清潔的寵物族群。\"\n    }\n  ]\n}", "input_tokens": 819, "output_tokens": 361}, "f074b086389e51d4df40": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「3件」套組能滿足日常需求，但未將其轉化為適合小坪數家庭或簡約廚房的優點。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提及不易殘留氣味（告別金屬味、去除異味），但未連結到寵物飼主的需求。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"採用高密度奈米陶瓷，鋒利度是傳統鋼刀的數倍，持久耐用，大幅減少磨刀頻率。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「極致鋒利」、「完美保留」、「頂級優勢」等主觀且誇大的形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"NT$3290 享終身技術支援。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"在「產品特色」的四個賣點中，有三個重複提及品牌名「Volta」或產品名「Volta 陶瓷刀」。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中未提供獨立的 SEO 描述欄位，而最接近的「副標題」雖提及核心優點（告別鈍刀與金屬味），但未包含目標客群。\"\n    }\n  ]\n}", "input_tokens": 1690, "output_tokens": 418}, "d9343c32d509c2d101d7": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"高純度氧化鋅，提供溫和物理性防護。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計，配方單純不易刺激。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"維持肌膚舒適與水潤感。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「必備」這個誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計，配方單純不易刺激。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"油狀質地好推勻，肌膚感受輕盈不厚重。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"溫和配方專為敏弱肌膚設計，質地清爽好延展。\"\n    }\n  ]\n}", "input_tokens": 814, "output_tokens": 310}, "8e80da651f9193aff4ac": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"我們選用最高規格的「分離乳清蛋白 (WPI)」，經過精密過濾，大幅降低乳糖、脂肪和碳水化合物含量。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案連結了「睡前食用」與「肌肉修復」，但完全沒有提及「美容」或肌膚相關的修護。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"高達 30g 蛋白質 / 份： 遠超市場平均，為重訓者提供足夠的修復基石。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"「嚴選高品質原料與製程」、「韓國原裝進口」\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中多處使用驚嘆號，例如「讓您在夢中也能持續成長！」、「夜間專屬！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"追求極致肌肉線條的專業健美人士。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"三大優勢「30g 極致純粹 WPI」、「夜間黃金修復期」、「頂級便利性」分別從產品成分、使用時機、包裝形式切入，各自獨立。\"\n    }\n  ]\n}", "input_tokens": 1691, "output_tokens": 403}, "d53a29b6ce5e5ad6d459": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"德國工藝食品級不鏽鋼保溫瓶 750ml | 樸實耐用\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"終身技術支援\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案提及了「寵物族群」與「小坪數家庭」，但未點出「帶寵物散步」或「小家庭外出」等具體使用情境。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「最佳選擇」，屬於相對誇大的宣傳詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"堅固耐用\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"採用食品級不鏽鋼，堅固耐用，不怕寵物碰撞。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"本木 BENMU 德國工藝食品級不鏽鋼保溫瓶 750ml\"\n    }\n  ]\n}", "input_tokens": 885, "output_tokens": 338}, "4df6e078139aef21dc99": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"選用高吸收率的甘胺酸亞鐵形式。每份足量提供亞鐵 20mg 溫和不刺激。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"90粒大容量包裝，上班族活力應援。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「日本專利原料」，但未提及產品的製造地為「日本製造」或「日本製」。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"每日持續補充好氣色。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"幫助您維持紅潤氣色與日常活力，輕鬆應對工作挑戰。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「90粒大容量」，但未換算成「三個月用量」或「90天份」等消費者易於理解的時間概念。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"選用高吸收率的甘胺酸亞鐵形式。\"\n    }\n  ]\n}", "input_tokens": 850, "output_tokens": 351}, "5dec3b81246a49584b96": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"本木 BENMU 堅持德國精密工藝，從結構設計到真空技術，確保卓越的保溫性能和耐用性。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"購買本木，即享終身技術支援，讓您買得安心，用得放心。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「長時間會議」、「健身訓練」、「戶外活動」等情境，但未提及「帶寵物散步」或「小家庭外出」的具體使用情境。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「超強保溫保冷」，「超強」一詞屬於較誇大的宣傳詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"確保卓越的保溫性能和耐用性。…確保您的每一口飲水都純淨、適溫、安心。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"採用最高標準的食品級不鏽鋼材質，不含 BPA，不殘留異味，不影響飲品風味。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"德國工藝，終身承諾。本木 BENMU 750ml 不鏽鋼保溫瓶：您值得擁有的「一輩子」好水壺。\"\n    }\n  ]\n}", "input_tokens": 1561, "output_tokens": 418}, "5dc2020898990e121108": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"擁有輕巧的 350ml 容量，輕鬆放入公事包。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用 PC+ABS 複合材質，堅固耐用。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"台灣製造，品質保證。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"輕量不鏽鋼保溫瓶 350ml | 商務差旅、多工辦公必備隨身保溫杯\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"350ml 精巧容量，輕鬆收納不佔位。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"商務差旅、多工辦公必備隨身保溫杯\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案使用「長效保溫」、「最佳選擇」等詞，但避免了「史上最強」、「絕對保溫」等無法證實的誇大宣稱。\"\n    }\n  ]\n}", "input_tokens": 858, "output_tokens": 344}, "6d56e57ae1df30b0e14e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"溫和高吸收率\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為女性與上班族設計\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「日本原裝進口」，未說明為「日本製造」或「日本製」。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溫和高吸收率 專為女性與上班族設計\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"溫和配方，空腹食用也舒適。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案未將「90粒」的規格換算成「三個月用量」或「90天份」。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"選用甘胺酸亞鐵，吸收率佳。\"\n    }\n  ]\n}", "input_tokens": 902, "output_tokens": 310}, "8f16cf93a1cd30cf32a5": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"高濃度積雪草萃取 10%，高效舒緩穩定膚況。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"輕盈凝露質地，混合肌秒吸收不黏膩、不悶痘。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"對抗環境壓力與敏弱不適。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"告別辦公室疲憊肌\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"SEO 描述中的『專為台灣混合肌設計！』使用了驚嘆號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"標籤中使用了『#混合肌救星』，屬於無法驗證的模糊行銷詞彙。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案直接以『專為台灣混合肌設計！』等重點開頭，沒有使用無意義的贅字。\"\n    }\n  ]\n}", "input_tokens": 930, "output_tokens": 331}, "8410f5ea1648e3732cb5": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"高規格胺基酸界面活性劑，溫和潔淨不刺激。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「凝露質地」，但未連結到「清爽」的膚觸，而是連結到「保濕」與「水嫩透亮」。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「最頂級」的絕對性詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現驚嘆號「！」，例如：「肌膚保濕是關鍵！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"30-45歲的您，肌膚保濕是關鍵！\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案中未出現「我們都知道」、「眾所皆知」等冗詞贅字。\"\n    }\n  ]\n}", "input_tokens": 863, "output_tokens": 337}, "273d21041e0eef96e0c2": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"水感質地輕盈，告別厚重黏膩感。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"方便學生族群隨身攜帶，隨時保持水潤狀態。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"蜂蠟與植物油，溫和潤澤雙唇。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"採用溫和的蜂蠟與植物油配方，提供雙唇水感滋潤。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"水感質地輕盈，告別厚重黏膩感。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"缺少將「蜂蠟與植物油」這個組合的作用原理，轉換成消費者能懂的好處（例如：形成鎖水保護膜）的說明。文案僅提及「溫和潤澤雙唇」，未解釋此組合如何達到該效果。\"\n    }\n  ]\n}", "input_tokens": 900, "output_tokens": 346}, "82e011f25841703a8e4c": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"350ml 精巧容量，差旅收納無負擔。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"PC+ABS 複合材質結構，輕巧耐用。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"台灣製造，原廠提供一年保固。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"PC+ABS 複合材質，結構抗衝擊。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"Volta PC複合材質 350ml 不鏽鋼保溫瓶 差旅隨行 輕巧保溫\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"專為出差族設計。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"高效能真空層，長效保溫保冷。\"\n    }\n  ]\n}", "input_tokens": 805, "output_tokens": 305}, "cd870886676e4716cf65": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"每份提供高達 100億CFU 的活性益生菌...發揮最大效益。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"60 粒裝（足足兩個月份量）\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"您是否也有這些困擾？...早上醒來總是感到腹部悶脹？...排便時間不固定，經常「卡關」？...換季時，保護力總是差人一截？\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了「最有效率」等過度誇大的詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了「搶購中！」等促銷感過強的字眼。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"我們不只追求數量，更注重品質！...採用多種專利菌株，確保益生菌能通過胃酸考驗...發揮最大效益。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"建議食用 每日1份（1粒），睡前搭配溫水食用。\"\n    }\n  ]\n}", "input_tokens": 1656, "output_tokens": 375}, "2cafd87ce8874a29be5e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"白日光 SUNDAY 舒緩精華液 10% 積雪草萃取\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"清爽凝露質地，秒吸收不黏膩\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"缺少具體的日常使用情境，例如：妝前保養、曬後安撫、換季調理等。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案於 SEO 描述中使用了驚嘆號『！』(混合肌上班族救星！)\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"高濃度 10% 積雪草萃取，溫和穩定肌膚。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"高濃度 10% 積雪草萃取，溫和穩定肌膚。\"\n    }\n  ]\n}", "input_tokens": 889, "output_tokens": 332}, "3c00d450aa9cbabf43b3": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"提供物理性防護\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用 溫和配方\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"質地滋潤服貼，為肌膚帶來日常保濕。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"給予肌膚透明且安心的防曬體驗。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"青研CHINGYEN礦物防曬乳，採用高純度氧化鋅，提供物理性防護。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"採用高純度氧化鋅，提供溫和防護。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌膚設計。\"\n    }\n  ]\n}", "input_tokens": 807, "output_tokens": 297}, "ab5297032068939e01ac": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"睡前輕盈補充，提供穩定飽足感。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"獨立包裝共30粒，方便攜帶，隨時享受。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"嚴選德國進口，優質全穀燕麥來源。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用『最強』、『唯一』、『保證見效』等過度誇大的最高級或承諾性詞語。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案完全沒有使用驚嘆號「！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"專為注重健康的家庭與健身族群設計。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"四條賣點分別以『嚴選德國進口』、『單包25g定量』、『睡前輕盈補充』、『快速沖泡即飲』開頭，皆為詞組。\"\n    }\n  ]\n}", "input_tokens": 891, "output_tokens": 344}, "342c3c9d6023e1a84e96": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及『30g 蛋白質』對『熟齡族群』維持肌肉量有幫助。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"我們堅持採用**德國進口**的頂級分離乳清蛋白（Whey Protein Isolate, WPI）。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"WPI 經過特殊工藝處理，有效去除多餘的脂肪和乳糖，提供極高的蛋白質純度，適合乳糖不耐者或追求極致低負擔的您。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了『最純粹、最高效』、『最佳』等極端誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及任何與產品風味或口感相關的描述。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"WPI 經過特殊工藝處理，有效去除多餘的脂肪和乳糖，提供極高的蛋白質純度，適合乳糖不耐者或追求極致低負擔的您。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇使用主動句，例如：『每份隨身包提供足足 30 克的純淨蛋白質。』\"\n    }\n  ]\n}", "input_tokens": 1430, "output_tokens": 409}, "494ff10cfa76a9442f34": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"每份足量添加 100億CFU 活性菌。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「60粒」，未說明這代表「兩個月份量」。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"幫助維持消化道機能平衡。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溫和穩定，適合長期持續補充，為您的健康打底。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"植物膠囊，溫和不刺激，睡前食用。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"嚴選專利益生菌配方，穩定性高。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"每日補充...睡前食用\"\n    }\n  ]\n}", "input_tokens": 827, "output_tokens": 298}, "39979402a33fe3681123": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"3件式精簡配置，有效節省小坪數廚房收納空間。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"陶瓷材質不殘留異味，生熟食分切衛生，適合寵物鮮食。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"高密度陶瓷刀刃，提供持久鋒利度與精準切割。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"輕量化高硬度刀刃\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"提供終身技術支援。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"3件式精簡配置，有效節省小坪數廚房收納空間。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"專為小坪數家庭與寵物族群設計。高硬度陶瓷刀刃輕巧易握，精準切割食材，不殘留氣味。\"\n    }\n  ]\n}", "input_tokens": 841, "output_tokens": 333}, "2b08bfde72e7a8b5a994": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及「長時間會議」或「半日外出」等具體情境，僅提及「滿足全日飲水需求」。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及 PC+ABS 複合材質的耐用或抗摔特性，僅說明其「輕盈易開」。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"台灣製造，品質保證。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「最佳飲品溫度」，「最佳」為主觀且空泛的形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案以規格（750ml 大容量、不鏽鋼材質、PC+ABS 複合瓶蓋）作為溝通重點。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案全文未使用驚嘆號。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「擁有 750ml 大容量」的冗餘句型。\"\n    }\n  ]\n}", "input_tokens": 860, "output_tokens": 354}, "a90eff55614c46c932c4": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案提及『30g 蛋白質』與『熟齡族群』，但未點出有助於『維持肌肉量』，僅提及維持體力與活力。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"德國原裝進口，嚴格品質把關，安心有保障。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"高純度分離乳清蛋白，乳糖低，好消化無負擔。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用『最強』、『冠軍』、『第一』等極端誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及任何與風味或口感相關的描述。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"高純度分離乳清蛋白，乳糖低，好消化無負擔。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案皆使用主動句，例如『高效補充日常所需活力』、『輕鬆完成日常保養』。\"\n    }\n  ]\n}", "input_tokens": 875, "output_tokens": 356}, "379e71c651dc51041760": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「維持日常活力」、「維持健康基礎」，但未明確點出有助於「維持肌肉量」。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"嚴選德國進口 分離乳清蛋白 (WPI)\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「高純度蛋白質」，但未提及「乳糖含量低」。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溯源焙所嚴選德國高品質分離乳清蛋白粉 (WPI)，每份提供 30g 純淨蛋白質。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及任何與風味或口感相關的描述。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"每份足量 30g 高純度蛋白質\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"輕鬆補充營養，維持健康基礎。\"\n    }\n  ]\n}", "input_tokens": 808, "output_tokens": 329}, "b954b6927c1c446206f5": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"輕盈水感質地，不黏膩好吸收，告別厚重感。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"一盒 7入設計，方便分裝與攜帶。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"採用蜂蠟與植物油配方，溫和滋潤唇部肌膚。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"聚焦於成分與質地本身帶來的益處，如「溫和滋潤唇部肌膚」、「不黏膩好吸收」。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"輕盈水感質地，不黏膩好吸收，告別厚重感。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「溫和滋潤」，未將「蜂蠟與植物油」這個組合的原理（如形成保護膜以鎖住水分）轉換成消費者能懂的好處。\"\n    }\n  ]\n}", "input_tokens": 877, "output_tokens": 354}, "22e3e673a7b0381daa5a": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"100% 物理性防曬：溫和氧化鋅配方\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌專屬！溫和氧化鋅配方\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"親膚滋養：油性基底有助於鎖住水分，提供額外的保濕效果，讓肌膚在防曬同時保持柔嫩。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「最溫和」、「首選」、「最高品質」、「頂級」、「最純淨」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了驚嘆號，例如：「敏弱肌專屬！」、「給肌膚最純淨的愛！」、「享受來自德國的溫和防護力！」。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案中的賣點（例如「零刺激： 專為敏弱肌、術後肌、甚至兒童設計。」）皆為能獨立閱讀的完整短句。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案與其他產品進行了負面比較，例如：「杜絕化學性防曬可能引起的刺激和過敏反應」及「許多物理性防曬產品常有厚重、難推勻或嚴重泛白的問題」。\"\n    }\n  ]\n}", "input_tokens": 1411, "output_tokens": 427}, "bf89b20de26d94aa3578": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"高效吸收率的甘胺酸亞鐵，溫和不刺激。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"長時間使用螢幕的上班族必備！...輕鬆對抗辦公室疲勞，是女性保健的首選。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"日本原裝進口\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「立即」、「最強」、「保證」等字眼，整體語氣為陳述事實，例如「每份足量 20mg，輕鬆補足一日所需」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案於 SEO 描述中使用了驚嘆號「！」。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"三個月份量\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案用詞簡潔，例如：「高效吸收率的甘胺酸亞鐵，溫和不刺激。」，沒有出現不必要的贅詞。\"\n    }\n  ]\n}", "input_tokens": 862, "output_tokens": 347}, "0041a025228d7a8f4cde": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案提及為熟齡族群設計，但未點出蛋白質有助於「維持肌肉量」。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"嚴選德國進口原料，品質安心有保障。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"高純度分離乳清蛋白...低乳糖好消化。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"品質安心有保障\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"缺少風味或口感的描述。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"高純度分離乳清蛋白，好吸收無負擔。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"補充優質蛋白質，維持活力與體力。\"\n    }\n  ]\n}", "input_tokens": 867, "output_tokens": 295}, "af1d6cca10cb4bedc434": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"蜂蠟與植物油，深度修護敏弱肌。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"日系水感質地，一抹爆水不黏膩。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案將「7入」連結到 CP 值與囤貨，未提及方便攜帶或隨處使用的優點\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「！」、「必囤」、「必備」等強烈銷售字眼與符號\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案聚焦於成分與質地本身帶來的益處，如「告別死皮」、「唇紋瞬間隱形」，未訴諸無關的感性情境。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案用詞具體，如「水感質地」、「蜂蠟與植物油」，沒有使用消費者不易理解的行銷詞彙。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「蜂蠟與植物油，深度修護敏弱肌」，未將成分組合的作用轉化為更具體的消費者利益（如：形成鎖水保護膜）。\"\n    }\n  ]\n}", "input_tokens": 848, "output_tokens": 396}, "f1f37af9e19050eceb23": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"採用親膚的胺基酸界面活性劑...洗後肌膚感到水潤舒適，不乾澀緊繃。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"清透凝露質地，洗後水潤不緊繃。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"是您日常清潔的理性選擇。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"採用溫和胺基酸界面活性劑，親膚潔淨。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"專為重視保濕與溫和清潔的您設計。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"配方單純，溫和低刺激，敏弱肌適用。\"\n    }\n  ]\n}", "input_tokens": 860, "output_tokens": 305}, "19de11ee8f71f820fc8a": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"打造居家清新工作空間\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"食品級不鏽鋼機身，堅固耐用\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"日本工藝製造，原廠一年保固，品質穩定可靠\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇採用如「靜音耐用設計」、「品質穩定可靠」等平實描述，未使用「驚爆」、「史上最強」等誇張詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案以描述產品本身為主，例如「食品級不鏽鋼機身，堅固耐用，體現樸實工藝」，未提及「限時特價」等促銷字眼。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"四條賣點各自聚焦，例如賣點3「低噪運轉設計，不干擾居家辦公或深夜休憩」專注於靜音單一概念。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案中的形容詞皆傳達具體資訊，例如「堅固耐用」、「低噪運轉」、「品質穩定可靠」，沒有空泛堆砌。\"\n    }\n  ]\n}", "input_tokens": 897, "output_tokens": 393}, "243045f355a7d5ac1ef4": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"我們選擇了吸收率高、溫和不刺激的頂級原料\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"上班族/學生族： 告別精神不濟，提升專注力與工作效率。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"本產品堅持日本製造、原裝進口\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「最高效」、「品質保證」、「立即下單」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中多處使用驚嘆號「！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"90粒 / 盒 (約三個月份)\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「我們深知...因此我們選擇了」、「專為現代女性打造的」等不必要的贅詞。\"\n    }\n  ]\n}", "input_tokens": 1691, "output_tokens": 328}, "5453255ecb76fa936995": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"適用 15坪 空間，滿足都會小家庭或個人工作室需求。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用食品級不鏽鋼機身，結構紮實，經久耐用。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"日本製造工藝，細節講究，提供穩定可靠的空氣淨化。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"本木 BENMU 15坪適用 空氣清淨機。食品級不鏽鋼機身，日系簡約設計，打造純淨工作環境。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"專為追求質感與耐用的您設計。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案的四條賣點各自聚焦於獨立好處，例如：「適用 15坪 空間，滿足都會小家庭或個人工作室需求。」聚焦於適用空間。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"結構紮實，經久耐用。\"\n    }\n  ]\n}", "input_tokens": 882, "output_tokens": 353}, "de5b597c1107eb4a9ab8": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"適用 15坪 空間，小套房/書房高效淨化首選。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"食品級不鏽鋼機身，質感耐用。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"本木 BENMU 日本頂級工藝 15坪 空氣清淨機\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「驚爆」、「史上最強」等誇張詞彙，整體語調平實，例如「日本製造，採用食品級不鏽鋼，品質可靠」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案以描述產品為主，例如「日本原裝進口，食品級不鏽鋼機身，質感耐用」，未營造限時促銷氛圍。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"未通過。第二點賣點混合了「日本產地」、「不鏽鋼材質」與「耐用」三個概念；第四點賣點混合了「輕巧設計」與「保固」兩個概念。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"輕巧不佔空間，原廠提供一年安心保固。\"\n    }\n  ]\n}", "input_tokens": 898, "output_tokens": 394}, "4f8bdc63ab6a9e0b7dfc": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"高純度分離乳清蛋白，輕盈無負擔好吸收。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"睡前黃金時段補給，為美麗與活力打下基礎。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"每份足量30g\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"嚴選韓國優質乳源，風味純淨，職人級選品。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用驚嘆號、或「必買」、「爆款」等詞彙。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"為美麗與活力打下基礎。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"四條賣點各自獨立，分別闡述產品成分、品質、份量與用法效益。\"\n    }\n  ]\n}", "input_tokens": 848, "output_tokens": 317}, "ce188dca8203e4708282": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「不佔用行李空間」與「隨身攜帶無負擔」，但未明確點出適合放入「公事包」或「隨身小包」。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"PC+ABS 複合材質外殼，結構堅固耐用。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"台灣製造，品質可靠，原廠提供一年保固。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"Volta 350ml 不鏽鋼保溫瓶專為商務人士與高效率使用者設計。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"容量 350ml，輕量化設計，不佔用行李空間。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"是您差旅、辦公、通勤時的最佳保溫夥伴。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"高效能不鏽鋼內膽，提供穩定保溫效果。\"\n    }\n  ]\n}", "input_tokens": 833, "output_tokens": 347}, "8ab2498e3f413747e844": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"高純度分離乳清蛋白，輕盈無負擔。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"睡前補充，為夜間美麗打底。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"每份足量 30g\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"韓國優質乳源，職人嚴選把關。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"專為熟齡女性設計的夜間保養配方。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"維持優雅體態與美麗光采。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"bullet 1: 高純度分離乳清蛋白，輕盈無負擔。 bullet 2: 每份足量 30g，獨立隨身包共 20粒。 bullet 3: 睡前補充，為夜間美麗打底。 bullet 4: 韓國優質乳源，職人嚴選把關。\"\n    }\n  ]\n}", "input_tokens": 826, "output_tokens": 354}, "0965a4cb01e7c3ab464d": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"積雪草是公認的肌膚修護專家，能有效鎮定不安定的膚況，快速舒緩泛紅、乾燥，讓肌膚重回健康平衡。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"清爽「凝露質地」，不黏膩：打破高濃度精華液的厚重印象！輕盈水感的凝露質地，一抹化水，快速吸收，不悶痘、不給肌膚造成負擔...\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"【美國原裝🇺🇸 敏弱肌救星】...就是為敏弱肌量身打造的「肌膚穩定器」！...專為敏弱肌設計的溫和配方...\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"口罩肌受害者：舒緩悶熱造成的泛紅與不適。換季敏感者：強化肌膚屏障，對抗環境刺激。醫美後修護者：溫和鎮定，加速肌膚穩定。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中多處使用驚嘆號「！」，例如：「NT$290超值入手！」、「肌膚穩定器！」、「現在就將...加入購物車...給您的肌膚一個喘息的機會！」等。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案使用「舒緩泛紅、穩定膚況」、「溫和修護受損屏障」、「鎮定不安定的膚況」等具體描述，避免了無法驗證的模糊詞彙。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"👉 你是否也有這些困擾？* 換季時臉頰總是泛紅、乾癢，怎麼擦都不對？* 戴口罩悶出小顆粒、痘痘，肌膚狀態極不穩定？\"\n    }\n  ]\n}", "input_tokens": 1712, "output_tokens": 532}, "8c0475394eea02a8917e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"高純度分離乳清蛋白，純淨無負擔\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"睡前黃金修護期飲用，啟動夜間代謝與美妍力。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"單包足量30g蛋白質，高效補給\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"韓國原裝進口\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"專為注重內外保養的熟齡女性設計。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"高效補給，維持優雅體態。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"賣點各自獨立，例如：1. 高規格分離乳清蛋白，純淨無負擔，好吸收。 3. 韓國原裝進口，20粒隨身包設計，輕巧方便攜帶。\"\n    }\n  ]\n}", "input_tokens": 894, "output_tokens": 333}, "bba9c4c4d8105042fc6d": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"無論是放進小包包、公事包，或隨手拿著，都毫無負擔。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"搭配耐用 PC+ABS 複合材質瓶蓋，堅固耐用，讓您用得更安心。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"【台灣製造，品質保證】我們堅持台灣在地生產製造，從原料選擇到製程控管，都經過嚴格把關。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了「好心情」等情感形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案多次使用「350ml」作為溝通核心，例如標題「350ml 台灣製不鏽鋼保溫瓶」以及內文「選擇 Volta 350ml，解決你的日常煩惱！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案在情境應用段落點出「辦公室夥伴：上班族必備！」、「通勤族首選」等具體使用情境。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案中沒有使用「史上最強」、「絕對保溫」等無法證實的誇大宣稱。\"\n    }\n  ]\n}", "input_tokens": 1672, "output_tokens": 408}, "1968ef9140f072c0e2ba": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"BENMU 鍵盤採用高規格的霧面鋁合金外殼，提供遠超塑膠材質的沉穩與耐用性。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"104鍵全尺寸配置，無論是數字輸入、文書處理或複雜指令，都能一氣呵成。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"提供原廠一年保固，我們為產品的耐用性與可靠性背書。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「頂級」、「無與倫比」、「奢華」、「極致」等華麗的形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了多個表情符號（如：💻、🌟、💎）與驚嘆號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了問句與對話語氣，例如：「您是否厭倦了普通鍵盤的遲鈍與模糊手感？」\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「最純粹的機械軸體」、「符合最高標準」、「最值得投資」等無法驗證的最高級宣稱。\"\n    }\n  ]\n}", "input_tokens": 1466, "output_tokens": 393}, "899925459530502e64ec": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專為亞洲敏弱肌設計，徹底顛覆您對「滋潤」的想像！\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"獨家「水感」質地，清爽零負擔\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"隨身攜帶零死角：一盒七入，您可以將它放置在辦公室、床頭、化妝包、甚至車上，隨時隨地為雙唇補水。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中多處使用驚嘆號，例如：「告別乾燥脫皮！」、「徹底顛覆您對「滋潤」的想像！」\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"天然蜂蠟與植物油精萃：提供長效鎖水屏障，深層修護乾燥受損的唇部肌膚。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"一抹瞬間化開，迅速被唇部吸收，形成一層隱形保護膜，不黏膩、不悶厚，即使在炎熱氣候下使用也極度舒適。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"天然蜂蠟與植物油精萃：提供長效鎖水屏障，深層修護乾燥受損的唇部肌膚。\"\n    }\n  ]\n}", "input_tokens": 1553, "output_tokens": 419}, "52efd41b8a8f252275ed": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"採用胺基酸界面活性劑，溫和潔淨不刺激\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"凝露質地，洗後清爽不緊繃\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"採用胺基酸界面活性劑，溫和潔淨不刺激\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"是敏弱肌日常保濕清潔的理想選擇。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案未使用第二人稱（你、您的肌膚）與目標受眾溝通\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"洗後不乾澀緊繃，維持肌膚水潤感\"\n    }\n  ]\n}", "input_tokens": 839, "output_tokens": 303}, "cecae5f93eff66ea0cff": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"核心添加高濃度神經醯胺，有效強化肌膚屏障。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"清爽不黏膩\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及「日常」、「每天」等生活化、融入情境的詞彙\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「成分黨」一詞，屬於美妝社群的流行語\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「首選」，屬於無法證實的最高級形容詞\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案主要以第三人稱描述產品，並未使用「你」或「你的」等第二人稱與消費者直接溝通\"\n    }\n  ]\n}", "input_tokens": 874, "output_tokens": 324}, "80dfeb83fc1c2236b166": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"霧面鋁合金機身，結構穩固，經久耐用。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"標準104鍵配置，滿足高頻率資料輸入需求。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"提供一年保固，品質有保障。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"強調材質與工法，耐用取向，是您桌面上的可靠夥伴。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用表情符號 (emoji) 或驚嘆號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"採用堅固的霧面鋁合金機身，提供紮實的打字手感。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用「最」、「第一」等無法驗證的最高級宣稱。\"\n    }\n  ]\n}", "input_tokens": 862, "output_tokens": 322}, "e8af42622fdc1bab50c7": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計，卸妝過程溫和不刺激。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用溫和植物油脂基底，輕盈不悶痘。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案中未提及『乳霜質地』，而是描述為『卸妝油』與『輕盈』質地。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用『最強』、『第一』等誇大詞彙，例如使用「給予肌膚清爽無負擔的卸妝體驗」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"能快速溶解彩妝髒污\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"青研 CHINGYEN 純淨卸妝油 100ml 敏弱肌適用 植物油脂溫和配方 1入\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"採用溫和植物油脂基底，輕盈不悶痘。\"\n    }\n  ]\n}", "input_tokens": 850, "output_tokens": 351}, "583370c4f64ffcd6de5e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"本產品採用高純度胺基酸界面活性劑...徹底告別傳統皂基或刺激性界面活性劑帶來的乾澀感。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計...適用於所有膚質，特別是敏弱肌、乾燥肌。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"透明凝露質地，輕盈滑順...泡沫極易沖洗，不殘留...\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「最溫和」、「最親膚」、「徹底告別」等絕對性詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"整份文案未使用任何驚嘆號，例如結尾句「選擇青研 CHINGYEN，讓清潔成為修護的起點。」僅使用句號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"您是否厭倦了洗後乾澀、緊繃，甚至泛紅的清潔產品？\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案內容直接，未出現「我們都知道」、「眾所皆知」等無助於傳達價值的冗詞贅字。\"\n    }\n  ]\n}", "input_tokens": 1445, "output_tokens": 392}, "9558fe0606d027c0aaff": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"高效氧化鋅物理防護\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌專用\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"輕盈油狀質地，保濕不黏膩。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案於 SEO 描述中使用「最佳選擇」，缺少對誇大詞彙的完全避免。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"全文未使用驚嘆號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"賣點各自獨立完整，例如：「高效氧化鋅，物理性溫和阻隔。」\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案專注於產品優點，例如：「採用高純度氧化鋅，提供高效物理防護，不刺激肌膚。」\"\n    }\n  ]\n}", "input_tokens": 808, "output_tokens": 311}, "239fcd83b69b879da454": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"25g黃金份量，高纖飽足，輕盈無負擔。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"30粒獨立包裝，隨沖即飲，忙碌生活神隊友。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"德國嚴選：純正全穀燕麥，保留完整營養價值。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"健身族群安心選。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"高纖維提供長效飽足感，輕鬆補充能量，健康無負擔。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"是健身族群與忙碌家庭的優質早餐或宵夜選擇。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"德國嚴選：純正全穀燕麥，保留完整營養價值。\"\n    }\n  ]\n}", "input_tokens": 876, "output_tokens": 318}, "f5b9649c02eb80793109": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"核心成分神經醯胺，幫助肌膚維持水潤與穩定。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"溫和配方設計，舒緩乾燥不適，適合敏弱膚質。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"輕盈乳霜質地，清爽不黏膩，快速滲透好吸收。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌日常保濕首選。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"溫和配方設計，舒緩乾燥不適，適合敏弱膚質。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"核心成分神經醯胺，幫助肌膚維持水潤與穩定。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案未使用『你』或『你的』人稱，未直接對消費者溝通。\"\n    }\n  ]\n}", "input_tokens": 844, "output_tokens": 324}, "d9eaad415c64cccce29f": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計的青研純淨卸妝油\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用溫和植物油脂配方\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"輕盈乳霜質地，快速乳化不油膩。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用『最強』、『第一』等誇大詞彙，僅使用如「溫和配方」、「敏弱肌友善」等客觀描述。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"好沖洗不殘留\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"純淨卸妝油...敏弱肌友善\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"嚴選多種親膚植物油脂基底。\"\n    }\n  ]\n}", "input_tokens": 890, "output_tokens": 312}, "ad0b20279e813f50e83a": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"適用坪數高達 15 坪，客廳、主臥室淨化無死角。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用食品級不鏽鋼機身，耐用、易潔、更衛生。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"日本原裝進口，精密工藝，穩定可靠。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「超乎想像的價格」等誇張詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「限時優惠」、「數量有限」、「限量供應」等營造短暫促銷氛圍的字眼。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"四條賣點分別聚焦於【獨家材質】、【高效覆蓋】、【品質保證】、【超值入手】，主題各自獨立。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「極致的耐用性」、「完美融入」、「頂級的材質」、「超乎想像的價格」等空泛或誇大的形容詞。\"\n    }\n  ]\n}", "input_tokens": 1479, "output_tokens": 373}, "0071a7c15782f5f04506": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"青研 CHINGYEN 專為亞洲敏弱肌設計\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"【敏弱肌專用配方】溫和淨化，不刺激。專為脆弱膚質設計，嚴選親膚的天然植物油脂。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"這不是一般的液態油！我們採用獨特的技術，將純淨植物油脂轉化為輕盈的「乳霜狀」質地。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用『最強』、『第一』等誇大詞彙，語氣為陳述事實，例如「雖然溫和，但卸妝力絕不打折！」\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"快速帶走彩妝與髒污\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"主標題：卸妝新革命！敏弱肌專屬的「奶油感」純淨卸妝油，溫和淨透，零負擔。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案以清晰的標題句總結賣點，例如：【獨家奶油感質地】打破卸妝油的油膩迷思\"\n    }\n  ]\n}", "input_tokens": 1499, "output_tokens": 402}, "cb64f5cc8410e37e5701": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"核心成分神經醯胺，提升肌膚防護力。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計，溫和日常呵護。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"輕盈乳霜質地，清爽保濕不黏膩。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溫和日常呵護。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了驚嘆號「！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"核心成分神經醯胺，提升肌膚防護力。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案主要以第三人稱描述產品，未以「你」或「你的」作為主要溝通人稱。\"\n    }\n  ]\n}", "input_tokens": 822, "output_tokens": 306}, "e1a740a0d6f403042e6f": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"頂級霧面鋁合金機身，穩固耐用，質感全面升級。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"標準104鍵配置，數字區讓資料輸入更有效率。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"享一年保固，品質安心有保障。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了「頂級」、「極致」等華麗形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"SEO 描述中使用了驚嘆號：「遠距工作者與通勤族必備！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案皆為直接陳述句，例如「標準104鍵配置，數字區讓資料輸入更有效率。」，沒有使用問句或對話語氣。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案使用了「頂級」、「極致」等無法驗證的最高級宣稱。\"\n    }\n  ]\n}", "input_tokens": 950, "output_tokens": 347}, "1b5071e080ae40442dfc": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"標準 750ml 容量，減少重複裝填。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"PC+ABS 複合外殼，耐衝擊設計。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"台灣製造，原廠提供一年保固。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"SEO 描述中出現「最佳」這個主觀形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案以「750ml」、「PC+ABS 複合外殼」、「台灣製造」等規格為溝通重點。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案完全沒有使用驚嘆號。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「提供...的功能」、「擁有...的設計」等冗餘句型。\"\n    }\n  ]\n}", "input_tokens": 799, "output_tokens": 313}, "ccac03ad427b0693c357": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"深度修護屏障 (神經醯胺) | 補充肌膚流失的神經醯胺，從根本強化肌膚防禦力，有效舒緩乾燥、泛紅等敏弱狀況，讓肌膚更穩定、更健康。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"無論是乾性肌、油性肌、混合肌，甚至是敏感肌，都能安心使用。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"雖然是乳霜質地，但輕盈好推開，不黏膩、不悶厚。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"讓您的肌膚在忙碌的生活中，也能重拾飽滿、水潤、透亮的健康光采。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案中多處使用「！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「頂級」、「最重要」、「最有效」等無法證實的最高級形容詞。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"【你是否也有這些肌膚困擾？】\"\n    }\n  ]\n}", "input_tokens": 1697, "output_tokens": 374}, "d26b7418f75bb7009fce": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"提供恰到好處的飽足感，避免空腹影響睡眠品質，同時又不會造成消化負擔。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用獨立小包裝設計...無論是辦公室加班、旅行，或是在家準備睡前飲品，只需倒入熱水即可享用。無需測量、無需等待，隨時隨地，輕鬆補充能量。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"我們堅持使用來自德國產地的優質全穀燕麥。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「最純粹」、「首選」、「極致」等最高級詞語。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案在標題與結尾的行動呼籲中使用了驚嘆號「！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"夜貓族與加班族、健康管理人士、重視儀式感者、追求效率者\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"賣點標題為「睡前輕食首選」、「德國嚴選，全穀營養」、「極致便利，即沖即飲」，皆為名詞或動詞詞組開頭。\"\n    }\n  ]\n}", "input_tokens": 1405, "output_tokens": 400}, "580cf0c5f545942f7695": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「滿足每日持續補充的飲食需求」，但未將 25g 的規格與「一餐所需」或「提供飽足感」等具體好處連結。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"獨立小包裝，每份 25g，方便快速沖泡攜帶。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"嚴選德國進口優質全穀燕麥，保留完整營養。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案整體用語平實，例如「嚴選德國進口優質全穀燕麥」，未出現最高級或誇大詞語。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用驚嘆號，所有句子皆以句號結尾，例如：「一盒 30粒，滿足每日持續補充的飲食需求。」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"提供原型食物的單純營養，適合家庭與健身族群。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"賣點皆以動詞或名詞詞組開頭，例如：「嚴選德國進口優質全穀燕麥...」、「獨立小包裝...」。\"\n    }\n  ]\n}", "input_tokens": 856, "output_tokens": 400}, "2c117e9b01ad4a2e4bd5": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"堅固霧面鋁合金機身，結構紮實，經久耐用。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"標準 104鍵配置，滿足專業工作者高效需求。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及一年保固的相關資訊。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"堅固霧面鋁合金機身，結構紮實，經久耐用。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"樸實耐用取向，適合通勤族與遠距工作者使用。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"經典機械式手感，提供精準回饋與穩定輸入。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案使用「首選」一詞，屬於無法驗證的最高級宣稱（原文：是專業人士的首選裝備）。\"\n    }\n  ]\n}", "input_tokens": 858, "output_tokens": 333}, "c78be1ed3cb5b6df7dd4": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"750ml 大容量，滿足全日飲水需求。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用 PC+ABS 複合材質，抗衝擊耐用。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"台灣製造，原廠提供一年保固服務。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"長效保溫，PC+ABS複合材質，台灣製造。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"Volta 不鏽鋼保溫瓶 750ml。長效保溫，PC+ABS複合材質，台灣製造。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"750ml 大容量，滿足全日飲水需求。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"採用 PC+ABS 複合材質，抗衝擊耐用。\"\n    }\n  ]\n}", "input_tokens": 826, "output_tokens": 317}, "f9bd18dc829807e3e9e4": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"無論是辦公室久坐、戶外運動，還是長時間通勤，750ml 的黃金容量設計，讓你輕鬆達到每日建議飲水量。減少重複裝水的次數，專注於你的重要事務。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"搭配 PC+ABS 複合型態，結構堅固耐用\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"台灣製造，品質保證！...保固 | 原廠一年保固\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"採用高效能真空隔熱技術，無論是炎炎夏日想喝冰涼的飲料，還是寒冷冬天想來一杯暖胃的熱茶，Volta 都能長時間維持最佳溫度\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"【規格速覽與保障】...容量 | 750ml...型態 | PC+ABS 複合...產地 | 台灣...保固 | 原廠一年保固\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中多處使用驚嘆號，例如：『大容量・長效保冰溫！』、『一次解決你的所有煩惱！』，不符合「完全沒有使用」的要求。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現『750ml 的黃金容量設計』、『簡約俐落的外型設計』等冗餘句型。\"\n    }\n  ]\n}", "input_tokens": 1582, "output_tokens": 443}, "32e52b28ecbd05ec140e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"高純度植物油脂配方，溫和溶解彩妝。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及『乳霜質地』，而是將產品描述為『卸妝油』。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現『最純淨』，屬於應避免的誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"溫和溶解彩妝\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"純淨植萃卸妝油 100ml 敏弱肌適用\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"賣點皆為短句，例如：「高純度植物油脂配方，溫和溶解彩妝。」\"\n    }\n  ]\n}", "input_tokens": 879, "output_tokens": 318}, "174642c34412fc0205c7": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"每份足量添加50億CFU，高活性直達，有感順暢。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為久坐族設計，幫助維持消化道機能，告別悶脹。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"每日一顆，輕鬆啟動腸道防護力。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案整體用詞（如：高品質、有感順暢）屬常見廣告用語，未包含『立即見效』、『最強』、『保證』等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案在 SEO 描述中使用了驚嘆號：「...上班族必備！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"嚴選日本專利菌株，高品質專利益生菌配方。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"四條賣點分別溝通不同面向，例如賣點1的「高品質」與賣點4的「專為久坐族設計」，並未重複說明同一件事情。\"\n    }\n  ]\n}", "input_tokens": 871, "output_tokens": 372}, "e45e40720e76926446f9": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"每份含 50億CFU 專利益生菌，高效定殖。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為長時間久坐、飲食不規律的上班族設計。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"方便每日持續補充。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案使用「幫助維持」、「調整體質」等中性詞彙，未出現誇大用語。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"全文未使用驚嘆號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"嚴選日本原裝進口，品質安心可靠。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"賣點 3「幫助維持消化道機能，輕鬆體內環保」中的「維持消化道機能」與「體內環保」意義相近，有重複說明同一件事情的疑慮。\"\n    }\n  ]\n}", "input_tokens": 900, "output_tokens": 332}, "35126fea673bcbfbb349": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"每粒提供 50億CFU 活性菌，幫助您維持消化道機能。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"長時間久坐、飲食不規律的上班族必備。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"方便每日持續食用。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「品質安心有保障」，其中的「有保障」與禁用詞「保證」意思相近，屬於應避免的誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案全篇未使用驚嘆號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"僅提及「專利益生菌」，但未將「專利」的優勢轉化為「品質信賴」或「來源安心」等消費者可直接理解的益處。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"四條賣點分別闡述規格、功能、品質、便利性，主題明確，沒有重複說明同一件事情。\"\n    }\n  ]\n}", "input_tokens": 811, "output_tokens": 356}, "fa0e9223766dd10b65ee": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"主要成分 | 專利益生菌 50億 CFU... 建議用量 | 每日 1-2 粒\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"久坐辦公室族群：經常感到腹部悶脹、排便不規律者。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"讓順暢不再是偶爾的幸運，而是每日的常態。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"缺少對『最佳』等最高級誇大詞彙的避免，文案中使用了『最佳的投資』。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"缺少對驚嘆號的完全避免，文案結尾使用了驚嘆號：「...開啟您的輕盈有感人生！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"產地 | 日本 (Japan) | 嚴謹製程，品質安心保證。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"1. 【日本專利技術】頂級菌株，穩定高效 2. 【複方協同作用】全面調理，不只求順暢 3. 【研選哲學】高規格錠劑，方便穩定\"\n    }\n  ]\n}", "input_tokens": 1587, "output_tokens": 397}, "38662ec662a619ffe7c1": {"text": "{\n  \"sentiment\": \"neutral\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"\",\n  \"urgency\": \"low\"\n}", "input_tokens": 342, "output_tokens": 62}, "5e9fbd86665495342d71": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"\",\n  \"urgency\": \"low\"\n}", "input_tokens": 367, "output_tokens": 56}, "d958833b4ddf8ce0f392": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"\",\n  \"urgency\": \"low\"\n}", "input_tokens": 385, "output_tokens": 62}, "6c928cfb27796fb69180": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"\",\n  \"urgency\": \"low\"\n}", "input_tokens": 374, "output_tokens": 56}, "bc8fafaa509ead9bb45d": {"text": "{\n\"sentiment\": \"negative\",\n\"aspects\": [\n\"包裝\"\n],\n\"is_about_product\": false,\n\"actionable_suggestion\": \"加強出貨時的外部保護包裝，避免商品外盒在運送過程中受損\",\n\"urgency\": \"medium\"\n}", "input_tokens": 377, "output_tokens": 71}, "2277e8e527fd1ad971f4": {"text": "{\n  \"sentiment\": \"negative\",\n  \"aspects\": [\n    \"物流配送\",\n    \"客服\"\n  ],\n  \"is_about_product\": false,\n  \"actionable_suggestion\": \"優化出貨流程以縮短配送時間，並要求客服提供明確的延遲原因與預計送達時間，而非僅回覆處理中。\",\n  \"urgency\": \"medium\"\n}", "input_tokens": 377, "output_tokens": 95}, "ba6dff6da87712e1b0e0": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"\",\n  \"urgency\": \"low\"\n}", "input_tokens": 379, "output_tokens": 62}, "58440a6086fab12a80e8": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"使用體驗\",\n    \"產品品質\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"考慮推出更大容量的產品規格\",\n  \"urgency\": \"low\"\n}", "input_tokens": 365, "output_tokens": 70}, "31e3c2063800a86f260a": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"更新掛耳咖啡沖泡說明，提供明確的水溫建議（例如：88°C）以避免澀感。\",\n  \"urgency\": \"low\"\n}", "input_tokens": 383, "output_tokens": 88}, "f83f9cbfb867fa1b36b9": {"text": "{\n\"sentiment\": \"neutral\",\n\"aspects\": [\"產品品質\", \"使用體驗\", \"價格\"],\n\"is_about_product\": true,\n\"actionable_suggestion\": \"評估產品定價策略，或考慮推出大包裝/多盒組合優惠以降低單價感\",\n\"urgency\": \"low\"\n}", "input_tokens": 380, "output_tokens": 76}, "fa917a5e62fb8cc2f406": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"使用體驗\",\n    \"包裝\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"檢查滴管長度與瓶口設計，避免滴管尖端容易沾染瓶口外部。\",\n  \"urgency\": \"low\"\n}", "input_tokens": 388, "output_tokens": 84}, "bac8771b1a183a650142": {"text": "{\n  \"sentiment\": \"negative\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"重新檢視面膜布的剪裁設計，特別是眼下區域，以改善服貼度；或調整精華液配方，增加黏稠度以減少滴落。\",\n  \"urgency\": \"medium\"\n}", "input_tokens": 373, "output_tokens": 101}, "2b8e7a0d3338a5a1d16a": {"text": "{\n  \"sentiment\": \"negative\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"增加集塵盒容量，並重新設計集塵盒傾倒機制，避免灰塵外噴\",\n  \"urgency\": \"medium\"\n}", "input_tokens": 366, "output_tokens": 84}, "80671142b38a72919fff": {"text": "{\n  \"sentiment\": \"neutral\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"1. 增加耳塞尺寸選項（如XS），改善小耳道用戶的配戴舒適度。2. 檢查通話麥克風的降噪功能，排除通話時對方聽到的雜音問題。\",\n  \"urgency\": \"medium\"\n}", "input_tokens": 372, "output_tokens": 110}, "5b1af89879928e397480": {"text": "{\n  \"sentiment\": \"negative\",\n  \"aspects\": [\n    \"使用體驗\",\n    \"產品品質\",\n    \"客服\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"在商品頁面或說明書中增加適用膚質的詳細說明或警語，並追蹤是否有其他客戶回報致痘問題。\",\n  \"urgency\": \"medium\"\n}", "input_tokens": 367, "output_tokens": 98}}, "token_counts": {"tok:9d37906d3581365ecfaf": 14, "tok:6caa929e1c491a647ff5": 13, "tok:199e5367866f34a4f260": 387, "tok:febcc7fe24883e573941": 24}}
print(f'已載入 {len(FIXTURES["calls"]):,} 筆錄製輸出' + ('　模式：離線重播' if OFFLINE_MODE else '　模式：連線'))

In [ ]:
import json, time, re
from dataclasses import dataclass, field, asdict
import pandas as pd

client = make_client(FIXTURES)

if OFFLINE_MODE:
    print("✓ 離線重播模式 — 不會連網")
elif RECORD_FIXTURES:
    print("✓ 錄製模式 — 正常呼叫並記錄輸出")
    print(f"  {'Vertex AI' if USE_VERTEX else 'Gemini API'}")
else:
    print(f"✓ {'Vertex AI — project=' + PROJECT_ID if USE_VERTEX else 'Gemini API (AI Studio key)'}")

print(f"  生成模型：{GEN_MODEL}")
print(f"  評審模型：{JUDGE_MODEL}  ← 刻意與生成模型不同，降低 self-preference bias")

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **30:00 — 捲動帶過，不要逐行讀。**
>
> 要說的一句：「注意這幾行 —— 切 Vertex 或 AI Studio 只差 client 的參數，
> 底下所有程式碼一行都不用改。這是部署決策，不是技術決策。」

</div>

## §1 商品資料

50 筆模擬台灣電商 PIM 匯出的商品資料 —— **這也是評測集本身**。
前 12 筆是手寫的（規格較完整，等一下看文案時用它們），後面是組合生成的，
用來補足統計需要的樣本量。

50 筆 × 7 條 rubric = 350 次檢查，雜訊底線 0.29 個百分點 ——
足以分辨 3 個百分點以上的差距。

刻意涵蓋三種法規類別：

- `food` — 保健食品／食品，受《食品安全衛生管理法》第 28 條規範
- `cosmetic` — 美妝保養，受《化粧品衛生安全管理法》第 10 條規範
- `general` — 3C／家電，無特殊廣告限制

**這個欄位是後面所有事情的關鍵。** 「護眼」用在葉黃素上是違規，
用在螢幕護目鏡上不是。規則必須跟著商品屬性走。

In [ ]:
PRODUCTS_RAW = {
 "_meta": {
  "description": "評測用商品資料集，前 12 筆為手寫，其餘由 seed=20260821 組合產生。",
  "note": "notebook 與 run_eval.py 共用這一份 —— demo 與評測是同一組數字，不再分裂。",
  "handwritten_count": 12,
  "count": 200,
  "seed": 20260821,
  "channel_limits": {
   "shopee_title_max": 60,
   "momo_title_max": 50,
   "seo_desc_max": 120,
   "bullet_max": 30
  }
 },
 "products": [
  {
   "sku": "HB-1001",
   "name": "金盞花萃取葉黃素膠囊 60粒",
   "brand": "晨光研選",
   "category": "保健食品 / 眼部保養",
   "regulated_category": "food",
   "price": 890,
   "specs": {
    "淨含量": "60粒 / 盒",
    "主成分": "游離型葉黃素 30mg、玉米黃素 6mg",
    "劑型": "植物膠囊",
    "產地": "台灣",
    "建議食用": "每日1粒，隨餐食用"
   },
   "must_include_keywords": [
    "葉黃素",
    "60粒",
    "30mg"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "25-45 歲長時間使用螢幕的上班族"
  },
  {
   "sku": "HB-1002",
   "name": "五益菌複方益生菌粉 30入",
   "brand": "晨光研選",
   "category": "保健食品 / 消化保健",
   "regulated_category": "food",
   "price": 1180,
   "specs": {
    "淨含量": "30包 / 盒，每包 2g",
    "主成分": "五株專利益生菌，每包 100 億 CFU",
    "劑型": "粉末隨身包",
    "產地": "台灣",
    "建議食用": "每日1包，空腹或睡前"
   },
   "must_include_keywords": [
    "益生菌",
    "30包",
    "100億"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "外食族、作息不規律的上班族"
  },
  {
   "sku": "HB-1003",
   "name": "小分子魚膠原蛋白粉 蔓越莓風味 30入",
   "brand": "晨光研選",
   "category": "保健食品 / 美容保養",
   "regulated_category": "food",
   "price": 1580,
   "specs": {
    "淨含量": "30包 / 盒，每包 3g",
    "主成分": "深海魚膠原蛋白胜肽 2500mg、維生素C 80mg",
    "劑型": "粉末隨身包",
    "產地": "台灣",
    "建議食用": "每日1包，睡前溫水沖泡"
   },
   "must_include_keywords": [
    "膠原蛋白",
    "2500mg",
    "蔓越莓"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "30-50 歲注重外在保養的女性"
  },
  {
   "sku": "CS-2001",
   "name": "積雪草舒緩精華液 30ml",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 精華液",
   "regulated_category": "cosmetic",
   "price": 780,
   "specs": {
    "容量": "30ml",
    "主成分": "積雪草萃取 5%、泛醇 2%、神經醯胺",
    "質地": "水感精華",
    "適用膚質": "所有膚質，敏弱肌適用",
    "產地": "韓國"
   },
   "must_include_keywords": [
    "積雪草",
    "30ml",
    "5%"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  },
  {
   "sku": "CS-2002",
   "name": "胺基酸氨基酸溫和洗面乳 120ml",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 清潔",
   "regulated_category": "cosmetic",
   "price": 420,
   "specs": {
    "容量": "120ml",
    "主成分": "胺基酸界面活性劑、洋甘菊萃取",
    "質地": "綿密泡沫",
    "pH值": "弱酸性 pH5.5",
    "產地": "台灣"
   },
   "must_include_keywords": [
    "胺基酸",
    "120ml",
    "pH5.5"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  },
  {
   "sku": "CS-2003",
   "name": "多重玻尿酸保濕面膜 5片入",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 面膜",
   "regulated_category": "cosmetic",
   "price": 350,
   "specs": {
    "容量": "5片 / 盒，每片 25ml 精華",
    "主成分": "五重玻尿酸、海藻醣",
    "膜布": "天絲蠶絲膜布",
    "適用膚質": "乾燥、缺水肌",
    "產地": "台灣"
   },
   "must_include_keywords": [
    "玻尿酸",
    "5片",
    "天絲"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  },
  {
   "sku": "EL-3001",
   "name": "無線降噪藍牙耳機 Pro 2",
   "brand": "Volta",
   "category": "3C / 音訊",
   "regulated_category": "general",
   "price": 3290,
   "specs": {
    "降噪深度": "最高 42dB 主動降噪",
    "續航": "單次 8 小時，含充電盒 32 小時",
    "連線": "藍牙 5.4，支援雙裝置切換",
    "防水": "IPX4",
    "重量": "單耳 4.8g"
   },
   "must_include_keywords": [
    "降噪",
    "32小時",
    "藍牙5.4"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "通勤族、遠距工作者"
  },
  {
   "sku": "EL-3002",
   "name": "65W 氮化鎵三孔快充充電器",
   "brand": "Volta",
   "category": "3C / 充電",
   "regulated_category": "general",
   "price": 990,
   "specs": {
    "輸出": "65W 總輸出，USB-C×2 + USB-A×1",
    "技術": "第三代氮化鎵 GaN",
    "尺寸": "5.2 × 5.2 × 2.9 cm",
    "重量": "112g",
    "保護": "過壓、過流、過熱保護"
   },
   "must_include_keywords": [
    "65W",
    "氮化鎵",
    "三孔"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "出差族、多裝置使用者"
  },
  {
   "sku": "EL-3003",
   "name": "手持無線吸塵器 Lite",
   "brand": "Volta",
   "category": "家電 / 清潔",
   "regulated_category": "general",
   "price": 4680,
   "specs": {
    "吸力": "18000Pa",
    "續航": "標準模式 35 分鐘",
    "集塵盒": "0.5L 可水洗",
    "重量": "1.4kg",
    "配件": "縫隙吸頭、床墊吸頭、軟絨滾筒"
   },
   "must_include_keywords": [
    "18000Pa",
    "35分鐘",
    "1.4kg"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "小坪數家庭、養寵物族群"
  },
  {
   "sku": "FD-4001",
   "name": "冷萃掛耳咖啡 淺焙水果調 10入",
   "brand": "溯源焙所",
   "category": "food / 咖啡",
   "regulated_category": "food",
   "price": 460,
   "specs": {
    "淨含量": "10包 / 盒，每包 12g",
    "產區": "衣索比亞 耶加雪菲",
    "烘焙度": "淺焙",
    "風味": "柑橘、莓果、花香",
    "保存": "室溫陰涼處，開封後盡快飲用"
   },
   "must_include_keywords": [
    "耶加雪菲",
    "10包",
    "淺焙"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "25-45 歲精品咖啡愛好者"
  },
  {
   "sku": "FD-4002",
   "name": "低溫烘焙綜合堅果 無調味 350g",
   "brand": "溯源焙所",
   "category": "food / 零食",
   "regulated_category": "food",
   "price": 590,
   "specs": {
    "淨含量": "350g / 罐",
    "內容": "腰果、核桃、杏仁、夏威夷豆",
    "製程": "低溫烘焙，無油炸",
    "調味": "完全無調味、無添加糖鹽",
    "保存": "開封後冷藏，兩週內食用完畢"
   },
   "must_include_keywords": [
    "綜合堅果",
    "350g",
    "無調味"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "注重原型食物的家庭與健身族群"
  },
  {
   "sku": "FD-4003",
   "name": "台灣紅烏龍茶包 三角立體茶包 20入",
   "brand": "溯源焙所",
   "category": "food / 茶葉",
   "regulated_category": "food",
   "price": 380,
   "specs": {
    "淨含量": "20包 / 盒，每包 3g",
    "產區": "台東鹿野",
    "發酵度": "重發酵",
    "風味": "蜜香、熟果、尾韻回甘",
    "沖泡": "95°C 熱水，浸泡 90 秒"
   },
   "must_include_keywords": [
    "紅烏龍",
    "鹿野",
    "20包"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "30-55 歲台灣茶愛好者、送禮需求"
  },
  {
   "sku": "HB-E0013",
   "name": "溯源焙所 魚膠原蛋白粉 3000mg 90粒",
   "brand": "溯源焙所",
   "category": "保健食品 / 美容保養",
   "regulated_category": "food",
   "price": 490,
   "specs": {
    "主要規格": "深海魚膠原蛋白胜肽 3000mg",
    "型態": "獨立包裝",
    "產地": "韓國",
    "包裝": "90粒 / 盒",
    "建議食用": "每日1份，睡前食用"
   },
   "must_include_keywords": [
    "深海魚膠原蛋白胜肽",
    "3000mg",
    "90粒"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "30-50 歲注重外在保養的女性"
  },
  {
   "sku": "CS-E0014",
   "name": "白日光 SUNDAY 純淨卸妝油 100ml 1入",
   "brand": "白日光 SUNDAY",
   "category": "美妝保養 / 卸妝",
   "regulated_category": "cosmetic",
   "price": 390,
   "specs": {
    "主要規格": "植物油脂 100ml",
    "型態": "油狀質地",
    "產地": "韓國",
    "包裝": "1入 / 盒",
    "適用膚質": "乾燥缺水肌"
   },
   "must_include_keywords": [
    "植物油脂",
    "100ml",
    "1入"
   ],
   "brand_tone": "明亮、生活感、口語但不輕浮",
   "target_audience": "30-45 歲重視保濕與溫和配方的族群"
  },
  {
   "sku": "EL-E0015",
   "name": "本木 BENMU 除濕機 16L",
   "brand": "本木 BENMU",
   "category": "家電 / 空氣",
   "regulated_category": "general",
   "price": 490,
   "specs": {
    "主要規格": "每日除濕量 16L",
    "型態": "陽極處理",
    "產地": "美國",
    "保固": "終身技術支援"
   },
   "must_include_keywords": [
    "除濕",
    "16L"
   ],
   "brand_tone": "樸實、耐用取向、強調材質與工法",
   "target_audience": "重視 CP 值的首購族"
  },
  {
   "sku": "HB-E0016",
   "name": "晨光研選 緩釋B群錠 100mg 20粒",
   "brand": "晨光研選",
   "category": "保健食品 / 精神體力",
   "regulated_category": "food",
   "price": 290,
   "specs": {
    "主要規格": "維生素B群複方 100mg",
    "型態": "錠劑",
    "產地": "台灣",
    "包裝": "20粒 / 盒",
    "建議食用": "每日1份，睡前食用"
   },
   "must_include_keywords": [
    "維生素B群複方",
    "100mg",
    "20粒"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "25-45 歲長時間使用螢幕的上班族"
  },
  {
   "sku": "CS-E0017",
   "name": "青研 CHINGYEN 保濕面膜 7片 1入",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 面膜",
   "regulated_category": "cosmetic",
   "price": 1580,
   "specs": {
    "主要規格": "多重玻尿酸 7片",
    "型態": "水感質地",
    "產地": "美國",
    "包裝": "1入 / 盒",
    "適用膚質": "敏弱肌適用"
   },
   "must_include_keywords": [
    "多重玻尿酸",
    "7片",
    "1入"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  },
  {
   "sku": "EL-E0018",
   "name": "本木 BENMU 螢幕掛燈 95Ra",
   "brand": "本木 BENMU",
   "category": "3C / 週邊",
   "regulated_category": "general",
   "price": 890,
   "specs": {
    "主要規格": "顯色指數 95Ra",
    "型態": "霧面鋁合金",
    "產地": "台灣",
    "保固": "終身技術支援"
   },
   "must_include_keywords": [
    "顯色",
    "95Ra"
   ],
   "brand_tone": "樸實、耐用取向、強調材質與工法",
   "target_audience": "通勤族、遠距工作者"
  },
  {
   "sku": "HB-E0019",
   "name": "晨光研選 凍乾果乾 50g 20粒",
   "brand": "晨光研選",
   "category": "food / 零食",
   "regulated_category": "food",
   "price": 1180,
   "specs": {
    "主要規格": "整顆凍乾水果 50g",
    "型態": "軟膠囊",
    "產地": "美國",
    "包裝": "20粒 / 盒",
    "建議食用": "每日1份，睡前食用"
   },
   "must_include_keywords": [
    "整顆凍乾水果",
    "50g",
    "20粒"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "外食族、作息不規律的上班族"
  },
  {
   "sku": "CS-E0020",
   "name": "青研 CHINGYEN 修護乳液 80ml 7入",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 乳液",
   "regulated_category": "cosmetic",
   "price": 890,
   "specs": {
    "主要規格": "神經醯胺 80ml",
    "型態": "水感質地",
    "產地": "日本",
    "包裝": "7入 / 盒",
    "適用膚質": "敏弱肌適用"
   },
   "must_include_keywords": [
    "神經醯胺",
    "80ml",
    "7入"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "30-45 歲重視保濕與溫和配方的族群"
  },
  {
   "sku": "EL-E0021",
   "name": "Volta 鑄鐵平底鍋 20cm",
   "brand": "Volta",
   "category": "生活 / 廚具",
   "regulated_category": "general",
   "price": 890,
   "specs": {
    "主要規格": "直徑 20cm",
    "型態": "PC+ABS 複合",
    "產地": "美國",
    "保固": "終身技術支援"
   },
   "must_include_keywords": [
    "鑄鐵",
    "20cm"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "出差族、多裝置使用者"
  },
  {
   "sku": "HB-E0022",
   "name": "溯源焙所 低溫烘焙堅果 350g 20粒",
   "brand": "溯源焙所",
   "category": "food / 零食",
   "regulated_category": "food",
   "price": 2290,
   "specs": {
    "主要規格": "綜合堅果 350g",
    "型態": "植物膠囊",
    "產地": "台灣",
    "包裝": "20粒 / 盒",
    "建議食用": "每日1份，睡前食用"
   },
   "must_include_keywords": [
    "綜合堅果",
    "350g",
    "20粒"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "外食族、作息不規律的上班族"
  },
  {
   "sku": "CS-E0023",
   "name": "青研 CHINGYEN 保濕面膜 5片 3入",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 面膜",
   "regulated_category": "cosmetic",
   "price": 390,
   "specs": {
    "主要規格": "多重玻尿酸 5片",
    "型態": "凝露質地",
    "產地": "德國",
    "包裝": "3入 / 盒",
    "適用膚質": "敏弱肌適用"
   },
   "must_include_keywords": [
    "多重玻尿酸",
    "5片",
    "3入"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  },
  {
   "sku": "EL-E0024",
   "name": "Volta 循環扇 8段",
   "brand": "Volta",
   "category": "家電 / 空調",
   "regulated_category": "general",
   "price": 490,
   "specs": {
    "主要規格": "風速段數 8段",
    "型態": "食品級不鏽鋼",
    "產地": "日本",
    "保固": "原廠一年保固"
   },
   "must_include_keywords": [
    "風速",
    "8段"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "小坪數家庭、養寵物族群"
  },
  {
   "sku": "HB-E0025",
   "name": "晨光研選 緩釋B群錠 75mg 20粒",
   "brand": "晨光研選",
   "category": "保健食品 / 精神體力",
   "regulated_category": "food",
   "price": 2290,
   "specs": {
    "主要規格": "維生素B群複方 75mg",
    "型態": "獨立包裝",
    "產地": "日本",
    "包裝": "20粒 / 盒",
    "建議食用": "每日1份，睡前食用"
   },
   "must_include_keywords": [
    "維生素B群複方",
    "75mg",
    "20粒"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "25-45 歲長時間使用螢幕的上班族"
  },
  {
   "sku": "CS-E0026",
   "name": "白日光 SUNDAY 唇部修護膏 15g 3入",
   "brand": "白日光 SUNDAY",
   "category": "美妝保養 / 唇部",
   "regulated_category": "cosmetic",
   "price": 290,
   "specs": {
    "主要規格": "蜂蠟與植物油 15g",
    "型態": "油狀質地",
    "產地": "美國",
    "包裝": "3入 / 盒",
    "適用膚質": "敏弱肌適用"
   },
   "must_include_keywords": [
    "蜂蠟與植物油",
    "15g",
    "3入"
   ],
   "brand_tone": "明亮、生活感、口語但不輕浮",
   "target_audience": "18-30 歲學生與社群重度使用者"
  },
  {
   "sku": "EL-E0027",
   "name": "本木 BENMU 鑄鐵平底鍋 24cm",
   "brand": "本木 BENMU",
   "category": "生活 / 廚具",
   "regulated_category": "general",
   "price": 890,
   "specs": {
    "主要規格": "直徑 24cm",
    "型態": "陽極處理",
    "產地": "台灣",
    "保固": "原廠一年保固"
   },
   "must_include_keywords": [
    "鑄鐵",
    "24cm"
   ],
   "brand_tone": "樸實、耐用取向、強調材質與工法",
   "target_audience": "出差族、多裝置使用者"
  },
  {
   "sku": "HB-E0028",
   "name": "溯源焙所 綜合維他命錠 25種 30粒",
   "brand": "溯源焙所",
   "category": "保健食品 / 綜合補充",
   "regulated_category": "food",
   "price": 390,
   "specs": {
    "主要規格": "23種維生素礦物質 25種",
    "型態": "軟膠囊",
    "產地": "台灣",
    "包裝": "30粒 / 盒",
    "建議食用": "每日1份，隨餐食用"
   },
   "must_include_keywords": [
    "23種維生素礦物質",
    "25種",
    "30粒"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "30-50 歲注重外在保養的女性"
  },
  {
   "sku": "CS-E0029",
   "name": "白日光 SUNDAY 護手霜 5% 5入",
   "brand": "白日光 SUNDAY",
   "category": "美妝保養 / 手部",
   "regulated_category": "cosmetic",
   "price": 490,
   "specs": {
    "主要規格": "尿素 5%",
    "型態": "乳霜質地",
    "產地": "台灣",
    "包裝": "5入 / 盒",
    "適用膚質": "乾燥缺水肌"
   },
   "must_include_keywords": [
    "尿素",
    "5%",
    "5入"
   ],
   "brand_tone": "明亮、生活感、口語但不輕浮",
   "target_audience": "30-45 歲重視保濕與溫和配方的族群"
  },
  {
   "sku": "EL-E0030",
   "name": "Volta 陶瓷刀具組 3件",
   "brand": "Volta",
   "category": "生活 / 廚具",
   "regulated_category": "general",
   "price": 3290,
   "specs": {
    "主要規格": "件數 3件",
    "型態": "食品級不鏽鋼",
    "產地": "韓國",
    "保固": "終身技術支援"
   },
   "must_include_keywords": [
    "陶瓷",
    "3件"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "小坪數家庭、養寵物族群"
  },
  {
   "sku": "HB-E0031",
   "name": "溯源焙所 深海魚油軟膠囊 800mg 20粒",
   "brand": "溯源焙所",
   "category": "保健食品 / 循環保健",
   "regulated_category": "food",
   "price": 390,
   "specs": {
    "主要規格": "Omega-3 800mg",
    "型態": "獨立包裝",
    "產地": "韓國",
    "包裝": "20粒 / 盒",
    "建議食用": "每日1份，睡前食用"
   },
   "must_include_keywords": [
    "Omega-3",
    "800mg",
    "20粒"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "30-50 歲注重外在保養的女性"
  },
  {
   "sku": "CS-E0032",
   "name": "青研 CHINGYEN 礦物防曬乳 50ml 3入",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 防曬",
   "regulated_category": "cosmetic",
   "price": 390,
   "specs": {
    "主要規格": "氧化鋅 50ml",
    "型態": "油狀質地",
    "產地": "韓國",
    "包裝": "3入 / 盒",
    "適用膚質": "敏弱肌適用"
   },
   "must_include_keywords": [
    "氧化鋅",
    "50ml",
    "3入"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  },
  {
   "sku": "EL-E0033",
   "name": "Volta USB-C 編織傳輸線 240W",
   "brand": "Volta",
   "category": "3C / 線材",
   "regulated_category": "general",
   "price": 290,
   "specs": {
    "主要規格": "過電瓦數 240W",
    "型態": "食品級不鏽鋼",
    "產地": "日本",
    "保固": "終身技術支援"
   },
   "must_include_keywords": [
    "USB-C",
    "240W"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "出差族、多裝置使用者"
  },
  {
   "sku": "HB-E0034",
   "name": "晨光研選 複方益生菌粉 100億CFU 60粒",
   "brand": "晨光研選",
   "category": "保健食品 / 消化保健",
   "regulated_category": "food",
   "price": 490,
   "specs": {
    "主要規格": "專利益生菌 100億CFU",
    "型態": "植物膠囊",
    "產地": "台灣",
    "包裝": "60粒 / 盒",
    "建議食用": "每日1份，睡前食用"
   },
   "must_include_keywords": [
    "專利益生菌",
    "100億CFU",
    "60粒"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "40 歲以上開始注意日常保養的族群"
  },
  {
   "sku": "CS-E0035",
   "name": "白日光 SUNDAY 舒緩精華液 10% 1入",
   "brand": "白日光 SUNDAY",
   "category": "美妝保養 / 精華液",
   "regulated_category": "cosmetic",
   "price": 290,
   "specs": {
    "主要規格": "積雪草萃取 10%",
    "型態": "凝露質地",
    "產地": "美國",
    "包裝": "1入 / 盒",
    "適用膚質": "敏弱肌適用"
   },
   "must_include_keywords": [
    "積雪草萃取",
    "10%",
    "1入"
   ],
   "brand_tone": "明亮、生活感、口語但不輕浮",
   "target_audience": "25-40 歲混合肌、重視質地的上班族"
  },
  {
   "sku": "EL-E0036",
   "name": "本木 BENMU 不鏽鋼保溫瓶 750ml",
   "brand": "本木 BENMU",
   "category": "生活 / 水壺",
   "regulated_category": "general",
   "price": 690,
   "specs": {
    "主要規格": "容量 750ml",
    "型態": "食品級不鏽鋼",
    "產地": "德國",
    "保固": "終身技術支援"
   },
   "must_include_keywords": [
    "保溫",
    "750ml"
   ],
   "brand_tone": "樸實、耐用取向、強調材質與工法",
   "target_audience": "小坪數家庭、養寵物族群"
  },
  {
   "sku": "HB-E0037",
   "name": "溯源焙所 乳清蛋白粉 30g 20粒",
   "brand": "溯源焙所",
   "category": "food / 運動營養",
   "regulated_category": "food",
   "price": 4680,
   "specs": {
    "主要規格": "分離乳清蛋白 30g",
    "型態": "粉末隨身包",
    "產地": "韓國",
    "包裝": "20粒 / 盒",
    "建議食用": "每日1份，睡前食用"
   },
   "must_include_keywords": [
    "分離乳清蛋白",
    "30g",
    "20粒"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "30-50 歲注重外在保養的女性"
  },
  {
   "sku": "CS-E0038",
   "name": "青研 CHINGYEN 礦物防曬乳 60ml 1入",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 防曬",
   "regulated_category": "cosmetic",
   "price": 890,
   "specs": {
    "主要規格": "氧化鋅 60ml",
    "型態": "油狀質地",
    "產地": "德國",
    "包裝": "1入 / 盒",
    "適用膚質": "敏弱肌適用"
   },
   "must_include_keywords": [
    "氧化鋅",
    "60ml",
    "1入"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "30-45 歲重視保濕與溫和配方的族群"
  },
  {
   "sku": "EL-E0039",
   "name": "Volta 不鏽鋼保溫瓶 350ml",
   "brand": "Volta",
   "category": "生活 / 水壺",
   "regulated_category": "general",
   "price": 290,
   "specs": {
    "主要規格": "容量 350ml",
    "型態": "PC+ABS 複合",
    "產地": "台灣",
    "保固": "原廠一年保固"
   },
   "must_include_keywords": [
    "保溫",
    "350ml"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "出差族、多裝置使用者"
  },
  {
   "sku": "HB-E0040",
   "name": "晨光研選 甘胺酸亞鐵錠 20mg 90粒",
   "brand": "晨光研選",
   "category": "保健食品 / 女性保健",
   "regulated_category": "food",
   "price": 1580,
   "specs": {
    "主要規格": "甘胺酸亞鐵 20mg",
    "型態": "軟膠囊",
    "產地": "日本",
    "包裝": "90粒 / 盒",
    "建議食用": "每日1份，隨餐食用"
   },
   "must_include_keywords": [
    "甘胺酸亞鐵",
    "20mg",
    "90粒"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "25-45 歲長時間使用螢幕的上班族"
  },
  {
   "sku": "CS-E0041",
   "name": "青研 CHINGYEN 唇部修護膏 8g 7入",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 唇部",
   "regulated_category": "cosmetic",
   "price": 1180,
   "specs": {
    "主要規格": "蜂蠟與植物油 8g",
    "型態": "水感質地",
    "產地": "日本",
    "包裝": "7入 / 盒",
    "適用膚質": "敏弱肌適用"
   },
   "must_include_keywords": [
    "蜂蠟與植物油",
    "8g",
    "7入"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "18-30 歲學生與社群重度使用者"
  },
  {
   "sku": "EL-E0042",
   "name": "本木 BENMU 空氣清淨機 15坪",
   "brand": "本木 BENMU",
   "category": "家電 / 空氣",
   "regulated_category": "general",
   "price": 890,
   "specs": {
    "主要規格": "適用坪數 15坪",
    "型態": "食品級不鏽鋼",
    "產地": "日本",
    "保固": "原廠一年保固"
   },
   "must_include_keywords": [
    "坪數",
    "15坪"
   ],
   "brand_tone": "樸實、耐用取向、強調材質與工法",
   "target_audience": "通勤族、遠距工作者"
  },
  {
   "sku": "HB-E0043",
   "name": "溯源焙所 乳清蛋白粉 30g 60粒",
   "brand": "溯源焙所",
   "category": "food / 運動營養",
   "regulated_category": "food",
   "price": 390,
   "specs": {
    "主要規格": "分離乳清蛋白 30g",
    "型態": "粉末隨身包",
    "產地": "德國",
    "包裝": "60粒 / 盒",
    "建議食用": "每日1份，隨餐食用"
   },
   "must_include_keywords": [
    "分離乳清蛋白",
    "30g",
    "60粒"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "40 歲以上開始注意日常保養的族群"
  },
  {
   "sku": "CS-E0044",
   "name": "青研 CHINGYEN 溫和洗面乳 150ml 1入",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 清潔",
   "regulated_category": "cosmetic",
   "price": 2290,
   "specs": {
    "主要規格": "胺基酸界面活性劑 150ml",
    "型態": "凝露質地",
    "產地": "台灣",
    "包裝": "1入 / 盒",
    "適用膚質": "敏弱肌適用"
   },
   "must_include_keywords": [
    "胺基酸界面活性劑",
    "150ml",
    "1入"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "30-45 歲重視保濕與溫和配方的族群"
  },
  {
   "sku": "EL-E0045",
   "name": "Volta 不鏽鋼保溫瓶 750ml",
   "brand": "Volta",
   "category": "生活 / 水壺",
   "regulated_category": "general",
   "price": 490,
   "specs": {
    "主要規格": "容量 750ml",
    "型態": "PC+ABS 複合",
    "產地": "台灣",
    "保固": "原廠一年保固"
   },
   "must_include_keywords": [
    "保溫",
    "750ml"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "出差族、多裝置使用者"
  },
  {
   "sku": "HB-E0046",
   "name": "晨光研選 即溶燕麥飲 25g 30粒",
   "brand": "晨光研選",
   "category": "food / 沖泡飲",
   "regulated_category": "food",
   "price": 490,
   "specs": {
    "主要規格": "全穀燕麥 25g",
    "型態": "獨立包裝",
    "產地": "德國",
    "包裝": "30粒 / 盒",
    "建議食用": "每日1份，睡前食用"
   },
   "must_include_keywords": [
    "全穀燕麥",
    "25g",
    "30粒"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "注重原型食物的家庭與健身族群"
  },
  {
   "sku": "CS-E0047",
   "name": "白日光 SUNDAY 修護乳液 50ml 1入",
   "brand": "白日光 SUNDAY",
   "category": "美妝保養 / 乳液",
   "regulated_category": "cosmetic",
   "price": 890,
   "specs": {
    "主要規格": "神經醯胺 50ml",
    "型態": "乳霜質地",
    "產地": "美國",
    "包裝": "1入 / 盒",
    "適用膚質": "所有膚質"
   },
   "must_include_keywords": [
    "神經醯胺",
    "50ml",
    "1入"
   ],
   "brand_tone": "明亮、生活感、口語但不輕浮",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  },
  {
   "sku": "EL-E0048",
   "name": "本木 BENMU 機械式鍵盤 104鍵",
   "brand": "本木 BENMU",
   "category": "3C / 週邊",
   "regulated_category": "general",
   "price": 3290,
   "specs": {
    "主要規格": "按鍵數 104鍵",
    "型態": "霧面鋁合金",
    "產地": "美國",
    "保固": "原廠一年保固"
   },
   "must_include_keywords": [
    "機械式",
    "104鍵"
   ],
   "brand_tone": "樸實、耐用取向、強調材質與工法",
   "target_audience": "通勤族、遠距工作者"
  },
  {
   "sku": "HB-E0049",
   "name": "晨光研選 複方益生菌粉 50億CFU 30粒",
   "brand": "晨光研選",
   "category": "保健食品 / 消化保健",
   "regulated_category": "food",
   "price": 4680,
   "specs": {
    "主要規格": "專利益生菌 50億CFU",
    "型態": "錠劑",
    "產地": "日本",
    "包裝": "30粒 / 盒",
    "建議食用": "每日1-2份"
   },
   "must_include_keywords": [
    "專利益生菌",
    "50億CFU",
    "30粒"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "25-45 歲長時間使用螢幕的上班族"
  },
  {
   "sku": "CS-E0050",
   "name": "青研 CHINGYEN 純淨卸妝油 100ml 1入",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 卸妝",
   "regulated_category": "cosmetic",
   "price": 390,
   "specs": {
    "主要規格": "植物油脂 100ml",
    "型態": "乳霜質地",
    "產地": "台灣",
    "包裝": "1入 / 盒",
    "適用膚質": "敏弱肌適用"
   },
   "must_include_keywords": [
    "植物油脂",
    "100ml",
    "1入"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  }
 ]
}

PRODUCTS = PRODUCTS_RAW['products']
CHANNEL_LIMITS = PRODUCTS_RAW['_meta']['channel_limits']

import pandas as pd
from collections import Counter
print(f'共 {len(PRODUCTS)} 筆　法規類別分佈：'
      + '　'.join(f'{k} {v}' for k, v in sorted(Counter(p['regulated_category'] for p in PRODUCTS).items())))
print('（下表只顯示前 12 筆手寫商品，等一下的文案示範會用它們）')
pd.DataFrame([{
    'SKU': p['sku'], '商品': p['name'][:24],
    '法規類別': p['regulated_category'], '售價': p['price'],
} for p in PRODUCTS[:12]])

### 禁詞清單

依台灣廣告法規整理，分成四組。**罰則差異很大**：

| 類別 | 法源 | 罰則 |
|---|---|---|
| 醫療效能 | 食安法 §28 第2項 | NT$60萬 – 500萬 |
| 不實誇張 | 食安法 §28 第1項 | NT$4萬 – 400萬 |
| 化粧品虛偽誇大 | 化粧品衛管法 §10 | NT$4萬 – 20萬 |

> 這不是工程潔癖，是不做會收罰單。這也是為什麼規則層要放在第一層。

⚠️ 本清單為示範用的簡化版，不構成法律意見；實際商用前應由法務或法規顧問審閱。

In [ ]:
BANNED_DATA = {
 "_meta": {
  "description": "商品文案禁詞清單，依台灣廣告法規分類。用於評測規則層。",
  "disclaimer": "本清單為示範用的簡化版，依主管機關公開認定準則的分類邏輯整理，不構成法律意見。實際商用前應由法務或法規顧問審閱，並以主管機關最新公告為準。",
  "legal_basis": {
   "food": {
    "law": "食品安全衛生管理法 第28條",
    "sub_regulation": "食品及相關產品標示宣傳廣告涉及不實誇張易生誤解或醫療效能認定準則",
    "penalty_exaggeration": "違反第1項（不實、誇張、易生誤解）：新臺幣 4 萬元以上 400 萬元以下罰鍰",
    "penalty_medical": "違反第2項（醫療效能）：新臺幣 60 萬元以上 500 萬元以下罰鍰",
    "criteria_ref": "認定準則 第4條（不實誇張易生誤解）、第5條（醫療效能）"
   },
   "cosmetic": {
    "law": "化粧品衛生安全管理法 第10條",
    "sub_regulation": "化粧品標示宣傳廣告涉及虛偽誇大或醫療效能認定準則",
    "note": "化粧品不得為虛偽或誇大之標示、宣傳、廣告，亦不得有醫療效能之標示、宣傳或廣告。",
    "penalty_exaggeration": "違反第1項（虛偽、誇大）：新臺幣 4 萬元以上 20 萬元以下罰鍰",
    "penalty_escalation": "若宣稱內容已使產品構成藥品，另可能違反藥事法（未經核准擅自製造販賣藥品／誇大醫療效能廣告），罰則顯著加重。說明此類升級風險時應指出其法源不同，勿逕稱『化粧品罰 500 萬』。"
   }
  },
  "important_nuance": "領有衛福部『健康食品』許可證（小綠人標章）的產品，得於許可範圍內宣稱經核可的保健功效。本清單假設商品『未取得』該許可，因此一律禁用。這正是規則層需要按商品屬性套用不同清單的原因。"
 },
 "medical_efficacy": {
  "_note": "涉及疾病之預防、改善、減輕、診斷或治療（認定準則第5條）。這是罰則最重的一類。",
  "applies_to": [
   "food",
   "cosmetic"
  ],
  "severity": "critical",
  "terms": [
   "治療",
   "療效",
   "醫療",
   "根治",
   "治癒",
   "痊癒",
   "藥效",
   "特效",
   "預防疾病",
   "抗癌",
   "防癌",
   "抑制腫瘤",
   "降血壓",
   "降血糖",
   "降膽固醇",
   "降尿酸",
   "降血脂",
   "消炎",
   "抗發炎",
   "殺菌",
   "抗菌",
   "抗病毒",
   "改善過敏",
   "治療過敏",
   "根治過敏",
   "改善失眠",
   "治療失眠",
   "護肝",
   "解毒",
   "排毒",
   "增強免疫力",
   "提升免疫力",
   "調節免疫",
   "抗憂鬱",
   "改善憂鬱",
   "壯陽",
   "生髮",
   "治療掉髮",
   "改善關節炎",
   "治療骨質疏鬆"
  ]
 },
 "exaggeration": {
  "_note": "不實、誇張或易生誤解（認定準則第4條）。含無證據支持之絕對化描述。",
  "applies_to": [
   "food",
   "cosmetic",
   "general"
  ],
  "severity": "high",
  "terms": [
   "最有效",
   "最強",
   "第一名",
   "唯一",
   "獨家專利保證",
   "百分之百有效",
   "100%有效",
   "保證有效",
   "絕對有效",
   "神奇",
   "奇蹟",
   "奇效",
   "驚人效果",
   "完全無副作用",
   "零副作用",
   "無任何副作用",
   "永久",
   "終身有效",
   "一勞永逸",
   "立即見效",
   "馬上見效",
   "三天見效",
   "七天見效",
   "全球最",
   "世界第一",
   "業界唯一"
  ]
 },
 "body_function_food": {
  "_note": "涉及維持或改變人體器官、組織、生理或外觀功能之描述（認定準則第4條第3款）。未取得健康食品許可者不得宣稱。",
  "applies_to": [
   "food"
  ],
  "severity": "high",
  "terms": [
   "美白",
   "淡斑",
   "除皺",
   "抗老",
   "逆齡",
   "回春",
   "瘦身",
   "減肥",
   "燃脂",
   "消脂",
   "瘦小腹",
   "阻斷澱粉",
   "豐胸",
   "長高",
   "改善視力",
   "恢復視力",
   "治療近視",
   "預防近視",
   "增強記憶力",
   "提升智力",
   "變聰明",
   "改善腸道",
   "整腸",
   "促進排便",
   "改善便秘",
   "護眼",
   "顧眼睛"
  ]
 },
 "cosmetic_medical": {
  "_note": "化粧品不得宣稱之醫療或類醫療效能（化粧品衛生安全管理法第10條）。",
  "applies_to": [
   "cosmetic"
  ],
  "severity": "critical",
  "terms": [
   "醫學美容",
   "醫美級",
   "藥用",
   "處方級",
   "換膚",
   "去角質換膚",
   "煥膚療程",
   "治療痘痘",
   "根治粉刺",
   "消除痘疤",
   "除皺",
   "撫平皺紋",
   "填補皺紋",
   "溶脂",
   "拉皮",
   "微整形效果",
   "生髮",
   "防止掉髮",
   "治療禿頭",
   "殺菌",
   "消毒",
   "抗菌配方",
   "修復受損肌膚細胞",
   "重建肌膚屏障"
  ]
 },
 "safe_alternatives": {
  "_note": "示範用的合規替代寫法。用來說明『規則層不只是擋，還能給修改方向』。",
  "mapping": {
   "護眼": "陪伴長時間用眼的日常",
   "改善視力": "日常補充葉黃素",
   "增強免疫力": "均衡飲食之外的日常補充",
   "整腸": "順暢的一天",
   "改善便秘": "維持日常規律作息",
   "美白": "亮采、勻淨膚況（化粧品限用，食品不可）",
   "抗老": "維持日常保養節奏",
   "醫學美容": "溫和配方",
   "殺菌": "清潔",
   "立即見效": "持續使用數週後感受變化",
   "保證有效": "多數使用者回饋"
  }
 }
}

for k, v in BANNED_DATA.items():
    if isinstance(v, dict) and 'terms' in v:
        print(f"{k:24s} {len(v['terms']):>3} 詞  適用：{'/'.join(v['applies_to'])}")

## §2 Prompt v0 → v3

**設計原則：每一版只加一件事。**

這樣評測表上每一欄的改善都能歸因到單一改動。若一次加三件事，
表格會一次全綠，就失去「哪個改動帶來哪個改善」的教學價值。

| 版本 | 加了什麼 | 預期修好 |
|---|---|---|
| v0 | 什麼都不給 | — （基準線）|
| v1 | 角色／受眾／字數／必含規格 | 長度、規格覆蓋 |
| v2 | 法規禁詞 + 品牌語調 few-shot | 禁詞、語調 |
| v3 | `responseSchema` 結構化輸出 | 可機器讀取 |

In [ ]:
"""Prompt v0 → v3 演進。

設計原則：**每一版只加一件事**，這樣評測表上每一欄的改善都能歸因到單一改動。
這是 §2 的核心。

    v0  naive          什麼都不給              → 基準線
    v1  + 通路約束      角色/受眾/字數/必含規格   → 修好「長度」與「規格覆蓋」
    v2  + 法規與語調    禁詞清單 + few-shot     → 修好「禁詞」與「語調」
    v3  + 結構化輸出    responseSchema         → 修好「可機器讀取」

刻意不做的事：不在 v1 就把法規塞進去。若一次加三件事，評測表會一次全綠，
就失去「哪個改動帶來哪個改善」的教學價值。
"""

import json


# --------------------------------------------------------------------------
# v3 的輸出 schema。同時給 Gemini 當 responseSchema，也給規則層當驗證依據。
# --------------------------------------------------------------------------
COPY_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": f"電商商品標題，最多 {SHOPEE_TITLE_MAX} 字",
        },
        "bullets": {
            "type": "array",
            "items": {"type": "string"},
            "description": f"{BULLET_COUNT} 條賣點，每條最多 {BULLET_MAX} 字",
        },
        "seo_description": {
            "type": "string",
            "description": f"SEO 描述，最多 {SEO_DESC_MAX} 字",
        },
        "hashtags": {
            "type": "array",
            "items": {"type": "string"},
            "description": "3-5 個標籤，不含 # 符號",
        },
    },
    "required": ["title", "bullets", "seo_description", "hashtags"],
}


def _product_block(product: dict) -> str:
    """把商品資料整理成 prompt 用的文字區塊。"""
    specs = "\n".join(f"  - {k}：{v}" for k, v in product["specs"].items())
    return (
        f"商品名稱：{product['name']}\n"
        f"品牌：{product['brand']}\n"
        f"分類：{product['category']}\n"
        f"售價：NT${product['price']}\n"
        f"規格：\n{specs}"
    )


# --------------------------------------------------------------------------
# v0 — naive
# --------------------------------------------------------------------------
def build_v0(product: dict, **_) -> str:
    """大多數人第一次寫的 prompt。刻意保持這麼爛。"""
    return f"幫我寫這個商品的電商文案。\n\n{_product_block(product)}"


# --------------------------------------------------------------------------
# v1 — 加通路約束
# --------------------------------------------------------------------------
def build_v1(product: dict, **_) -> str:
    """加入角色、受眾、通路硬限制、必含規格。

    這一版修好的是「可上架」：長度符合平台規則、規格沒有漏。
    還沒處理法規，所以禁詞仍會出現 —— 這是刻意的。
    """
    must = "、".join(product["must_include_keywords"])
    return f"""你是台灣電商平台的資深商品文案編輯。

請為以下商品撰寫文案，目標受眾是{product['target_audience']}。

【格式要求】
- 商品標題：**不超過 {SHOPEE_TITLE_MAX} 個字**
- 賣點：{BULLET_COUNT} 條，每條**不超過 {BULLET_MAX} 個字**
- SEO 描述：**不超過 {SEO_DESC_MAX} 個字**
- 標籤：3-5 個

【必須包含的規格資訊】
以下關鍵字必須出現在文案中，不可省略或改寫：{must}

【商品資料】
{_product_block(product)}"""


# --------------------------------------------------------------------------
# v2 — 加法規約束與品牌語調
# --------------------------------------------------------------------------
def build_v2(product: dict, banned_terms: list[str], tone_examples: list[dict] = None, **_) -> str:
    """在 v1 之上加兩件事：法規禁令、品牌語調 few-shot。

    禁詞清單依商品的 regulated_category 動態帶入 —— 保健食品和 3C 適用的規則不同，
    這是規則層必須「按商品屬性套用」的原因。
    """
    base = build_v1(product)

    banned_display = "、".join(banned_terms[:60])
    legal = {
        "food": "《食品安全衛生管理法》第 28 條",
        "cosmetic": "《化粧品衛生安全管理法》第 10 條",
        "general": "《公平交易法》關於不實廣告之規範",
    }[product["regulated_category"]]

    block = f"""

【法規限制 — 這是硬性要求，違反會被開罰】
本商品受{legal}規範。文案**絕對不可**出現下列詞彙或其同義表達：
{banned_display}

具體而言，不可宣稱：
- 疾病的預防、改善、減輕、診斷或治療
- 維持或改變人體器官、組織、生理或外觀之功能
- 無證據支持的絕對化描述（最有效、保證、100%、立即見效等）

若某個賣點只能用上述詞彙表達，請改用「描述使用情境」或「描述成分事實」的方式改寫，
不要為了避開禁詞而寫出空洞的句子。

【品牌語調】
{product['brand_tone']}"""

    if tone_examples:
        samples = "\n\n".join(
            f"範例 {i + 1}（商品：{ex['product']}）\n標題：{ex['title']}\n賣點：{ex['bullet']}"
            for i, ex in enumerate(tone_examples)
        )
        block += f"""

以下是本品牌既有的合規文案，請模仿其語氣與用字習慣：

{samples}"""

    return base + block


# --------------------------------------------------------------------------
# v3 — 加結構化輸出
# --------------------------------------------------------------------------
def build_v3(product: dict, banned_terms: list[str], tone_examples: list[dict] = None, **_) -> str:
    """v2 + 結構化輸出。

    注意：prompt 本身只多一句話，真正的工作交給 API 的 responseSchema 參數
    （見 generation.py）。這是重點 —— 不要用 prompt 硬凹 JSON 格式，
    要用 API 原生的 structured output，模型才會被約束在 schema 內。
    """
    return build_v2(product, banned_terms, tone_examples) + """

【輸出格式】
請直接輸出符合指定 schema 的 JSON，不要加上 markdown 程式碼區塊標記，不要加任何說明文字。"""


PROMPT_VERSIONS = {
    "v0": build_v0,
    "v1": build_v1,
    "v2": build_v2,
    "v3": build_v3,
}

# v3 是唯一啟用 API 層 structured output 的版本
STRUCTURED_VERSIONS = {"v3"}


# --------------------------------------------------------------------------
# 品牌語調 few-shot 範例（人工撰寫，已確認合規）
# --------------------------------------------------------------------------
TONE_EXAMPLES = {
    "晨光研選": [
        {
            "product": "B群緩釋錠",
            "title": "晨光研選 緩釋B群錠 90錠 每日一錠 全素可食",
            "bullet": "8種B群一次補齊，緩釋設計",
        },
        {
            "product": "鎂錠",
            "title": "晨光研選 甘胺酸鎂錠 120錠 睡前補充 無鎮靜成分",
            "bullet": "選用好吸收的甘胺酸螯合形式",
        },
    ],
    "青研 CHINGYEN": [
        {
            "product": "神經醯胺乳液",
            "title": "青研 神經醯胺修護乳液 50ml 敏弱肌適用 無香料",
            "bullet": "三種神經醯胺，質地清爽不黏膩",
        },
        {
            "product": "溫和卸妝油",
            "title": "青研 純淨卸妝油 150ml 好沖洗 不致粉刺測試",
            "bullet": "乳化快速，沖水後不留油感",
        },
    ],
    "Volta": [
        {
            "product": "行動電源",
            "title": "Volta 20000mAh 行動電源 45W雙向快充 可上飛機",
            "bullet": "45W輸出，筆電也充得動",
        },
        {
            "product": "USB-C 傳輸線",
            "title": "Volta 240W USB-C 編織線 2M 支援40Gbps傳輸",
            "bullet": "240W過電，8K螢幕直出",
        },
    ],
    "溯源焙所": [
        {
            "product": "日曬西達摩掛耳",
            "title": "溯源焙所 日曬西達摩掛耳咖啡 10入 中淺焙",
            "bullet": "日曬處理，草莓與黑糖尾韻",
        },
        {
            "product": "阿里山烏龍",
            "title": "溯源焙所 阿里山高山烏龍 三角茶包 20入 海拔1400m",
            "bullet": "清香型輕發酵，冷泡熱泡皆宜",
        },
    ],
}


def get_banned_terms_for(product: dict, banned_data: dict) -> list[str]:
    """依商品的 regulated_category 取出適用的禁詞。

    保健食品要擋『護眼』，3C 不用 —— 規則必須跟著商品屬性走，
    這是規則層在真實系統裡最容易被做錯的地方。
    """
    category = product["regulated_category"]
    terms: list[str] = []
    for group in banned_data.values():
        if not isinstance(group, dict) or "terms" not in group:
            continue
        if category in group.get("applies_to", []):
            terms.extend(group["terms"])
    return sorted(set(terms))

### 看一下四個版本的實際差異

In [ ]:
p = PRODUCTS[0]   # 葉黃素 — 保健食品，法規風險最高的類別
terms = get_banned_terms_for(p, BANNED_DATA)

for name, fn in PROMPT_VERSIONS.items():
    text = fn(p, banned_terms=terms, tone_examples=TONE_EXAMPLES[p['brand']])
    print(f"{name}: {len(text):>5,} 字元")

print("\n" + "=" * 72)
print("v0 全文（這就是大多數人第一次寫的 prompt）")
print("=" * 72)
print(PROMPT_VERSIONS['v0'](p))

In [ ]:
print("=" * 72)
print("v3 全文 — 注意法規段落與 few-shot 是怎麼寫的")
print("=" * 72)
print(PROMPT_VERSIONS['v3'](p, banned_terms=terms, tone_examples=TONE_EXAMPLES[p['brand']]))

### 生成

`generate(..., structured=True)` 時啟用 API 原生的 `responseSchema` ——
**這是 v3 與 v2 的唯一差別**（呼叫層的程式碼在 §0）。

**重點：不要用 prompt 硬凹 JSON 格式，要用 API 參數約束。**
prompt 裡寫「請輸出 JSON」模型會照做九成的時間；用 `responseSchema`
模型是被解碼器約束在 schema 內，那一成的失敗才會消失。

In [ ]:
ledger = UsageLedger()

# 12 商品 × 4 版 = 48 次呼叫，彼此獨立，所以並行跑。
jobs = [(p, v) for p in PRODUCTS for v in PROMPT_VERSIONS]


def _generate_one(job):
    product, version = job
    prompt = PROMPT_VERSIONS[version](
        product,
        banned_terms=get_banned_terms_for(product, BANNED_DATA),
        tone_examples=TONE_EXAMPLES.get(product['brand']),
    )
    res = generate(client, prompt, structured=(version in STRUCTURED_VERSIONS))
    ledger.record(f"gen:{version}", res.usage)
    return (product['sku'], version), res


t0 = time.time()
outputs = {}   # (sku, version) -> 文案原文
for key, res in run_parallel(_generate_one, jobs, label="生成"):
    outputs[key] = res.text
    if res.error:
        print(f"  ⚠ {key}: {res.error}")

print(f"\n共 {len(outputs)} 筆，耗時 {time.time() - t0:.1f} 秒")

### 肉眼比對：同一個商品，v0 vs v3

In [ ]:
sku = PRODUCTS[0]['sku']
print("=" * 72); print(f"{sku}  v0"); print("=" * 72)
print(outputs[(sku, 'v0')])
print("\n" + "=" * 72); print(f"{sku}  v3"); print("=" * 72)
print(outputs[(sku, 'v3')])

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **33:30 — ▶ 重跑 v0。**「看起來不錯對吧？文筆很好。」（停頓）「但它不能用。」
>
> **35:00 — ▶ 重跑 v3。**「同一個商品。」
>
> 先不要解釋為什麼 v3 比較好 —— 讓他們自己看。解釋留到對比表。

</div>

## §3 評測 — 這一節是重點

三層評測，由便宜到貴：

1. **規則層** — 免費、毫秒級、確定性，可以進 CI
2. **LLM judge** — 有成本、有偏誤，只評規則層評不了的東西
3. **人工抽樣** — 最貴，只看前兩層有分歧的

**順序很重要。** 多數團隊一上來就做 LLM judge，又貴又不穩。
先建規則層：它擋掉大部分明確錯誤，而且結果可重現。

### 第一層：規則

注意 `parse_freeform()` —— 為了評測 v0～v2 的自由文字輸出，
我們被迫寫一個靠正則猜測的 parser。**模型換個排版它就壞掉。**

這段脆弱的程式碼本身就是「為什麼要用 structured output」最好的論據。
v3 之後這整段可以刪掉。

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **36:30 — 秀 `parse_freeform()`，這段工程師最有共鳴。**
>
> 「為了評 v0，我得寫這種靠正則猜測的 parser。模型換個排版它就壞掉。
> **v3 之後這段可以整個刪掉。**」

</div>

In [ ]:
"""評測第一層：規則層。

不呼叫任何模型，毫秒級，零成本。重點：**先建這一層**。
多數團隊一上來就做 LLM judge，又貴又不穩；規則層能擋掉大部分明確錯誤，
而且結果是確定性的 —— 同樣的輸入永遠得到同樣的判定，可以進 CI。

這個模組刻意不 import 任何 GCP 套件，因此離線可測、可單元測試。
"""

from __future__ import annotations

import json
import re
from dataclasses import dataclass, field



# --------------------------------------------------------------------------
# 把模型輸出正規化成統一結構
# --------------------------------------------------------------------------
@dataclass
class ParsedCopy:
    """從模型輸出解析出的文案結構。"""

    title: str = ""
    bullets: list[str] = field(default_factory=list)
    seo_description: str = ""
    hashtags: list[str] = field(default_factory=list)
    raw: str = ""
    is_structured: bool = False  # True = 直接來自合法 JSON，不需猜測


_TITLE_PAT = re.compile(
    r"^\s*(?:\**)\s*(?:商品)?標題\s*(?:\**)\s*[:：]\s*(?:\**)\s*(.+?)\s*(?:\**)\s*$",
    re.MULTILINE,
)
_BULLET_PAT = re.compile(r"^\s*(?:[-*•・‧]|\d+[.、)])\s*(.+?)\s*$", re.MULTILINE)
_SEO_PAT = re.compile(
    r"^\s*(?:\**)\s*(?:SEO\s*)?(?:描述|說明)\s*(?:\**)\s*[:：]\s*(?:\**)\s*(.+?)\s*(?:\**)\s*$",
    re.MULTILINE | re.IGNORECASE,
)
_HASHTAG_PAT = re.compile(r"#([^\s#,，、]+)")
_FENCE_PAT = re.compile(r"^\s*```(?:json)?\s*|\s*```\s*$", re.MULTILINE)


def parse_structured(text: str) -> ParsedCopy | None:
    """嘗試把輸出當成 JSON 解析。成功才算 schema 合法。"""
    cleaned = _FENCE_PAT.sub("", text).strip()
    try:
        data = json.loads(cleaned)
    except (json.JSONDecodeError, ValueError):
        return None
    if not isinstance(data, dict):
        return None

    required = {"title", "bullets", "seo_description", "hashtags"}
    if not required.issubset(data.keys()):
        return None
    if not isinstance(data["bullets"], list) or not isinstance(data["hashtags"], list):
        return None
    if not isinstance(data["title"], str) or not isinstance(data["seo_description"], str):
        return None

    return ParsedCopy(
        title=data["title"].strip(),
        bullets=[str(b).strip() for b in data["bullets"]],
        seo_description=data["seo_description"].strip(),
        hashtags=[str(h).lstrip("#").strip() for h in data["hashtags"]],
        raw=text,
        is_structured=True,
    )


def parse_freeform(text: str) -> ParsedCopy:
    """從自由文字裡「猜」出結構。

    ⚠️ 這個函式是脆弱的，而那正是重點。
    為了評測 v0～v2 的輸出，我們被迫寫這種靠正則猜測的 parser：
    模型換個排版它就壞掉。v3 用 responseSchema 之後這段就可以整個刪掉。
    這段脆弱的程式碼本身，就是「為什麼要 structured output」最好的論據。
    """
    title_match = _TITLE_PAT.search(text)
    if title_match:
        title = title_match.group(1)
    else:
        # 退而求其次：第一行非空、非 markdown 標記的文字
        title = ""
        for line in text.splitlines():
            stripped = line.strip().lstrip("#").strip()
            if stripped and not stripped.startswith("```"):
                title = stripped
                break

    bullets = [b for b in _BULLET_PAT.findall(text) if len(b) > 2]

    seo_match = _SEO_PAT.search(text)
    seo = seo_match.group(1) if seo_match else ""

    hashtags = _HASHTAG_PAT.findall(text)

    return ParsedCopy(
        title=title.strip(),
        bullets=[b.strip() for b in bullets],
        seo_description=seo.strip(),
        hashtags=hashtags,
        raw=text,
        is_structured=False,
    )


def parse_output(text: str) -> ParsedCopy:
    """先試結構化，失敗才退回猜測。"""
    return parse_structured(text) or parse_freeform(text)


# --------------------------------------------------------------------------
# 規則檢查
# --------------------------------------------------------------------------
@dataclass
class RuleResult:
    sku: str
    version: str
    schema_valid: bool
    title_length_ok: bool
    title_length: int
    bullet_length_ok: bool
    seo_length_ok: bool
    spec_coverage: float          # 0.0 - 1.0
    spec_missing: list[str]
    banned_clean: bool
    banned_hits: list[str]

    @property
    def all_pass(self) -> bool:
        return (
            self.schema_valid
            and self.title_length_ok
            and self.bullet_length_ok
            and self.seo_length_ok
            and self.spec_coverage == 1.0
            and self.banned_clean
        )


def _searchable_text(parsed: ParsedCopy) -> str:
    """組出用來搜尋禁詞與規格關鍵字的文字範圍。

    這裡有一個容易做錯、而且錯了會出事的決定：**掃描範圍要多大？**

    - 結構化輸出（v3）：掃 parsed 後的欄位就夠，範圍精確，不會把 JSON key
      名稱或模型的客套話算進去。
    - 自由文字（v0～v2）：**必須掃 raw 全文**。因為脆弱的 parser 常常只抓到
      第一行，真正的違規詞往往躲在後面幾段。只掃 parsed 欄位會讓違規漏網 ——
      而漏檢比誤判危險得多（誤判只是多花人力複查，漏檢是收罰單）。

    這條差異本身就是「為什麼要用 structured output」的另一個論據：
    有 schema 才能精確掃描，沒有 schema 就只能全文粗掃、誤判率上升。
    """
    if parsed.is_structured:
        return " ".join(
            [parsed.title, *parsed.bullets, parsed.seo_description, *parsed.hashtags]
        ).strip()
    return parsed.raw


def check_banned(text: str, banned_terms: list[str]) -> list[str]:
    """回傳所有命中的禁詞。

    用長詞優先比對，避免「治療」和「治療過敏」重複計數同一段文字。
    """
    hits: list[str] = []
    consumed: list[tuple[int, int]] = []
    for term in sorted(banned_terms, key=len, reverse=True):
        for m in re.finditer(re.escape(term), text):
            span = (m.start(), m.end())
            if any(span[0] >= s and span[1] <= e for s, e in consumed):
                continue  # 已被更長的禁詞涵蓋
            consumed.append(span)
            hits.append(term)
            break  # 同一個詞只記一次
    return hits


def evaluate_rules(
    output_text: str,
    product: dict,
    version: str,
    banned_terms: list[str],
) -> RuleResult:
    """對單一筆模型輸出跑完整規則層。"""
    parsed = parse_output(output_text)
    text = _searchable_text(parsed)

    missing = [kw for kw in product["must_include_keywords"] if kw not in text]
    coverage = 1.0 - len(missing) / max(len(product["must_include_keywords"]), 1)

    bullets_ok = bool(parsed.bullets) and all(
        len(b) <= BULLET_MAX for b in parsed.bullets
    )
    seo_ok = 0 < len(parsed.seo_description) <= SEO_DESC_MAX

    hits = check_banned(text, banned_terms)

    return RuleResult(
        sku=product["sku"],
        version=version,
        schema_valid=parsed.is_structured,
        title_length_ok=0 < len(parsed.title) <= SHOPEE_TITLE_MAX,
        title_length=len(parsed.title),
        bullet_length_ok=bullets_ok,
        seo_length_ok=seo_ok,
        spec_coverage=round(coverage, 3),
        spec_missing=missing,
        banned_clean=not hits,
        banned_hits=hits,
    )


def suggest_fix(hits: list[str], banned_data: dict) -> dict[str, str]:
    """對命中的禁詞給出合規替代寫法。

    重點：規則層不只是「擋」，還能「給修改方向」。
    這讓它從一個惹人厭的 linter 變成一個有用的工具。
    """
    mapping = banned_data.get("safe_alternatives", {}).get("mapping", {})
    return {h: mapping[h] for h in hits if h in mapping}

In [ ]:
"""把評測結果整理成「版本 × 指標」對比表。

這張表是整份分析的結論。前面所有步驟都是為了讓它可信。

設計原則：
- 每一欄對應一個 prompt 改動所修好的問題，欄位順序＝改動順序，階梯感才會出來。
- 顯示通過率（%）而非平均分數，因為「12 個商品裡有幾個能直接上架」比
  「平均 3.8 分」更接近商業決策者要的答案。
"""

from __future__ import annotations


METRIC_LABELS = {
    "schema_valid": "可機器讀取",
    "title_length_ok": "標題長度合規",
    "bullet_length_ok": "賣點長度合規",
    "seo_length_ok": "SEO描述合規",
    "spec_full": "規格完整覆蓋",
    "banned_clean": "法規禁詞 0 命中",
}

# 「這個指標是被哪一版修好的」**必須從資料算出來，不可以硬寫。**
#
# 這裡原本是一份手寫的對照表（title→v1、banned→v2⋯⋯），寫的時候看起來
# 理所當然。後來實跑 50 筆與 200 筆，六條裡有三條被資料推翻：
#
#     賣點長度   標成 v1，實際是 v3 修好的（+62pp）
#     SEO描述    標成 v1，實際是 v3 修好的（+88pp）—— v1 完全沒動
#     法規禁詞   標成 v2，實際 v1 就做掉大部分（+20pp），v2 只是收尾
#
# 一個硬寫的因果宣稱，就擺在真實數字旁邊，而且是錯的 —— 在一份講
# 「怎麼知道它有沒有變好」的材料裡，這是最不該犯的錯。
#
# 所以改成：看哪一版帶來最大的單步改善，就歸因給那一版。
# 資料換了，標註自動跟著換。


def attribution(table: dict[str, dict[str, float]]) -> dict[str, str]:
    """依實測資料判斷每個指標是被哪一版修好的。

    作法很簡單：比較相鄰版本的差距，取單步改善最大的那一版。
    改善幅度太小（<5pp）就標成「—」，不硬掰因果。
    """
    versions = sorted(table)
    out: dict[str, str] = {}
    for metric in METRIC_LABELS:
        gains = [
            (table[versions[i]][metric] - table[versions[i - 1]][metric], versions[i])
            for i in range(1, len(versions))
        ]
        if not gains:
            out[metric] = "—"
            continue
        best_gain, best_version = max(gains)
        out[metric] = best_version if best_gain >= 5 else "—"
    return out


def _metric_values(r: RuleResult) -> dict[str, bool]:
    return {
        "schema_valid": r.schema_valid,
        "title_length_ok": r.title_length_ok,
        "bullet_length_ok": r.bullet_length_ok,
        "seo_length_ok": r.seo_length_ok,
        "spec_full": r.spec_coverage == 1.0,
        "banned_clean": r.banned_clean,
    }


def build_table(results: list[RuleResult]) -> dict[str, dict[str, float]]:
    """results 為所有 (商品 × 版本) 的規則結果，回傳 {版本: {指標: 通過率}}。"""
    by_version: dict[str, list[RuleResult]] = {}
    for r in results:
        by_version.setdefault(r.version, []).append(r)

    table: dict[str, dict[str, float]] = {}
    for version in sorted(by_version):
        rows = by_version[version]
        agg = {m: 0 for m in METRIC_LABELS}
        for r in rows:
            for m, ok in _metric_values(r).items():
                agg[m] += int(ok)
        table[version] = {m: round(100 * c / len(rows), 1) for m, c in agg.items()}
        table[version]["_全數通過"] = round(
            100 * sum(r.all_pass for r in rows) / len(rows), 1
        )
    return table


def to_dataframe(results: list[RuleResult]):
    """回傳 pandas DataFrame，notebook 裡直接顯示用。"""
    import pandas as pd

    table = build_table(results)
    attr = attribution(table)
    ordered = [
        "title_length_ok",
        "bullet_length_ok",
        "seo_length_ok",
        "spec_full",
        "banned_clean",
        "schema_valid",
    ]
    df = pd.DataFrame(
        {
            METRIC_LABELS[m] + f"\n({attr[m]}修好)": [
                table[v][m] for v in sorted(table)
            ]
            for m in ordered
        },
        index=sorted(table),
    )
    df["★ 全數通過"] = [table[v]["_全數通過"] for v in sorted(table)]
    df.index.name = "Prompt 版本"
    return df


def render_text_table(results: list[RuleResult]) -> str:
    """純文字版對比表 —— 不依賴 pandas，離線也能看。"""
    table = build_table(results)
    attr = attribution(table)
    ordered = [
        "title_length_ok",
        "bullet_length_ok",
        "seo_length_ok",
        "spec_full",
        "banned_clean",
        "schema_valid",
    ]
    versions = sorted(table)

    def pad_to(text: str, width: int) -> str:
        """CJK 字元在等寬終端佔兩格，用顯示寬度而非字元數對齊。"""
        display = sum(2 if ord(c) > 127 else 1 for c in text)
        return text + " " * max(width - display, 1)

    width = 22 + 12 + 8 * len(versions)
    lines = [
        pad_to("指標", 22) + pad_to("實測歸因", 12) + "".join(f"{v:>8}" for v in versions),
        "─" * width,
    ]

    for m in ordered:
        lines.append(
            pad_to(METRIC_LABELS[m], 22)
            + pad_to(attr[m] + ("※" if m in PARSER_DEPENDENT else ""), 12)
            + "".join(f"{table[v][m]:>7.0f}%" for v in versions)
        )

    lines.append("─" * width)
    lines.append(
        pad_to("★ 全數通過", 22)
        + pad_to("", 12)
        + "".join(f"{table[v]['_全數通過']:>7.0f}%" for v in versions)
    )
    return "\n".join(lines)


def noise_floor(n_items: int, n_checks_each: int = 1) -> float:
    """這張表上「多少百分點以內的差距應該當成雜訊」。

    用最粗的估計：一次檢查翻轉所造成的百分點變化。12 個商品 × 7 條 rubric
    = 84 次檢查，翻一條就是 1.2 個百分點；所以 2 個百分點以內的差距
    不該當成改善。

    為什麼要放這個函式：demo 的樣本數很小（12 個商品），
    很容易看到 92.9% vs 95.2% 就宣稱「v2 比較好」，但那其實只差一次檢查。
    **把雜訊當成訊號，是評測工作最容易犯、也最傷的錯。**
    """
    total = max(n_items * n_checks_each, 1)
    return round(100.0 / total, 1)


def paired_compare(reports, version_a: str, version_b: str, n_boot: int = 5000, seed: int = 42):
    """比較兩個版本的 rubric 通過率，用**配對**比較 + bootstrap 信賴區間。

    為什麼不能只看 noise_floor()：

    那個函式算的是「翻轉一次檢查等於幾個百分點」，那是**解析度**，
    不是**統計顯著性**。兩者常被混為一談。解析度 0.29pp 不代表
    1.4pp 的差距就一定是真的 —— 還要看樣本之間的變異有多大。

    這裡用的是正確的工具：因為 v0～v3 是在**同一批商品**上評的，
    屬於配對設計。對每個商品算 (B 的通過率 − A 的通過率)，再對這組
    差值做 bootstrap，得到平均差的 95% 信賴區間。

    **若區間跨過 0，就不能宣稱誰比較好。**
    """
    import random as _random

    by_sku_a = {r.sku: r.pass_rate for r in reports if r.version == version_a and not r.error}
    by_sku_b = {r.sku: r.pass_rate for r in reports if r.version == version_b and not r.error}
    common = sorted(set(by_sku_a) & set(by_sku_b))
    if len(common) < 2:
        return None

    diffs = [by_sku_b[s] - by_sku_a[s] for s in common]
    mean = sum(diffs) / len(diffs)

    rng = _random.Random(seed)
    boots = []
    for _ in range(n_boot):
        sample = [diffs[rng.randrange(len(diffs))] for _ in range(len(diffs))]
        boots.append(sum(sample) / len(sample))
    boots.sort()
    lo, hi = boots[int(0.025 * n_boot)], boots[int(0.975 * n_boot)]

    return {
        "比較": f"{version_b} − {version_a}",
        "配對商品數": len(common),
        "平均差(pp)": round(mean, 2),
        "95%CI": (round(lo, 2), round(hi, 2)),
        "顯著": not (lo <= 0 <= hi),
    }


def render_paired(reports, versions: list[str]) -> str:
    """把相鄰版本的配對比較整理成一段可直接唸的結論。"""
    lines = ["版本之間的差距是真的還是雜訊？（配對比較 + bootstrap 95% CI）", ""]
    for a, b in zip(versions, versions[1:], strict=False):
        r = paired_compare(reports, a, b)
        if not r:
            continue
        lo, hi = r["95%CI"]
        verdict = "✓ 有差異" if r["顯著"] else "✗ 落在雜訊內，不能宣稱"
        lines.append(
            f"  {r['比較']:<10}{r['平均差(pp)']:>+7.2f} pp   "
            f"95%CI [{lo:>+6.2f}, {hi:>+6.2f}]   {verdict}"
        )
    lines += [
        "",
        "  註：區間跨過 0 就代表在這個樣本量下分不出差別 —— 不是「一樣好」，",
        "      是「還沒有證據說誰比較好」。要下更細的結論就要加大評測集。",
    ]
    return "\n".join(lines)


def significance_note(n_items: int, n_checks_each: int = 1) -> str:
    """樣本規模說明。

    ⚠️ 這裡只講**解析度**，不講顯著性 —— 兩者常被混為一談，而混淆的代價
    是把雜訊當成訊號。

    解析度是「翻轉一次檢查等於幾個百分點」，那是這張表能表示的最小刻度。
    顯著性是「這個差距有沒有超出樣本變異」，那要看配對比較的信賴區間
    （見 render_paired）。實測 50 筆時，解析度是 0.29pp，但配對比較的
    95% 區間寬達 ±5pp —— **差了一個數量級**。只看解析度會嚴重高估精度。
    """
    floor = noise_floor(n_items, n_checks_each)
    return (
        f"樣本規模：{n_items} 筆 × 每筆 {n_checks_each} 項 = {n_items * n_checks_each} 次檢查\n"
        f"  表格解析度：單次檢查翻轉 ≈ {floor} 個百分點\n"
        f"\n"
        f"  ⚠ **解析度不等於顯著性。** 能表示到 {floor}pp，不代表 {floor}pp 的差距是真的。\n"
        f"  要判斷差距是否成立，看上面的配對比較信賴區間 —— 那才是正確的工具。"
    )


# 這兩個指標**只在結構化輸出上可靠**。
#
# 自由文字要靠 parse_freeform() 猜出哪幾句是賣點、哪一段是 SEO 描述，
# 而那個 parser 是脆弱的。實測 12 筆時它剛好都猜中，換到 200 筆就大量失效 ——
# 於是 v0～v2 的「賣點長度合規」掉到個位數。
#
# 但那個數字測到的是 **parser 的成功率**，不是模型的合規率。
# 把它當成 0% 呈現會誤導成「模型寫超長」，實際上是我們根本量不到。
# 所以非結構化的版本一律標成「不可測」（NaN），不填 0。
#
# 這件事本身就是 structured output 最有力的論據：
# 沒有 schema，連「有沒有做對」都無法誠實測量。
PARSER_DEPENDENT = ("bullet_length_ok", "seo_length_ok")


def combined_table(rule_results: list[RuleResult], rubric_summary: dict):
    """規則層 + rubric 層的合併對比表。

    每一版只負責修好一件事，這張表要能讓那件事一眼看出來。
    依賴 parser 的指標在非結構化版本上以「—」呈現，不假裝量得到。
    """
    import pandas as pd

    table = build_table(rule_results)
    versions = sorted(table)
    attr = attribution(table)  # 歸因一律由資料算出，欄名不硬寫版本

    # 某一版是否為結構化輸出：用該版 schema_valid 是否過半判定，
    # 不寫死版本名，這樣改了版本命名也不會壞。
    structured = {v: table[v]["schema_valid"] >= 50 for v in versions}

    def parser_dependent(metric: str) -> list:
        return [table[v][metric] if structured[v] else float("nan") for v in versions]

    def rubric_col(key: str) -> list:
        return [rubric_summary.get(v, {}).get(key, float("nan")) for v in versions]

    def rubric_attr(key: str) -> str:
        """rubric 欄位的歸因同樣用實測算，不預設是哪一版。"""
        vals = [rubric_summary.get(v, {}).get(key) for v in versions]
        gains = [
            (vals[i] - vals[i - 1], versions[i])
            for i in range(1, len(versions))
            if vals[i] is not None and vals[i - 1] is not None
        ]
        if not gains:
            return "—"
        best, ver = max(gains)
        return ver if best >= 5 else "—"

    rows = {
        f"標題長度合規 ({attr['title_length_ok']})": [
            table[v]["title_length_ok"] for v in versions
        ],
        f"規格完整覆蓋 ({attr['spec_full']})": [table[v]["spec_full"] for v in versions],
        f"法規禁詞 0 命中 ({attr['banned_clean']})": [
            table[v]["banned_clean"] for v in versions
        ],
        "賣點長度 ※": parser_dependent("bullet_length_ok"),
        "SEO描述長度 ※": parser_dependent("seo_length_ok"),
        f"rubric 通過率 ({rubric_attr('rubric通過率%')})": rubric_col("rubric通過率%"),
        f"可直接上架 ({rubric_attr('可直接上架%')})": rubric_col("可直接上架%"),
        f"可機器讀取 ({attr['schema_valid']})": [table[v]["schema_valid"] for v in versions],
    }
    df = pd.DataFrame(rows, index=versions)
    df.index.name = "Prompt 版本"
    return df


PARSER_CAVEAT = """※ 這兩欄只在結構化輸出上可靠。

自由文字要靠正則猜出哪幾句是賣點、哪一段是 SEO 描述，而那個 parser 很脆弱。
非結構化的版本標成「—」，代表**量不到**，不是「不合格」。

這正是 structured output 的核心價值：沒有 schema，
連「有沒有做對」都無法誠實測量 —— 而測不到的東西就管不了。"""


def violation_detail(results: list[RuleResult], version: str, limit: int = 5) -> str:
    """列出某一版的違規細節 —— demo 中用來證明『這不是假資料』。"""
    rows = [r for r in results if r.version == version and not r.banned_clean]
    if not rows:
        return f"{version}：無禁詞違規 ✓"
    out = [f"{version} 的禁詞命中（前 {limit} 筆）："]
    for r in rows[:limit]:
        out.append(f"  {r.sku}  →  {'、'.join(r.banned_hits)}")
    if len(rows) > limit:
        out.append(f"  ...另有 {len(rows) - limit} 筆")
    return "\n".join(out)

### ★ 規則層對比表

In [ ]:
rule_results = []
for product in PRODUCTS:
    terms = get_banned_terms_for(product, BANNED_DATA)
    for version in PROMPT_VERSIONS:
        rule_results.append(
            evaluate_rules(outputs[(product['sku'], version)], product, version, terms)
        )

print(render_text_table(rule_results))

In [ ]:
# 同一張表的 DataFrame 版本（投影用，數字大一點好讀）
to_dataframe(rule_results).style.format("{:.0f}%").background_gradient(cmap="RdYlGn", vmin=0, vmax=100)

### 違規細節 — 證明這不是假資料

規則層不只能擋，還能給修改方向。這讓它從一個惹人厭的 linter
變成文案人員真的願意用的工具。

In [ ]:
for version in ['v0', 'v1', 'v2', 'v3']:
    print(violation_detail(rule_results, version))
    print()

# 對命中的禁詞給出合規替代寫法
worst = max((r for r in rule_results if r.version == 'v0'),
            key=lambda r: len(r.banned_hits), default=None)
if worst and worst.banned_hits:
    print("=" * 60)
    print(f"{worst.sku} 的修改建議：")
    for bad, good in suggest_fix(worst.banned_hits, BANNED_DATA).items():
        print(f"  ✗ {bad}  →  ✓ {good}")

### 第二層：LLM-as-judge —— 用二元 rubric，不用 1–5 分

只評規則層評不了的：語調、賣點覆蓋。

#### 為什麼不用 1–5 分？

早期的 LLM-as-judge 幾乎都用 Likert 量表。實務上它有幾個難以修復的問題：

| 問題 | 後果 |
|---|---|
| 分數擠在 3–4 分 | 鑑別度低，看不出差異 |
| 同一份文案今天 4 分明天 3 分 | 不可重現，無法判斷是模型變差還是評審漂移 |
| 換評審模型整組分數平移 | 歷史數據作廢 |
| 寫得長、寫得華麗容易拿高分 | verbosity bias |
| **「3.8 分」無法行動** | 你不知道要改什麼 |

二元 rubric 把一個模糊的大問題，拆成一組**具體、可檢查的 yes/no 小問題**：

```
✗  「這份文案的品質有幾分？」          → 3.8 / 5，然後呢？
✓  「文案是否寫出游離型葉黃素 30mg？」  → 否 → 知道要補什麼
```

這也是 Vertex AI Gen AI Evaluation Service 把 adaptive rubrics
形容成**「像單元測試」**的原因。

#### 公平性：rubric 只從商品資料生成，不看被評的文案

**每個商品產生一次，v0～v3 共用同一組。**
如果讓評審看著文案即興出題，每個版本會被問不同的問題 ——
那等於每個考生考不同的考卷，對比表就沒有意義了。

In [ ]:
"""評測第二層：LLM-as-judge，採用**二元自適應 rubric**（pass / fail 檢查清單）。

只評「規則層評不了」的東西 —— 語調、賣點覆蓋這類需要語意理解的維度。
順序很重要：**先跑規則層，被擋下來的不必送 judge**，省下的是真金白銀。

---

## 為什麼不用 1–5 分？

這是這個模組最重要的設計決定。

早期的 LLM-as-judge 幾乎都用 Likert 量表（1–5 分）。實務上它有幾個難以修復的問題：

- **分數擠在中間。** 模型很少給 1 或 5，結果多數樣本都是 3–4 分，鑑別度低。
- **不可重現。** 同一份文案今天 4 分明天 3 分，你無法判斷是模型變差還是評審漂移。
- **跨模型不一致。** 換一個評審模型，整組分數就平移，歷史數據全部作廢。
- **容易被長度騙。** 寫得長、寫得華麗容易拿高分，這正是 verbosity bias。
- **「3.8 分」無法行動。** 你不知道要改什麼。

二元 rubric 把一個模糊的大問題拆成一組**具體、可檢查的 yes/no 小問題**：

    ✗  「這份文案的品質有幾分？」        → 3.8 / 5，然後呢？
    ✓  「文案是否提到 30mg 這個含量？」  → 否 → 知道要補什麼

好處是直接的：**判準明確、可重現、可累積、能直接對應到修改動作**，
而且 pass/fail 很難用堆字數來灌水。這也是為什麼 Vertex AI 的
Gen AI Evaluation Service 把 adaptive rubrics 形容成「像單元測試」。

> 註：Vertex AI 有託管版本的 rubric 評測。本模組選擇手寫，原因有二：
> 一是要看得到 rubric 長什麼樣子，那才是可帶走的概念；
> 二是託管服務的 SDK 介面仍在變動（本機安裝的 `vertexai.evaluation`
> 只有舊的 Likert 型 PointwiseMetric，新的 adaptive rubric 走另一個介面）。
> 概念完全相同，要換成託管版本時，換的是呼叫方式，不是方法論。

---

## 公平性：rubric 只從商品資料生成，不看被評的文案

這點很容易做錯，做錯了整張對比表就沒有意義。

rubric **對每個商品產生一次，v0～v3 共用同一組**。
如果讓評審看著文案即興出題，每個版本會被問不同的問題，
分數就不可比 —— 那等於每個考生考不同的考卷。
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field


# --------------------------------------------------------------------------
# 已知偏誤與本專案的緩解方式
# --------------------------------------------------------------------------
JUDGE_BIASES = {
    "self-preference": {
        "說明": "模型傾向給自己產出的內容較高分。",
        "緩解": f"評審模型（{JUDGE_MODEL}）與生成模型（{GEN_MODEL}）刻意選不同的。",
    },
    "verbosity": {
        "說明": "較長、較華麗的回答容易被評為較好，即使內容沒有更好。",
        "緩解": "改用 pass/fail 判準，「有沒有提到 30mg」無法靠寫得長來灌水。",
    },
    "position": {
        "說明": "成對比較時，先出現的選項容易獲勝。",
        "緩解": "採 pointwise（單篇對照 rubric），不做 pairwise，從源頭避開。",
    },
    "score-clustering": {
        "說明": "Likert 分數容易全部擠在 3–4 分，鑑別度低且不可重現。",
        "緩解": "不用分數。二元判準沒有中間地帶可以躲。",
    },
    "criteria-drift": {
        "說明": "評審每次自己想判準，等於每個版本考不同的考卷。",
        "緩解": "rubric 只從商品資料生成一次，v0～v3 共用同一組。",
    },
}

# --------------------------------------------------------------------------
# Schemas
# --------------------------------------------------------------------------
RUBRIC_GEN_SCHEMA = {
    "type": "object",
    "properties": {
        "rubrics": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "string", "description": "R1, R2, ..."},
                    "criterion": {
                        "type": "string",
                        "description": "一句可用是／否回答的具體判準",
                    },
                    "dimension": {
                        "type": "string",
                        "enum": ["賣點覆蓋", "品牌語調", "消費者可讀性"],
                    },
                    "critical": {
                        "type": "boolean",
                        "description": "未通過就不該上架者為 true",
                    },
                },
                "required": ["id", "criterion", "dimension", "critical"],
            },
        }
    },
    "required": ["rubrics"],
}

RUBRIC_CHECK_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "string"},
                    "passed": {"type": "boolean"},
                    "evidence": {
                        "type": "string",
                        "description": "通過就引用文案中的原文；未通過就說明缺什麼",
                    },
                },
                "required": ["id", "passed", "evidence"],
            },
        }
    },
    "required": ["results"],
}

# --------------------------------------------------------------------------
# Prompts
# --------------------------------------------------------------------------
RUBRIC_GEN_PROMPT = """你是資深電商文案主管，要為一個商品訂出「文案驗收清單」。

這份清單會用來檢查 AI 產出的文案能不能上架。請產生 7 條判準。

【⚠️ 最重要：字數預算】
被檢查的文案有嚴格的通路字數限制，總共只有約 {total_budget} 個字：

  - 標題：最多 {title_max} 字
  - 賣點：{bullet_count} 條，每條最多 {bullet_max} 字
  - SEO 描述：最多 {seo_max} 字

**你的每一條判準都必須是在這個字數預算內可以達成的。**

這是最容易出錯的地方。不要寫出需要長篇說明才能滿足的判準：

  ✗ 「文案是否說明為何隨餐食用有助吸收？」    （解釋機制，30 字的賣點塞不下）
  ✗ 「文案是否說明黃金比例能提供更全面的保護？」（同上）
  ✓ 「文案是否寫出葉黃素 30mg 的含量？」       （幾個字就能達成）
  ✓ 「文案是否點出適合長時間看螢幕的族群？」    （一句話能達成）

**如果一條判準只有把文案寫長才可能通過，它就是壞判準。**
壞判準會讓冗長雜亂的文案得高分、讓精簡合規的文案被扣分，
整張評測表的結論就會顛倒。

【判準的寫法】
- 每一條都必須能用「是」或「否」回答，不可以是程度問題。
  ✗ 「文案的語氣是否夠專業？」（程度問題，無法一致判定）
  ✓ 「文案是否避免使用驚嘆號？」（可以直接看出來）
- 判準要**具體指向這個商品**，不要寫成通用的空話。

【⚠️ 不可與規則層衝突】
以下規格關鍵字是**硬性要求**，文案一定會出現它們：

  {must_include}

**絕對不可以寫出「避免使用某某專有名詞」這類會懲罰上述關鍵字的判準。**

這是真實踩過的坑：規則層要求文案必須寫出「游離型葉黃素 30mg」，
rubric 卻出了一條「是否避免使用『游離型』這種需要解釋的專有名詞」——
兩層互相打架，文案怎麼寫都會被扣分，整張評測表就失去意義。

你可以要求「把專有名詞轉成好懂的說法」，但不可以要求「不要出現它」。

【其他不要納入的範圍】
以下已由程式規則自動檢查，**不要**寫成判準：字數是否超標、JSON 格式是否合法、
法規禁詞、必含關鍵字是否存在。你只負責語意層面。

【維度分配 — 必須照這個比例，不可偏重賣點覆蓋】
- 賣點覆蓋：3 條（把規格轉成消費者看得懂的好處，但要在字數內可達成）
- 品牌語調：2 條（是否符合下面的品牌語調設定）
- 消費者可讀性：2 條（是否精簡、無冗詞、無誇大語氣、可直接上架）

【critical 的判定】
未通過就不該上架的設為 true，請設 2 條為 critical。

【商品資料】
商品名稱：{name}
品牌：{brand}
品牌語調設定：{brand_tone}
目標受眾：{audience}
規格：
{specs}

請依 schema 輸出判準清單。"""

RUBRIC_CHECK_PROMPT = """你是電商文案審核員。請對照驗收清單逐條檢查下面這份文案。

【文案的字數預算】
這份文案受通路限制，總共只有約 {total_budget} 個字可用
（標題 {title_max} 字、{bullet_count} 條賣點各 {bullet_max} 字、SEO 描述 {seo_max} 字）。

**請在這個前提下判斷。** 文案簡潔不是缺點，那是通路要求。
不要因為文案沒有展開說明、沒有解釋原理就判定不通過 ——
只要在字數內把該講的講到了，就算通過。

【檢查原則】
- 每一條**只回答通過或不通過**，不要給分數。
- 判斷依據只能是文案中實際出現的內容，不要腦補。
- 通過時，`evidence` 請引用文案中的原文片段。
- 不通過時，`evidence` 請說明缺少什麼。
- **絕對不要因為文案較長、較華麗就傾向通過。**
  冗長、堆砌形容詞、重複同一個賣點，在「消費者可讀性」維度應判為不通過。
- 文案可能是自由文字，也可能是 JSON，兩者一視同仁看內容。

【驗收清單】
{rubric_list}

【待檢查的文案】
{copy_text}

請依 schema 逐條回覆，results 的長度必須與清單條數相同。"""


# --------------------------------------------------------------------------
# 資料結構
# --------------------------------------------------------------------------
@dataclass
class Rubric:
    id: str
    criterion: str
    dimension: str
    critical: bool


@dataclass
class RubricSet:
    sku: str
    rubrics: list[Rubric] = field(default_factory=list)
    error: str | None = None

    def as_prompt_list(self) -> str:
        return "\n".join(
            f"{r.id}. [{r.dimension}]{'（critical）' if r.critical else ''} {r.criterion}"
            for r in self.rubrics
        )


@dataclass
class RubricReport:
    sku: str
    version: str
    passed_ids: list[str] = field(default_factory=list)
    failed: list[tuple[str, str, bool]] = field(default_factory=list)  # (id, 原因, critical)
    total: int = 0
    error: str | None = None

    @property
    def pass_rate(self) -> float:
        return round(100 * len(self.passed_ids) / self.total, 1) if self.total else 0.0

    @property
    def critical_failures(self) -> int:
        return sum(1 for _, _, crit in self.failed if crit)

    @property
    def would_publish(self) -> bool:
        """critical 全過，且整體通過率達 80% 才算能上架。"""
        return self.total > 0 and self.critical_failures == 0 and self.pass_rate >= 80.0


# --------------------------------------------------------------------------
# 呼叫
# --------------------------------------------------------------------------
def _call_json(client, prompt: str, schema: dict, tag: str, ledger=None, max_retries: int = 4):
    """走 structured output 呼叫評審模型並解析 JSON。

    評審本身也用 responseSchema —— 解析失敗會讓整張評測表出現空洞，
    比生成失敗更難察覺。

    **必須重試。** 這一層在高並行下會撞到配額限制（429）。原本沒有重試，
    失敗的那一筆會被上層 except 接住、變成一個帶 error 的空結果 ——
    評測表上看不出來，只會發現受評筆數少了幾筆。那比直接報錯更危險。

    429 的退避拉得比一般錯誤久：配額是時間窗限制，馬上重試只會再撞一次。
    """
    import random
    import time


    last_err: Exception | None = None
    for attempt in range(max_retries):
        started = time.time()
        try:
            resp = client.models.generate_content(
                model=JUDGE_MODEL,
                contents=prompt,
                config=gen_config(
                    temperature=JUDGE_TEMPERATURE,
                    response_mime_type="application/json",
                    response_schema=schema,
                ),
            )
            meta = resp.usage_metadata
            if ledger is not None:
                ledger.record(
                    tag,
                    Usage(
                        model=JUDGE_MODEL,
                        input_tokens=getattr(meta, "prompt_token_count", 0) or 0,
                        output_tokens=getattr(meta, "candidates_token_count", 0) or 0,
                        latency_s=round(time.time() - started, 2),
                    ),
                )
            return json.loads(resp.text)
        except Exception as e:  # noqa: BLE001
            last_err = e
            if attempt == max_retries - 1:
                break
            text = str(e)
            throttled = "429" in text or "RESOURCE_EXHAUSTED" in text or "quota" in text.lower()
            base = 8.0 if throttled else 1.5
            # 加抖動，避免所有執行緒在同一刻一起重試又一起撞牆
            time.sleep(base * (2**attempt) + random.uniform(0, 1.5))

    raise RuntimeError(f"評審呼叫失敗（重試 {max_retries} 次）：{last_err}")


def _budget() -> dict:
    """文案的字數預算。

    rubric 生成與檢查都必須知道這個數字，否則會產生「解釋原理」這種
    在 30 字賣點裡不可能達成的判準 —— 那會讓冗長的文案得高分、
    精簡合規的文案被扣分，整張評測表的結論顛倒過來。
    """
    total = (
        SHOPEE_TITLE_MAX
        + BULLET_COUNT * BULLET_MAX
        + SEO_DESC_MAX
    )
    return {
        "title_max": SHOPEE_TITLE_MAX,
        "bullet_count": BULLET_COUNT,
        "bullet_max": BULLET_MAX,
        "seo_max": SEO_DESC_MAX,
        "total_budget": total,
    }


def generate_rubrics(client, product: dict, ledger=None) -> RubricSet:
    """為單一商品產生驗收清單。

    ⚠️ 每個商品只做一次，v0～v3 共用。不要對每個版本重新生成 —— 那會讓
    每個版本被問不同的問題，對比表就失去意義。
    """
    specs = "\n".join(f"  - {k}：{v}" for k, v in product["specs"].items())
    prompt = RUBRIC_GEN_PROMPT.format(
        name=product["name"],
        brand=product["brand"],
        brand_tone=product["brand_tone"],
        audience=product["target_audience"],
        specs=specs,
        # 讓 rubric 生成知道哪些字是規則層的硬性要求，否則會出現
        # 「避免使用這個專有名詞」這種與規則層打架的判準。
        must_include="、".join(product["must_include_keywords"]),
        **_budget(),
    )
    try:
        data = _call_json(client, prompt, RUBRIC_GEN_SCHEMA, "judge:rubric_gen", ledger)
        return RubricSet(
            sku=product["sku"],
            rubrics=[
                Rubric(
                    id=str(r["id"]),
                    criterion=str(r["criterion"]),
                    dimension=str(r["dimension"]),
                    critical=bool(r["critical"]),
                )
                for r in data["rubrics"]
            ],
        )
    except Exception as e:  # noqa: BLE001
        return RubricSet(sku=product["sku"], error=str(e))


def check_against_rubrics(
    client,
    copy_text: str,
    rubric_set: RubricSet,
    version: str,
    ledger=None,
) -> RubricReport:
    """逐條檢查文案。"""
    if rubric_set.error or not rubric_set.rubrics:
        return RubricReport(
            sku=rubric_set.sku,
            version=version,
            error=rubric_set.error or "沒有可用的 rubric",
        )

    by_id = {r.id: r for r in rubric_set.rubrics}
    prompt = RUBRIC_CHECK_PROMPT.format(
        rubric_list=rubric_set.as_prompt_list(),
        copy_text=copy_text[:4000],  # 防止異常長輸出把成本吃掉
        **_budget(),
    )
    try:
        data = _call_json(client, prompt, RUBRIC_CHECK_SCHEMA, f"judge:{version}", ledger)
        passed, failed = [], []
        for row in data["results"]:
            rid = str(row["id"])
            rub = by_id.get(rid)
            if rub is None:
                continue  # 評審捏造了不存在的 id，忽略
            if bool(row["passed"]):
                passed.append(rid)
            else:
                failed.append((rid, str(row.get("evidence", "")), rub.critical))
        return RubricReport(
            sku=rubric_set.sku,
            version=version,
            passed_ids=passed,
            failed=failed,
            total=len(by_id),
        )
    except Exception as e:  # noqa: BLE001
        return RubricReport(sku=rubric_set.sku, version=version, error=str(e))


def should_judge(rule_result) -> bool:
    """**生產環境**用的過濾：規則層擋下的就不送 judge。

    語意檢查對一份已經違反法規的文案沒有意義，而 judge 用的是比生成更貴的
    模型。上線後 prompt 越爛、擋掉越多、省越多，這是真的有價值的。

    ⚠️ **但評測時不要用這個函式** —— 見下方 should_judge_for_eval()。
    """
    return rule_result.banned_clean and rule_result.spec_coverage > 0.5


def should_judge_for_eval(rule_result) -> bool:
    """**評測時一律全部送審。**

    這是實測踩到的坑，值得記下來：

    原本評測也沿用 should_judge() 的過濾，結果四個版本的分母不一樣 ——
    v0 只有 8 筆受評、v1 有 11 筆、v2/v3 各 12 筆。因為被擋下的正是最爛的
    那幾筆，v0 的平均分被「倖存者偏差」灌水，反而看起來比 v1 高
    （78.5% vs 74.0%）。

    然後去算過濾到底省了多少：**7–10%，金額是 US$0.08。**

    用 0.08 美金換掉「四個版本能不能互相比較」—— 而那正是整個評測存在的
    理由。這是很差的交換。

    所以生產與評測要分開看：

        生產環境   擋下的不送審    prompt 越爛省越多，值得
        評測環境   全部送審        分母一致比省 7% 重要得多

    這個區分本身就是可以講的東西：**同一個機制，在不同目的下的取捨相反。**
    """
    return True


def summarize_rubrics(
    reports: list[RubricReport], total_items: int | None = None
) -> dict[str, dict[str, float]]:
    """彙整成 {版本: 指標}，供對比表使用。

    `total_items` 是**每個版本應有的商品總數**（例如 12）。給了它，
    「可直接上架%」就會用固定分母計算，把規則層擋下、沒送評分的那些
    一律算成不可上架。

    為什麼一定要這樣做 —— 這是實測踩到的坑：

    規則層擋下的都是最爛的那幾筆，它們不會進 rubric 評分。結果 v0 只有
    10 筆受評、v1～v3 各 12 筆，v0 的平均分被「倖存者偏差」灌水成
    74.3%，看起來比實際好。**兩個版本用不同分母，就不能直接比。**

    處理方式：
    - 「可直接上架%」用固定分母。被規則層擋下的本來就不能上架，
      算成不通過完全合理，這樣跨版本才可比，也是真正該拿來做決策的數字。
    - 「rubric通過率%」維持只在受評項目上平均（否則無法區分「沒評」與
      「評了但沒過」），但一定要一起看「受評筆數」。分母不同時不要直接比。
    """
    by_version: dict[str, list[RubricReport]] = {}
    for r in reports:
        if r.error:
            continue
        by_version.setdefault(r.version, []).append(r)

    out = {}
    for version, rows in sorted(by_version.items()):
        if not rows:
            continue
        denom = total_items or len(rows)
        out[version] = {
            "rubric通過率%": round(sum(r.pass_rate for r in rows) / len(rows), 1),
            "critical違反數": sum(r.critical_failures for r in rows),
            "可直接上架%": round(100 * sum(r.would_publish for r in rows) / denom, 1),
            "受評筆數": len(rows),
            "分母": denom,
        }
    return out


def most_common_failures(reports: list[RubricReport], rubric_sets: dict, limit: int = 5):
    """哪些判準最常沒過 —— 這才是能拿去改 prompt 的資訊。

    這正是二元 rubric 勝過分數的地方：「3.8 分」不能行動，
    「有 9 個商品沒寫出含量」可以。
    """
    counts: dict[tuple[str, str], int] = {}
    for rep in reports:
        if rep.error:
            continue
        rs = rubric_sets.get(rep.sku)
        if not rs:
            continue
        text_by_id = {r.id: r.criterion for r in rs.rubrics}
        for rid, _reason, _crit in rep.failed:
            key = (rid, text_by_id.get(rid, rid))
            counts[key] = counts.get(key, 0) + 1
    ranked = sorted(counts.items(), key=lambda kv: -kv[1])[:limit]
    return [(criterion, n) for (_rid, criterion), n in ranked]

In [ ]:
for name, info in JUDGE_BIASES.items():
    print(f"● {name}")
    print(f"    問題：{info['說明']}")
    print(f"    緩解：{info['緩解']}\n")

#### 步驟一：為每個商品產生驗收清單

這一步和被評的文案無關，只看商品資料。

In [ ]:
t0 = time.time()
rubric_sets = dict(
    run_parallel(
        lambda p: (p['sku'], generate_rubrics(client, p, ledger)),
        PRODUCTS,
        label="產生 rubric",
    )
)
print(f"  耗時 {time.time() - t0:.1f} 秒")

demo = rubric_sets[PRODUCTS[0]['sku']]
print(f"{PRODUCTS[0]['name']} 的驗收清單（{len(demo.rubrics)} 條）\n")
for r in demo.rubrics:
    mark = "★" if r.critical else " "
    print(f"  {mark} {r.id}  [{r.dimension}] {r.criterion}")
print("\n★ = critical，未通過就不該上架")

#### 步驟二：逐條檢查

只送規則層過關的文案去評審 —— **這是省錢的關鍵**。
對一份已經違反法規的文案做語意檢查沒有意義，而 judge 用的是比生成更貴的模型。

In [ ]:
by_key = {(r.sku, r.version): r for r in rule_results}

# 評測時全部送審 —— 分母一致才能比較。
to_judge = [(p, v) for p in PRODUCTS for v in PROMPT_VERSIONS]

# 生產環境會怎麼做：規則層擋下的不送審。這裡只統計、不實際跳過，
# 用來對照「同一個機制在不同目的下取捨相反」。
would_skip = sum(
    1 for p in PRODUCTS for v in PROMPT_VERSIONS
    if not should_judge(by_key[(p['sku'], v)])
)
total = len(PRODUCTS) * len(PROMPT_VERSIONS)
print(f"送審 {len(to_judge)}/{total} 筆 —— 評測全送，四個版本的分母才一致")
print(f"生產環境會擋下其中 {would_skip} 筆，省 {would_skip/total*100:.0f}% 評審成本。")
print("但評測時不值得：省下的是零頭，換掉的是「版本之間能不能比較」。")

# 這是整份 notebook 最慢的一段：評審模型單次十幾秒。
# 序列跑約 9 分鐘，並行後約 1 分鐘。
t0 = time.time()
rubric_reports = run_parallel(
    lambda job: check_against_rubrics(
        client, outputs[(job[0]['sku'], job[1])],
        rubric_sets[job[0]['sku']], job[1], ledger
    ),
    to_judge,
    label="評審",
)
print(f"  耗時 {time.time() - t0:.1f} 秒\n")
pd.DataFrame(summarize_rubrics(rubric_reports, total_items=len(PRODUCTS))).T

#### 最常沒過的判準 —— 這才是能拿去改 prompt 的東西

這正是二元 rubric 勝過分數的地方：「3.8 分」不能行動，
「有 9 個商品沒寫出含量」可以。

In [ ]:
for criterion, n in most_common_failures(rubric_reports, rubric_sets):
    print(f"  {n:>2} 次未通過 — {criterion}")

worst = [r for r in rubric_reports if r.version == 'v0' and r.failed]
if worst:
    r = worst[0]
    print(f"\n{r.sku} v0 未通過的細節（{r.pass_rate}% 通過）：")
    for rid, reason, crit in r.failed[:4]:
        print(f"  {'★' if crit else ' '} {rid}: {reason}")

### ★★ 合併對比表 —— 每一版修好一件事

這張表是整份 notebook 的結論。左邊三欄由 v1 修好、中間兩欄由 v2 修好、
最後一欄由 v3 修好 —— **每一版的貢獻都能單獨看見**。

In [ ]:
print(render_paired(rubric_reports, list(PROMPT_VERSIONS)))
print()
print(significance_note(len(PRODUCTS), 7))

combined = combined_table(
    rule_results, summarize_rubrics(rubric_reports, total_items=len(PRODUCTS))
)
print(PARSER_CAVEAT)
combined.style.format("{:.1f}%", na_rep="—").background_gradient(
    cmap="RdYlGn", vmin=0, vmax=100)

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **38:00 — ★ 全場高潮。表格出來後停 5 秒不要說話，讓大家看。**
>
> 然後：「每一版修好一件事。左邊三欄是 v1、中間兩欄是 v2、最後一欄是 v3。」
>
> **接著一定要講這兩句，這是誠信也是專業度：**
>
> 1.「台上這張是 12 個商品，樣本太小。**真正的結論來自另外跑的 50 筆評測集**
>    —— 就是投影片 S17c 那組數字。demo 給你看流程，結論要看評測集。」
> 2.「50 筆的結果：v0→v1、v1→v2 是真的改善；**v2→v3 只差 0.3 個百分點，
>    那是雜訊。** v3 贏的不是文案品質，是可機器讀取從 0 變成 100%。」
>
> **不要**說「你看全部都變好了」。台下有工程師，會扣分。

</div>

#### 讀這張表的時候要小心兩件事

**⚠️ 先看這個：rubric 那一列不是遞增的，v0（78.5%）比 v1（74.0%）高。**

台上一定會有人注意到。**先講，不要等被問。** 台詞：

> 「注意 rubric 這一列 —— v0 竟然比 v1 高。是 v0 比較好嗎？
> 看旁邊那欄『受評筆數』：v0 只有 8 筆，v1 有 11 筆。
> **v0 最爛的 4 筆被規則層擋下來，根本沒進評分。**
> 分母不同，這兩個數字不能比。
>
> 這就是為什麼我把受評筆數印在表上 —— 不然這格看起來就像 v1 讓事情變糟了。」

這是全場最好的評測方法論教材：**倖存者偏差，而且是活生生出現在自己的表上。**
「可直接上架%」那一列就沒有這個問題，因為它用固定分母（12）計算，
被擋下的一律算不通過。**那才是能拿來做決策的數字。**

---

**一、這裡的樣本太小，只能看趨勢，不能下結論。**
12 個商品、rubric 每個 7 條，共 84 次檢查，翻轉一次就是 1.2 個百分點。
把雜訊當成訊號，是評測工作最容易犯、也最傷的錯。

真正的結論來自另外跑的 **50 筆評測集**（`poc/run_eval.py`）——
350 次檢查、雜訊底線 0.29 個百分點。實測結果：

| | rubric 通過率 | 可直接上架 | 可機器讀取 |
|---|---|---|---|
| v0 | 68.4% | 14% | 0% |
| v1 | 79.1% | 42% | 0% |
| v2 | 86.3% | 72% | 0% |
| v3 | 86.6% | 68% | **100%** |

**二、v2 → v3 只差 0.3 個百分點 —— 那是雜訊，不能說 v3 讓文案變好。**

v3 真正的價值在最後一欄。**v3 不是讓文案變好，是讓文案變得可用。**
這是兩件不同的事，而你兩個都需要。

一張每格都完美遞增的表，通常代表有人在調數字。

**三、上面那一欄「實測歸因」是算出來的，不是寫死的。**
它比較相鄰版本的差距、取單步改善最大的那一版。這件事本身有教訓：
最早這裡是一份手寫對照表，後來實跑 50 筆，六條裡有三條被資料推翻
（例如「SEO描述」原本標成 v1 修好，實際上 v1 完全沒動，是 v3 修的）。
**在講評測的材料裡放一個硬寫的因果宣稱，是最不該犯的錯。**

## §4 場景 B：評論洞察

方法論完全相同，方向相反：

    場景 A   結構化資料 → 非結構化文字   （生成）
    場景 B   非結構化文字 → 結構化資料   （抽取）

**這一節的目的不是教評論分析**，而是證明你剛學的是一套流程，不是一個文案技巧。
同樣要 schema、同樣要評測、同樣要算成本。

In [ ]:
"""場景 B：評論洞察分析。

方法論與場景 A 完全相同，方向相反：
    場景 A  結構化資料 → 非結構化文字   （生成）
    場景 B  非結構化文字 → 結構化資料   （抽取）

這一節的重點不是評論分析本身，而是證明**這是一套可遷移的流程，
不是一個文案技巧**。同樣要 schema、同樣要評測、同樣要算成本。

商業價值一句話：把「客服每天讀 500 則評論」變成「BigQuery 一句 SQL」。
"""

from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from datetime import UTC


ASPECTS = ["產品品質", "價格", "物流配送", "包裝", "客服", "使用體驗", "其他"]

REVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "sentiment": {
            "type": "string",
            "enum": ["positive", "neutral", "negative"],
        },
        "aspects": {
            "type": "array",
            "items": {"type": "string", "enum": ASPECTS},
            "description": "這則評論實際談到的面向，可多選",
        },
        "is_about_product": {
            "type": "boolean",
            "description": "評論主體是商品本身，還是物流／客服等非商品因素",
        },
        "actionable_suggestion": {
            "type": "string",
            "description": "可執行的改善建議；若無則填空字串",
        },
        "urgency": {"type": "string", "enum": ["low", "medium", "high"]},
    },
    "required": [
        "sentiment",
        "aspects",
        "is_about_product",
        "actionable_suggestion",
        "urgency",
    ],
}

EXTRACT_PROMPT = """你是電商營運分析師，要把客戶評論轉成結構化資料進資料倉儲。

【重要判準】
- `is_about_product`：評論在抱怨物流慢、客服態度、包裝破損時，**主體不是商品**，
  應為 false。這一欄的用途是把「商品不好」和「服務不好」分開 ——
  兩者要交給不同部門處理，混在一起會導致採購端誤判商品品質。
- `urgency`：只有涉及安全疑慮、法規風險、或大量客戶可能受影響時才是 high。
  單純的個人喜好不合不算。
- `actionable_suggestion`：必須是團隊真的能執行的動作（改包裝、補說明、調規格），
  不要寫「提升品質」這種無法執行的空話。無明確建議就留空字串。

【評論】
商品：{product_name}
評分：{rating} 星
內容：{text}

請依 schema 輸出。"""


@dataclass
class ReviewInsight:
    review_id: str
    sku: str
    rating: int
    sentiment: str
    aspects: list[str]
    is_about_product: bool
    actionable_suggestion: str
    urgency: str
    error: str | None = None


def extract_one(client, review: dict, product_name: str, ledger=None) -> ReviewInsight:
    """把單則評論轉成結構化資料。"""
    import time


    prompt = EXTRACT_PROMPT.format(
        product_name=product_name,
        rating=review["rating"],
        text=review["text"],
    )

    started = time.time()
    try:
        resp = client.models.generate_content(
            model=GEN_MODEL,
            contents=prompt,
            config=gen_config(
                temperature=0.0,  # 抽取任務要可重現，不要創意
                response_mime_type="application/json",
                response_schema=REVIEW_SCHEMA,
            ),
        )
        meta = resp.usage_metadata
        if ledger is not None:
            ledger.record(
                "gen:insight",
                Usage(
                    model=GEN_MODEL,
                    input_tokens=getattr(meta, "prompt_token_count", 0) or 0,
                    output_tokens=getattr(meta, "candidates_token_count", 0) or 0,
                    latency_s=round(time.time() - started, 2),
                ),
            )
        data = json.loads(resp.text)
        return ReviewInsight(
            review_id=review["review_id"],
            sku=review["sku"],
            rating=review["rating"],
            sentiment=data["sentiment"],
            aspects=list(data["aspects"]),
            is_about_product=bool(data["is_about_product"]),
            actionable_suggestion=data["actionable_suggestion"],
            urgency=data["urgency"],
        )
    except Exception as e:  # noqa: BLE001
        return ReviewInsight(
            review_id=review["review_id"],
            sku=review["sku"],
            rating=review["rating"],
            sentiment="neutral",
            aspects=[],
            is_about_product=True,
            actionable_suggestion="",
            urgency="low",
            error=str(e),
        )


def validate_insights(insights: list[ReviewInsight]) -> dict:
    """場景 B 也要評測 —— 這是本節的重點。

    抽取任務沒有「好文案」這種主觀標準，但仍有可自動檢查的規則：
    - enum 值是否合法
    - 星等與情緒是否矛盾（1 星卻標 positive，通常是抽取錯誤）
    - 高星等卻標 high urgency（同樣可疑）

    這些規則抓到的不是「模型寫得不好」，而是「模型理解錯了」—— 更嚴重。
    """
    issues = []
    for r in insights:
        if r.error:
            issues.append((r.review_id, f"抽取失敗：{r.error}"))
            continue
        if r.sentiment not in {"positive", "neutral", "negative"}:
            issues.append((r.review_id, f"sentiment 值非法：{r.sentiment}"))
        if bad := [a for a in r.aspects if a not in ASPECTS]:
            issues.append((r.review_id, f"aspects 出現未定義值：{bad}"))
        if r.rating <= 2 and r.sentiment == "positive":
            issues.append((r.review_id, f"{r.rating} 星卻判為 positive，疑似抽取錯誤"))
        if r.rating >= 4 and r.sentiment == "negative":
            issues.append((r.review_id, f"{r.rating} 星卻判為 negative，疑似抽取錯誤"))
        if r.rating == 5 and r.urgency == "high":
            issues.append((r.review_id, "5 星卻標記 high urgency，請人工確認"))

    return {
        "總筆數": len(insights),
        "問題筆數": len(issues),
        "通過率": round(100 * (len(insights) - len(issues)) / max(len(insights), 1), 1),
        "明細": issues,
    }


def to_dataframe(insights: list[ReviewInsight]):
    """轉成 DataFrame。欄位刻意設計成可直接進 BigQuery 的扁平結構。"""
    import pandas as pd

    rows = []
    for r in insights:
        d = asdict(r)
        d["aspects"] = "|".join(r.aspects)  # BigQuery 可用 SPLIT() 還原
        rows.append(d)
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# BigQuery —— 把「AI 當 ETL」這件事真的做完
# --------------------------------------------------------------------------
BQ_SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS `{project}.{dataset}.{table}` (
  review_id             STRING  NOT NULL,
  sku                   STRING  NOT NULL,
  rating                INT64,
  sentiment             STRING,          -- positive / neutral / negative
  aspects               ARRAY<STRING>,   -- 一則評論可能談到多個面向
  is_about_product      BOOL,            -- 把商品問題與服務問題分開的關鍵欄位
  actionable_suggestion STRING,
  urgency               STRING,          -- low / medium / high
  ingested_at           TIMESTAMP
)
PARTITION BY DATE(ingested_at)
CLUSTER BY sku, sentiment
"""

# 這句 SQL 就是整個場景 B 的商業價值：客服原本要讀 500 則評論，現在是一句查詢。
BQ_INSIGHT_SQL = """
SELECT
  sku,
  COUNTIF(is_about_product AND sentiment = 'negative') AS 商品負評,
  COUNTIF(NOT is_about_product AND sentiment = 'negative') AS 服務負評,
  COUNTIF(urgency = 'high')                             AS 高急迫,
  ROUND(AVG(rating), 2)                                 AS 平均星等
FROM `{project}.{dataset}.{table}`
GROUP BY sku
HAVING 商品負評 > 0 OR 服務負評 > 0
ORDER BY 商品負評 DESC, 服務負評 DESC
"""


def load_to_bigquery(insights: list[ReviewInsight], verbose: bool = True):
    """把抽取結果寫進 BigQuery。

    這一步讓「非結構化文字 → 結構化資料」這句話變成真的 —— 資料真的落地、
    真的能被 SQL 查詢、真的能接上既有的 BI 報表。

    schema 的設計重點在 `is_about_product`：把「商品不好」與「服務不好」
    分開儲存。混在一起統計，採購會以為商品品質有問題，實際上是物流慢。
    """
    from datetime import datetime

    from google.cloud import bigquery

    client = bigquery.Client(project=PROJECT_ID)
    ds_id = f"{PROJECT_ID}.{BQ_DATASET}"

    ds = bigquery.Dataset(ds_id)
    ds.location = BQ_LOCATION
    client.create_dataset(ds, exists_ok=True)

    table_ref = f"{ds_id}.{BQ_TABLE}"
    client.query(
        BQ_SCHEMA_SQL.format(
            project=PROJECT_ID, dataset=BQ_DATASET, table=BQ_TABLE
        )
    ).result()

    now = datetime.now(UTC).isoformat()
    rows = [
        {
            "review_id": r.review_id,
            "sku": r.sku,
            "rating": r.rating,
            "sentiment": r.sentiment,
            "aspects": r.aspects,
            "is_about_product": r.is_about_product,
            "actionable_suggestion": r.actionable_suggestion,
            "urgency": r.urgency,
            "ingested_at": now,
        }
        for r in insights
        if not r.error
    ]
    if not rows:
        raise RuntimeError("沒有可寫入的資料（全部抽取失敗）")

    errors = client.insert_rows_json(table_ref, rows)
    if errors:
        raise RuntimeError(f"BigQuery 寫入失敗：{errors}")

    if verbose:
        print(f"✓ 已寫入 {len(rows)} 列 → {table_ref}")
        print("  分區：DATE(ingested_at)　叢集：sku, sentiment")
    return table_ref


def query_bigquery_insights():
    """跑那句「取代人工讀 500 則評論」的 SQL。"""
    from google.cloud import bigquery

    client = bigquery.Client(project=PROJECT_ID)
    sql = BQ_INSIGHT_SQL.format(
        project=PROJECT_ID, dataset=BQ_DATASET, table=BQ_TABLE
    )
    return client.query(sql).to_dataframe()


def business_summary(insights: list[ReviewInsight]) -> str:
    """重點在這裡：資料變成決策。"""
    ok = [r for r in insights if not r.error]
    if not ok:
        return "無有效資料"

    product_issues = [r for r in ok if r.is_about_product and r.sentiment == "negative"]
    service_issues = [
        r for r in ok if not r.is_about_product and r.sentiment == "negative"
    ]
    urgent = [r for r in ok if r.urgency == "high"]

    aspect_counts: dict[str, int] = {}
    for r in ok:
        if r.sentiment == "negative":
            for a in r.aspects:
                aspect_counts[a] = aspect_counts.get(a, 0) + 1

    top = sorted(aspect_counts.items(), key=lambda kv: -kv[1])[:3]
    suggestions = [r.actionable_suggestion for r in ok if r.actionable_suggestion]

    lines = [
        f"有效評論 {len(ok)} 則",
        "",
        f"  商品相關負評   {len(product_issues):>3} 則  → 採購／研發要看",
        f"  服務相關負評   {len(service_issues):>3} 則  → 物流／客服要看",
        f"  高急迫案件     {len(urgent):>3} 則  → 今天就要處理",
        "",
        "負評集中的面向：",
    ]
    lines += [f"  {a}：{c} 則" for a, c in top] or ["  （無）"]
    lines += ["", f"可執行建議 {len(suggestions)} 條，例如："]
    lines += [f"  - {s}" for s in suggestions[:3]]
    lines += [
        "",
        "↑ 這就是價值所在：把「客服每天讀 500 則評論」變成一句 SQL。",
        "  注意『商品負評』與『服務負評』已經分開 —— 混在一起會讓採購誤判商品品質。",
    ]
    return "\n".join(lines)

In [ ]:
REVIEWS = {
 "_meta": {
  "description": "示範用假評論資料，模擬台灣電商評論區的真實樣態：長短不一、有錯字、有離題、有物流抱怨混在商品評價裡。",
  "note": "刻意保留雜訊（注音、錯字、只有標點的短評、抱怨物流而非商品），用來展示模型在髒資料上的表現，這才是真實情況。",
  "target_schema": {
   "sentiment": "positive | neutral | negative",
   "aspects": "list of 產品品質 | 價格 | 物流配送 | 包裝 | 客服 | 使用體驗 | 其他",
   "is_about_product": "bool — 用來把物流客訴從商品評價中分離",
   "actionable_suggestion": "string | null",
   "urgency": "low | medium | high"
  }
 },
 "reviews": [
  {
   "review_id": "R001",
   "sku": "HB-1001",
   "rating": 5,
   "text": "吃了快兩個月，本來只是想說試試看，結果現在每天都會記得吃。膠囊不大顆很好吞，也沒有什麼味道。會回購。"
  },
  {
   "review_id": "R002",
   "sku": "HB-1001",
   "rating": 2,
   "text": "東西還沒拆就先說，寄來的時候盒子整個壓扁了…雖然裡面膠囊看起來沒事但這樣送禮很尷尬耶，希望包裝可以加強"
  },
  {
   "review_id": "R003",
   "sku": "HB-1002",
   "rating": 4,
   "text": "粉末沖水很好溶解不會結塊這點加分，味道偏淡幾乎沒味道我可以接受。唯一想抱怨的是一盒才30包但價格快1200，有點貴"
  },
  {
   "review_id": "R004",
   "sku": "HB-1002",
   "rating": 1,
   "text": "下單到收到花了九天！！！客服問了三次都說在處理中，最後也沒給我明確時間。產品本身還沒吃不評論，但這個出貨速度真的不行"
  },
  {
   "review_id": "R005",
   "sku": "HB-1003",
   "rating": 5,
   "text": "蔓越莓口味比我想像中好喝很多，不會有腥味這點很重要！！睡前泡一杯變成習慣了"
  },
  {
   "review_id": "R006",
   "sku": "HB-1003",
   "rating": 3,
   "text": "還可以吧"
  },
  {
   "review_id": "R007",
   "sku": "CS-2001",
   "rating": 5,
   "text": "敏感肌用了三週沒有刺痛感，這對我來說已經是很高的評價了。質地是水感的很好推開，上妝前用也不會搓泥。唯一小缺點是滴管有時候會沾到瓶口"
  },
  {
   "review_id": "R008",
   "sku": "CS-2001",
   "rating": 2,
   "text": "我用了長痘痘...不知道是不是我不適合，成分表看起來是很溫和沒錯啦。退貨流程倒是很順利客服人很好"
  },
  {
   "review_id": "R009",
   "sku": "CS-2002",
   "rating": 4,
   "text": "泡沫很綿密洗完不緊繃，但120ml用得有點快，希望出大容量版本"
  },
  {
   "review_id": "R010",
   "sku": "CS-2003",
   "rating": 1,
   "text": "膜布跟我臉根本不合，眼睛下面那塊一直翹起來，敷到一半就掉了。精華液倒是蠻多的但都流到脖子去了= ="
  },
  {
   "review_id": "R011",
   "sku": "EL-3001",
   "rating": 5,
   "text": "降噪真的有感，捷運上開最強模式幾乎聽不到廣播聲。續航也很夠用，通勤來回一週充一次盒子就好。雙裝置切換超方便，手機筆電來回切不用重連"
  },
  {
   "review_id": "R012",
   "sku": "EL-3001",
   "rating": 3,
   "text": "音質沒話說但戴久了耳朵會痛，我耳道比較小可能不適合。另外通話時對方說我這邊有雜音，不知道是不是個案"
  },
  {
   "review_id": "R013",
   "sku": "EL-3002",
   "rating": 5,
   "text": "體積真的小！！之前那顆65W是這個的兩倍大。三個孔同時插筆電手機耳機都還能跑滿速，出差只帶這顆就夠"
  },
  {
   "review_id": "R014",
   "sku": "EL-3003",
   "rating": 2,
   "text": "吸力是不錯啦可是集塵盒也太小了吧0.5L根本掃不到一半就要倒，而且倒的時候灰塵會噴出來設計有問題"
  },
  {
   "review_id": "R015",
   "sku": "FD-4001",
   "rating": 4,
   "text": "耶加雪菲的花香有出來，掛耳能做到這樣不容易。不過我覺得建議沖法的水溫寫得有點模糊，第一次沖太燙有澀感，後來自己調到88度才對"
  }
 ]
}['reviews']

name_by_sku = {p['sku']: p['name'] for p in PRODUCTS}
insights = run_parallel(
    lambda r: extract_one(client, r, name_by_sku.get(r['sku'], '未知商品'), ledger),
    REVIEWS,
    label='抽取評論',
)

to_dataframe(insights)[['review_id','rating','sentiment','aspects',
                        'is_about_product','urgency']]

### 抽取任務也要評測

沒有「好文案」這種主觀標準，但仍有可自動檢查的規則：
星等與情緒矛盾、enum 值非法、5 星卻標高急迫。

這些抓到的不是「模型寫得不好」，而是**「模型理解錯了」—— 更嚴重**。

In [ ]:
check = validate_insights(insights)
print(f"通過率 {check['通過率']}%  （{check['總筆數'] - check['問題筆數']}/{check['總筆數']}）")
for rid, msg in check['明細']:
    print(f"  ⚠ {rid}: {msg}")

print("\n" + "=" * 60)
print(business_summary(insights))

#### 落地到 BigQuery — 讓「變成結構化資料」不只是說說

到這裡為止，結構化資料還只存在於記憶體裡。真正的價值要等它**進到資料倉儲、
能被 SQL 查詢、能接上既有的 BI 報表**才會發生。

注意 schema 的設計重點：`is_about_product` 這個欄位把「商品不好」與
「服務不好」分開儲存。混在一起統計，採購會以為商品品質有問題，
實際上是物流慢。

> 預設 `USE_BIGQUERY = False`。要真的寫入請在 CONFIG 打開，
> 並確認專案已啟用 BigQuery API 且帳號有 `bigquery.dataEditor` 權限。
> 離線重播模式不涵蓋 BigQuery 呼叫。

In [ ]:
print("建表 SQL：")
print(BQ_SCHEMA_SQL.format(project=PROJECT_ID, dataset=BQ_DATASET, table=BQ_TABLE))
print("\n分析 SQL —— 這句就是整個場景 B 的商業價值：")
print(BQ_INSIGHT_SQL.format(project=PROJECT_ID, dataset=BQ_DATASET, table=BQ_TABLE))

In [ ]:
if USE_BIGQUERY:
    load_to_bigquery(insights)
    bq_df = query_bigquery_insights()
    display(bq_df)
else:
    print("USE_BIGQUERY = False — 只展示 schema 與 SQL，未實際寫入。")
    print("要真的跑，請在 CONFIG 把 USE_BIGQUERY 設為 True。")

## §5 成本

**教公式，不背數字。**

    單次成本 = (input_tokens × 單價_in + output_tokens × 單價_out) / 1,000,000
    月成本  = 單次 × SKU數 × 每SKU重生成次數 × (1 + judge 開銷比)

下面印出來的是**這次執行的真實花費**，比任何通用估算都準 ——
因為 prompt 長度、輸出長度、中文 token 密度全是這個專案的實際值。

In [ ]:
"""成本估算：從 demo 的實際用量外推到正式營運規模。

原則：**用公式，不要背數字。**
模型與價格每季在變（例如 Gemini 2.5 Flash-Lite 已公告 2026-10-16 退役），
任何寫死的價格很快就會過期。公式不會。

公式是穩定的，價格常數集中在 config，實際金額由執行結果給出。
"""

from __future__ import annotations

from dataclasses import dataclass



@dataclass
class ScaleEstimate:
    """把 demo 用量外推到正式規模。"""

    sku_count: int
    regen_per_sku: float          # 每個 SKU 平均重新生成幾次（改版、A/B、季節性換檔）
    judge_ratio: float            # 送 judge 的比例（規則層擋掉的不送）
    gen_cost_per_item: float      # 單次生成成本 USD
    judge_cost_per_item: float    # 單次評審成本 USD

    @property
    def total_generations(self) -> float:
        return self.sku_count * self.regen_per_sku

    @property
    def gen_cost(self) -> float:
        return self.total_generations * self.gen_cost_per_item

    @property
    def judge_cost(self) -> float:
        return self.total_generations * self.judge_ratio * self.judge_cost_per_item

    @property
    def total_usd(self) -> float:
        return self.gen_cost + self.judge_cost

    @property
    def total_twd(self) -> float:
        return self.total_usd * USD_TO_TWD

    def render(self) -> str:
        return (
            f"規模假設：{self.sku_count:,} 個 SKU × 每個重生成 {self.regen_per_sku} 次"
            f" = {self.total_generations:,.0f} 次生成\n"
            f"           其中 {self.judge_ratio:.0%} 通過規則層、需要送 LLM 評審\n\n"
            f"  生成成本   {self.total_generations:>10,.0f} 次 × US${self.gen_cost_per_item:.6f}"
            f"  = US${self.gen_cost:>10,.2f}\n"
            f"  評審成本   {self.total_generations * self.judge_ratio:>10,.0f} 次 "
            f"× US${self.judge_cost_per_item:.6f}  = US${self.judge_cost:>10,.2f}\n"
            f"  {'─' * 58}\n"
            f"  合計                                       US${self.total_usd:>10,.2f}"
            f"  （約 NT${self.total_twd:,.0f}）"
        )


def estimate_from_ledger(
    ledger,
    sku_count: int = 100_000,
    regen_per_sku: float = 1.5,
    judge_ratio: float = 0.7,
) -> ScaleEstimate:
    """用 demo 實際跑出來的平均單次成本外推。

    這比任何通用估算都準，因為 prompt 長度、輸出長度、中文 token 密度
    全部是這個專案的真實值，不是別人部落格上的數字。
    """
    gen_calls = [u for t, u in ledger.calls if t.startswith("gen")]
    judge_calls = [u for t, u in ledger.calls if t.startswith("judge")]

    gen_avg = sum(u.cost_usd for u in gen_calls) / len(gen_calls) if gen_calls else 0.0
    judge_avg = (
        sum(u.cost_usd for u in judge_calls) / len(judge_calls) if judge_calls else 0.0
    )

    return ScaleEstimate(
        sku_count=sku_count,
        regen_per_sku=regen_per_sku,
        judge_ratio=judge_ratio,
        gen_cost_per_item=gen_avg,
        judge_cost_per_item=judge_avg,
    )


# --------------------------------------------------------------------------
# 降本四招
# --------------------------------------------------------------------------
LEVERS = [
    {
        "名稱": "模型選型降級",
        "作法": "簡單任務改用更小的模型；只有需要判斷力的環節才用大模型。",
        "省下": "視價差而定，通常是最大的一筆",
        "代價": "品質下降風險 —— **必須有評測表才敢降級**，否則是在賭。",
        "適用": "文案生成這種格式固定、判準明確的任務，通常降級後評測分數不會掉。",
    },
    {
        "名稱": "Batch API",
        "作法": "非即時的工作改走批次介面。商品文案本來就不需要即時。",
        "省下": "約 5 折",
        "代價": "延遲從秒級變成小時級。",
        "適用": "初次匯入全站商品、季節性全面改版 —— 這些場景根本不在乎即時性。",
    },
    {
        "名稱": "Context caching",
        "作法": "把共用的前綴（品牌指南、法規禁詞清單、few-shot 範例）快取起來。",
        "省下": "快取命中部分的 input token 大幅折扣",
        "代價": "有最低 token 門檻與存活時間，前綴要夠長、呼叫要夠密集才划算。",
        "適用": "本專案的 v2/v3 prompt 有大量固定前綴，是典型適用情境。",
    },
    {
        "名稱": "結果快取",
        "作法": "商品規格沒變就不要重新生成。用規格的 hash 當 key。",
        "省下": "取決於商品異動率，通常這是最被低估的一招",
        "代價": "幾乎沒有，只需要一個 key-value 存放。",
        "適用": "所有場景。很多團隊每天全量重跑，其實九成商品根本沒變。",
    },
]


def render_levers() -> str:
    out = []
    for i, lever in enumerate(LEVERS, 1):
        out.append(
            f"{i}. {lever['名稱']}（省下：{lever['省下']}）\n"
            f"   作法：{lever['作法']}\n"
            f"   代價：{lever['代價']}\n"
            f"   適用：{lever['適用']}"
        )
    return "\n\n".join(out)


def measure_chinese_token_density(client, samples: list[str]) -> dict:
    """實測中文的 token 密度，不要用經驗法則。

    直接量出來。中文的 token 密度會隨內容（純中文／中英混雜／
    含數字規格）明顯變動，背一個「一字約 X token」的數字是不可靠的。
    """

    rows = []
    for s in samples:
        tokens = count_tokens(client, s)
        chars = len(s)
        rows.append(
            {
                "字元數": chars,
                "token數": tokens,
                "每字token": round(tokens / chars, 3) if chars else 0,
                "預覽": s[:24] + ("..." if len(s) > 24 else ""),
            }
        )
    densities = [r["每字token"] for r in rows if r["每字token"]]
    return {
        "明細": rows,
        "平均每字token": round(sum(densities) / len(densities), 3) if densities else 0,
        "最低": min(densities) if densities else 0,
        "最高": max(densities) if densities else 0,
    }


def price_check_reminder() -> str:
    """執行前要印出來的提醒。"""
    return (
        f"⚠ 價格常數最後查證時間：{PRICE_LAST_CHECKED}\n"
        f"  上場前請到 https://cloud.google.com/vertex-ai/generative-ai/pricing "
        f"確認並更新 PRICING，\n"
        f"  然後把 PRICE_LAST_CHECKED 改成當天日期。\n"
        f"  不要引用固定的價格數字，以本 notebook 實際印出的金額為準。"
    )

In [ ]:
print(price_check_reminder())
print("\n" + "=" * 60)
print("這次 demo 的實際用量")
print("=" * 60)
print(ledger.summary())

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **41:00 — 唸出實際的「評審佔總成本 __%」。**
>
> 這個數字每次跑都會變，**唸畫面上的，不要唸背下來的**。
> （前一天錄製時先看過一次，心裡有個底。）
>
> 「評審佔了這麼多。這是最常被漏算的一筆 —— 大家算成本只算生成。」

</div>

### 實測中文 token 密度 — 不要背經驗法則

In [ ]:
samples = [
    PRODUCTS[0]['name'],
    "游離型葉黃素30mg，吸收更直接",
    PROMPT_VERSIONS['v3'](PRODUCTS[0],
                          banned_terms=get_banned_terms_for(PRODUCTS[0], BANNED_DATA),
                          tone_examples=TONE_EXAMPLES[PRODUCTS[0]['brand']])[:500],
    "Volta 240W USB-C 編織線 2M 支援40Gbps傳輸",   # 中英數混雜
]
density = measure_chinese_token_density(client, samples)
pd.DataFrame(density['明細'])

In [ ]:
print(f"平均每字 {density['平均每字token']} tokens"
      f"（範圍 {density['最低']} – {density['最高']}）")
print("\n↑ 注意範圍有多寬。中英數混雜的文字密度和純中文差很多，")
print("  所以『中文一個字約 X 個 token』這種經驗法則不可靠，要自己量。")

### 外推到正式規模

In [ ]:
est = estimate_from_ledger(ledger, sku_count=100_000, regen_per_sku=1.5, judge_ratio=0.7)
print(est.render())

### 降本四招

In [ ]:
print(render_levers())

---

## §6 錄製 fixtures（需要離線重播時才執行）

這一格把剛剛所有真實 API 輸出存成檔案，之後就能零網路重播。

**流程：**

1. 在**有網路**的環境把 `RECORD_FIXTURES` 設為 `True`，Run all
2. 執行下面這格，下載 `demo_outputs.json`
3. 把檔案放到 `poc/data/`，執行 `python3 poc/build_notebook.py` 重新產生 notebook
4. 把 `OFFLINE_MODE` 設為 `True`，再 Run all 驗證一次 —— **這次應該完全不連網**
5. 之後在任何沒有網路的環境都能完整重跑

In [ ]:
if RECORD_FIXTURES:
    dump = client.dump()
    with open(FIXTURES_FILE, "w", encoding="utf-8") as f:
        json.dump(dump, f, ensure_ascii=False)
    print(f"✓ 已錄製 {len(dump['calls']):,} 筆呼叫、"
          f"{len(dump['token_counts']):,} 筆 token 計數 → {FIXTURES_FILE}")
    try:
        from google.colab import files
        files.download(FIXTURES_FILE)
    except ImportError:
        print("  （本機執行，檔案已寫在當前目錄）")
elif OFFLINE_MODE:
    print(f"離線重播模式：本次共命中 {client.models.hits:,} 筆錄製輸出，全程未連網。")
else:
    print("目前是一般連線模式。若要錄製以供離線重播，請把 RECORD_FIXTURES 設為 True。")

---

## 收尾

回到一開始的問題：**你怎麼知道它有沒有變好？**

現在你有一張表可以回答。而且那張表是自動產生的，
下次改 prompt、換模型、降級省錢的時候，重跑一次就知道有沒有退步。

**這才是能上線的東西。**

---

### 帶回去的三件事

1. **先建規則層。** 免費、確定性、可以進 CI。不要一開始就做 LLM judge。
2. **用 API 原生的 structured output**，不要用 prompt 硬凹 JSON。
3. **評測本身要花錢**，記得算進去 —— 這是最常被漏掉的一筆。

### 90 天導入路徑

| 階段 | 時間 | 產出 |
|---|---|---|
| Vertex AI Studio 驗證可行性 | 2 週 | 這件事到底做不做得成 |
| 建 golden set + API 串接 | 4 週 | 30–50 筆就足以開始 |
| 小流量上線 + 監控 | 6 週 | 有數字可以對董事會報告 |

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **41:45 — 捲回對比表收尾。**
>
> 「所以：你怎麼知道它有沒有變好？看這張表。」
>
> 然後切回投影片，講最後三件事帶回去。
>
> ---
>
> ⏱ **超時的話砍 §4（場景 B）**，不影響主論點。

</div>